# KDIC V1.5 · B/C/D 개별 호출 · Action Link Registry · v4

기존 B·C·D 답변 비교, 공통 검색 캐시, Fact Index, 상세 레이턴시를 유지하면서 후속 행동 링크를 규칙 기반으로 추가한 버전입니다.

- 답변 본문: LLM이 Evidence 범위에서 생성하되 URL 생성 금지
- 공식 출처: 답변에 사용한 원문 페이지를 기존 방식으로 표시
- 관련 공식 서비스: Action Link Registry가 신청·조회·서류·상담 의도를 확인해 일반 링크로 표시
- 안전성: HTTPS·공식 호스트 allowlist·승인 상태·역할 조건·회귀 테스트 하드 게이트
- 후속 UI: 구조화된 `action_links`를 보존하여 추후 Streamlit `st.link_button()`에 연결 가능


## 실행 방법

1. 새 Colab 런타임에서 위에서부터 순서대로 실행합니다.
2. 기존과 동일하게 KDIC 문서 ZIP, Dense structured V2 캐시, Fact Index JSON을 업로드합니다.
3. Action Link Registry는 노트북에 검증본이 내장되어 별도 업로드가 필요하지 않습니다.
4. 질문을 입력하고 B·C·D 중 하나를 누르면 공통 검색 후 해당 답변만 생성합니다.
5. 신청·조회·서류·상담 의도가 명확하면 답변 아래에 공식 서비스 링크가 표시됩니다.
6. 기술 정보 표시를 켜면 Action Link 선택·탈락 사유와 매칭 레이턴시를 확인할 수 있습니다.

현재 Colab 버전은 버튼 UI를 만들지 않습니다. 구조화된 `action_links`를 결과에 보존하며, 실제 신청·조회는 사용자가 공식 링크로 이동해 본인인증 후 직접 진행합니다.


## 1. 의존성 설치

Elasticsearch 서버는 `8.15.3`, Python 클라이언트는 실제 배포되어 있는 같은 minor 계열의 `8.15.1`을 사용합니다. `elasticsearch==8.15.3`이라는 Python 패키지는 배포되어 있지 않으므로 해당 핀을 사용하면 설치 셀에서 바로 실패합니다.


In [ ]:
!pip -q install "openai>=1.68,<2" "elasticsearch==8.15.1" "tqdm>=4.66,<5" "ipywidgets>=8.1,<9" "markdown>=3.6,<4" "pandas>=2.0,<3" "requests>=2.31,<3" "sentence-transformers>=3.0,<4" "openpyxl>=3.1,<4"

## 2. Elasticsearch 8.15.3 + Nori 준비

이 셀은 Colab에서 자주 발생하는 다음 문제를 피하도록 구성했습니다.

- Elasticsearch를 root로 실행해서 발생하는 시작 실패
- 일반 사용자가 `/content`에 PID 파일을 쓰지 못하는 권한 오류
- 노트북 재실행 때 `elasticsearch.yml` 설정이 계속 중복되는 문제
- 기존 9200 포트 프로세스와의 충돌
- 부분 다운로드·부분 압축 해제로 인한 실행 파일 손상
- Nori 플러그인이 없는 상태에서 인덱스를 생성하는 문제
- Colab cgroup v2 경로가 샌드박스 밖으로 해석되어 `AccessControlException`이 발생하는 문제

설치 폴더, 데이터, 로그, PID 파일을 모두 A안 전용 경로로 분리합니다. Colab의 cgroup 경로는
Elasticsearch 컨테이너 실행 방식과 동일하게 루트(`/`)로 명시하여, Elasticsearch가
`/sys/fs/cgroup/../../jupyter-children/cpu.stat` 같은 잘못된 경로를 읽지 않도록 합니다.


In [ ]:
%%bash
set -Eeuo pipefail

ES_VERSION="8.15.3"
ES_USER="kdic_es_a"
INSTALL_ROOT="/content/kdic_es_a_dist"
ES_HOME="${INSTALL_ROOT}/elasticsearch-${ES_VERSION}"
ES_RUNTIME="/content/kdic_es_a_runtime"
ES_ARCHIVE="${INSTALL_ROOT}/elasticsearch-${ES_VERSION}.tar.gz"
ES_PID_FILE="${ES_RUNTIME}/elasticsearch.pid"
ES_LOG_FILE="${ES_RUNTIME}/logs/kdic-a.log"
ES_HTTP_URL="http://127.0.0.1:9220"

show_diagnostics() {
  echo "[Elasticsearch 진단]" >&2
  if [ -f "${ES_PID_FILE}" ]; then
    echo "PID file: $(cat "${ES_PID_FILE}" 2>/dev/null || true)" >&2
  fi
  if [ -f "${ES_LOG_FILE}" ]; then
    tail -n 160 "${ES_LOG_FILE}" >&2 || true
  elif [ -d "${ES_RUNTIME}/logs" ]; then
    tail -n 160 "${ES_RUNTIME}"/logs/*.log >&2 2>/dev/null || true
  fi
}
trap show_diagnostics ERR

case "$(uname -m)" in
  x86_64) ES_ARCH="x86_64" ;;
  aarch64|arm64) ES_ARCH="aarch64" ;;
  *) echo "지원하지 않는 CPU 아키텍처: $(uname -m)" >&2; exit 1 ;;
esac

mkdir -p "${INSTALL_ROOT}" "${ES_RUNTIME}/data" "${ES_RUNTIME}/logs" "${ES_RUNTIME}/tmp"

if ! id "${ES_USER}" >/dev/null 2>&1; then
  useradd --system --create-home --home-dir "/content/${ES_USER}" --shell /usr/sbin/nologin "${ES_USER}"
fi

if curl -fsS "${ES_HTTP_URL}" >/dev/null 2>&1; then
  RUNNING_VERSION="$(curl -fsS "${ES_HTTP_URL}" | python3 -c 'import json,sys; print(json.load(sys.stdin)["version"]["number"])')"
  if [ "${RUNNING_VERSION}" != "${ES_VERSION}" ]; then
    echo "9220 포트에 Elasticsearch ${RUNNING_VERSION}가 실행 중입니다. Colab 런타임을 재시작하세요." >&2
    exit 1
  fi
  if ! curl -fsS "${ES_HTTP_URL}/_nodes/plugins" | python3 -c 'import json,sys; d=json.load(sys.stdin); assert any(p.get("name")=="analysis-nori" for n in d["nodes"].values() for p in n.get("plugins", []))'; then
    echo "실행 중인 9220 Elasticsearch에 analysis-nori가 없습니다. Colab 런타임을 재시작하세요." >&2
    exit 1
  fi
  echo "Elasticsearch ${RUNNING_VERSION} + analysis-nori 재사용"
  exit 0
fi

# A안 전용 PID는 남아 있지만 HTTP가 열리지 않으면 해당 프로세스만 정리한 뒤 재시작합니다.
if [ -f "${ES_PID_FILE}" ]; then
  OLD_PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
  if [ -n "${OLD_PID}" ] && kill -0 "${OLD_PID}" 2>/dev/null; then
    kill "${OLD_PID}" 2>/dev/null || true
    for _ in $(seq 1 15); do
      if ! kill -0 "${OLD_PID}" 2>/dev/null; then
        break
      fi
      sleep 1
    done
    if kill -0 "${OLD_PID}" 2>/dev/null; then
      kill -9 "${OLD_PID}" 2>/dev/null || true
    fi
  fi
  rm -f "${ES_PID_FILE}"
fi

if [ ! -x "${ES_HOME}/bin/elasticsearch" ]; then
  DOWNLOAD_URL="https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-${ES_ARCH}.tar.gz"
  TEMP_ARCHIVE="${ES_ARCHIVE}.part"
  rm -f "${TEMP_ARCHIVE}"
  curl -fL --retry 5 --retry-delay 3 --connect-timeout 20 \
    "${DOWNLOAD_URL}" -o "${TEMP_ARCHIVE}"
  tar -tzf "${TEMP_ARCHIVE}" >/dev/null
  mv "${TEMP_ARCHIVE}" "${ES_ARCHIVE}"
  tar -xzf "${ES_ARCHIVE}" -C "${INSTALL_ROOT}"
fi

chown -R "${ES_USER}:${ES_USER}" "${ES_HOME}" "${ES_RUNTIME}" "/content/${ES_USER}"

if ! runuser -u "${ES_USER}" -- "${ES_HOME}/bin/elasticsearch-plugin" list | grep -qx "analysis-nori"; then
  runuser -u "${ES_USER}" -- "${ES_HOME}/bin/elasticsearch-plugin" install --batch analysis-nori
fi

CONFIG_FILE="${ES_HOME}/config/elasticsearch.yml"
python3 - "${CONFIG_FILE}" "${ES_RUNTIME}" <<'PY'
from pathlib import Path
import sys

config_path = Path(sys.argv[1])
runtime = Path(sys.argv[2])
config_path.write_text(
    "\n".join([
        "cluster.name: kdic-colab-answer-a",
        "node.name: kdic-colab-answer-a-node",
        f"path.data: {runtime / 'data'}",
        f"path.logs: {runtime / 'logs'}",
        "network.host: 127.0.0.1",
        "http.port: 9220",
        "transport.port: 9320",
        "discovery.type: single-node",
        "xpack.security.enabled: false",
        "xpack.security.enrollment.enabled: false",
        "xpack.security.http.ssl.enabled: false",
        "xpack.security.transport.ssl.enabled: false",
        "xpack.ml.enabled: false",
        "ingest.geoip.downloader.enabled: false",
        "cluster.routing.allocation.disk.threshold_enabled: false",
        "node.store.allow_mmap: false",
        "bootstrap.memory_lock: false",
        "",
    ]),
    encoding="utf-8",
)
PY
chown "${ES_USER}:${ES_USER}" "${CONFIG_FILE}"

if [ -f "${ES_PID_FILE}" ]; then
  OLD_PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
  if [ -n "${OLD_PID}" ] && kill -0 "${OLD_PID}" 2>/dev/null; then
    echo "기존 Elasticsearch PID ${OLD_PID}의 시작을 기다립니다."
  else
    rm -f "${ES_PID_FILE}"
  fi
fi

if [ ! -f "${ES_PID_FILE}" ]; then
  runuser -u "${ES_USER}" -- env \
    ES_JAVA_OPTS="-Xms512m -Xmx512m -Djava.io.tmpdir=${ES_RUNTIME}/tmp -Des.cgroups.hierarchy.override=/" \
    "${ES_HOME}/bin/elasticsearch" -d -p "${ES_PID_FILE}"
fi

for _ in $(seq 1 120); do
  if curl -fsS "${ES_HTTP_URL}" >/dev/null 2>&1; then
    break
  fi
  if [ -f "${ES_PID_FILE}" ]; then
    PID="$(cat "${ES_PID_FILE}" 2>/dev/null || true)"
    if [ -n "${PID}" ] && ! kill -0 "${PID}" 2>/dev/null; then
      echo "Elasticsearch 프로세스가 시작 중 종료되었습니다." >&2
      exit 1
    fi
  fi
  sleep 1
done

curl -fsS "${ES_HTTP_URL}" >/dev/null
curl -fsS "${ES_HTTP_URL}/_nodes/jvm" | python3 -c '
import json, sys
data = json.load(sys.stdin)
args = [arg for node in data["nodes"].values() for arg in node["jvm"].get("input_arguments", [])]
assert "-Des.cgroups.hierarchy.override=/" in args, args
'
curl -fsS "${ES_HTTP_URL}/_nodes/plugins" | python3 -c 'import json,sys; d=json.load(sys.stdin); assert any(p.get("name")=="analysis-nori" for n in d["nodes"].values() for p in n.get("plugins", []))'
curl -fsS -X POST "${ES_HTTP_URL}/_analyze" \
  -H 'Content-Type: application/json' \
  -d '{"tokenizer":{"type":"nori_tokenizer","decompound_mode":"none"},"text":"예금자보호제도"}' >/dev/null

echo "Elasticsearch ${ES_VERSION} + analysis-nori 준비 완료: ${ES_HTTP_URL}"


## 1. 설정

In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import math
import os
import re
import shutil
import zipfile
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Literal

import ipywidgets as widgets
import numpy as np
from elasticsearch import Elasticsearch, helpers
from IPython.display import JSON, Markdown, clear_output, display
from openai import BadRequestError, OpenAI
from tqdm.auto import tqdm

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass


# ---------- 데이터 / 캐시 ----------
# 경로를 직접 지정하지 않으면 ZIP 업로드 창이 열립니다.
DATA_SOURCE: str | None = None
DENSE_CACHE_FILENAME = "kdic_dense_structured_v2_embeddings.jsonl"
DENSE_CACHE_PATH = Path("/content") / DENSE_CACHE_FILENAME

# ---------- HCX ----------
HCX_BASE_URL = "https://clovastudio.stream.ntruss.com/v1/openai"
HCX_EMBEDDING_MODEL = "bge-m3"
HCX_CHAT_MODEL = "HCX-005"
HCX_ENCODING_FORMAT = "float"
HCX_REQUEST_TIMEOUT = 120.0
HCX_MAX_RETRIES = 4

# ---------- 확정 검색 조건 ----------
# 문서 Dense 벡터는 Elasticsearch dense_vector에 저장하고 kNN으로 검색합니다.
# NUMPY_EXACT는 Elasticsearch kNN 비교 및 장애 fallback에만 사용합니다.
DENSE_BACKEND: Literal["ELASTICSEARCH_KNN", "NUMPY_EXACT"] = "ELASTICSEARCH_KNN"
DENSE_KNN_NUM_CANDIDATES = 200
ALLOW_NUMPY_DENSE_FALLBACK = True
DENSE_WEIGHT = 0.7
BM25_WEIGHT = 0.3
QUERY_FUSION_RRF_K = 10
CANDIDATE_DEPTH = 20
FINAL_TOP_K = 5

# ---------- Reranker ----------
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_CANDIDATE_DEPTH = 20
RERANKER_BATCH_SIZE = 8
RERANKER_MAX_LENGTH = 512

# ---------- Parent-Child ----------
# 검색 순위는 Child 청크 기준으로 유지하고, Reranker Top-K 확정 뒤
# parent_doc_id가 같은 형제 청크를 Evidence Context로 확장합니다.
PARENT_CHILD_ENABLED = True
# None이면 전체 Parent를 사용합니다. 숫자를 넣으면 문자 수 기준으로
# matched child 우선 + 가까운 sibling 순서로 선택합니다.
PARENT_CONTEXT_MAX_CHARS: int | None = 8192

# ---------- 버전 / Elasticsearch ----------
DENSE_INPUT_VERSION = "kdic-dense-structured-v2-title-section-content-newline"
DENSE_CACHE_VERSION = "kdic-hcx-dense-structured-v2-cache-v1"
ES_EXPECTED_VERSION = "8.15.3"
ES_URL = "http://127.0.0.1:9220"
ES_ANALYZER_NAME = "kdic_nori_none"
ES_INDEX_SCHEMA_VERSION = "kdic-hybrid-bm25-dense-v3"
FORCE_REBUILD_HYBRID_INDEX = False

assert math.isclose(DENSE_WEIGHT + BM25_WEIGHT, 1.0)
assert QUERY_FUSION_RRF_K > 0 and CANDIDATE_DEPTH > 0 and FINAL_TOP_K > 0
assert RERANKER_CANDIDATE_DEPTH == CANDIDATE_DEPTH
assert RERANKER_CANDIDATE_DEPTH >= FINAL_TOP_K
assert DENSE_KNN_NUM_CANDIDATES >= CANDIDATE_DEPTH

print({
    "answer_method": "B_BASIC_EVIDENCE_PACK",
    "dense": "HCX bge-m3 Dense-structured-v2 + Elasticsearch kNN",
    "dense_backend": DENSE_BACKEND,
    "dense_knn_num_candidates": DENSE_KNN_NUM_CANDIDATES,
    "sparse": "Elasticsearch BM25 + Nori-none",
    "weights": [DENSE_WEIGHT, BM25_WEIGHT],
    "fusion": "MINMAX",
    "query_fusion_rrf_k": QUERY_FUSION_RRF_K,
    "candidate_depth": CANDIDATE_DEPTH,
    "final_top_k": FINAL_TOP_K,
    "reranker": RERANKER_MODEL_NAME,
    "parent_child": PARENT_CHILD_ENABLED,
    "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
    "evidence_pack": True,
    "answer_skeleton": False,
    "fact_index": False,
    "fact_sheet": False,
})


# ---------- V1.5 질의분석 ----------
HCX_DECOMPOSITION_MODEL = "HCX-007"
V15_ORIGINAL_WEIGHT = 0.40
V15_SUBQUERY_TOTAL_WEIGHT = 0.60
V15_MIN_CONFIDENCE = 0.80
V15_MAX_SUBQUERIES = 4
V15_CACHE_PATH = Path("/content/kdic_v15_chat_decomposition_cache.jsonl")

assert math.isclose(V15_ORIGINAL_WEIGHT + V15_SUBQUERY_TOTAL_WEIGHT, 1.0)

## 4. KDIC ZIP 업로드와 청크 로딩

필수 파일은 ZIP 내부의 `processed/chunks.jsonl`입니다. 기존 `chunk_embeddings_hcx.jsonl`은 `content` 중심 임베딩이므로 Dense-structured-v2 캐시로 사용하지 않습니다.


In [ ]:
def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"JSONL 파싱 실패: {path}, line={line_number}") from error
            if not isinstance(record, dict):
                raise TypeError(f"JSONL 레코드가 객체가 아닙니다: {path}, line={line_number}")
            records.append(record)
    return records


def _safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"손상된 ZIP 항목: {bad_member}")
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"안전하지 않은 ZIP 경로: {member.filename}")
        archive.extractall(destination)
    return destination


def _find_unique_file(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))
    processed = [path for path in matches if path.parent.name == "processed"]
    candidates = processed or matches
    if not candidates:
        raise FileNotFoundError(f"{filename}을 찾지 못했습니다: {root}")
    if len(candidates) != 1:
        raise RuntimeError(f"{filename} 후보가 여러 개입니다: {candidates}")
    return candidates[0]


def resolve_data_source(configured_path: str | None) -> Path:
    if configured_path:
        path = Path(configured_path)
        if path.exists():
            return path
        raise FileNotFoundError(f"DATA_SOURCE 경로가 없습니다: {path}")

    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            "DATA_SOURCE에 KDIC_output ZIP 또는 압축 해제 폴더 경로를 지정하세요."
        ) from error

    print("KDIC_output ZIP 파일을 업로드하세요.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError(f"ZIP 파일을 정확히 1개 업로드해야 합니다: {list(uploaded)}")
    return Path("/content") / zip_names[0]


def prepare_data_root(source: Path) -> Path:
    if source.is_dir():
        return source
    if not zipfile.is_zipfile(source):
        raise ValueError(f"ZIP 파일이 아닙니다: {source}")
    digest = hashlib.sha256(source.read_bytes()).hexdigest()[:16]
    destination = Path("/content/kdic_data_a") / digest
    marker = destination / ".ready"
    if marker.exists():
        return destination
    if destination.exists():
        shutil.rmtree(destination)
    _safe_extract_zip(source, destination)
    marker.write_text("ready", encoding="utf-8")
    return destination


def _clean_text(value: Any) -> str:
    text = str(value or "").replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_chunks(data_root: Path) -> list[dict[str, Any]]:
    chunks_path = _find_unique_file(data_root, "chunks.jsonl")
    chunks = _read_jsonl(chunks_path)
    if not chunks:
        raise RuntimeError("chunks.jsonl이 비어 있습니다.")

    chunk_ids = [str(chunk.get("chunk_id") or "").strip() for chunk in chunks]
    if any(not chunk_id for chunk_id in chunk_ids):
        raise RuntimeError("빈 chunk_id가 있습니다.")
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("중복 chunk_id가 있습니다.")
    if any(not _clean_text(chunk.get("content")) for chunk in chunks):
        raise RuntimeError("본문이 비어 있는 청크가 있습니다.")
    return chunks


def build_dense_structured_v2_text(chunk: dict[str, Any]) -> str:
    parts = [
        _clean_text(chunk.get("title")),
        _clean_text(chunk.get("section_title")),
        _clean_text(chunk.get("content")),
    ]
    text = "\n".join(part for part in parts if part)
    if not text:
        raise ValueError(f"Dense 입력이 비었습니다: {chunk.get('chunk_id')}")
    return text


DATA_PATH = resolve_data_source(DATA_SOURCE)
DATA_ROOT = prepare_data_root(DATA_PATH)
CHUNKS = load_chunks(DATA_ROOT)
CHUNKS_BY_ID = {str(chunk["chunk_id"]): chunk for chunk in CHUNKS}

# Parent-Child 인덱스: parent_doc_id가 없으면 document_id, 그것도 없으면 chunk_id를 사용합니다.
PARENT_CHILDREN_BY_ID: dict[str, list[dict[str, Any]]] = {}
CHUNK_PARENT_ID: dict[str, str] = {}
for chunk in CHUNKS:
    chunk_id = str(chunk["chunk_id"])
    parent_id = (
        _clean_text(chunk.get("parent_doc_id"))
        or _clean_text(chunk.get("document_id"))
        or chunk_id
    )
    CHUNK_PARENT_ID[chunk_id] = parent_id
    PARENT_CHILDREN_BY_ID.setdefault(parent_id, []).append(chunk)

for parent_id, children in PARENT_CHILDREN_BY_ID.items():
    children.sort(key=lambda row: (
        int(row.get("chunk_index") or 0),
        str(row.get("chunk_id") or ""),
    ))

dataset_hash = hashlib.sha256()
for chunk in CHUNKS:
    dataset_hash.update(str(chunk["chunk_id"]).encode("utf-8"))
    dataset_hash.update(b"\0")
    dataset_hash.update(build_dense_structured_v2_text(chunk).encode("utf-8"))
    dataset_hash.update(b"\0")
DATASET_FINGERPRINT = dataset_hash.hexdigest()
ES_INDEX_NAME = f"kdic-hybrid-nori-none-dense-v3-{DATASET_FINGERPRINT[:12]}"

print("데이터 경로:", DATA_ROOT)
print("청크 수:", len(CHUNKS))
print("Parent 문서 수:", len(PARENT_CHILDREN_BY_ID))
print("데이터 지문:", DATASET_FINGERPRINT[:16])
print("업무:", sorted({_clean_text(chunk.get("business_function")) for chunk in CHUNKS}))


## 4-1. 기존 Dense-structured-v2 캐시 업로드

**최초 생성 시에는 이 셀을 건너뜁니다.** 이미 만들어 둔 캐시를 재사용할 때만 실행하고,
정확히 `kdic_dense_structured_v2_embeddings.jsonl` 파일 하나를 업로드하세요.

업로드한 파일은 뒤의 Dense 준비 셀에서 청크 ID, 모델, 입력 구조 버전, 입력 SHA-256,
임베딩 차원을 검증합니다. 검증을 통과하지 못한 항목이 있더라도 기본 설정에서는 자동으로
재임베딩하지 않고 중단합니다.


In [ ]:
def upload_dense_cache_from_browser(
    target_path: Path = DENSE_CACHE_PATH,
    *,
    reuse_existing: bool = True,
) -> Path:
    if reuse_existing and target_path.is_file() and target_path.stat().st_size > 0:
        print("이미 런타임에 있는 캐시를 재사용합니다:", target_path)
        return target_path

    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError(
            f"Colab이 아니면 {target_path} 경로에 {DENSE_CACHE_FILENAME}을 직접 복사하세요."
        ) from error

    print(f"{DENSE_CACHE_FILENAME} 파일 하나를 업로드하세요.")
    uploaded = files.upload()
    if set(uploaded) != {DENSE_CACHE_FILENAME}:
        raise RuntimeError(
            "업로드 파일명이 정확하지 않습니다. "
            f"expected={DENSE_CACHE_FILENAME}, uploaded={list(uploaded)}"
        )

    payload = uploaded[DENSE_CACHE_FILENAME]
    if not payload:
        raise RuntimeError("업로드한 Dense 캐시 파일이 비어 있습니다.")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_bytes(payload)
    print(f"Dense 캐시 업로드 완료: {target_path} ({target_path.stat().st_size:,} bytes)")
    return target_path


# 두 번째 런타임부터 이 셀을 실행합니다. 최초 생성 시에는 셀 자체를 건너뛰세요.
UPLOADED_DENSE_CACHE_PATH = upload_dense_cache_from_browser()


## 5. HCX 클라이언트와 Dense-structured-v2 임베딩 캐시

Dense Structured V2 캐시를 먼저 검증해 `DENSE_MATRIX`와 `DENSE_VECTOR_BY_ID`를 만듭니다. 검증된 벡터만 다음 Elasticsearch 통합 인덱스에 저장합니다.


In [ ]:
def load_hcx_api_key() -> str:
    key: str | None = None
    try:
        from google.colab import userdata
        key = userdata.get("HCX_API_KEY")
    except Exception:
        key = os.environ.get("HCX_API_KEY")
    if not key:
        key = getpass.getpass("HCX_API_KEY: ")

    key = str(key or "").strip()
    if not key:
        raise ValueError("HCX_API_KEY가 비어 있습니다.")
    if key.lower().startswith("bearer "):
        raise ValueError("HCX_API_KEY 앞에 'Bearer '를 붙이지 마세요.")
    if any(character.isspace() for character in key):
        raise ValueError("HCX_API_KEY 안에 공백 또는 줄바꿈이 있습니다.")
    return key


HCX_API_KEY = load_hcx_api_key()
HCX_CLIENT = OpenAI(
    api_key=HCX_API_KEY,
    base_url=HCX_BASE_URL,
    timeout=HCX_REQUEST_TIMEOUT,
    max_retries=HCX_MAX_RETRIES,
)


def embed_hcx_single(text: str) -> np.ndarray:
    cleaned = _clean_text(text)
    if not cleaned:
        raise ValueError("임베딩 입력이 비어 있습니다.")
    response = HCX_CLIENT.embeddings.create(
        model=HCX_EMBEDDING_MODEL,
        input=cleaned,
        encoding_format=HCX_ENCODING_FORMAT,
    )
    if len(response.data) != 1:
        raise RuntimeError(f"단일 임베딩 응답 개수가 1이 아닙니다: {len(response.data)}")
    vector = np.asarray(response.data[0].embedding, dtype=np.float32)
    if vector.ndim != 1 or vector.size == 0:
        raise RuntimeError(f"잘못된 임베딩 shape: {vector.shape}")
    if not np.all(np.isfinite(vector)):
        raise RuntimeError("임베딩에 NaN 또는 무한대가 있습니다.")
    return vector


print("HCX 클라이언트 준비 완료")
print("Dense 입력 예시:\n", build_dense_structured_v2_text(CHUNKS[0])[:500])


In [ ]:
def structured_input_sha256(chunk: dict[str, Any]) -> str:
    text = build_dense_structured_v2_text(chunk)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _load_valid_dense_cache(path: Path) -> dict[str, dict[str, Any]]:
    if not path.exists():
        return {}
    valid: dict[str, dict[str, Any]] = {}
    for record in _read_jsonl(path):
        chunk_id = str(record.get("chunk_id") or "")
        chunk = CHUNKS_BY_ID.get(chunk_id)
        if chunk is None:
            continue
        if record.get("model") != HCX_EMBEDDING_MODEL:
            continue
        if record.get("input_version") != DENSE_INPUT_VERSION:
            continue
        if record.get("cache_version") != DENSE_CACHE_VERSION:
            continue
        if record.get("input_sha256") != structured_input_sha256(chunk):
            continue
        vector = np.asarray(record.get("embedding"), dtype=np.float32)
        if vector.ndim != 1 or vector.size == 0 or not np.all(np.isfinite(vector)):
            continue
        if int(record.get("dimensions") or 0) != vector.size:
            continue
        valid[chunk_id] = record
    return valid


def _write_dense_cache_atomic(path: Path, records_by_id: dict[str, dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as file:
        for chunk in CHUNKS:
            record = records_by_id.get(str(chunk["chunk_id"]))
            if record is not None:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")
    os.replace(temp_path, path)


def prepare_dense_embeddings(
    cache_path: Path = DENSE_CACHE_PATH,
    checkpoint_every: int = 5,
    allow_generate_missing: bool = False,
) -> tuple[np.ndarray, list[str]]:
    cache = _load_valid_dense_cache(cache_path)
    missing = [chunk for chunk in CHUNKS if str(chunk["chunk_id"]) not in cache]
    print(f"Dense cache: valid={len(cache)}, missing={len(missing)}")

    if missing and not allow_generate_missing:
        raise RuntimeError(
            "Dense 캐시에 유효한 문서 임베딩이 부족하므로 자동 생성을 중단했습니다. "
            f"valid={len(cache)}, missing={len(missing)}. "
            "기존 캐시 파일을 올바르게 업로드하거나, 최초 생성일 때만 "
            "CREATE_DENSE_CACHE_ONCE=True로 바꾼 뒤 이 셀을 다시 실행하세요."
        )

    try:
        for index, chunk in enumerate(
            tqdm(missing, desc="Dense-structured-v2 embedding"),
            start=1,
        ):
            chunk_id = str(chunk["chunk_id"])
            input_text = build_dense_structured_v2_text(chunk)
            vector = embed_hcx_single(input_text)
            cache[chunk_id] = {
                "chunk_id": chunk_id,
                "model": HCX_EMBEDDING_MODEL,
                "encoding_format": HCX_ENCODING_FORMAT,
                "input_version": DENSE_INPUT_VERSION,
                "input_sha256": hashlib.sha256(input_text.encode("utf-8")).hexdigest(),
                "cache_version": DENSE_CACHE_VERSION,
                "dimensions": int(vector.size),
                "embedding": vector.tolist(),
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            if index % checkpoint_every == 0:
                _write_dense_cache_atomic(cache_path, cache)
    finally:
        if cache:
            _write_dense_cache_atomic(cache_path, cache)

    ordered_vectors: list[np.ndarray] = []
    dimensions: set[int] = set()
    chunk_ids: list[str] = []
    for chunk in CHUNKS:
        chunk_id = str(chunk["chunk_id"])
        record = cache.get(chunk_id)
        if record is None:
            raise RuntimeError(f"Dense 캐시 누락: {chunk_id}")
        vector = np.asarray(record["embedding"], dtype=np.float32)
        norm = float(np.linalg.norm(vector))
        if norm == 0.0:
            raise RuntimeError(f"영벡터 임베딩: {chunk_id}")
        ordered_vectors.append(vector / norm)
        dimensions.add(int(vector.size))
        chunk_ids.append(chunk_id)

    if len(dimensions) != 1:
        raise RuntimeError(f"임베딩 차원 불일치: {dimensions}")
    return np.vstack(ordered_vectors), chunk_ids


def download_dense_cache_to_browser(
    cache_path: Path = DENSE_CACHE_PATH,
) -> None:
    if not cache_path.is_file() or cache_path.stat().st_size == 0:
        raise FileNotFoundError(f"다운로드할 Dense 캐시가 없습니다: {cache_path}")
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError(f"Colab 외부에서는 이 파일을 직접 가져가세요: {cache_path}") from error
    print(f"Dense 캐시 다운로드를 시작합니다: {cache_path.name}")
    files.download(str(cache_path))


# 최초 캐시를 만드는 단 한 번만 True로 바꾸세요.
# 이후 A~E 노트북에서는 False를 유지하고 캐시 업로드 셀을 실행합니다.
CREATE_DENSE_CACHE_ONCE = False

DENSE_MATRIX, DENSE_CHUNK_IDS = prepare_dense_embeddings(
    allow_generate_missing=CREATE_DENSE_CACHE_ONCE,
)
DENSE_DIMENSION = int(DENSE_MATRIX.shape[1])
DENSE_VECTOR_BY_ID = dict(zip(DENSE_CHUNK_IDS, DENSE_MATRIX))
if len(DENSE_VECTOR_BY_ID) != len(CHUNKS):
    raise RuntimeError(
        f"Dense 벡터 매핑 건수 불일치: vectors={len(DENSE_VECTOR_BY_ID)}, chunks={len(CHUNKS)}"
    )
missing_dense_ids = sorted(set(CHUNKS_BY_ID) - set(DENSE_VECTOR_BY_ID))
if missing_dense_ids:
    raise RuntimeError(f"Dense 벡터가 없는 청크가 있습니다: {missing_dense_ids[:10]}")

print("Dense matrix:", DENSE_MATRIX.shape)
print("Dense vector map:", len(DENSE_VECTOR_BY_ID))
print("Dense cache:", DENSE_CACHE_PATH)
if CREATE_DENSE_CACHE_ONCE:
    download_dense_cache_to_browser()
    print("다운로드한 파일을 보관하고 A~E 실험에서 공통으로 업로드해 사용하세요.")
else:
    print("문서 임베딩 API 호출 없이 업로드된 Dense 캐시를 사용했습니다.")


## 6. Elasticsearch BM25 Nori-none + Dense kNN 통합 인덱스

같은 Elasticsearch 인덱스에 `search_text`와 `embedding`을 함께 저장합니다. BM25는 Nori-none, Dense는 `dense_vector`의 dot-product kNN을 사용합니다. 새 인덱스 이름을 사용하므로 기존 BM25 전용 인덱스는 건드리지 않습니다.


In [ ]:
def connect_elasticsearch() -> Elasticsearch:
    client = Elasticsearch(
        ES_URL,
        request_timeout=120,
        max_retries=5,
        retry_on_timeout=True,
    )
    try:
        info = client.info()
    except Exception as error:
        raise RuntimeError(
            "Elasticsearch 연결 실패입니다. 2번 준비 셀의 마지막 로그를 확인하세요. "
            f"원인={type(error).__name__}: {error}"
        ) from error

    running_version = str(info["version"]["number"])
    if running_version != ES_EXPECTED_VERSION:
        raise RuntimeError(
            f"Elasticsearch 버전 불일치: running={running_version}, expected={ES_EXPECTED_VERSION}"
        )

    nodes = client.nodes.info(metric="plugins")
    plugin_names = {
        str(plugin.get("name") or "")
        for node in nodes["nodes"].values()
        for plugin in node.get("plugins", [])
    }
    if "analysis-nori" not in plugin_names:
        raise RuntimeError(f"analysis-nori 플러그인이 없습니다: {sorted(plugin_names)}")
    return client


def _hybrid_index_is_reusable(client: Elasticsearch) -> bool:
    if FORCE_REBUILD_HYBRID_INDEX:
        return False
    if not client.indices.exists(index=ES_INDEX_NAME):
        return False
    count = int(client.count(index=ES_INDEX_NAME)["count"])
    mapping = client.indices.get_mapping(index=ES_INDEX_NAME)
    mappings = mapping[ES_INDEX_NAME]["mappings"]
    metadata = mappings.get("_meta", {})
    properties = mappings.get("properties", {})
    embedding = properties.get("embedding", {})
    return (
        count == len(CHUNKS)
        and metadata.get("schema_version") == ES_INDEX_SCHEMA_VERSION
        and metadata.get("dataset_fingerprint") == DATASET_FINGERPRINT
        and metadata.get("dense_input_version") == DENSE_INPUT_VERSION
        and metadata.get("dense_model") == HCX_EMBEDDING_MODEL
        and int(metadata.get("dense_dimension") or 0) == DENSE_DIMENSION
        and embedding.get("type") == "dense_vector"
        and int(embedding.get("dims") or 0) == DENSE_DIMENSION
    )


def prepare_hybrid_nori_dense_index(client: Elasticsearch) -> None:
    if _hybrid_index_is_reusable(client):
        print(f"기존 BM25 + Dense 통합 인덱스 재사용: {ES_INDEX_NAME}")
        return

    # 이름에 schema v3와 데이터 지문이 포함된 전용 인덱스만 재생성합니다.
    if client.indices.exists(index=ES_INDEX_NAME):
        client.indices.delete(index=ES_INDEX_NAME)

    client.indices.create(
        index=ES_INDEX_NAME,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "similarity": {
                "kdic_bm25": {
                    "type": "BM25",
                    "k1": 1.2,
                    "b": 0.75,
                }
            },
            "analysis": {
                "tokenizer": {
                    "kdic_nori_none_tokenizer": {
                        "type": "nori_tokenizer",
                        "decompound_mode": "none",
                    }
                },
                "analyzer": {
                    ES_ANALYZER_NAME: {
                        "type": "custom",
                        "tokenizer": "kdic_nori_none_tokenizer",
                    }
                },
            },
        },
        mappings={
            "_meta": {
                "schema_version": ES_INDEX_SCHEMA_VERSION,
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "dense_input_version": DENSE_INPUT_VERSION,
                "dense_model": HCX_EMBEDDING_MODEL,
                "dense_dimension": DENSE_DIMENSION,
            },
            "properties": {
                "chunk_id": {"type": "keyword"},
                "search_text": {
                    "type": "text",
                    "analyzer": ES_ANALYZER_NAME,
                    "search_analyzer": ES_ANALYZER_NAME,
                    "similarity": "kdic_bm25",
                },
                "embedding": {
                    "type": "dense_vector",
                    "dims": DENSE_DIMENSION,
                    "index": True,
                    "similarity": "dot_product",
                },
            },
        },
    )

    actions = (
        {
            "_op_type": "index",
            "_index": ES_INDEX_NAME,
            "_id": str(chunk["chunk_id"]),
            "_source": {
                "chunk_id": str(chunk["chunk_id"]),
                "search_text": build_dense_structured_v2_text(chunk),
                "embedding": DENSE_VECTOR_BY_ID[str(chunk["chunk_id"])].tolist(),
            },
        }
        for chunk in CHUNKS
    )
    bulk_client = client.options(request_timeout=120)
    success, errors = helpers.bulk(
        bulk_client,
        actions,
        chunk_size=100,
        max_retries=4,
        initial_backoff=1,
        max_backoff=8,
        raise_on_error=False,
        raise_on_exception=False,
    )
    client.indices.refresh(index=ES_INDEX_NAME)

    if errors:
        preview = json.dumps(errors[:3], ensure_ascii=False, default=str)[:3000]
        raise RuntimeError(f"Hybrid 인덱싱 실패 {len(errors)}건: {preview}")
    if int(success) != len(CHUNKS):
        raise RuntimeError(f"Hybrid 인덱싱 건수 불일치: success={success}, chunks={len(CHUNKS)}")

    actual_count = int(client.count(index=ES_INDEX_NAME)["count"])
    dense_count = int(
        client.count(
            index=ES_INDEX_NAME,
            query={"exists": {"field": "embedding"}},
        )["count"]
    )
    if actual_count != len(CHUNKS) or dense_count != len(CHUNKS):
        raise RuntimeError(
            "Hybrid 저장 건수 불일치: "
            f"documents={actual_count}, dense_vectors={dense_count}, chunks={len(CHUNKS)}"
        )

    analysis = client.indices.analyze(
        index=ES_INDEX_NAME,
        analyzer=ES_ANALYZER_NAME,
        text="예금자보호제도",
    )
    if not analysis.get("tokens"):
        raise RuntimeError("Nori 분석 결과가 비어 있습니다.")


ES = connect_elasticsearch()
prepare_hybrid_nori_dense_index(ES)

print("Elasticsearch:", ES.info()["version"]["number"])
print("BM25 + Dense 인덱스:", ES_INDEX_NAME)
print("통합 문서 수:", ES.count(index=ES_INDEX_NAME)["count"])
print("Dense 벡터 수:", ES.count(
    index=ES_INDEX_NAME,
    query={"exists": {"field": "embedding"}},
)["count"])
print("Nori-none 토큰:", [
    token["token"]
    for token in ES.indices.analyze(
        index=ES_INDEX_NAME,
        analyzer=ES_ANALYZER_NAME,
        text="예금자보호제도",
    )["tokens"]
])


## 2. V1.5 질의분석 모듈

In [ ]:
%%writefile kdic_integrated_eval_core.py
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass
class QueryPlan:
    need_id: str
    variant_id: str
    dense_query: str
    bm25_query: str
    filter_mode: str = "NONE"
    business_filters: list[str] = field(default_factory=list)
    soft_business_hints: list[str] = field(default_factory=list)
    query_weight: float = 1.0
    query_source: str = "ORIGINAL"


@dataclass
class AnalyzerCase:
    evaluation_id: str
    analyzer: str
    original_question: str
    route: str
    analysis_latency_ms: float
    plans: list[QueryPlan]
    raw_result: dict[str, Any]


def normalize_route(value: Any) -> str:
    text = str(value or "").strip().upper()
    mapping = {
        "SIMPLE_RETRIEVE": "RETRIEVE",
        "MULTI_RETRIEVE": "RETRIEVE",
        "OUT_OF_SCOPE": "OUT_OF_SCOPE",
        "OOS": "OUT_OF_SCOPE",
        "DIRECT": "DIRECT_RESPONSE",
        "DIRECT_RESPONSE": "DIRECT_RESPONSE",
        "CLARIFY": "CLARIFY",
        "RETRIEVE": "RETRIEVE",
    }
    return mapping.get(text, text)

In [ ]:
%%writefile kdic_lightweight_router_v1.py
from __future__ import annotations

"""KDIC 간편 라우터 V1.

설계 목표
---------
1. 명백한 DIRECT/OUT_OF_SCOPE/CLARIFY만 규칙으로 차단하고 나머지는 검색으로 보낸다.
2. 단일질의의 검색 문자열은 사용자 원문을 그대로 보존한다.
3. 실제로 독립 검색이 필요한 복합질의만 보수적으로 분해한다.
4. 업무 필터는 SOFT/NONE만 사용해 잘못된 HARD 필터를 구조적으로 막는다.
5. 외부 API나 모델을 호출하지 않아 라우팅 지연을 최소화한다.
"""

import json
import re
import time
import unicodedata
from dataclasses import dataclass
from typing import Any, Iterable, Mapping, Sequence


PIPELINE_VERSION = "KDIC_LIGHTWEIGHT_ROUTER_V1_2026_08_13"

BUSINESS_FUNCTIONS = (
    "예금자보호제도",
    "예금보험금 안내",
    "고객 미수령금 신청",
    "착오송금 반환 신청",
    "채무조정 안내",
    "은닉재산 신고",
)

INTENTS = (
    "AMOUNT",
    "ELIGIBILITY",
    "TIME",
    "APPLICATION",
    "OVERVIEW",
    "STATUS",
    "DOCUMENTS",
    "CONTACT",
)

BUSINESS_KEYWORDS: dict[str, tuple[str, ...]] = {
    "예금자보호제도": (
        "예금자보호", "보호한도", "보호 한도", "보호대상", "예금 보호",
        "예금은 얼마까지 보호", "금융상품이 보호", "금융회사가 보호 대상",
    ),
    "예금보험금 안내": (
        "예금보험금", "보험금 지급", "보험사고", "가지급금", "개산지급금",
        "1종 보험사고", "2종 보험사고",
    ),
    "고객 미수령금 신청": (
        "고객 미수령금", "미수령금", "파산배당금", "개산지급금 정산금",
        "지급대행점", "상속인 금융거래 조회", "상속인 금융거래 조회서비스",
    ),
    "착오송금 반환 신청": (
        "착오송금", "착오 송금", "잘못 보낸 돈", "잘못 송금", "반환지원",
        "매입계약", "지급명령", "강제집행", "송금인", "수취인",
        "계좌번호를 잘못", "엉뚱한 사람에게 보낸 돈",
    ),
    "채무조정 안내": (
        "채무조정", "신용회복지원", "파산선고", "채무감면", "개인회생",
        "개인파산", "워크아웃", "변제기간", "부채증명원", "채무정보", "면책",
    ),
    "은닉재산 신고": (
        "은닉재산", "은닉 재산", "금융부실관련자", "부실관련자", "차명재산",
        "차명 재산", "신고 포상금",
    ),
}

STRONG_BUSINESS_KEYWORDS: dict[str, tuple[str, ...]] = {
    "예금자보호제도": ("예금자보호", "보호한도", "보호 한도", "예금 보호"),
    "예금보험금 안내": ("예금보험금", "보험금 지급", "1종 보험사고", "2종 보험사고"),
    "고객 미수령금 신청": (
        "고객 미수령금", "미수령금", "파산배당금", "지급대행점", "상속인 금융거래 조회",
    ),
    "착오송금 반환 신청": (
        "착오송금", "착오 송금", "반환지원", "잘못 보낸 돈", "지급명령", "강제집행",
    ),
    "채무조정 안내": (
        "채무조정", "신용회복지원", "개인회생", "개인파산", "워크아웃", "부채증명원",
    ),
    "은닉재산 신고": (
        "은닉재산", "은닉 재산", "금융부실관련자", "차명재산", "차명 재산", "신고 포상금",
    ),
}

TYPO_MAP = (
    ("예금보헝금", "예금보험금"),
    ("예금보혐금", "예금보험금"),
    ("착오송금반한", "착오송금 반환"),
    ("반한지원", "반환지원"),
    ("반환지웜", "반환지원"),
    ("미수령금신정", "미수령금 신청"),
    ("통합신정", "통합신청"),
    ("검새", "검색"),
    ("발샐", "발생"),
    ("관게", "관계"),
    ("요정", "요청"),
    ("언재", "언제"),
    ("제외돼는", "제외되는"),
)

DIRECT_META_PATTERN = re.compile(
    r"^(?:안녕|안녕하세요|반갑습니다|반가워요|고마워요|고맙습니다|감사합니다|도움이 됐어요|"
    r"알겠습니다|무슨 질문을 할 수 있나요|지원하는 업무를 (?:알려주세요|목록으로 보여주세요)|"
    r"이 챗봇은 어떻게 사용하면 되나요|답변을 쉽게 설명해 줄 수 있나요|"
    r"전문가 수준으로 자세히 설명해 주세요|긴 설명보다 핵심 내용만 먼저 알려주세요|"
    r"질문을 잘못 입력했어요[.]? 다시 물어볼게요)[.!?]*$",
    re.I,
)

REFORMAT_PATTERN = re.compile(
    r"^(?:쉽게 설명해 주세요|핵심만 알려주세요|표로 정리해 주세요|더 자세히 설명해 주세요)[.!?]*$",
    re.I,
)

OOS_PATTERN = re.compile(
    r"(?:코스피|비트코인|주택담보대출\s*금리|대출금리|신용점수|실손보험|국민연금|"
    r"보이스피싱.*(?:경찰|신고)|은행\s*계좌를\s*새로|계좌\s*개설|해외송금\s*수수료|"
    r"카드\s*결제.*환불|전세대출|세금\s*환급|퇴직금|개인정보\s*유출|상속세|"
    r"환율|환전소|주식\s*(?:투자|포트폴리오)|신용카드\s*연회비|사업자등록|"
    r"(?:서울|오늘|내일|이번\s*주말)?.{0,8}(?:날씨|기온|미세먼지))",
    re.I,
)

GENERIC_BUSINESS_CLARIFY_PATTERN = re.compile(
    r"^(?:신청\s*(?:방법|자격|기한|과정|대상)|접수\s*(?:방법|절차)|제출해야\s*하는\s*서류|"
    r"필요한\s*서류|조회는\s*어디에서|온라인으로\s*신청|방문해서\s*접수|접수\s*후\s*처리\s*기간|"
    r"신청\s*자격과\s*제외\s*조건|신청\s*과정에서\s*수수료나\s*비용|"
    r"처리\s*결과는\s*어디에서\s*확인|이미\s*접수한\s*신청을\s*취소|"
    r"문의하거나\s*접수하려면\s*어느\s*기관|제가\s*신청\s*대상에\s*해당|"
    r"본인\s*대신\s*대리인이\s*신청|상속인이\s*신청하거나\s*받을\s*수|"
    r"처리\s*기간|문의처|신청\s*비용).*$",
    re.I,
)

TARGET_DEMONSTRATIVE_PATTERN = re.compile(
    r"(?:^|[\s,.(])(?:제가\s*(?:말한|가입한|가진|본)\s*)?"
    r"이\s*(?:금융상품|계좌|상품|돈|송금|거래|금액|채무|재산)"
    r"(?:을|를|이|가|도|은|는|의|이나|과|와|\s|[,.!?]|$)",
    re.I,
)

TARGET_REFERENCE_PHRASES = (
    "제가 가입한 상품", "어떤 송금 건", "어떤 예금에 대해", "어떤 돈을 신청",
    "어떤 예금이나 금융상품",
)

APPLICANT_REFERENCE_PHRASES = (
    "제 신청 유형", "제 신청 자격", "제 경우", "누구를 신청인", "누가 방문",
    "신고 주체 유형", "신청인란",
)

CASE_REFERENCE_PHRASES = (
    "제 상황", "현재 상황", "제 채무 상태", "제 신고 상황", "반려", "거절",
    "보완 요청", "진행되지 않", "여러 금융회사에 예금", "여러 계좌에 나뉘",
)

HIGH_PRECISION_INTENT_RULES: tuple[tuple[str, tuple[str, ...]], ...] = (
    ("DOCUMENTS", (
        r"필요한?\s*(?:서류|증빙)", r"제출(?:해야\s*하는|할)?\s*서류", r"구비\s*서류",
        r"신분증", r"위임장", r"준비(?:해야\s*할|할)?\s*서류", r"무엇을\s*(?:더\s*)?준비",
    )),
    ("STATUS", (
        r"어디(?:서|에서)\s*(?:확인|조회|검색)", r"조회\s*(?:방법|결과)", r"처리\s*결과",
        r"진행\s*(?:상황|상태)", r"지급\s*정보.*보는\s*방법", r"있는지.*조회",
    )),
    ("APPLICATION", (
        r"신청\s*(?:방법|절차)", r"접수\s*(?:방법|절차)", r"제출\s*방법", r"신고\s*채널",
        r"어떻게\s*(?:신청|접수|청구)", r"(?:온라인|방문|직접).*신청.*(?:가능|할\s*수)",
        r"취소.*방법", r"철회.*방법", r"무엇을\s*해야", r"어디에\s*접수",
    )),
    ("TIME", (
        r"언제(?:부터|까지)", r"신청.*(?:기한|기간|시점)", r"처리\s*기간", r"소요\s*(?:기간|시간)",
        r"얼마나\s*걸", r"언제\s*(?:지급|찾)",
    )),
    ("AMOUNT", (
        r"보호\s*한도", r"지급\s*금액", r"금액\s*계산", r"금액.*얼마(?:여야|이어야)",
        r"계산\s*(?:기준|방법)", r"수수료\s*(?:금액|비용)", r"얼마나\s*(?:감면|지급|보상|돌려|받|보호)",
        r"비용\s*차감", r"최종\s*보호금액",
    )),
    ("CONTACT", (
        r"연락처", r"전화번호", r"문의처", r"어디로\s*연락", r"어느\s*기관.*문의",
    )),
    ("ELIGIBILITY", (
        r"신청\s*(?:대상|자격|요건)", r"가능한\s*대상", r"제외되는?\s*경우", r"받을\s*수\s*있",
        r"신청할\s*수\s*있", r"포함되", r"어떤\s*경우.*(?:지급|지원|보호)",
        r"(?:대상|자격)에\s*해당", r"(?:예금|계좌|금융상품|상품|원금|이자|채권).{0,30}보호(?:가)?\s*되",
        r"지원\s*대상", r"누가\s*(?:신청|수령)", r"보호\s*대상",
    )),
    ("OVERVIEW", (
        r"무엇(?:인가요|인지|이며)", r"뭐예요", r"의미", r"정의", r"차이", r"종류", r"개요",
        r"설명", r"관계", r"왜\s*(?:발생|제외)", r"어떤\s*성격",
    )),
)

WEAK_INTENT_RULES: tuple[tuple[str, tuple[str, ...]], ...] = (
    ("DOCUMENTS", (r"서류", r"증빙", r"준비")),
    ("STATUS", (r"조회", r"확인", r"검색")),
    ("APPLICATION", (r"신청", r"접수", r"절차", r"제출", r"청구", r"신고")),
    ("TIME", (r"기간", r"기한", r"시점", r"언제")),
    ("AMOUNT", (r"한도", r"금액", r"계산", r"비용", r"포상금")),
    ("CONTACT", (r"연락", r"문의", r"전화")),
    ("ELIGIBILITY", (r"대상", r"자격", r"요건", r"조건", r"가능", r"보호")),
    ("OVERVIEW", (r"설명", r"관계", r"방식", r"종류", r"의미")),
)

# 두 정보가 서로 밀접한 하나의 검색 문서에서 함께 해결될 가능성이 높은 결합입니다.
# 이런 결합은 요구가 두 개여도 원문을 유지합니다.
COHESIVE_NO_SPLIT_PATTERNS = (
    re.compile(r"(?:대상|자격|조건).{0,28}(?:서류|준비)|(?:서류|준비).{0,28}(?:제출|신청|접수)\s*방법", re.I),
    re.compile(r"(?:누가|대리인|상속인|본인|법인).{0,35}(?:서류|준비)", re.I),
    re.compile(r"(?:한도|금액).{0,30}(?:포함|계산|합산|상계)|(?:포함|계산|합산|상계).{0,30}(?:한도|금액)", re.I),
    re.compile(r"(?:사유|이유).{0,25}(?:조건|해제)|(?:의미|무엇).{0,25}(?:차이|관계|종류)", re.I),
    re.compile(r"(?:언제까지|기한|기간).{0,25}(?:어디에서|어디에)\s*(?:신청|확인)", re.I),
    re.compile(r"(?:조회|확인)한?\s*(?:뒤|다음).{0,35}(?:신청|지급)", re.I),
    re.compile(r"(?:퇴직연금).{0,45}(?:연금저축).{0,45}(?:보호|한도)", re.I),
)

# 서로 다른 사전 용어가 잡혀도 목적이 관계·차이 또는 결합 가능성 확인이면 원문을 유지합니다.
CROSS_TERM_RELATION_KEEP_PATTERN = re.compile(
    r"(?:와|과|및).{0,38}(?:관계|차이)|(?:관계|차이).{0,38}(?:와|과|및)|"
    r"(?:와|과|및).{0,38}(?:함께|동시에)\s*(?:조회|확인|신청|보호).*(?:가능|할\s*수)",
    re.I,
)

# 독립된 대상·상황·처리 단계가 명시된 경우에만 같은 업무 안에서도 분해합니다.
STRONG_SAME_BUSINESS_SPLIT_PATTERNS = (
    re.compile(r"(?:때|경우)와.{0,55}(?:때|경우)", re.I),
    re.compile(r"(?:외화예금|간편송금|온라인\s*신청).{0,45}(?:후순위채권|해외\s*계좌|방문\s*신청)", re.I),
    re.compile(r"(?:1종\s*보험사고).{0,45}(?:2종\s*보험사고)", re.I),
    re.compile(r"(?:영업정지).{0,35}(?:기존\s*대출|대출\s*거래)", re.I),
    re.compile(r"(?:미리\s*신청|신청\s*전).{0,45}(?:실제\s*보험사고|접수\s*후)", re.I),
    re.compile(r"(?:지급되는\s*조건|지급\s*조건).{0,35}(?:실제\s*)?신청\s*절차", re.I),
    re.compile(r"(?:온라인\s*신청).{0,45}(?:지급대행점\s*)?방문\s*신청", re.I),
    re.compile(r"(?:신청(?:하기)?\s*전|신청\s*전에).{0,45}(?:접수|신청)\s*후", re.I),
    re.compile(r"(?:접수|신청).{0,25}(?:결과|진행\s*절차).{0,25}(?:확인|진행)", re.I),
    re.compile(r"(?:파산\s*금융회사).{0,40}(?:남은\s*)?미수령금.*신청", re.I),
    re.compile(r"(?:기간|얼마나\s*걸).{0,40}(?:비용\s*차감|차감\s*방식)", re.I),
    re.compile(r"(?:금융회사).{0,30}보호\s*대상.{0,30}(?:금융상품).{0,20}보호", re.I),
    re.compile(r"(?:미성년자).{0,35}보호되.{0,35}(?:누가|수령)", re.I),
    re.compile(r"(?:상속인).{0,30}(?:조회).{0,20}(?:뒤|다음).{0,30}(?:지급을\s*)?신청", re.I),
    re.compile(r"(?:제외되는\s*경우).{0,35}(?:제외되는\s*이유|왜\s*제외)", re.I),
)

CLAUSE_BOUNDARY_PATTERN = re.compile(
    r"\s*(?:[.!?;]+|,?\s*(?:그리고|또|혹시|별도로|그와\s*별개로|반면에|반면|뿐만\s*아니라)\s+)\s*",
    re.I,
)

CONJUNCTION_BOUNDARY_PATTERN = re.compile(
    r"\s*(?:,\s*|\s+)(?:그리고|또|혹시|별도로|반면에|반면)\s*",
    re.I,
)

NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)*(?:\s*(?:원|만원|억원|개월|년|일|%))?")
NEGATION_TERMS = ("아니", "못", "제외", "불가", "없", "않", "전혀")


@dataclass(frozen=True)
class RouterConfig:
    max_subqueries: int = 4
    include_original_anchor_for_multi: bool = True
    allow_hard_filter: bool = False
    min_subquery_chars: int = 5


def normalize_query(text: Any) -> dict[str, Any]:
    original = str(text or "")
    value = unicodedata.normalize("NFKC", original)
    changes: list[str] = []
    cleaned = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", value)
    if cleaned != value:
        changes.append("CONTROL_CHARACTER")
        value = cleaned
    cleaned = re.sub(r"([!?ㅋㅎㅠㅜ])\1{2,}", r"\1\1", value)
    if cleaned != value:
        changes.append("REPEATED_CHARACTER")
        value = cleaned
    for wrong, correct in TYPO_MAP:
        if wrong in value:
            value = value.replace(wrong, correct)
            changes.append("EXPLICIT_TYPO")
    cleaned = re.sub(r"\s+", " ", value).strip()
    if cleaned != value:
        changes.append("WHITESPACE")
    if not cleaned:
        raise ValueError("사용자 질의가 비어 있습니다.")
    return {"original_query": original, "normalized_query": cleaned, "changes": list(dict.fromkeys(changes))}


def _ordered_unique(values: Iterable[Any]) -> list[Any]:
    output: list[Any] = []
    seen: set[Any] = set()
    for value in values:
        if value is None or value in seen:
            continue
        seen.add(value)
        output.append(value)
    return output


def _compact(text: str) -> str:
    return re.sub(r"\s+", "", text).lower()


def find_business_matches(text: str) -> list[dict[str, Any]]:
    compact = _compact(text)
    found: list[dict[str, Any]] = []
    for business, keywords in BUSINESS_KEYWORDS.items():
        evidence = [keyword for keyword in keywords if _compact(keyword) in compact]
        if not evidence:
            continue
        strong = [term for term in STRONG_BUSINESS_KEYWORDS[business] if _compact(term) in compact]
        found.append({
            "business_function": business,
            "evidence": _ordered_unique(evidence),
            "strong_evidence": _ordered_unique(strong),
            "confidence": 0.99 if strong else 0.80,
        })
    return found


def find_businesses(text: str) -> list[str]:
    return [row["business_function"] for row in find_business_matches(text)]


def find_intent_matches(text: str) -> list[dict[str, Any]]:
    matches: list[dict[str, Any]] = []
    for source, rules in (("HIGH_PRECISION_RULE", HIGH_PRECISION_INTENT_RULES), ("WEAK_RULE", WEAK_INTENT_RULES)):
        for intent, patterns in rules:
            hit = next((m for pattern in patterns if (m := re.search(pattern, text, flags=re.I))), None)
            if hit and intent not in {row["intent"] for row in matches}:
                matches.append({
                    "intent": intent,
                    "source": source,
                    "evidence": hit.group(0),
                    "start": hit.start(),
                    "end": hit.end(),
                })
    return matches


def _parse_previous_turns(value: Any) -> list[dict[str, str]]:
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() == "nan":
            return []
        try:
            value = json.loads(text)
        except json.JSONDecodeError:
            return [{"user": text, "assistant": ""}]
    if not isinstance(value, list):
        return []
    output = []
    for row in value[-3:]:
        if isinstance(row, Mapping):
            user = str(row.get("user") or row.get("question") or "").strip()
            assistant = str(row.get("assistant") or row.get("answer") or "").strip()
            if user or assistant:
                output.append({"user": user, "assistant": assistant})
        elif str(row).strip():
            output.append({"user": str(row).strip(), "assistant": ""})
    return output


def build_context(previous_turns: Any = None, conversation_state: Mapping[str, Any] | None = None) -> dict[str, Any]:
    state = dict(conversation_state or {})
    turns = _parse_previous_turns(previous_turns if previous_turns is not None else state.get("recent_turns"))
    confirmed = dict(state.get("confirmed") or {}) if isinstance(state.get("confirmed"), Mapping) else {}
    context_text = " ".join(row["user"] for row in turns if row.get("user"))
    context_businesses = find_businesses(context_text)
    if len(context_businesses) == 1 and not confirmed.get("business_function"):
        confirmed["business_function"] = context_businesses[0]
    return {
        "used": bool(turns or confirmed),
        "recent_turns": turns,
        "confirmed": confirmed,
        "context_businesses": context_businesses,
    }


def detect_route(
    query: str,
    *,
    context: Mapping[str, Any],
) -> tuple[str, list[str], list[str], str | None]:
    """보수적 라우팅. 반환값은 route, reasons, missing, direct_action 순서입니다."""
    businesses = find_businesses(query)
    context_business = str((context.get("confirmed") or {}).get("business_function") or "")
    has_context = bool(context.get("used"))

    if DIRECT_META_PATTERN.fullmatch(query):
        return "DIRECT", ["EXPLICIT_META_OR_SOCIAL"], [], "META_OR_SOCIAL"
    if REFORMAT_PATTERN.fullmatch(query) and has_context:
        return "DIRECT", ["REFORMAT_PREVIOUS_ANSWER"], [], "REFORMAT_PREVIOUS_ANSWER"
    if OOS_PATTERN.search(query) and not businesses:
        return "OUT_OF_SCOPE", ["EXPLICIT_NON_KDIC_TOPIC"], [], None

    if GENERIC_BUSINESS_CLARIFY_PATTERN.fullmatch(query) and not businesses and not context_business:
        return "CLARIFY", ["BUSINESS_NOT_SPECIFIED"], ["business_function"], None

    has_resolved_context = has_context or bool(context_business)
    if any(phrase in query for phrase in APPLICANT_REFERENCE_PHRASES) and not has_resolved_context:
        return "CLARIFY", ["UNRESOLVED_APPLICANT_REFERENCE"], ["applicant_type"], None
    if (any(phrase in query for phrase in TARGET_REFERENCE_PHRASES) or TARGET_DEMONSTRATIVE_PATTERN.search(query)) and not has_resolved_context:
        return "CLARIFY", ["UNRESOLVED_TARGET_REFERENCE"], ["target_type"], None
    if any(phrase in query for phrase in CASE_REFERENCE_PHRASES) and not has_resolved_context:
        return "CLARIFY", ["PERSONAL_CASE_REQUIRES_DETAILS"], ["case_details"], None

    return "RETRIEVE", ["DEFAULT_FAIL_OPEN_RETRIEVAL"], [], None


def _is_cohesive_no_split(query: str) -> bool:
    return any(pattern.search(query) for pattern in COHESIVE_NO_SPLIT_PATTERNS)


def _has_strong_same_business_split_signal(query: str) -> tuple[bool, str | None]:
    for index, pattern in enumerate(STRONG_SAME_BUSINESS_SPLIT_PATTERNS, 1):
        if pattern.search(query):
            return True, f"SAME_BUSINESS_STRONG_PATTERN_{index:02d}"
    return False, None


def detect_complexity(query: str) -> dict[str, Any]:
    businesses = find_businesses(query)
    intents = find_intent_matches(query)
    strong_same, strong_rule = _has_strong_same_business_split_signal(query)
    sentence_clauses = [part.strip(" ,") for part in CLAUSE_BOUNDARY_PATTERN.split(query) if part.strip(" ,")]

    reasons: list[str] = []
    cross_relation_keep = bool(CROSS_TERM_RELATION_KEEP_PATTERN.search(query))
    if len(businesses) >= 2 and not cross_relation_keep:
        reasons.append("MULTIPLE_BUSINESS_FUNCTIONS")
    elif len(businesses) >= 2 and cross_relation_keep:
        reasons.append("CROSS_TERM_RELATION_KEEP_ORIGINAL")
    if strong_same:
        reasons.append(strong_rule or "STRONG_SAME_BUSINESS_SIGNAL")
    if len(sentence_clauses) >= 2:
        clause_businesses = [find_businesses(part) for part in sentence_clauses]
        if sum(bool(values) for values in clause_businesses) >= 2:
            reasons.append("INDEPENDENT_BUSINESS_CLAUSES")
        elif len(intents) >= 2 and not _is_cohesive_no_split(query):
            reasons.append("INDEPENDENT_INTENT_CLAUSES")

    is_multi = (len(businesses) >= 2 and not cross_relation_keep) or strong_same or "INDEPENDENT_INTENT_CLAUSES" in reasons
    if _is_cohesive_no_split(query) and len(businesses) <= 1 and not strong_same:
        is_multi = False
        reasons.append("COHESIVE_SAME_BUSINESS_KEEP_ORIGINAL")
    if not is_multi:
        reasons.append("NO_SAFE_SPLIT_EVIDENCE")

    return {
        "question_type": "MULTI" if is_multi else "SINGLE",
        "businesses": businesses,
        "intents": [row["intent"] for row in intents],
        "clause_count": len(sentence_clauses),
        "reasons": _ordered_unique(reasons),
    }


def _clean_clause(text: str) -> str:
    text = re.sub(r"^(?:그리고|또|혹시|별도로|그럼|그러면)\s*", "", text.strip(), flags=re.I)
    text = re.sub(r"\s+", " ", text).strip(" ,.;")
    if text and not re.search(r"[?요다까]$", text):
        text += " 관련 정보"
    return text


def _sentence_clauses(query: str) -> list[str]:
    return [_clean_clause(part) for part in CLAUSE_BOUNDARY_PATTERN.split(query) if _clean_clause(part)]


def _business_anchor_positions(query: str) -> list[tuple[int, int, str, str]]:
    positions: list[tuple[int, int, str, str]] = []
    compact_query = query.lower()
    for business, keywords in BUSINESS_KEYWORDS.items():
        for keyword in sorted(keywords, key=len, reverse=True):
            start = compact_query.find(keyword.lower())
            if start >= 0:
                positions.append((start, start + len(keyword), business, keyword))
                break
    positions.sort(key=lambda row: row[0])
    return positions


def _split_cross_business(query: str, businesses: Sequence[str]) -> list[str]:
    clauses = _sentence_clauses(query)
    if len(clauses) >= 2:
        enriched: list[str] = []
        for clause in clauses:
            local_businesses = find_businesses(clause)
            if local_businesses:
                enriched.append(clause)
        if len(enriched) >= 2:
            return enriched

    anchors = _business_anchor_positions(query)
    if len(anchors) < 2:
        return []
    output: list[str] = []
    for index, (start, _end, business, keyword) in enumerate(anchors):
        next_start = anchors[index + 1][0] if index + 1 < len(anchors) else len(query)
        previous_end = anchors[index - 1][1] if index > 0 else 0
        raw = query[previous_end:next_start]
        raw = re.sub(r"^(?:와|과|및|하고|,|\s)+", "", raw)
        raw = re.sub(r"(?:와|과|및|하고|,|\s)+$", "", raw)
        clause = _clean_clause(raw)
        if business not in find_businesses(clause):
            clause = f"{business} {clause}".strip()
        # 너무 짧거나 명사구뿐이면 전체 문장의 해당 업무 관련 의도 단서를 붙입니다.
        local_intents = find_intent_matches(clause)
        if not local_intents:
            global_intents = find_intent_matches(query)
            if global_intents:
                clause = f"{clause} {global_intents[min(index, len(global_intents)-1)]['evidence']}"
        output.append(_clean_clause(clause))
    return output


def _split_same_business(query: str, business: str | None) -> list[str]:
    # 처리 전후나 서로 다른 처리 대상을 한 문장에 묶은 대표 구조는 의미 단위로 직접 분리합니다.
    match = re.search(
        r"^(?P<context>.*?영업정지되면)\s*(?P<first>예금은.*?)(?:고|며)\s*(?P<second>기존\s*대출\s*거래.*?)(?:[?]?)$",
        query,
        flags=re.I,
    )
    if match:
        context = match.group("context").strip()
        return [
            _clean_clause(f"{context} {match.group('first')}"),
            _clean_clause(f"{context} {match.group('second')}"),
        ]

    match = re.search(
        r"^(?P<actor>상속인이\s*고인의)\s*(?P<target>미수령금)을?\s*조회한?\s*(?:뒤|다음)\s*"
        r"(?P<action>지급을\s*신청하는\s*방법).*?$",
        query,
        flags=re.I,
    )
    if match:
        prefix = f"{match.group('actor')} {match.group('target')}"
        return [
            _clean_clause(f"{prefix} 조회 방법"),
            _clean_clause(f"{prefix} {match.group('action')}"),
        ]

    clauses = _sentence_clauses(query)
    if len(clauses) >= 2:
        output = []
        for clause in clauses:
            if business and business not in find_businesses(clause):
                clause = f"{business} {clause}"
            output.append(_clean_clause(clause))
        return output

    # 쉼표·연결 표현을 먼저 이용합니다.
    candidates = [part.strip(" ,") for part in re.split(r"\s*(?:,|이고|이며|인지,?|는지와|과|와)\s*", query) if part.strip(" ,")]
    if len(candidates) >= 2:
        candidates = candidates[:3]
        output = []
        for clause in candidates:
            if len(clause) < 4:
                continue
            if business and business not in find_businesses(clause):
                clause = f"{business} {clause}"
            output.append(_clean_clause(clause))
        if len(output) >= 2:
            return output

    # 안전한 절단점을 못 찾으면 분해 실패로 두고 원문 fallback을 사용합니다.
    return []


def validate_decomposition(original: str, subqueries: Sequence[str], expected_businesses: Sequence[str]) -> dict[str, Any]:
    queries = [_clean_clause(str(value)) for value in subqueries if _clean_clause(str(value))]
    issues: list[str] = []
    if len(queries) < 2:
        issues.append("TOO_FEW_SUBQUERIES")
    if len(set(_compact(value) for value in queries)) != len(queries):
        issues.append("DUPLICATE_SUBQUERIES")
    if any(len(value) < 5 for value in queries):
        issues.append("SUBQUERY_TOO_SHORT")

    reconstructed = " ".join(queries)
    missing_businesses = [business for business in expected_businesses if business not in find_businesses(reconstructed)]
    if missing_businesses:
        issues.append("MISSING_BUSINESS_COVERAGE")

    original_numbers = NUMBER_PATTERN.findall(original)
    missing_numbers = [value for value in original_numbers if value not in reconstructed]
    if missing_numbers:
        issues.append("MISSING_NUMERIC_CONSTRAINT")

    original_negations = [term for term in NEGATION_TERMS if term in original]
    missing_negations = [term for term in original_negations if term not in reconstructed]
    if missing_negations:
        issues.append("MISSING_NEGATION")

    status = "COMPLETE" if not issues else ("PARTIAL" if len(queries) >= 2 else "FAILED")
    return {
        "status": status,
        "issues": issues,
        "subqueries": queries,
        "missing_businesses": missing_businesses,
        "missing_numbers": missing_numbers,
        "missing_negations": missing_negations,
    }


def decompose_query(query: str, complexity: Mapping[str, Any], config: RouterConfig) -> dict[str, Any]:
    businesses = list(complexity.get("businesses") or [])
    if complexity.get("question_type") != "MULTI":
        return {"status": "NOT_REQUIRED", "subqueries": [query], "issues": [], "fallback_to_original": False}

    if len(businesses) >= 2:
        candidates = _split_cross_business(query, businesses)
    else:
        candidates = _split_same_business(query, businesses[0] if businesses else None)
    candidates = _ordered_unique(candidates)[: config.max_subqueries]
    validation = validate_decomposition(query, candidates, businesses)
    validation["fallback_to_original"] = validation["status"] != "COMPLETE"
    return validation


def _business_filter_for_query(query: str) -> dict[str, Any]:
    matches = find_business_matches(query)
    if len(matches) == 1:
        match = matches[0]
        return {
            "mode": "SOFT",
            "value": None,
            "soft_hint": match["business_function"],
            "confidence": match["confidence"],
            "evidence": match["evidence"],
            "hard_filter_eligible": False,
            "hard_filter_denial_reasons": ["HARD_DISABLED_BY_ROUTER_POLICY"],
        }
    return {
        "mode": "NONE",
        "value": None,
        "soft_hint": None,
        "confidence": 0.0,
        "evidence": [],
        "hard_filter_eligible": False,
        "hard_filter_denial_reasons": [
            "HARD_DISABLED_BY_ROUTER_POLICY",
            "MULTIPLE_OR_UNKNOWN_BUSINESS_CANDIDATES",
        ],
    }


def _make_need(need_id: str, query: str, *, source: str) -> dict[str, Any]:
    business_matches = find_business_matches(query)
    intent_matches = find_intent_matches(query)
    return {
        "need_id": need_id,
        "query": query,
        "query_source": source,
        "business_function": business_matches[0]["business_function"] if len(business_matches) == 1 else None,
        "business_candidates": business_matches,
        "intents": [row["intent"] for row in intent_matches],
        "intent_evidence": intent_matches,
    }


def build_query_plans(
    original_query: str,
    route: str,
    complexity: Mapping[str, Any],
    decomposition: Mapping[str, Any],
    config: RouterConfig,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if route != "RETRIEVE":
        return [], []

    is_multi = complexity.get("question_type") == "MULTI"
    decomposition_complete = decomposition.get("status") == "COMPLETE"
    if is_multi and decomposition_complete:
        retrieval_queries = list(decomposition.get("subqueries") or [])
        source = "CONSERVATIVE_DECOMPOSITION"
    else:
        retrieval_queries = [original_query]
        source = "ORIGINAL_PASSTHROUGH" if not is_multi else "ORIGINAL_FALLBACK"

    needs = [_make_need(f"N{index}", query, source=source) for index, query in enumerate(retrieval_queries, 1)]
    plans: list[dict[str, Any]] = []
    for need in needs:
        query = need["query"]
        plans.append({
            "need_id": need["need_id"],
            "retrieval_mode": "STANDARD",
            "semantic_query": query,
            "keyword_query": query,
            "query_source": need["query_source"],
            "business_filter": _business_filter_for_query(query),
            "fallback_policy": {
                "enabled": True,
                "on": ["NO_RESULTS", "LOW_TOP_SCORE", "LOW_COVERAGE"],
                "next_filter_modes": ["NONE"],
                "fail_open": True,
                "original_anchor_query": original_query if is_multi and config.include_original_anchor_for_multi else None,
            },
            "intent_boost": {
                "mode": "SOFT" if need["intents"] else "NONE",
                "values": need["intents"],
                "weight": 0.10 if need["intents"] else 0.0,
            },
        })
    return needs, plans


def query_plan_is_valid(result: Mapping[str, Any]) -> bool:
    route = str((result.get("analysis") or {}).get("route") or "")
    plans = result.get("query_plans") or []
    if route == "RETRIEVE":
        if not plans:
            return False
        for plan in plans:
            if not str(plan.get("semantic_query") or "").strip():
                return False
            if not str(plan.get("keyword_query") or "").strip():
                return False
            if (plan.get("business_filter") or {}).get("mode") == "HARD":
                return False
    elif plans:
        return False
    return True


class KDICLightweightRouterV1:
    def __init__(self, config: RouterConfig | None = None):
        self.config = config or RouterConfig()

    def run(
        self,
        query: str,
        *,
        previous_turns: Any = None,
        conversation_state: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        started = time.perf_counter()
        normalized = normalize_query(query)
        original = normalized["original_query"]
        normalized_text = normalized["normalized_query"]
        context = build_context(previous_turns, conversation_state)
        route, route_reasons, missing, direct_action = detect_route(normalized_text, context=context)

        if route == "RETRIEVE":
            complexity = detect_complexity(normalized_text)
            decomposition = decompose_query(normalized_text, complexity, self.config)
        else:
            complexity = {
                "question_type": "NONE",
                "businesses": [],
                "intents": [],
                "clause_count": 0,
                "reasons": ["NO_RETRIEVAL_ROUTE"],
            }
            decomposition = {
                "status": "NOT_APPLICABLE",
                "subqueries": [],
                "issues": [],
                "fallback_to_original": False,
            }

        needs, plans = build_query_plans(original, route, complexity, decomposition, self.config)
        analysis = {
            "route": route,
            "question_type": complexity["question_type"],
            "business_functions": complexity["businesses"],
            "intents": complexity["intents"],
            "needs": needs,
            "missing_information": missing,
            "decomposition_status": decomposition["status"],
        }
        result = {
            "pipeline_version": PIPELINE_VERSION,
            "analysis_status": "OK",
            "original_query": original,
            "normalized_query": normalized_text,
            "normalization_changes": normalized["changes"],
            "context": context,
            "route_reasons": route_reasons,
            "direct_action": direct_action,
            "complexity": complexity,
            "decomposition": decomposition,
            "analysis": analysis,
            "query_plans": plans,
            "validation_warnings": [],
            "runtime": {
                "api_request_count": 0,
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "total_tokens": 0,
                "latency_ms": round((time.perf_counter() - started) * 1000, 3),
            },
        }
        if not query_plan_is_valid(result):
            result["analysis_status"] = "INVALID_PLAN"
            result["validation_warnings"].append("QUERY_PLAN_VALIDATION_FAILED")
        return result


def route_query(
    query: str,
    *,
    previous_turns: Any = None,
    conversation_state: Mapping[str, Any] | None = None,
    config: RouterConfig | None = None,
) -> dict[str, Any]:
    return KDICLightweightRouterV1(config).run(
        query,
        previous_turns=previous_turns,
        conversation_state=conversation_state,
    )


if __name__ == "__main__":
    examples = (
        "예금자보호 한도는 얼마인가요?",
        "예금자보호 한도는 얼마인가요? 그리고 착오송금 반환지원은 누가 신청할 수 있나요?",
        "신청 방법을 알려주세요.",
        "안녕하세요.",
    )
    router = KDICLightweightRouterV1()
    for example in examples:
        print(json.dumps(router.run(example), ensure_ascii=False, indent=2))


In [ ]:
%%writefile kdic_lightweight_query_ablation_core.py
from __future__ import annotations

import hashlib
import json
import re
import time
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import pandas as pd
import requests

import kdic_lightweight_router_v1 as light_router
from kdic_integrated_eval_core import AnalyzerCase, QueryPlan, normalize_route


VERSION_V10 = "LIGHT_V1.0_ORIGINAL"
VERSION_V11 = "LIGHT_V1.1_RULE_FALLBACK_ORIGINAL"
VERSION_V12 = "LIGHT_V1.2_RULE_THEN_LLM"
VERSION_V13 = "LIGHT_V1.3_LLM_ON_COMPLEX"
VERSION_ORDER = (VERSION_V10, VERSION_V11, VERSION_V12, VERSION_V13)

VERSION_DESCRIPTIONS = {
    VERSION_V10: "공통 라우팅 후 RETRIEVE 질의는 원문 하나만 검색",
    VERSION_V11: "복합 가능성이 높으면 규칙 분해, 실패 시 원문 검색",
    VERSION_V12: "복합 가능성이 높으면 규칙 분해, 실패 시 LLM 구조화 분해, 다시 실패하면 원문 검색",
    VERSION_V13: "복합 가능성이 높으면 규칙을 건너뛰고 LLM 구조화 분해, 실패 시 원문 검색",
}

DECOMPOSITION_PROMPT_VERSION = "KDIC_DECOMPOSE_STRUCTURED_V1_2026_08_13"
NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)?")
NEGATION_TERMS = ("아닌", "아니", "않", "못", "제외", "불가", "없이", "없", "미해당")
TOKEN_PATTERN = re.compile(r"[가-힣A-Za-z0-9]+")


@dataclass(frozen=True)
class AblationConfig:
    original_anchor_weight: float = 0.60
    decomposition_weight: float = 0.40
    max_subqueries: int = 4
    llm_min_confidence: float = 0.80
    llm_model: str = "HCX-007"
    llm_endpoint: str = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
    llm_timeout_seconds: float = 90.0
    llm_max_retries: int = 2
    request_delay_seconds: float = 0.0

    def __post_init__(self) -> None:
        total = self.original_anchor_weight + self.decomposition_weight
        if abs(total - 1.0) > 1e-9:
            raise ValueError(f"검색 질의 가중치 합은 1이어야 합니다: {total}")
        if self.max_subqueries < 2:
            raise ValueError("max_subqueries는 2 이상이어야 합니다.")


def _now_ms() -> float:
    return time.perf_counter() * 1000.0


def _ordered_unique(values: Sequence[Any]) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for value in values:
        text = str(value or "").strip()
        if text and text not in seen:
            output.append(text)
            seen.add(text)
    return output


def _parse_previous_turns(value: Any) -> Any:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (list, dict)):
        return value
    text = str(value).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return text


def normalize_gold_route(value: Any) -> str:
    route = normalize_route(value)
    if route in {"SIMPLE_RETRIEVE", "MULTI_RETRIEVE", "RETRIEVE_RELAXED"}:
        return "RETRIEVE"
    return route


def analyze_common(
    evaluation_id: str,
    question: str,
    *,
    previous_turns: Any = None,
    router_config: light_router.RouterConfig | None = None,
) -> dict[str, Any]:
    """네 버전이 공유하는 정규화·라우팅·복합가능성 판별을 한 번 수행한다."""
    config = router_config or light_router.RouterConfig()
    common_started = _now_ms()
    normalized = light_router.normalize_query(question)
    context = light_router.build_context(_parse_previous_turns(previous_turns), None)
    route_raw, route_reasons, missing, direct_action = light_router.detect_route(
        normalized["normalized_query"], context=context
    )
    route = normalize_route(route_raw)
    if route == "RETRIEVE":
        complexity = light_router.detect_complexity(normalized["normalized_query"])
    else:
        complexity = {
            "question_type": "NONE",
            "businesses": [],
            "intents": [],
            "clause_count": 0,
            "reasons": ["NO_RETRIEVAL_ROUTE"],
        }
    common_latency_ms = _now_ms() - common_started

    rule_started = _now_ms()
    if route == "RETRIEVE" and complexity.get("question_type") == "MULTI":
        rule_decomposition = light_router.decompose_query(
            normalized["normalized_query"], complexity, config
        )
    else:
        rule_decomposition = {
            "status": "NOT_REQUIRED" if route == "RETRIEVE" else "NOT_APPLICABLE",
            "subqueries": [normalized["normalized_query"]] if route == "RETRIEVE" else [],
            "issues": [],
            "fallback_to_original": False,
        }
    rule_latency_ms = _now_ms() - rule_started

    return {
        "evaluation_id": str(evaluation_id),
        "original_question": str(question).strip(),
        "normalized_question": normalized["normalized_query"],
        "normalization_changes": normalized.get("changes") or [],
        "context": context,
        "route": route,
        "route_reasons": route_reasons,
        "missing_information": missing,
        "direct_action": direct_action,
        "complexity": complexity,
        "complex_candidate": route == "RETRIEVE" and complexity.get("question_type") == "MULTI",
        "rule_decomposition": rule_decomposition,
        "common_latency_ms": round(common_latency_ms, 3),
        "rule_latency_ms": round(rule_latency_ms, 3),
    }


def llm_required(version: str, common: Mapping[str, Any]) -> bool:
    if common.get("route") != "RETRIEVE" or not common.get("complex_candidate"):
        return False
    if version == VERSION_V12:
        return (common.get("rule_decomposition") or {}).get("status") != "COMPLETE"
    return version == VERSION_V13


def decomposition_json_schema(max_subqueries: int = 4) -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            "decomposable": {"type": "boolean"},
            "subqueries": {
                "type": "array",
                "maxItems": int(max_subqueries),
                "items": {
                    "type": "object",
                    "properties": {"query": {"type": "string"}},
                    "required": ["query"],
                    "additionalProperties": False,
                },
            },
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reason": {"type": "string"},
        },
        "required": ["decomposable", "subqueries", "confidence", "reason"],
        "additionalProperties": False,
    }


def build_decomposition_messages(question: str) -> list[dict[str, str]]:
    system = """
당신은 예금보험공사 검색 파이프라인의 복합질의 분해기입니다.
이 작업은 질의 재작성이나 검색어 최적화가 아니라, 원문에 실제로 들어 있는 독립 정보 요구를 구조적으로 분리하는 작업입니다.

규칙:
1. 서로 따로 검색하고 답할 수 있는 정보 요구가 2개 이상일 때만 decomposable=true로 판단합니다.
2. 단일 업무의 하나의 응집된 질문, 용어 정의, 비교 관계 자체를 묻는 질문은 분리하지 않습니다.
3. 원문의 업무명, 대상, 조건, 숫자, 기간, 부정 표현을 빠뜨리거나 바꾸지 않습니다.
4. 원문에 없는 업무, 조건, 숫자, 예외, 의도를 추가하지 않습니다.
5. 문체 개선, 요약, 동의어 확장, 검색 키워드 생성은 하지 않습니다.
6. 각 하위질문은 단독으로 이해 가능한 한국어 질문이어야 합니다.
7. 분리할 수 없거나 확신이 낮으면 decomposable=false, subqueries=[]로 반환합니다.
8. 하위질문은 2개 이상 4개 이하로 제한합니다.
""".strip()
    user = f"원문 질문:\n{question}\n\n원문의 독립 정보 요구만 판별하고 JSON 스키마에 맞춰 반환하세요."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def _extract_hcx_payload(response_json: Mapping[str, Any]) -> tuple[dict[str, Any], dict[str, int]]:
    result = response_json.get("result") or {}
    message = result.get("message") or {}
    content = message.get("content")
    if isinstance(content, Mapping):
        payload = dict(content)
    else:
        text = str(content or "").strip()
        if text.startswith("```"):
            text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.S).strip()
        payload = json.loads(text)
    usage = result.get("usage") or response_json.get("usage") or {}
    prompt_tokens = int(usage.get("promptTokens") or usage.get("prompt_tokens") or 0)
    completion_tokens = int(usage.get("completionTokens") or usage.get("completion_tokens") or 0)
    total_tokens = int(usage.get("totalTokens") or usage.get("total_tokens") or prompt_tokens + completion_tokens)
    return payload, {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
    }


def _token_overlap_ratio(original: str, candidate: str) -> float:
    original_tokens = set(TOKEN_PATTERN.findall(original.lower()))
    candidate_tokens = set(TOKEN_PATTERN.findall(candidate.lower()))
    if not candidate_tokens:
        return 0.0
    return len(original_tokens & candidate_tokens) / len(candidate_tokens)


def validate_llm_decomposition(
    original: str,
    payload: Mapping[str, Any],
    *,
    expected_businesses: Sequence[str],
    config: AblationConfig,
) -> dict[str, Any]:
    issues: list[str] = []
    decomposable = payload.get("decomposable") is True
    confidence = float(payload.get("confidence") or 0.0)
    reason = str(payload.get("reason") or "").strip()
    raw_subqueries = payload.get("subqueries") or []
    if not isinstance(raw_subqueries, list):
        raw_subqueries = []
        issues.append("SUBQUERIES_NOT_ARRAY")
    raw_subquery_count = len(raw_subqueries)
    subqueries = _ordered_unique(
        [item.get("query") if isinstance(item, Mapping) else item for item in raw_subqueries]
    )[: config.max_subqueries]

    if not decomposable:
        return {
            "status": "DECLINED",
            "accepted": False,
            "subqueries": [],
            "confidence": confidence,
            "reason": reason,
            "issues": ["LLM_DECLINED_DECOMPOSITION"],
        }
    if confidence < config.llm_min_confidence:
        issues.append("LOW_LLM_CONFIDENCE")

    base_validation = light_router.validate_decomposition(original, subqueries, expected_businesses)
    issues.extend(base_validation.get("issues") or [])
    if raw_subquery_count > config.max_subqueries:
        issues.append("TOO_MANY_SUBQUERIES")

    reconstructed = " ".join(subqueries)
    original_numbers = set(NUMBER_PATTERN.findall(original))
    generated_numbers = set(NUMBER_PATTERN.findall(reconstructed))
    if generated_numbers - original_numbers:
        issues.append("INVENTED_NUMERIC_CONSTRAINT")
    original_negations = {term for term in NEGATION_TERMS if term in original}
    generated_negations = {term for term in NEGATION_TERMS if term in reconstructed}
    if generated_negations - original_negations:
        issues.append("INVENTED_NEGATION")
    generated_businesses = set(light_router.find_businesses(reconstructed))
    expected_business_set = set(expected_businesses)
    if expected_business_set and generated_businesses - expected_business_set:
        issues.append("INVENTED_BUSINESS")
    for subquery in subqueries:
        if _token_overlap_ratio(original, subquery) < 0.25:
            issues.append("LOW_SOURCE_TERM_OVERLAP")
            break

    issues = _ordered_unique(issues)
    accepted = not issues and 2 <= len(subqueries) <= config.max_subqueries
    return {
        "status": "COMPLETE" if accepted else "FAILED",
        "accepted": accepted,
        "subqueries": subqueries if accepted else [],
        "confidence": confidence,
        "reason": reason,
        "issues": issues,
    }


class HCXStructuredDecomposer:
    def __init__(
        self,
        api_key: str,
        *,
        config: AblationConfig | None = None,
        cache_path: str | Path | None = None,
        session: requests.Session | None = None,
    ) -> None:
        key = str(api_key or "").strip()
        if not key or key.lower().startswith("bearer ") or any(ch.isspace() for ch in key):
            raise ValueError("HCX_API_KEY에는 Bearer 접두사나 공백을 넣지 않습니다.")
        self.api_key = key
        self.config = config or AblationConfig()
        self.cache_path = Path(cache_path) if cache_path else None
        self.session = session or requests.Session()
        self.cache: dict[str, dict[str, Any]] = {}
        if self.cache_path and self.cache_path.exists():
            with self.cache_path.open(encoding="utf-8") as handle:
                for line in handle:
                    if line.strip():
                        row = json.loads(line)
                        self.cache[str(row["cache_key"])] = row

    def _cache_key(self, question: str, expected_businesses: Sequence[str]) -> str:
        raw = json.dumps(
            {
                "prompt_version": DECOMPOSITION_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
                "min_confidence": self.config.llm_min_confidence,
            },
            ensure_ascii=False,
            sort_keys=True,
        )
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _append_cache(self, row: Mapping[str, Any]) -> None:
        if not self.cache_path:
            return
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        with self.cache_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    def decompose(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        cache_key = self._cache_key(question, expected_businesses)
        if cache_key in self.cache:
            cached = dict(self.cache[cache_key])
            cached["cache_hit"] = True
            cached["actual_api_latency_ms"] = 0.0
            return cached

        body = {
            "messages": build_decomposition_messages(question),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 700,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": decomposition_json_schema(self.config.max_subqueries)},
        }
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        }
        last_error: Exception | None = None
        for attempt in range(self.config.llm_max_retries + 1):
            started = _now_ms()
            try:
                response = self.session.post(
                    self.config.llm_endpoint,
                    headers=headers,
                    json=body,
                    timeout=self.config.llm_timeout_seconds,
                )
                response.raise_for_status()
                payload, usage = _extract_hcx_payload(response.json())
                api_latency_ms = _now_ms() - started
                validation = validate_llm_decomposition(
                    question,
                    payload,
                    expected_businesses=expected_businesses,
                    config=self.config,
                )
                row = {
                    "cache_key": cache_key,
                    "question": question,
                    "model": self.config.llm_model,
                    "prompt_version": DECOMPOSITION_PROMPT_VERSION,
                    "raw_payload": payload,
                    **validation,
                    **usage,
                    "effective_api_latency_ms": round(api_latency_ms, 3),
                    "actual_api_latency_ms": round(api_latency_ms, 3),
                    "cache_hit": False,
                    "error_type": "",
                    "error_message": "",
                }
                self.cache[cache_key] = row
                self._append_cache(row)
                if self.config.request_delay_seconds > 0:
                    time.sleep(self.config.request_delay_seconds)
                return dict(row)
            except Exception as error:
                last_error = error
                if attempt < self.config.llm_max_retries:
                    time.sleep(min(2 ** attempt, 4))

        row = {
            "cache_key": cache_key,
            "question": question,
            "model": self.config.llm_model,
            "prompt_version": DECOMPOSITION_PROMPT_VERSION,
            "raw_payload": {},
            "status": "ERROR",
            "accepted": False,
            "subqueries": [],
            "confidence": 0.0,
            "reason": "",
            "issues": ["LLM_REQUEST_FAILED"],
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "effective_api_latency_ms": 0.0,
            "actual_api_latency_ms": 0.0,
            "cache_hit": False,
            "error_type": type(last_error).__name__ if last_error else "UnknownError",
            "error_message": str(last_error or "unknown error"),
        }
        self.cache[cache_key] = row
        self._append_cache(row)
        return dict(row)


def _make_plans(original: str, subqueries: Sequence[str], config: AblationConfig) -> list[QueryPlan]:
    original_compact = re.sub(r"\s+", "", original).lower()
    valid_subqueries = [
        value for value in _ordered_unique(subqueries)
        if re.sub(r"\s+", "", value).lower() != original_compact
    ]
    if len(valid_subqueries) < 2:
        return [
            QueryPlan(
                need_id="FUSED",
                variant_id="ORIGINAL",
                dense_query=original,
                bm25_query=original,
                filter_mode="NONE",
                business_filters=[],
                soft_business_hints=[],
                query_weight=1.0,
                query_source="ORIGINAL",
            )
        ]
    sub_weight = config.decomposition_weight / len(valid_subqueries)
    plans = [
        QueryPlan(
            need_id="FUSED",
            variant_id="ORIGINAL_ANCHOR",
            dense_query=original,
            bm25_query=original,
            filter_mode="NONE",
            business_filters=[],
            soft_business_hints=[],
            query_weight=config.original_anchor_weight,
            query_source="ORIGINAL_ANCHOR",
        )
    ]
    for index, subquery in enumerate(valid_subqueries, 1):
        plans.append(
            QueryPlan(
                need_id="FUSED",
                variant_id=f"SUBQUERY_{index:02d}",
                dense_query=subquery,
                bm25_query=subquery,
                filter_mode="NONE",
                business_filters=[],
                soft_business_hints=[],
                query_weight=sub_weight,
                query_source="DECOMPOSED",
            )
        )
    return plans


def build_version_case(
    common: Mapping[str, Any],
    version: str,
    *,
    llm_record: Mapping[str, Any] | None = None,
    config: AblationConfig | None = None,
) -> AnalyzerCase:
    if version not in VERSION_ORDER:
        raise ValueError(f"지원하지 않는 버전: {version}")
    cfg = config or AblationConfig()
    route = str(common["route"])
    original = str(common["original_question"])
    rule = dict(common.get("rule_decomposition") or {})
    candidate = bool(common.get("complex_candidate"))
    subqueries: list[str] = []
    source = "ORIGINAL_POLICY"
    fallback_reason = ""
    policy_rule_used = False
    policy_llm_called = False

    if route == "RETRIEVE" and candidate:
        if version in {VERSION_V11, VERSION_V12}:
            policy_rule_used = True
            if rule.get("status") == "COMPLETE":
                subqueries = list(rule.get("subqueries") or [])
                source = "RULE"
            elif version == VERSION_V12:
                policy_llm_called = True
                if llm_record and llm_record.get("accepted"):
                    subqueries = list(llm_record.get("subqueries") or [])
                    source = "LLM"
                else:
                    source = "ORIGINAL_FALLBACK"
                    fallback_reason = "LLM_FAILED_OR_DECLINED"
            else:
                source = "ORIGINAL_FALLBACK"
                fallback_reason = "RULE_DECOMPOSITION_FAILED"
        elif version == VERSION_V13:
            policy_llm_called = True
            if llm_record and llm_record.get("accepted"):
                subqueries = list(llm_record.get("subqueries") or [])
                source = "LLM"
            else:
                source = "ORIGINAL_FALLBACK"
                fallback_reason = "LLM_FAILED_OR_DECLINED"

    if route == "RETRIEVE":
        plans = _make_plans(original, subqueries, cfg)
    else:
        plans = []

    analysis_latency_ms = float(common.get("common_latency_ms") or 0.0)
    if policy_rule_used:
        analysis_latency_ms += float(common.get("rule_latency_ms") or 0.0)
    if policy_llm_called and llm_record:
        analysis_latency_ms += float(llm_record.get("effective_api_latency_ms") or 0.0)

    raw_result = {
        "pipeline_version": version,
        "analysis_status": "OK",
        "original_query": original,
        "normalized_query": common.get("normalized_question"),
        "route_reasons": common.get("route_reasons") or [],
        "direct_action": common.get("direct_action"),
        "blocking_slot": (common.get("missing_information") or [None])[0],
        "complexity": common.get("complexity") or {},
        "complex_candidate": candidate,
        "rule_decomposition": rule,
        "llm_decomposition": dict(llm_record or {}),
        "decomposition_source": source,
        "final_subqueries": subqueries,
        "fallback_reason": fallback_reason,
        "model_needs": [
            {"business_function": value}
            for value in (common.get("complexity") or {}).get("businesses") or []
        ],
        "runtime": {
            "api_request_count": int(policy_llm_called),
            "prompt_tokens": int((llm_record or {}).get("prompt_tokens") or 0) if policy_llm_called else 0,
            "completion_tokens": int((llm_record or {}).get("completion_tokens") or 0) if policy_llm_called else 0,
            "total_tokens": int((llm_record or {}).get("total_tokens") or 0) if policy_llm_called else 0,
            "latency_ms": round(analysis_latency_ms, 3),
        },
    }
    return AnalyzerCase(
        evaluation_id=str(common["evaluation_id"]),
        analyzer=version,
        original_question=original,
        route=route,
        analysis_latency_ms=round(analysis_latency_ms, 3),
        plans=plans,
        raw_result=raw_result,
    )


def build_all_cases(
    eval_df: pd.DataFrame,
    *,
    decomposer: HCXStructuredDecomposer | None,
    config: AblationConfig | None = None,
    previous_turns_column: str = "previous_turns",
) -> tuple[dict[tuple[str, str], AnalyzerCase], pd.DataFrame]:
    cfg = config or AblationConfig()
    common_by_id: dict[str, dict[str, Any]] = {}
    for row in eval_df.to_dict(orient="records"):
        evaluation_id = str(row["evaluation_id"])
        common_by_id[evaluation_id] = analyze_common(
            evaluation_id,
            str(row["question"]),
            previous_turns=row.get(previous_turns_column),
        )

    llm_by_id: dict[str, dict[str, Any]] = {}
    required_ids = [
        evaluation_id
        for evaluation_id, common in common_by_id.items()
        if any(llm_required(version, common) for version in (VERSION_V12, VERSION_V13))
    ]
    if required_ids and decomposer is None:
        raise ValueError("V1.2/V1.3 평가에는 HCXStructuredDecomposer가 필요합니다.")
    for evaluation_id in required_ids:
        common = common_by_id[evaluation_id]
        llm_by_id[evaluation_id] = decomposer.decompose(
            str(common["normalized_question"]),
            list((common.get("complexity") or {}).get("businesses") or []),
        )

    cases: dict[tuple[str, str], AnalyzerCase] = {}
    audit_rows: list[dict[str, Any]] = []
    for evaluation_id, common in common_by_id.items():
        llm_record = llm_by_id.get(evaluation_id)
        for version in VERSION_ORDER:
            case = build_version_case(common, version, llm_record=llm_record, config=cfg)
            cases[(version, evaluation_id)] = case
            runtime = case.raw_result["runtime"]
            audit_rows.append({
                "evaluation_id": evaluation_id,
                "question": case.original_question,
                "version": version,
                "route": case.route,
                "complex_candidate": bool(common.get("complex_candidate")),
                "rule_status": (common.get("rule_decomposition") or {}).get("status"),
                "rule_subqueries": (common.get("rule_decomposition") or {}).get("subqueries") or [],
                "llm_policy_call": int(llm_required(version, common)),
                "llm_actual_api_call": int(bool(llm_record) and not bool(llm_record.get("cache_hit"))) if llm_required(version, common) else 0,
                "llm_cache_hit": bool((llm_record or {}).get("cache_hit")) if llm_required(version, common) else False,
                "llm_status": (llm_record or {}).get("status", "NOT_CALLED"),
                "llm_confidence": float((llm_record or {}).get("confidence") or 0.0),
                "llm_issues": (llm_record or {}).get("issues") or [],
                "decomposition_source": case.raw_result["decomposition_source"],
                "final_subqueries": case.raw_result["final_subqueries"],
                "query_plan_count": len(case.plans),
                "query_plan_weight_sum": round(sum(plan.query_weight for plan in case.plans), 10),
                "hard_filter_count": sum(plan.filter_mode == "HARD" for plan in case.plans),
                "analysis_latency_ms": case.analysis_latency_ms,
                "prompt_tokens": runtime["prompt_tokens"],
                "completion_tokens": runtime["completion_tokens"],
                "total_tokens": runtime["total_tokens"],
            })
    return cases, pd.DataFrame(audit_rows)


def summarize_router_ablation(audit_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for version, frame in audit_df.groupby("version", sort=False):
        retrieve = frame[frame["route"].eq("RETRIEVE")]
        rows.append({
            "version": version,
            "question_count": len(frame),
            "retrieve_count": int(frame["route"].eq("RETRIEVE").sum()),
            "clarify_count": int(frame["route"].eq("CLARIFY").sum()),
            "out_of_scope_count": int(frame["route"].eq("OUT_OF_SCOPE").sum()),
            "direct_response_count": int(frame["route"].eq("DIRECT_RESPONSE").sum()),
            "complex_candidate_count": int(frame["complex_candidate"].sum()),
            "decomposed_count": int(frame["decomposition_source"].isin(["RULE", "LLM"]).sum()),
            "rule_decomposed_count": int(frame["decomposition_source"].eq("RULE").sum()),
            "llm_decomposed_count": int(frame["decomposition_source"].eq("LLM").sum()),
            "original_fallback_count": int(frame["decomposition_source"].eq("ORIGINAL_FALLBACK").sum()),
            "llm_policy_call_count": int(frame["llm_policy_call"].sum()),
            "llm_total_tokens": int(frame["total_tokens"].sum()),
            "analysis_latency_ms_mean_all": float(frame["analysis_latency_ms"].mean()),
            "analysis_latency_ms_p95_all": float(frame["analysis_latency_ms"].quantile(0.95)),
            "retrieval_query_count_mean": float(retrieve["query_plan_count"].mean()) if len(retrieve) else 0.0,
            "hard_filter_count": int(frame["hard_filter_count"].sum()),
            "invalid_query_plan_weight_count": int((retrieve["query_plan_weight_sum"].sub(1.0).abs() > 1e-9).sum()),
        })
    summary = pd.DataFrame(rows)
    summary["version"] = pd.Categorical(summary["version"], VERSION_ORDER, ordered=True)
    return summary.sort_values("version").reset_index(drop=True)


def route_hard_gate_report(audit_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
    gold = eval_df[["evaluation_id", "gold_route_v6"]].copy() if "gold_route_v6" in eval_df.columns else pd.DataFrame()
    if not gold.empty:
        gold["gold_route_v6"] = gold["gold_route_v6"].map(normalize_gold_route)
    rows: list[dict[str, Any]] = []
    for version, frame in audit_df.groupby("version", sort=False):
        merged = frame.merge(gold, on="evaluation_id", how="left") if not gold.empty else frame.assign(gold_route_v6="")
        route_known = merged["gold_route_v6"].fillna("").ne("")
        normal_retrieve = merged["gold_route_v6"].eq("RETRIEVE")
        predicted_retrieve = merged["route"].eq("RETRIEVE")
        query_valid_rate = float(merged.loc[predicted_retrieve, "query_plan_count"].gt(0).mean()) if predicted_retrieve.any() else 1.0
        wrong_oos = int((normal_retrieve & merged["route"].eq("OUT_OF_SCOPE")).sum())
        wrong_direct = int((normal_retrieve & merged["route"].eq("DIRECT_RESPONSE")).sum())
        route_accuracy = float((merged.loc[route_known, "route"] == merged.loc[route_known, "gold_route_v6"]).mean()) if route_known.any() else float("nan")
        predicted_clarify = merged["route"].eq("CLARIFY") & route_known
        clarify_precision = float(merged.loc[predicted_clarify, "gold_route_v6"].eq("CLARIFY").mean()) if predicted_clarify.any() else 1.0
        rows.extend([
            {"version": version, "gate": "실행 성공률", "value": 1.0, "threshold": ">=0.995", "passed": True},
            {"version": version, "gate": "검색 질의 생성 유효율", "value": query_valid_rate, "threshold": ">=0.99", "passed": query_valid_rate >= 0.99},
            {"version": version, "gate": "정상 질문의 잘못된 OOS", "value": wrong_oos, "threshold": "=0", "passed": wrong_oos == 0},
            {"version": version, "gate": "정상 질문의 잘못된 DIRECT", "value": wrong_direct, "threshold": "=0", "passed": wrong_direct == 0},
            {"version": version, "gate": "Hard Filter", "value": int(frame["hard_filter_count"].sum()), "threshold": "=0", "passed": int(frame["hard_filter_count"].sum()) == 0},
            {"version": version, "gate": "검색 불가능 질문의 추가질문 Precision", "value": clarify_precision, "threshold": ">=0.95", "passed": clarify_precision >= 0.95},
            {"version": version, "gate": "최종 라우팅 정확도", "value": route_accuracy, "threshold": "참고", "passed": True},
        ])
    return pd.DataFrame(rows)


def query_analysis_quality_summary(audit_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
    """라우팅과 최종 분해 여부를 gold와 비교한다. gold 열이 없으면 해당 값은 NaN이다."""
    gold_columns = [column for column in ("evaluation_id", "gold_route_v6", "split_needed") if column in eval_df.columns]
    gold = eval_df[gold_columns].copy()
    if "gold_route_v6" in gold.columns:
        gold["gold_route"] = gold["gold_route_v6"].map(normalize_gold_route)
    else:
        gold["gold_route"] = ""
    if "split_needed" in gold.columns:
        gold["gold_multi"] = gold["split_needed"].map(
            lambda value: str(value).strip().lower() in {"1", "true", "y", "yes", "multi", "복합", "필요"}
            if not pd.isna(value) and str(value).strip() else pd.NA
        )
    else:
        gold["gold_multi"] = pd.NA

    rows: list[dict[str, Any]] = []
    for version, frame in audit_df.groupby("version", sort=False):
        merged = frame.merge(gold[["evaluation_id", "gold_route", "gold_multi"]], on="evaluation_id", how="left")
        known_route = merged["gold_route"].fillna("").ne("")
        route_accuracy = float(merged.loc[known_route, "route"].eq(merged.loc[known_route, "gold_route"]).mean()) if known_route.any() else float("nan")
        gold_retrieve = merged["gold_route"].eq("RETRIEVE")
        retrieve_recall = float(merged.loc[gold_retrieve, "route"].eq("RETRIEVE").mean()) if gold_retrieve.any() else float("nan")
        predicted_clarify = merged["route"].eq("CLARIFY") & known_route
        clarify_precision = float(merged.loc[predicted_clarify, "gold_route"].eq("CLARIFY").mean()) if predicted_clarify.any() else 1.0

        known_multi = merged["gold_multi"].notna() & gold_retrieve
        predicted_multi = merged["decomposition_source"].isin(["RULE", "LLM"])
        tp = int((known_multi & merged["gold_multi"].astype("boolean").fillna(False) & predicted_multi).sum())
        fp = int((known_multi & ~merged["gold_multi"].astype("boolean").fillna(False) & predicted_multi).sum())
        fn = int((known_multi & merged["gold_multi"].astype("boolean").fillna(False) & ~predicted_multi).sum())
        if not known_multi.any():
            precision = recall = f1 = float("nan")
        else:
            precision = tp / (tp + fp) if tp + fp else 1.0
            recall = tp / (tp + fn) if tp + fn else 1.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({
            "version": version,
            "route_accuracy": route_accuracy,
            "retrieve_recall": retrieve_recall,
            "clarify_precision": clarify_precision,
            "multi_precision": precision,
            "multi_recall": recall,
            "multi_f1": f1,
            "multi_tp": tp,
            "multi_fp": fp,
            "multi_fn": fn,
            "analysis_latency_ms_mean": float(frame["analysis_latency_ms"].mean()),
            "llm_policy_call_count": int(frame["llm_policy_call"].sum()),
            "llm_total_tokens": int(frame["total_tokens"].sum()),
        })
    summary = pd.DataFrame(rows)
    summary["version"] = pd.Categorical(summary["version"], VERSION_ORDER, ordered=True)
    return summary.sort_values("version").reset_index(drop=True)


def case_signature(case: AnalyzerCase) -> str:
    payload = {
        "route": case.route,
        "plans": [asdict(plan) for plan in case.plans],
        "source": case.raw_result.get("decomposition_source"),
    }
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:16]


In [ ]:
%%writefile kdic_decomposition_quality_core.py
from __future__ import annotations

import hashlib
import json
import math
import re
import time
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence

import pandas as pd
try:
    import requests
except ImportError:  # 로컬 정적 검증 환경에서는 HTTP 호출을 사용하지 않을 수 있습니다.
    requests = None  # type: ignore[assignment]

import kdic_lightweight_router_v1 as light_router


DATE_PREFIX = "2026-08-14"

BASELINE = "V1.5_BASELINE"
QUALITY = "V1.5_Q"
RETRY = "V1.5_R"
QUALITY_RETRY = "V1.5_QR"
CONDITION_ORDER = (BASELINE, QUALITY, RETRY, QUALITY_RETRY)
CONDITION_LABELS = {
    BASELINE: "V1.5 Baseline",
    QUALITY: "V1.5-Q 품질개선",
    RETRY: "V1.5-R 교정재시도",
    QUALITY_RETRY: "V1.5-QR 품질개선+재시도",
}

PROMPT_VERSION = "KDIC_DECOMPOSITION_QUALITY_V1_2026_08_14"
REPAIR_PROMPT_VERSION = "KDIC_DECOMPOSITION_REPAIR_V1_2026_08_14"

INTENT_VALUES = (
    "OVERVIEW", "ELIGIBILITY", "AMOUNT", "APPLICATION", "DOCUMENTS",
    "TIME", "STATUS", "CALCULATION", "EXCEPTION", "OTHER",
)
INTENT_PATTERNS: dict[str, tuple[str, ...]] = {
    "AMOUNT": (r"한도", r"얼마", r"금액", r"최대", r"최소", r"몇\s*원", r"비율"),
    "APPLICATION": (
        r"신청\s*(?:방법|절차)", r"신청하려면", r"접수\s*(?:방법|절차)?",
        r"어떻게\s*(?:신청|받|진행)",
    ),
    "DOCUMENTS": (r"서류", r"준비물", r"증빙", r"제출"),
    "TIME": (r"언제", r"기간", r"기한", r"시점", r"며칠", r"몇\s*개월"),
    "STATUS": (r"조회", r"확인", r"찾(?:는|을|아)", r"남았(?:는지|나요)"),
    "CALCULATION": (r"계산", r"산정", r"합산"),
    "EXCEPTION": (r"제외", r"예외", r"불가", r"해당하지", r"받지\s*못"),
    "ELIGIBILITY": (r"대상", r"자격", r"조건", r"누가", r"가능한지", r"받을\s*수\s*있"),
    "OVERVIEW": (r"무엇(?:인가요|인지|이죠)?", r"의미", r"차이", r"어떤\s*제도", r"설명"),
}
SUBJECT_TERMS = (
    "상속인", "본인", "대리인", "법인", "개인", "채무자", "송금인", "수취인",
    "미성년자", "친권자", "외국인", "고인", "피상속인", "금융회사",
)
NEGATION_TERMS = ("아닌", "아니", "않", "못", "제외", "불가", "없이", "없", "미해당", "뿐 아니라")
NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)?(?:\s*(?:원|만원|천만원|억원|%|퍼센트|년|개월|일|회))?")
TOKEN_PATTERN = re.compile(r"[가-힣A-Za-z0-9]+")
UNRESOLVED_REFERENCE_PATTERN = re.compile(r"(?:그것|그거|이것|해당\s*(?:제도|경우|업무)|그\s*제도|앞의\s*내용)")


@dataclass(frozen=True)
class QualityConfig:
    llm_endpoint: str = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
    llm_model: str = "HCX-007"
    llm_timeout_seconds: float = 120.0
    transport_retries: int = 2
    semantic_retries: int = 1
    llm_min_confidence: float = 0.80
    max_subqueries: int = 4
    request_delay_seconds: float = 0.0
    original_weight: float = 0.40
    subquery_total_weight: float = 0.60

    def __post_init__(self) -> None:
        if abs(self.original_weight + self.subquery_total_weight - 1.0) > 1e-9:
            raise ValueError("원문과 하위질의 가중치 합은 1이어야 합니다.")
        if self.semantic_retries != 1:
            raise ValueError("이번 실험의 의미 교정 재시도는 정확히 1회로 고정합니다.")


def ordered_unique(values: Sequence[Any]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for value in values:
        text = re.sub(r"\s+", " ", str(value or "")).strip()
        if text and text not in seen:
            seen.add(text)
            result.append(text)
    return result


def find_intents(text: str) -> list[str]:
    output: list[str] = []
    for intent, patterns in INTENT_PATTERNS.items():
        if any(re.search(pattern, text) for pattern in patterns):
            output.append(intent)
    if len(output) > 1 and "OVERVIEW" in output:
        output.remove("OVERVIEW")
    return output


def source_features(question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
    matches = light_router.find_business_matches(question)
    evidence_by_business = {
        str(row["business_function"]): ordered_unique(row.get("evidence") or [])
        for row in matches
    }
    return {
        "expected_businesses": ordered_unique(expected_businesses),
        "expected_intents": find_intents(question),
        "numbers": ordered_unique(NUMBER_PATTERN.findall(question)),
        "negations": [term for term in NEGATION_TERMS if term in question],
        "subjects": [term for term in SUBJECT_TERMS if term in question],
        "business_evidence": evidence_by_business,
    }


def quality_json_schema(max_subqueries: int) -> dict[str, Any]:
    business_values = sorted(light_router.BUSINESS_FUNCTIONS)
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "decomposable": {"type": "boolean"},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reason": {"type": "string"},
            "subqueries": {
                "type": "array",
                "minItems": 0,
                "maxItems": max_subqueries,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "query": {"type": "string"},
                        "business_function": {"type": "string", "enum": business_values},
                        "intent": {"type": "string", "enum": list(INTENT_VALUES)},
                        "preserved_terms": {"type": "array", "items": {"type": "string"}},
                        "preserved_constraints": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": [
                        "query", "business_function", "intent",
                        "preserved_terms", "preserved_constraints",
                    ],
                },
            },
        },
        "required": ["decomposable", "confidence", "reason", "subqueries"],
    }


def baseline_json_schema(max_subqueries: int) -> dict[str, Any]:
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "decomposable": {"type": "boolean"},
            "subqueries": {
                "type": "array",
                "maxItems": max_subqueries,
                "items": {
                    "oneOf": [
                        {"type": "string"},
                        {
                            "type": "object",
                            "additionalProperties": False,
                            "properties": {"query": {"type": "string"}},
                            "required": ["query"],
                        },
                    ]
                },
            },
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reason": {"type": "string"},
        },
        "required": ["decomposable", "subqueries", "confidence", "reason"],
    }


def build_quality_messages(question: str, expected_businesses: Sequence[str]) -> list[dict[str, str]]:
    features = source_features(question, expected_businesses)
    system = """당신은 예금보험공사 검색용 교차업무 질의 구조화 분해기입니다.
라우터가 제시한 업무들은 확정 정답이 아니라 분해 필요성을 검토할 후보입니다.
서로 다른 업무 용어가 보여도 하나의 사건·절차·대상을 비교하거나 설명하는 단일 정보요구라면 decomposable=false와 빈 subqueries를 반환합니다.
서로 독립적으로 검색해야 할 업무별 정보요구가 둘 이상일 때만 decomposable=true로 분해합니다.
원문을 요약하거나 일반화하지 말고, 서로 다른 업무별 독립 검색 질의로만 분리합니다.
각 하위질의는 하나의 업무와 하나의 주된 요청 의도만 담당해야 합니다.
원문의 전문용어, 숫자, 금액, 기간, 부정·제외 표현, 사용자 주체와 조건을 보존합니다.
원문에 없는 업무·숫자·조건을 추가하지 않습니다.
'그것', '해당 경우'처럼 원문 없이 이해할 수 없는 표현을 사용하지 않습니다.
동일 의미의 하위질의를 중복 생성하지 않습니다.
반드시 지정된 JSON Schema만 출력합니다."""
    user = json.dumps({
        "question": question,
        "router_expected_businesses": list(expected_businesses),
        "source_features_to_preserve": features,
        "instruction": "먼저 실제 독립 정보요구가 둘 이상인지 판정하세요. 맞을 때만 각 업무를 담당하는 2~4개 하위질의로 분해하세요.",
    }, ensure_ascii=False, indent=2)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_repair_messages(
    question: str,
    expected_businesses: Sequence[str],
    previous_payload: Mapping[str, Any],
    issues: Sequence[str],
    *,
    quality_mode: bool,
) -> list[dict[str, str]]:
    system = """당신은 검색용 질의 분해 결과 교정기입니다.
직전 결과 전체를 새로 창작하지 말고, 검증기가 지적한 오류만 수정합니다.
누락된 업무·요청·전문용어·숫자·부정·주체를 복원하고 새 정보는 만들지 않습니다.
라우터 업무 후보는 확정 정답이 아닙니다. 하나의 정보요구라면 decomposable=false로 판단하며 억지로 분해하지 않습니다.
교정 결과도 검증을 통과하지 못하면 폐기되므로 억지로 분해하지 않습니다.
반드시 지정된 JSON Schema만 출력합니다."""
    user = json.dumps({
        "question": question,
        "router_expected_businesses": list(expected_businesses),
        "source_features_to_preserve": source_features(question, expected_businesses),
        "previous_payload": dict(previous_payload),
        "validation_issues": list(issues),
        "output_mode": "quality_structured" if quality_mode else "baseline_structured",
        "instruction": "검증 오류를 정확히 수정하여 2~4개의 독립 검색 질의를 반환하세요.",
    }, ensure_ascii=False, indent=2)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def extract_queries(payload: Mapping[str, Any]) -> list[str]:
    raw = payload.get("subqueries") or []
    if not isinstance(raw, list):
        return []
    return ordered_unique([
        item.get("query") if isinstance(item, Mapping) else item
        for item in raw
    ])


def _token_overlap_ratio(original: str, candidate: str) -> float:
    original_tokens = set(TOKEN_PATTERN.findall(original.lower()))
    candidate_tokens = set(TOKEN_PATTERN.findall(candidate.lower()))
    if not candidate_tokens:
        return 0.0
    return len(original_tokens & candidate_tokens) / len(candidate_tokens)


def validate_baseline_decomposition(
    question: str,
    payload: Mapping[str, Any],
    *,
    expected_businesses: Sequence[str],
    config: QualityConfig,
) -> dict[str, Any]:
    """기존 V1.5 검증 규칙을 독립적으로 재현한다.

    이 함수는 개선 검증기의 비교 기준이므로 기존 실험 모듈을 import하지 않는다.
    """
    issues: list[str] = []
    decomposable = payload.get("decomposable") is True
    confidence = float(payload.get("confidence") or 0.0)
    reason = str(payload.get("reason") or "").strip()
    raw_subqueries = payload.get("subqueries") or []
    if not isinstance(raw_subqueries, list):
        raw_subqueries = []
        issues.append("SUBQUERIES_NOT_ARRAY")
    raw_subquery_count = len(raw_subqueries)
    subqueries = extract_queries({"subqueries": raw_subqueries})[: config.max_subqueries]

    if not decomposable:
        return {
            "status": "DECLINED",
            "accepted": False,
            "subqueries": [],
            "candidate_subqueries": subqueries,
            "confidence": confidence,
            "reason": reason,
            "issues": ["LLM_DECLINED_DECOMPOSITION"],
            "checks": content_checks(question, payload, expected_businesses),
        }
    if confidence < config.llm_min_confidence:
        issues.append("LOW_LLM_CONFIDENCE")

    base_validation = light_router.validate_decomposition(
        question, subqueries, expected_businesses
    )
    issues.extend(base_validation.get("issues") or [])
    if raw_subquery_count > config.max_subqueries:
        issues.append("TOO_MANY_SUBQUERIES")

    reconstructed = " ".join(subqueries)
    original_numbers = set(NUMBER_PATTERN.findall(question))
    generated_numbers = set(NUMBER_PATTERN.findall(reconstructed))
    if generated_numbers - original_numbers:
        issues.append("INVENTED_NUMERIC_CONSTRAINT")
    original_negations = {term for term in NEGATION_TERMS if term in question}
    generated_negations = {term for term in NEGATION_TERMS if term in reconstructed}
    if generated_negations - original_negations:
        issues.append("INVENTED_NEGATION")
    generated_businesses = set(light_router.find_businesses(reconstructed))
    expected_business_set = set(expected_businesses)
    if expected_business_set and generated_businesses - expected_business_set:
        issues.append("INVENTED_BUSINESS")
    if any(_token_overlap_ratio(question, subquery) < 0.25 for subquery in subqueries):
        issues.append("LOW_SOURCE_TERM_OVERLAP")

    issues = ordered_unique(issues)
    accepted = not issues and 2 <= len(subqueries) <= config.max_subqueries
    return {
        "status": "COMPLETE" if accepted else "FAILED",
        "accepted": accepted,
        "subqueries": subqueries if accepted else [],
        "candidate_subqueries": subqueries,
        "confidence": confidence,
        "reason": reason,
        "issues": issues,
        "checks": content_checks(question, payload, expected_businesses),
    }


def content_checks(
    question: str,
    payload: Mapping[str, Any],
    expected_businesses: Sequence[str],
) -> dict[str, Any]:
    queries = extract_queries(payload)
    reconstructed = " ".join(queries)
    features = source_features(question, expected_businesses)
    generated_businesses = set(light_router.find_businesses(reconstructed))
    expected_business_set = set(features["expected_businesses"])
    generated_intents = set(find_intents(reconstructed))
    expected_intents = set(features["expected_intents"])
    numbers_preserved = all(value in reconstructed for value in features["numbers"])
    negations_preserved = all(value in reconstructed for value in features["negations"])
    subjects_preserved = all(value in reconstructed for value in features["subjects"])
    standalone = all(
        not UNRESOLVED_REFERENCE_PATTERN.search(query)
        or bool(light_router.find_businesses(query))
        for query in queries
    )
    atomic = all(len(set(light_router.find_businesses(query)) & expected_business_set) <= 1 for query in queries)
    return {
        "query_count": len(queries),
        "business_coverage": expected_business_set.issubset(generated_businesses),
        "request_coverage": expected_intents.issubset(generated_intents),
        "numeric_preservation": numbers_preserved,
        "negation_preservation": negations_preserved,
        "subject_preservation": subjects_preserved,
        "standalone_subqueries": standalone,
        "atomic_subqueries": atomic,
        "invented_business_count": len(generated_businesses - expected_business_set),
    }


def validate_quality_decomposition(
    question: str,
    payload: Mapping[str, Any],
    *,
    expected_businesses: Sequence[str],
    config: QualityConfig,
) -> dict[str, Any]:
    base = validate_baseline_decomposition(
        question, payload, expected_businesses=expected_businesses, config=config
    )
    issues = list(base.get("issues") or [])
    queries = extract_queries(payload)
    raw_items = payload.get("subqueries") if isinstance(payload.get("subqueries"), list) else []
    structured_items = [item for item in raw_items if isinstance(item, Mapping)]
    if len(structured_items) != len(raw_items):
        issues.append("MISSING_STRUCTURED_SUBQUERY_FIELDS")

    assigned_businesses: list[str] = []
    assigned_intents: list[str] = []
    pairs: list[tuple[str, str]] = []
    for item in structured_items:
        business = str(item.get("business_function") or "").strip()
        intent = str(item.get("intent") or "").strip()
        query = str(item.get("query") or "").strip()
        assigned_businesses.append(business)
        assigned_intents.append(intent)
        pairs.append((business, intent))
        if business not in expected_businesses:
            issues.append("INVENTED_OR_WRONG_ASSIGNED_BUSINESS")
        if intent not in INTENT_VALUES:
            issues.append("INVALID_ASSIGNED_INTENT")
        detected = set(light_router.find_businesses(query))
        if business and business not in detected:
            issues.append("ASSIGNED_BUSINESS_NOT_EXPLICIT_IN_QUERY")
        if len(detected & set(expected_businesses)) > 1:
            issues.append("NON_ATOMIC_SUBQUERY")
        if UNRESOLVED_REFERENCE_PATTERN.search(query) and not detected:
            issues.append("NON_STANDALONE_SUBQUERY")

    if set(expected_businesses) - set(assigned_businesses):
        issues.append("MISSING_ASSIGNED_BUSINESS_COVERAGE")
    expected_intents = set(find_intents(question))
    if expected_intents - set(assigned_intents):
        issues.append("MISSING_REQUEST_COVERAGE")
    if len(pairs) != len(set(pairs)):
        issues.append("DUPLICATE_BUSINESS_INTENT_PAIR")

    checks = content_checks(question, payload, expected_businesses)
    if not checks["request_coverage"]:
        issues.append("MISSING_REQUEST_TERMS_IN_QUERY")
    if not checks["subject_preservation"]:
        issues.append("MISSING_SUBJECT_CONSTRAINT")
    if not checks["standalone_subqueries"]:
        issues.append("NON_STANDALONE_SUBQUERY")
    if not checks["atomic_subqueries"]:
        issues.append("NON_ATOMIC_SUBQUERY")
    issues = ordered_unique(issues)
    accepted = bool(payload.get("decomposable") is True) and not issues and 2 <= len(queries) <= config.max_subqueries
    return {
        "status": "COMPLETE" if accepted else "FAILED",
        "accepted": accepted,
        "subqueries": queries if accepted else [],
        "candidate_subqueries": queries,
        "confidence": float(payload.get("confidence") or 0.0),
        "reason": str(payload.get("reason") or "").strip(),
        "issues": issues,
        "checks": checks,
    }


class HCXQualityCaller:
    def __init__(
        self,
        api_key: str,
        *,
        config: QualityConfig | None = None,
        cache_path: str | Path | None = None,
        session: Any | None = None,
    ) -> None:
        key = str(api_key or "").strip()
        if not key or key.lower().startswith("bearer ") or any(char.isspace() for char in key):
            raise ValueError("HCX_API_KEY 형식을 확인하세요.")
        self.api_key = key
        self.config = config or QualityConfig()
        self.cache_path = Path(cache_path) if cache_path else None
        if session is None and requests is None:
            raise RuntimeError("HCX 호출에는 requests 패키지가 필요합니다.")
        self.session = session or requests.Session()
        self.cache: dict[str, dict[str, Any]] = {}
        if self.cache_path and self.cache_path.exists():
            for line in self.cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    row = json.loads(line)
                    self.cache[str(row["cache_key"])] = row

    def _append(self, row: Mapping[str, Any]) -> None:
        if not self.cache_path:
            return
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        with self.cache_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    def _call(
        self,
        *,
        cache_payload: Mapping[str, Any],
        messages: Sequence[Mapping[str, str]],
        schema: Mapping[str, Any],
        validator: Callable[[Mapping[str, Any]], dict[str, Any]],
        prompt_version: str,
    ) -> dict[str, Any]:
        from kdic_lightweight_query_ablation_core import _extract_hcx_payload

        cache_key = hashlib.sha256(
            json.dumps(cache_payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
        ).hexdigest()
        if cache_key in self.cache:
            row = dict(self.cache[cache_key])
            row["cache_hit"] = True
            row["actual_api_latency_ms"] = 0.0
            return row

        body = {
            "messages": list(messages),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 900,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": dict(schema)},
        }
        last_error: Exception | None = None
        for attempt in range(self.config.transport_retries + 1):
            started = time.perf_counter()
            try:
                response = self.session.post(
                    self.config.llm_endpoint,
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "Content-Type": "application/json",
                        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
                    },
                    json=body,
                    timeout=self.config.llm_timeout_seconds,
                )
                response.raise_for_status()
                payload, usage = _extract_hcx_payload(response.json())
                validation = validator(payload)
                latency = (time.perf_counter() - started) * 1000
                row = {
                    "cache_key": cache_key,
                    "model": self.config.llm_model,
                    "prompt_version": prompt_version,
                    "raw_payload": payload,
                    **validation,
                    **usage,
                    "effective_api_latency_ms": round(latency, 3),
                    "actual_api_latency_ms": round(latency, 3),
                    "cache_hit": False,
                    "transport_attempts": attempt + 1,
                    "error_type": "",
                    "error_message": "",
                }
                self.cache[cache_key] = row
                self._append(row)
                if self.config.request_delay_seconds:
                    time.sleep(self.config.request_delay_seconds)
                return dict(row)
            except Exception as error:
                last_error = error
                if attempt < self.config.transport_retries:
                    time.sleep(min(2 ** attempt, 4))

        row = {
            "cache_key": cache_key,
            "model": self.config.llm_model,
            "prompt_version": prompt_version,
            "raw_payload": {},
            "status": "ERROR",
            "accepted": False,
            "subqueries": [],
            "candidate_subqueries": [],
            "confidence": 0.0,
            "reason": "",
            "issues": ["LLM_REQUEST_FAILED"],
            "checks": {},
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "effective_api_latency_ms": 0.0,
            "actual_api_latency_ms": 0.0,
            "cache_hit": False,
            "transport_attempts": self.config.transport_retries + 1,
            "error_type": type(last_error).__name__ if last_error else "UnknownError",
            "error_message": str(last_error or "unknown error"),
        }
        self.cache[cache_key] = row
        self._append(row)
        return dict(row)

    def quality_first(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        return self._call(
            cache_payload={
                "prompt_version": PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
            },
            messages=build_quality_messages(question, expected_businesses),
            schema=quality_json_schema(self.config.max_subqueries),
            validator=lambda payload: validate_quality_decomposition(
                question, payload, expected_businesses=expected_businesses, config=self.config
            ),
            prompt_version=PROMPT_VERSION,
        )

    def repair(
        self,
        question: str,
        expected_businesses: Sequence[str],
        first_record: Mapping[str, Any],
        *,
        quality_mode: bool,
    ) -> dict[str, Any]:
        issues = list(first_record.get("issues") or [])
        previous_payload = dict(first_record.get("raw_payload") or {})
        if quality_mode:
            schema = quality_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_quality_decomposition(
                question, payload, expected_businesses=expected_businesses, config=self.config
            )
        else:
            schema = baseline_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_baseline_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            )
        return self._call(
            cache_payload={
                "prompt_version": REPAIR_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
                "quality_mode": quality_mode,
                "issues": issues,
                "previous_payload": previous_payload,
            },
            messages=build_repair_messages(
                question, expected_businesses, previous_payload, issues, quality_mode=quality_mode
            ),
            schema=schema,
            validator=validator,
            prompt_version=REPAIR_PROMPT_VERSION,
        )


def normalize_baseline_record(
    question: str,
    expected_businesses: Sequence[str],
    record: Mapping[str, Any],
) -> dict[str, Any]:
    output = dict(record)
    output.setdefault("candidate_subqueries", extract_queries(record.get("raw_payload") or {}))
    output.setdefault("checks", content_checks(question, record.get("raw_payload") or {}, expected_businesses))
    return output


def _condition_record(
    condition: str,
    first: Mapping[str, Any],
    final: Mapping[str, Any],
    *,
    retry_called: bool,
) -> dict[str, Any]:
    first_latency = float(first.get("effective_api_latency_ms") or 0.0)
    retry_latency = float(final.get("effective_api_latency_ms") or 0.0) if retry_called else 0.0
    first_tokens = int(first.get("total_tokens") or 0)
    retry_tokens = int(final.get("total_tokens") or 0) if retry_called else 0
    return {
        "condition": condition,
        "condition_label": CONDITION_LABELS[condition],
        "first_status": first.get("status"),
        "first_accepted": bool(first.get("accepted")),
        "first_confidence": float(first.get("confidence") or 0.0),
        "first_issues": list(first.get("issues") or []),
        "first_candidate_subqueries": list(first.get("candidate_subqueries") or first.get("subqueries") or []),
        "first_checks": dict(first.get("checks") or {}),
        "retry_called": retry_called,
        "retry_count": int(retry_called),
        "retry_success": bool(retry_called and final.get("accepted")),
        "retry_status": final.get("status") if retry_called else "NOT_CALLED",
        "retry_issues": list(final.get("issues") or []) if retry_called else [],
        "final_status": final.get("status"),
        "final_accepted": bool(final.get("accepted")),
        "final_confidence": float(final.get("confidence") or 0.0),
        "final_issues": list(final.get("issues") or []),
        "final_subqueries": list(final.get("subqueries") or []),
        "final_candidate_subqueries": list(final.get("candidate_subqueries") or final.get("subqueries") or []),
        "final_checks": dict(final.get("checks") or {}),
        "fallback_to_original": not bool(final.get("accepted")),
        "analysis_api_latency_ms": first_latency + retry_latency,
        "prompt_tokens": int(first.get("prompt_tokens") or 0) + (int(final.get("prompt_tokens") or 0) if retry_called else 0),
        "completion_tokens": int(first.get("completion_tokens") or 0) + (int(final.get("completion_tokens") or 0) if retry_called else 0),
        "total_tokens": first_tokens + retry_tokens,
        "api_request_count": 1 + int(retry_called),
        "first_raw_payload": dict(first.get("raw_payload") or {}),
        "retry_raw_payload": dict(final.get("raw_payload") or {}) if retry_called else {},
    }


def should_semantic_retry(record: Mapping[str, Any]) -> bool:
    """검증 가능한 생성 오류만 한 번 교정하고, 모델의 분해 거절은 존중한다."""
    status = str(record.get("status") or "").upper()
    issues = set(record.get("issues") or [])
    if bool(record.get("accepted")):
        return False
    if status in {"DECLINED", "ERROR"}:
        return False
    if "LLM_DECLINED_DECOMPOSITION" in issues or "LLM_REQUEST_FAILED" in issues:
        return False
    return True


def run_candidate_conditions(
    question: str,
    expected_businesses: Sequence[str],
    *,
    baseline_decomposer: Any,
    quality_caller: HCXQualityCaller,
) -> list[dict[str, Any]]:
    baseline_first = normalize_baseline_record(
        question, expected_businesses,
        baseline_decomposer.decompose(question, expected_businesses),
    )
    quality_first = quality_caller.quality_first(question, expected_businesses)

    if not should_semantic_retry(baseline_first):
        baseline_final = baseline_first
        baseline_retry_called = False
    else:
        baseline_final = quality_caller.repair(
            question, expected_businesses, baseline_first, quality_mode=False
        )
        baseline_retry_called = True

    if not should_semantic_retry(quality_first):
        quality_final = quality_first
        quality_retry_called = False
    else:
        quality_final = quality_caller.repair(
            question, expected_businesses, quality_first, quality_mode=True
        )
        quality_retry_called = True

    return [
        _condition_record(BASELINE, baseline_first, baseline_first, retry_called=False),
        _condition_record(QUALITY, quality_first, quality_first, retry_called=False),
        _condition_record(RETRY, baseline_first, baseline_final, retry_called=baseline_retry_called),
        _condition_record(QUALITY_RETRY, quality_first, quality_final, retry_called=quality_retry_called),
    ]


def make_query_plans(original: str, subqueries: Sequence[str], config: QualityConfig) -> list[Any]:
    from kdic_integrated_eval_core import QueryPlan

    cleaned = ordered_unique(subqueries)
    compact_original = re.sub(r"\s+", "", original).lower()
    cleaned = [item for item in cleaned if re.sub(r"\s+", "", item).lower() != compact_original]
    if len(cleaned) < 2:
        return [QueryPlan(
            need_id="FUSED", variant_id="ORIGINAL", dense_query=original, bm25_query=original,
            filter_mode="NONE", business_filters=[], soft_business_hints=[], query_weight=1.0,
            query_source="ORIGINAL_FALLBACK",
        )]
    each = config.subquery_total_weight / len(cleaned)
    plans = [QueryPlan(
        need_id="FUSED", variant_id="ORIGINAL_ANCHOR", dense_query=original, bm25_query=original,
        filter_mode="NONE", business_filters=[], soft_business_hints=[],
        query_weight=config.original_weight, query_source="ORIGINAL_ANCHOR",
    )]
    plans.extend(QueryPlan(
        need_id="FUSED", variant_id=f"SUBQUERY_{index:02d}", dense_query=query, bm25_query=query,
        filter_mode="NONE", business_filters=[], soft_business_hints=[],
        query_weight=each, query_source="DECOMPOSED",
    ) for index, query in enumerate(cleaned, 1))
    return plans


def build_condition_case(
    common: Mapping[str, Any],
    condition_record: Mapping[str, Any] | None,
    condition: str,
    *,
    config: QualityConfig,
) -> Any:
    from kdic_integrated_eval_core import AnalyzerCase

    route = str(common["route"])
    original = str(common["original_question"])
    cross_candidate = bool(common.get("complex_candidate")) and len(
        ordered_unique((common.get("complexity") or {}).get("businesses") or [])
    ) >= 2
    record = dict(condition_record or {})
    accepted = bool(cross_candidate and record.get("final_accepted"))
    subqueries = list(record.get("final_subqueries") or []) if accepted else []
    plans = make_query_plans(original, subqueries, config) if route == "RETRIEVE" else []
    analysis_latency = float(common.get("common_latency_ms") or 0.0) + float(record.get("analysis_api_latency_ms") or 0.0)
    raw_result = {
        "pipeline_version": condition,
        "analysis_status": "OK",
        "original_query": original,
        "normalized_query": common.get("normalized_question"),
        "route_reasons": common.get("route_reasons") or [],
        "complex_candidate": bool(common.get("complex_candidate")),
        "cross_business_candidate": cross_candidate,
        "businesses": (common.get("complexity") or {}).get("businesses") or [],
        "decomposition_condition": condition,
        "decomposition_record": record,
        "decomposition_source": "LLM" if accepted else ("ORIGINAL_FALLBACK" if cross_candidate else "ORIGINAL_POLICY"),
        "final_subqueries": subqueries,
        "fusion_policy": "WEIGHTED_RRF",
        "original_weight": config.original_weight if accepted else 1.0,
        "subquery_total_weight": config.subquery_total_weight if accepted else 0.0,
        "runtime": {
            "api_request_count": int(record.get("api_request_count") or 0),
            "prompt_tokens": int(record.get("prompt_tokens") or 0),
            "completion_tokens": int(record.get("completion_tokens") or 0),
            "total_tokens": int(record.get("total_tokens") or 0),
            "latency_ms": analysis_latency,
        },
    }
    return AnalyzerCase(
        evaluation_id=str(common["evaluation_id"]), analyzer=condition,
        original_question=original, route=route,
        analysis_latency_ms=round(analysis_latency, 3), plans=plans, raw_result=raw_result,
    )


def summarize_decomposition(audit_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for condition in CONDITION_ORDER:
        frame = audit_df[audit_df["condition"].eq(condition)]
        candidates = frame[frame["cross_business_candidate"]]
        true_cross = candidates[candidates["gold_cross_business"]]
        boundary = candidates[~candidates["gold_cross_business"]]
        retries = candidates[candidates["retry_called"]]
        rows.append({
            "condition": condition,
            "condition_label": CONDITION_LABELS[condition],
            "all_question_count": len(frame),
            "predicted_cross_business_count": len(candidates),
            "gold_cross_business_count": int(frame["gold_cross_business"].sum()),
            "cross_business_true_positive_count": int((frame["cross_business_candidate"] & frame["gold_cross_business"]).sum()),
            "cross_business_false_positive_count": int((frame["cross_business_candidate"] & ~frame["gold_cross_business"]).sum()),
            "cross_business_false_negative_count": int((~frame["cross_business_candidate"] & frame["gold_cross_business"]).sum()),
            "first_accept_count": int(candidates["first_accepted"].sum()),
            "final_accept_count": int(candidates["final_accepted"].sum()),
            "true_cross_first_accept_rate": float(true_cross["first_accepted"].mean()) if len(true_cross) else math.nan,
            "true_cross_final_accept_rate": float(true_cross["final_accepted"].mean()) if len(true_cross) else math.nan,
            "boundary_wrong_accept_count": int(boundary["final_accepted"].sum()),
            "retry_call_count": int(candidates["retry_called"].sum()),
            "retry_success_count": int(candidates["retry_success"].sum()),
            "retry_success_rate": float(retries["retry_success"].mean()) if len(retries) else math.nan,
            "fallback_count": int(candidates["fallback_to_original"].sum()),
            "business_coverage_rate": float(candidates["check_business_coverage"].mean()),
            "request_coverage_rate": float(candidates["check_request_coverage"].mean()),
            "constraint_preservation_rate": float((
                candidates["check_numeric_preservation"]
                & candidates["check_negation_preservation"]
                & candidates["check_subject_preservation"]
            ).mean()),
            "standalone_rate": float(candidates["check_standalone_subqueries"].mean()),
            "atomic_rate": float(candidates["check_atomic_subqueries"].mean()),
            "analysis_latency_ms_mean_all": float(frame["analysis_latency_ms"].mean()),
            "analysis_latency_ms_p95_all": float(frame["analysis_latency_ms"].quantile(.95)),
            "analysis_latency_ms_mean_candidates": float(candidates["analysis_latency_ms"].mean()),
            "total_tokens": int(frame["total_tokens"].sum()),
        })
    return pd.DataFrame(rows)


def decomposition_hard_gates(audit_df: pd.DataFrame, eval_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for condition in CONDITION_ORDER:
        frame = audit_df[audit_df["condition"].eq(condition)]
        candidates = frame[frame["cross_business_candidate"]]
        boundary = candidates[~candidates["gold_cross_business"]]
        retry_failed = candidates[candidates["retry_called"] & ~candidates["retry_success"]]
        accepted = candidates[candidates["final_accepted"]]
        gates = [
            ("실행 성공률", float(frame["execution_success"].mean()), ">=0.995", float(frame["execution_success"].mean()) >= .995),
            ("검색 질의 생성 유효율", float(frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_valid"].mean()), ">=0.99", float(frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_valid"].mean()) >= .99),
            ("정상 질문의 잘못된 OOS", int(frame["false_oos"].sum()), "=0", int(frame["false_oos"].sum()) == 0),
            ("정상 질문의 잘못된 DIRECT", int(frame["false_direct"].sum()), "=0", int(frame["false_direct"].sum()) == 0),
            ("Hard Filter", int(frame["hard_filter_count"].sum()), "=0", int(frame["hard_filter_count"].sum()) == 0),
            ("비교차 업무 분해 승인", int(boundary["final_accepted"].sum()), "=0", int(boundary["final_accepted"].sum()) == 0),
            ("승인 결과 검증 오류", int(accepted["final_issues"].map(bool).sum()), "=0", int(accepted["final_issues"].map(bool).sum()) == 0),
            ("재시도 최대 1회 초과", int((candidates["retry_count"] > 1).sum()), "=0", int((candidates["retry_count"] > 1).sum()) == 0),
            ("재시도 실패 후 원문 fallback", float(retry_failed["fallback_to_original"].mean()) if len(retry_failed) else 1.0, "=1.0", bool(retry_failed["fallback_to_original"].all()) if len(retry_failed) else True),
            (
                "질의 가중치 합 오류",
                int((
                    frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_weight_sum"]
                    .sub(1.0).abs() > 1e-9
                ).sum()),
                "=0",
                int((
                    frame.loc[frame["route"].eq("RETRIEVE"), "query_plan_weight_sum"]
                    .sub(1.0).abs() > 1e-9
                ).sum()) == 0,
            ),
        ]
        for gate, value, threshold, passed in gates:
            rows.append({
                "condition": condition,
                "condition_label": CONDITION_LABELS[condition],
                "gate": gate,
                "value": value,
                "threshold": threshold,
                "passed": bool(passed),
            })
    return pd.DataFrame(rows)


def serialize_nested(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    for column in output.columns:
        if output[column].map(lambda value: isinstance(value, (list, dict, tuple))).any():
            output[column] = output[column].map(
                lambda value: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (list, dict, tuple)) else value
            )
    return output


In [ ]:
%%writefile kdic_hcx007_resumable_decomposition_core.py
from __future__ import annotations

import email.utils
import hashlib
import json
import random
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence

import requests

import kdic_decomposition_quality_core as quality_core
from kdic_decomposition_quality_core import (
    BASELINE,
    CONDITION_LABELS,
    QUALITY,
    QUALITY_RETRY,
    RETRY,
    QualityConfig,
    baseline_json_schema,
    build_quality_messages,
    build_repair_messages,
    content_checks,
    extract_queries,
    quality_json_schema,
    should_semantic_retry,
    validate_baseline_decomposition,
    validate_quality_decomposition,
)
from kdic_lightweight_query_ablation_core import (
    AblationConfig,
    DECOMPOSITION_PROMPT_VERSION,
    _extract_hcx_payload,
    build_decomposition_messages,
    decomposition_json_schema,
    validate_llm_decomposition,
)


HCX_DECOMPOSITION_MODEL = "HCX-007"
HCX_DECOMPOSITION_ENDPOINT = (
    "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
)
REDESIGN_PROMPT_VERSION = "KDIC_DECOMPOSITION_QUALITY_V2_HCX007_2026_08_14"
REDESIGN_REPAIR_PROMPT_VERSION = "KDIC_DECOMPOSITION_REPAIR_V2_HCX007_2026_08_14"


@dataclass(frozen=True)
class TransportPolicy:
    request_delay_seconds: float = 1.5
    max_transport_retries: int = 5
    base_backoff_seconds: float = 2.0
    max_backoff_seconds: float = 32.0
    jitter_seconds: float = 0.5
    consecutive_429_cooldown_threshold: int = 3
    cooldown_seconds: float = 60.0
    timeout_seconds: float = 120.0


def _valid_api_key(value: str) -> str:
    key = str(value or "").strip()
    if not key or key.lower().startswith("bearer ") or any(ch.isspace() for ch in key):
        raise ValueError("HCX_API_KEY에는 Bearer 접두사나 공백을 넣지 않습니다.")
    return key


def _cache_key(payload: Mapping[str, Any]) -> str:
    raw = json.dumps(dict(payload), ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


class ValidOnlyJsonlCache:
    """정상 응답만 재사용하고 ERROR 행은 감사 기록으로만 남긴다."""

    def __init__(
        self,
        active_path: str | Path,
        *,
        seed_paths: Sequence[str | Path] = (),
    ) -> None:
        self.active_path = Path(active_path)
        self.rows: dict[str, dict[str, Any]] = {}
        self.origin_by_key: dict[str, str] = {}
        for source in [*map(Path, seed_paths), self.active_path]:
            if not source.is_file():
                continue
            for line in source.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                row = json.loads(line)
                key = str(row.get("cache_key") or "")
                if not key or str(row.get("status") or "").upper() == "ERROR":
                    continue
                self.rows[key] = row
                self.origin_by_key[key] = str(source)

    def get(self, key: str) -> dict[str, Any] | None:
        if key not in self.rows:
            return None
        row = dict(self.rows[key])
        row["cache_hit"] = True
        row["actual_api_latency_ms"] = 0.0
        row["cache_origin"] = self.origin_by_key.get(key, "")
        if Path(self.origin_by_key.get(key, "")) != self.active_path:
            promoted = dict(row)
            promoted["promoted_from_seed_cache"] = True
            self.append(promoted, reusable=True)
        return row

    def append(self, row: Mapping[str, Any], *, reusable: bool) -> None:
        payload = dict(row)
        self.active_path.parent.mkdir(parents=True, exist_ok=True)
        with self.active_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
        key = str(payload.get("cache_key") or "")
        if reusable and key:
            self.rows[key] = payload
            self.origin_by_key[key] = str(self.active_path)


def _retry_after_seconds(response: requests.Response) -> float | None:
    raw = str(response.headers.get("Retry-After") or "").strip()
    if not raw:
        return None
    try:
        return max(0.0, float(raw))
    except ValueError:
        try:
            parsed = email.utils.parsedate_to_datetime(raw)
            if parsed.tzinfo is None:
                parsed = parsed.replace(tzinfo=timezone.utc)
            return max(0.0, (parsed - datetime.now(timezone.utc)).total_seconds())
        except Exception:
            return None


class RobustHCXTransport:
    def __init__(
        self,
        api_key: str,
        *,
        endpoint: str = HCX_DECOMPOSITION_ENDPOINT,
        policy: TransportPolicy | None = None,
        session: requests.Session | None = None,
        sleep_fn: Callable[[float], None] = time.sleep,
        monotonic_fn: Callable[[], float] = time.monotonic,
        random_fn: Callable[[], float] = random.random,
    ) -> None:
        self.api_key = _valid_api_key(api_key)
        self.endpoint = endpoint
        self.policy = policy or TransportPolicy()
        self.session = session or requests.Session()
        self.sleep_fn = sleep_fn
        self.monotonic_fn = monotonic_fn
        self.random_fn = random_fn
        self.last_request_at: float | None = None
        self.consecutive_429 = 0

    def _pace(self) -> float:
        if self.last_request_at is None:
            return 0.0
        remaining = self.policy.request_delay_seconds - (
            self.monotonic_fn() - self.last_request_at
        )
        if remaining > 0:
            self.sleep_fn(remaining)
            return remaining
        return 0.0

    def post_json(self, body: Mapping[str, Any]) -> dict[str, Any]:
        logical_started = self.monotonic_fn()
        total_sleep_seconds = 0.0
        service_latency_ms = 0.0
        attempts = 0
        last_status: int | None = None
        last_error_type = ""
        last_error_message = ""
        last_response_body = ""

        for retry_index in range(self.policy.max_transport_retries + 1):
            total_sleep_seconds += self._pace()
            attempts += 1
            request_started = self.monotonic_fn()
            try:
                response = self.session.post(
                    self.endpoint,
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "Content-Type": "application/json",
                        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
                    },
                    json=dict(body),
                    timeout=self.policy.timeout_seconds,
                )
                self.last_request_at = self.monotonic_fn()
                service_latency_ms += (self.last_request_at - request_started) * 1000
                last_status = int(response.status_code)
                last_response_body = str(response.text or "")[:8000]

                if 200 <= response.status_code < 300:
                    self.consecutive_429 = 0
                    payload, usage = _extract_hcx_payload(response.json())
                    return {
                        "transport_ok": True,
                        "payload": payload,
                        "usage": usage,
                        "http_status": last_status,
                        "transport_attempts": attempts,
                        "service_latency_ms": round(service_latency_ms, 3),
                        "transport_sleep_ms": round(total_sleep_seconds * 1000, 3),
                        "actual_api_latency_ms": round(
                            (self.monotonic_fn() - logical_started) * 1000, 3
                        ),
                        "error_type": "",
                        "error_message": "",
                        "error_response_body": "",
                    }

                last_error_type = "HTTP_ERROR"
                last_error_message = f"HTTP {response.status_code}"
                retryable = response.status_code == 429 or response.status_code >= 500
                if response.status_code == 429:
                    self.consecutive_429 += 1
                else:
                    self.consecutive_429 = 0
                if not retryable or retry_index >= self.policy.max_transport_retries:
                    break

                if (
                    response.status_code == 429
                    and self.consecutive_429
                    >= self.policy.consecutive_429_cooldown_threshold
                ):
                    wait_seconds = self.policy.cooldown_seconds
                    self.consecutive_429 = 0
                else:
                    retry_after = _retry_after_seconds(response)
                    exponential = min(
                        self.policy.max_backoff_seconds,
                        self.policy.base_backoff_seconds * (2**retry_index),
                    )
                    wait_seconds = retry_after if retry_after is not None else exponential
                    wait_seconds += self.random_fn() * self.policy.jitter_seconds
                self.sleep_fn(wait_seconds)
                total_sleep_seconds += wait_seconds
            except Exception as error:
                self.last_request_at = self.monotonic_fn()
                service_latency_ms += (self.last_request_at - request_started) * 1000
                last_error_type = type(error).__name__
                last_error_message = str(error)
                if retry_index >= self.policy.max_transport_retries:
                    break
                wait_seconds = min(
                    self.policy.max_backoff_seconds,
                    self.policy.base_backoff_seconds * (2**retry_index),
                ) + self.random_fn() * self.policy.jitter_seconds
                self.sleep_fn(wait_seconds)
                total_sleep_seconds += wait_seconds

        return {
            "transport_ok": False,
            "payload": {},
            "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
            "http_status": last_status,
            "transport_attempts": attempts,
            "service_latency_ms": round(service_latency_ms, 3),
            "transport_sleep_ms": round(total_sleep_seconds * 1000, 3),
            "actual_api_latency_ms": round(
                (self.monotonic_fn() - logical_started) * 1000, 3
            ),
            "error_type": last_error_type or "UNKNOWN_TRANSPORT_ERROR",
            "error_message": last_error_message or "unknown transport error",
            "error_response_body": last_response_body,
        }


def _error_row(
    cache_key: str,
    *,
    question: str,
    model: str,
    prompt_version: str,
    transport: Mapping[str, Any],
) -> dict[str, Any]:
    return {
        "cache_key": cache_key,
        "question": question,
        "model": model,
        "prompt_version": prompt_version,
        "raw_payload": {},
        "status": "ERROR",
        "accepted": False,
        "subqueries": [],
        "candidate_subqueries": [],
        "confidence": 0.0,
        "reason": "",
        "issues": ["LLM_REQUEST_FAILED"],
        "checks": {},
        **dict(transport.get("usage") or {}),
        "effective_api_latency_ms": float(transport.get("actual_api_latency_ms") or 0.0),
        "cache_hit": False,
        **{
            key: transport.get(key)
            for key in (
                "http_status", "transport_attempts", "service_latency_ms",
                "transport_sleep_ms", "actual_api_latency_ms", "error_type",
                "error_message", "error_response_body",
            )
        },
    }


class ResumableBaselineDecomposer:
    def __init__(
        self,
        api_key: str,
        *,
        cache_path: str | Path,
        seed_cache_paths: Sequence[str | Path] = (),
        config: AblationConfig | None = None,
        transport_policy: TransportPolicy | None = None,
        session: requests.Session | None = None,
    ) -> None:
        self.config = config or AblationConfig(
            llm_endpoint=HCX_DECOMPOSITION_ENDPOINT,
            llm_model=HCX_DECOMPOSITION_MODEL,
        )
        if self.config.llm_model != HCX_DECOMPOSITION_MODEL:
            raise ValueError("Baseline 구조화 분해 모델은 HCX-007이어야 합니다.")
        self.cache = ValidOnlyJsonlCache(cache_path, seed_paths=seed_cache_paths)
        self.transport = RobustHCXTransport(
            api_key, endpoint=HCX_DECOMPOSITION_ENDPOINT,
            policy=transport_policy, session=session,
        )

    def _key(self, question: str, expected_businesses: Sequence[str]) -> str:
        return _cache_key({
            "prompt_version": DECOMPOSITION_PROMPT_VERSION,
            "model": self.config.llm_model,
            "question": question,
            "expected_businesses": list(expected_businesses),
            "min_confidence": self.config.llm_min_confidence,
        })

    def decompose(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        key = self._key(question, expected_businesses)
        cached = self.cache.get(key)
        if cached is not None:
            return cached
        body = {
            "messages": build_decomposition_messages(question),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 700,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {
                "type": "json",
                "schema": decomposition_json_schema(self.config.max_subqueries),
            },
        }
        transport = self.transport.post_json(body)
        if not transport["transport_ok"]:
            row = _error_row(
                key, question=question, model=self.config.llm_model,
                prompt_version=DECOMPOSITION_PROMPT_VERSION, transport=transport,
            )
            self.cache.append(row, reusable=False)
            return row
        validation = validate_llm_decomposition(
            question,
            transport["payload"],
            expected_businesses=expected_businesses,
            config=self.config,
        )
        row = {
            "cache_key": key,
            "question": question,
            "model": self.config.llm_model,
            "prompt_version": DECOMPOSITION_PROMPT_VERSION,
            "raw_payload": transport["payload"],
            **validation,
            **transport["usage"],
            "effective_api_latency_ms": transport["actual_api_latency_ms"],
            "cache_hit": False,
            **{
                field: transport.get(field)
                for field in (
                    "http_status", "transport_attempts", "service_latency_ms",
                    "transport_sleep_ms", "actual_api_latency_ms", "error_type",
                    "error_message", "error_response_body",
                )
            },
        }
        self.cache.append(row, reusable=True)
        return row


class ResumableQualityCaller:
    def __init__(
        self,
        api_key: str,
        *,
        cache_path: str | Path,
        config: QualityConfig,
        transport_policy: TransportPolicy | None = None,
        session: requests.Session | None = None,
    ) -> None:
        if config.llm_model != HCX_DECOMPOSITION_MODEL:
            raise ValueError("개선 구조화 분해 모델은 HCX-007이어야 합니다.")
        self.config = config
        self.cache = ValidOnlyJsonlCache(cache_path)
        self.transport = RobustHCXTransport(
            api_key, endpoint=HCX_DECOMPOSITION_ENDPOINT,
            policy=transport_policy, session=session,
        )

    def _call(
        self,
        *,
        key_payload: Mapping[str, Any],
        question: str,
        prompt_version: str,
        messages: Sequence[Mapping[str, str]],
        schema: Mapping[str, Any],
        validator: Callable[[Mapping[str, Any]], dict[str, Any]],
    ) -> dict[str, Any]:
        key = _cache_key(key_payload)
        cached = self.cache.get(key)
        if cached is not None:
            return cached
        body = {
            "messages": list(messages),
            "topP": 0.1,
            "topK": 0,
            "maxCompletionTokens": 900,
            "temperature": 0.0,
            "repetitionPenalty": 1.0,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": dict(schema)},
        }
        transport = self.transport.post_json(body)
        if not transport["transport_ok"]:
            row = _error_row(
                key, question=question, model=self.config.llm_model,
                prompt_version=prompt_version, transport=transport,
            )
            self.cache.append(row, reusable=False)
            return row
        validation = validator(transport["payload"])
        row = {
            "cache_key": key,
            "question": question,
            "model": self.config.llm_model,
            "prompt_version": prompt_version,
            "raw_payload": transport["payload"],
            **validation,
            **transport["usage"],
            "effective_api_latency_ms": transport["actual_api_latency_ms"],
            "cache_hit": False,
            **{
                field: transport.get(field)
                for field in (
                    "http_status", "transport_attempts", "service_latency_ms",
                    "transport_sleep_ms", "actual_api_latency_ms", "error_type",
                    "error_message", "error_response_body",
                )
            },
        }
        self.cache.append(row, reusable=True)
        return row

    def quality_first(self, question: str, expected_businesses: Sequence[str]) -> dict[str, Any]:
        return self._call(
            key_payload={
                "prompt_version": REDESIGN_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
            },
            question=question,
            prompt_version=REDESIGN_PROMPT_VERSION,
            messages=build_quality_messages(question, expected_businesses),
            schema=quality_json_schema(self.config.max_subqueries),
            validator=lambda payload: validate_quality_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            ),
        )

    def repair(
        self,
        question: str,
        expected_businesses: Sequence[str],
        first_record: Mapping[str, Any],
        *,
        quality_mode: bool,
    ) -> dict[str, Any]:
        issues = list(first_record.get("issues") or [])
        previous_payload = dict(first_record.get("raw_payload") or {})
        if quality_mode:
            schema = quality_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_quality_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            )
        else:
            schema = baseline_json_schema(self.config.max_subqueries)
            validator = lambda payload: validate_baseline_decomposition(
                question, payload,
                expected_businesses=expected_businesses,
                config=self.config,
            )
        return self._call(
            key_payload={
                "prompt_version": REDESIGN_REPAIR_PROMPT_VERSION,
                "model": self.config.llm_model,
                "question": question,
                "expected_businesses": list(expected_businesses),
                "quality_mode": quality_mode,
                "issues": issues,
                "previous_payload": previous_payload,
            },
            question=question,
            prompt_version=REDESIGN_REPAIR_PROMPT_VERSION,
            messages=build_repair_messages(
                question, expected_businesses, previous_payload, issues,
                quality_mode=quality_mode,
            ),
            schema=schema,
            validator=validator,
        )


def _normalize_baseline_record(
    question: str,
    expected_businesses: Sequence[str],
    record: Mapping[str, Any],
) -> dict[str, Any]:
    output = dict(record)
    output.setdefault(
        "candidate_subqueries",
        extract_queries(record.get("raw_payload") or {}),
    )
    output.setdefault(
        "checks",
        content_checks(question, record.get("raw_payload") or {}, expected_businesses),
    )
    return output


def _condition_record(
    condition: str,
    first: Mapping[str, Any],
    final: Mapping[str, Any],
    *,
    retry_called: bool,
) -> dict[str, Any]:
    final_record = dict(final)
    first_latency = float(first.get("effective_api_latency_ms") or 0.0)
    retry_latency = float(final.get("effective_api_latency_ms") or 0.0) if retry_called else 0.0
    return {
        "condition": condition,
        "condition_label": CONDITION_LABELS[condition],
        "first_status": first.get("status"),
        "first_accepted": bool(first.get("accepted")),
        "first_confidence": float(first.get("confidence") or 0.0),
        "first_issues": list(first.get("issues") or []),
        "first_candidate_subqueries": list(
            first.get("candidate_subqueries") or first.get("subqueries") or []
        ),
        "first_checks": dict(first.get("checks") or {}),
        "first_http_status": first.get("http_status"),
        "first_error_type": first.get("error_type") or "",
        "first_error_message": first.get("error_message") or "",
        "first_error_response_body": first.get("error_response_body") or "",
        "first_transport_attempts": int(first.get("transport_attempts") or 0),
        "first_cache_hit": bool(first.get("cache_hit")),
        "first_cache_origin": first.get("cache_origin") or "",
        "retry_called": retry_called,
        "retry_count": int(retry_called),
        "retry_success": bool(retry_called and final.get("accepted")),
        "retry_status": final.get("status") if retry_called else "NOT_CALLED",
        "retry_issues": list(final.get("issues") or []) if retry_called else [],
        "retry_http_status": final.get("http_status") if retry_called else None,
        "retry_error_type": final.get("error_type") if retry_called else "",
        "retry_transport_attempts": int(final.get("transport_attempts") or 0) if retry_called else 0,
        "final_status": final.get("status"),
        "final_accepted": bool(final.get("accepted")),
        "final_confidence": float(final.get("confidence") or 0.0),
        "final_issues": list(final.get("issues") or []),
        "final_subqueries": list(final.get("subqueries") or []),
        "final_candidate_subqueries": list(
            final.get("candidate_subqueries") or final.get("subqueries") or []
        ),
        "final_checks": dict(final.get("checks") or {}),
        "fallback_to_original": not bool(final.get("accepted")),
        "analysis_api_latency_ms": first_latency + retry_latency,
        "actual_api_latency_ms": float(first.get("actual_api_latency_ms") or 0.0)
        + (float(final.get("actual_api_latency_ms") or 0.0) if retry_called else 0.0),
        "prompt_tokens": int(first.get("prompt_tokens") or 0)
        + (int(final.get("prompt_tokens") or 0) if retry_called else 0),
        "completion_tokens": int(first.get("completion_tokens") or 0)
        + (int(final.get("completion_tokens") or 0) if retry_called else 0),
        "total_tokens": int(first.get("total_tokens") or 0)
        + (int(final.get("total_tokens") or 0) if retry_called else 0),
        "logical_api_request_count": 1 + int(retry_called),
        # 기존 build_condition_case가 읽는 호환 필드입니다. 의미는 HTTP 재시도
        # 횟수가 아니라 첫 구조화 호출 + 선택적 의미 교정 호출 수입니다.
        "api_request_count": 1 + int(retry_called),
        "transport_attempt_count": int(first.get("transport_attempts") or 0)
        + (int(final.get("transport_attempts") or 0) if retry_called else 0),
        "first_raw_payload": dict(first.get("raw_payload") or {}),
        "retry_raw_payload": dict(final_record.get("raw_payload") or {}) if retry_called else {},
    }


def run_resumable_candidate_conditions(
    question: str,
    expected_businesses: Sequence[str],
    *,
    baseline_decomposer: ResumableBaselineDecomposer,
    quality_caller: ResumableQualityCaller,
) -> list[dict[str, Any]]:
    baseline_first = _normalize_baseline_record(
        question,
        expected_businesses,
        baseline_decomposer.decompose(question, expected_businesses),
    )
    quality_first = quality_caller.quality_first(question, expected_businesses)

    if should_semantic_retry(baseline_first):
        baseline_final = quality_caller.repair(
            question, expected_businesses, baseline_first, quality_mode=False
        )
        baseline_retry_called = True
    else:
        baseline_final = baseline_first
        baseline_retry_called = False

    if should_semantic_retry(quality_first):
        quality_final = quality_caller.repair(
            question, expected_businesses, quality_first, quality_mode=True
        )
        quality_retry_called = True
    else:
        quality_final = quality_first
        quality_retry_called = False

    return [
        _condition_record(BASELINE, baseline_first, baseline_first, retry_called=False),
        _condition_record(QUALITY, quality_first, quality_first, retry_called=False),
        _condition_record(RETRY, baseline_first, baseline_final, retry_called=baseline_retry_called),
        _condition_record(
            QUALITY_RETRY, quality_first, quality_final,
            retry_called=quality_retry_called,
        ),
    ]


def component_gate_rows(audit_df: Any) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for condition, frame in audit_df.groupby("condition", sort=False):
        candidates = frame[frame["cross_business_candidate"].astype(bool)]
        first_success = candidates["first_status"].ne("ERROR")
        http_400_count = int(candidates["first_http_status"].eq(400).sum())
        final_error_count = int(candidates["final_status"].eq("ERROR").sum())
        error_latency_missing = int((
            candidates["first_status"].eq("ERROR")
            & candidates["actual_api_latency_ms"].le(0)
        ).sum())
        checks = [
            ("structured_call_success_rate", float(first_success.mean()), 0.995, float(first_success.mean()) >= .995),
            ("candidate_evaluable_count", int(first_success.sum()), len(candidates), int(first_success.sum()) == len(candidates)),
            ("http_400_count", http_400_count, 0, http_400_count == 0),
            ("final_transport_error_count", final_error_count, 0, final_error_count == 0),
            ("error_latency_missing_count", error_latency_missing, 0, error_latency_missing == 0),
            ("retry_limit_exceeded_count", int((candidates["retry_count"] > 1).sum()), 0, int((candidates["retry_count"] > 1).sum()) == 0),
        ]
        for gate, value, threshold, passed in checks:
            rows.append({
                "condition": condition,
                "condition_label": CONDITION_LABELS.get(condition, condition),
                "gate": gate,
                "value": value,
                "threshold": threshold,
                "passed": bool(passed),
            })
    return rows


## 3. M3 Hybrid 7:3 Min-Max + Reranker 검색

In [ ]:
def _normalize_vector(vector: np.ndarray) -> np.ndarray:
    vector = np.asarray(vector, dtype=np.float32)
    norm = float(np.linalg.norm(vector))
    if norm == 0.0:
        raise RuntimeError("질문 임베딩이 영벡터입니다.")
    return vector / norm


def dense_search_numpy_by_vector(
    query_vector: np.ndarray,
    depth: int = CANDIDATE_DEPTH,
) -> list[dict[str, Any]]:
    """현재 corpus 전체를 대상으로 한 NumPy exact dot-product 기준 검색."""
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    vector = _normalize_vector(query_vector)
    if vector.shape != (DENSE_DIMENSION,):
        raise RuntimeError(
            f"질문 임베딩 차원 불일치: query={vector.shape}, stored={DENSE_DIMENSION}"
        )
    scores = DENSE_MATRIX @ vector
    limit = min(depth, len(DENSE_CHUNK_IDS))
    candidate_indices = np.argpartition(-scores, limit - 1)[:limit]
    ordered_indices = sorted(
        candidate_indices.tolist(),
        key=lambda index: (-float(scores[index]), DENSE_CHUNK_IDS[index]),
    )
    return [
        {
            "chunk_id": DENSE_CHUNK_IDS[index],
            "score": float(scores[index]),
            "rank": rank,
        }
        for rank, index in enumerate(ordered_indices, start=1)
    ]


def dense_search_by_vector(
    query_vector: np.ndarray,
    depth: int = CANDIDATE_DEPTH,
) -> list[dict[str, Any]]:
    """Elasticsearch dense_vector kNN 검색."""
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    vector = _normalize_vector(query_vector)
    if vector.shape != (DENSE_DIMENSION,):
        raise RuntimeError(
            f"질문 임베딩 차원 불일치: query={vector.shape}, stored={DENSE_DIMENSION}"
        )
    response = ES.search(
        index=ES_INDEX_NAME,
        knn={
            "field": "embedding",
            "query_vector": vector.tolist(),
            "k": min(depth, len(CHUNKS)),
            "num_candidates": min(
                len(CHUNKS),
                max(depth, DENSE_KNN_NUM_CANDIDATES),
            ),
        },
        size=min(depth, len(CHUNKS)),
        source=["chunk_id"],
    )
    results = []
    seen: set[str] = set()
    for hit in response["hits"]["hits"]:
        chunk_id = str(hit["_source"]["chunk_id"])
        if chunk_id in seen:
            continue
        if chunk_id not in CHUNKS_BY_ID:
            raise RuntimeError(f"Dense kNN 결과의 chunk_id가 원본에 없습니다: {chunk_id}")
        seen.add(chunk_id)
        results.append({
            "chunk_id": chunk_id,
            "score": float(hit["_score"]),
            "rank": len(results) + 1,
        })
    if not results:
        raise RuntimeError("Elasticsearch Dense kNN 결과가 없습니다.")
    return results


_LAST_DENSE_BACKEND_USED = ""
_LAST_DENSE_FALLBACK_ERROR = ""


def dense_search_from_vector(
    query_vector: np.ndarray,
    depth: int = CANDIDATE_DEPTH,
) -> list[dict[str, Any]]:
    global _LAST_DENSE_BACKEND_USED, _LAST_DENSE_FALLBACK_ERROR
    _LAST_DENSE_FALLBACK_ERROR = ""
    if DENSE_BACKEND == "NUMPY_EXACT":
        _LAST_DENSE_BACKEND_USED = "NUMPY_EXACT"
        return dense_search_numpy_by_vector(query_vector, depth)
    if DENSE_BACKEND != "ELASTICSEARCH_KNN":
        raise ValueError(f"지원하지 않는 DENSE_BACKEND입니다: {DENSE_BACKEND}")
    try:
        results = dense_search_by_vector(query_vector, depth)
        _LAST_DENSE_BACKEND_USED = "ELASTICSEARCH_KNN"
        return results
    except Exception as error:
        if not ALLOW_NUMPY_DENSE_FALLBACK:
            raise
        _LAST_DENSE_BACKEND_USED = "NUMPY_EXACT_FALLBACK"
        _LAST_DENSE_FALLBACK_ERROR = f"{type(error).__name__}: {error}"
        return dense_search_numpy_by_vector(query_vector, depth)


def dense_search(question: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    query_vector = _normalize_vector(embed_hcx_single(question))
    return dense_search_from_vector(query_vector, depth)

def bm25_search(question: str, depth: int = CANDIDATE_DEPTH) -> list[dict[str, Any]]:
    if depth < 1:
        raise ValueError("depth는 1 이상이어야 합니다.")
    response = ES.search(
        index=ES_INDEX_NAME,
        size=depth,
        query={"match": {"search_text": {"query": question}}},
    )
    results = []
    for rank, hit in enumerate(response["hits"]["hits"], start=1):
        chunk_id = str(hit["_source"]["chunk_id"])
        if chunk_id not in CHUNKS_BY_ID:
            raise RuntimeError(f"BM25 결과의 chunk_id가 원본에 없습니다: {chunk_id}")
        results.append({
            "chunk_id": chunk_id,
            "score": float(hit["_score"]),
            "rank": rank,
        })
    return results


def _minmax_by_chunk(results: list[dict[str, Any]]) -> dict[str, float]:
    if not results:
        return {}
    scores = np.asarray([float(row["score"]) for row in results], dtype=np.float64)
    low, high = float(scores.min()), float(scores.max())
    if abs(high - low) <= 1e-12:
        normalized = np.ones_like(scores)
    else:
        normalized = (scores - low) / (high - low)
    return {
        str(row["chunk_id"]): float(score)
        for row, score in zip(results, normalized)
    }


def weighted_minmax(
    dense_results: list[dict[str, Any]],
    bm25_results: list[dict[str, Any]],
    *,
    dense_weight: float = DENSE_WEIGHT,
    bm25_weight: float = BM25_WEIGHT,
    top_k: int = FINAL_TOP_K,
) -> list[dict[str, Any]]:
    if not math.isclose(dense_weight + bm25_weight, 1.0):
        raise ValueError("Dense/BM25 가중치 합은 1이어야 합니다.")
    dense_norm = _minmax_by_chunk(dense_results)
    bm25_norm = _minmax_by_chunk(bm25_results)
    dense_by_id = {str(row["chunk_id"]): row for row in dense_results}
    bm25_by_id = {str(row["chunk_id"]): row for row in bm25_results}
    candidates = []
    for chunk_id in sorted(set(dense_norm) | set(bm25_norm)):
        dense_row = dense_by_id.get(chunk_id)
        bm25_row = bm25_by_id.get(chunk_id)
        score = dense_weight * dense_norm.get(chunk_id, 0.0) + bm25_weight * bm25_norm.get(chunk_id, 0.0)
        candidates.append({
            "chunk_id": chunk_id,
            "minmax_score": float(score),
            "dense_rank": dense_row.get("rank") if dense_row else None,
            "dense_score": dense_row.get("score") if dense_row else None,
            "bm25_rank": bm25_row.get("rank") if bm25_row else None,
            "bm25_score": bm25_row.get("score") if bm25_row else None,
        })
    infinity = float("inf")
    ordered = sorted(candidates, key=lambda row: (
        -row["minmax_score"],
        row["dense_rank"] or infinity,
        row["bm25_rank"] or infinity,
        row["chunk_id"],
    ))
    return [
        {**row, "rank": rank, "chunk": CHUNKS_BY_ID[row["chunk_id"]]}
        for rank, row in enumerate(ordered[:top_k], start=1)
    ]


def hybrid_minmax_search(question: str, *, top_k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    cleaned = _clean_text(question)
    if not cleaned:
        raise ValueError("검색 질문이 비어 있습니다.")
    dense_results = dense_search(cleaned, CANDIDATE_DEPTH)
    bm25_results = bm25_search(cleaned, CANDIDATE_DEPTH)
    results = weighted_minmax(
        dense_results,
        bm25_results,
        dense_weight=DENSE_WEIGHT,
        bm25_weight=BM25_WEIGHT,
        top_k=top_k,
    )
    if not results:
        raise RuntimeError("Hybrid Min-Max 검색 결과가 없습니다.")
    return results


def fuse_query_results(
    plans: list[dict[str, Any]],
    *,
    top_k: int = FINAL_TOP_K,
    rrf_k: int = QUERY_FUSION_RRF_K,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if not plans:
        raise ValueError("검색 계획이 없습니다.")
    if not math.isclose(sum(float(plan["weight"]) for plan in plans), 1.0, abs_tol=1e-9):
        raise ValueError("검색 계획 가중치 합은 1이어야 합니다.")
    per_query = []
    fused: dict[str, dict[str, Any]] = {}
    for plan_index, plan in enumerate(plans, start=1):
        started = time.perf_counter()
        hits = hybrid_minmax_search(str(plan["query"]), top_k=CANDIDATE_DEPTH)
        elapsed_ms = (time.perf_counter() - started) * 1000
        per_query.append({**plan, "latency_ms": elapsed_ms, "hits": hits})
        for hit in hits:
            chunk_id = str(hit["chunk_id"])
            row = fused.setdefault(chunk_id, {
                "chunk_id": chunk_id,
                "query_fusion_score": 0.0,
                "best_minmax_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
                "matched_queries": [],
                "chunk": hit["chunk"],
            })
            row["query_fusion_score"] += float(plan["weight"]) / (rrf_k + int(hit["rank"]))
            row["best_minmax_score"] = max(row["best_minmax_score"], float(hit["minmax_score"]))
            if row["dense_rank"] is None or (hit["dense_rank"] is not None and hit["dense_rank"] < row["dense_rank"]):
                row["dense_rank"] = hit["dense_rank"]
            if row["bm25_rank"] is None or (hit["bm25_rank"] is not None and hit["bm25_rank"] < row["bm25_rank"]):
                row["bm25_rank"] = hit["bm25_rank"]
            row["matched_queries"].append({
                "plan_index": plan_index,
                "query": plan["query"],
                "source": plan["source"],
                "rank": hit["rank"],
                "weight": plan["weight"],
            })
    ordered = sorted(fused.values(), key=lambda row: (
        -row["query_fusion_score"], -row["best_minmax_score"], row["chunk_id"]
    ))
    final = []
    for rank, row in enumerate(ordered[:top_k], start=1):
        final.append({
            **row,
            "rank": rank,
            "minmax_score": row["best_minmax_score"],
        })
    return final, per_query


print("M3 Hybrid 7:3 Min-Max + Reranker 검색기 준비 완료")

### 3-1. 질문별 검색 세부 레이턴시

In [ ]:
# 상세 검색 레이턴시 버전으로 기존 함수를 재정의합니다.
_V15_LAST_QUERY_TRACE: dict[str, Any] = {}


def hybrid_minmax_search(question: str, *, top_k: int = FINAL_TOP_K) -> list[dict[str, Any]]:
    global _V15_LAST_QUERY_TRACE
    cleaned = _clean_text(question)
    if not cleaned:
        raise ValueError("검색 질문이 비어 있습니다.")
    total_started = time.perf_counter()

    embedding_started = time.perf_counter()
    query_vector = _normalize_vector(embed_hcx_single(cleaned))
    embedding_latency_ms = (time.perf_counter() - embedding_started) * 1000
    if query_vector.shape != (DENSE_DIMENSION,):
        raise RuntimeError(
            f"질문 임베딩 차원 불일치: query={query_vector.shape}, stored={DENSE_DIMENSION}"
        )

    dense_started = time.perf_counter()
    dense_results = dense_search_from_vector(query_vector, CANDIDATE_DEPTH)
    dense_compute_latency_ms = (time.perf_counter() - dense_started) * 1000

    bm25_started = time.perf_counter()
    bm25_results = bm25_search(cleaned, CANDIDATE_DEPTH)
    bm25_latency_ms = (time.perf_counter() - bm25_started) * 1000

    minmax_started = time.perf_counter()
    results = weighted_minmax(
        dense_results,
        bm25_results,
        dense_weight=DENSE_WEIGHT,
        bm25_weight=BM25_WEIGHT,
        top_k=top_k,
    )
    minmax_latency_ms = (time.perf_counter() - minmax_started) * 1000
    total_latency_ms = (time.perf_counter() - total_started) * 1000

    _V15_LAST_QUERY_TRACE = {
        "question": cleaned,
        "embedding_latency_ms": embedding_latency_ms,
        "dense_compute_latency_ms": dense_compute_latency_ms,
        "dense_backend_requested": DENSE_BACKEND,
        "dense_backend_used": _LAST_DENSE_BACKEND_USED,
        "dense_fallback_error": _LAST_DENSE_FALLBACK_ERROR,
        "dense_knn_num_candidates": DENSE_KNN_NUM_CANDIDATES,
        "bm25_latency_ms": bm25_latency_ms,
        "minmax_latency_ms": minmax_latency_ms,
        "query_total_latency_ms": total_latency_ms,
        "dense_candidate_count": len(dense_results),
        "bm25_candidate_count": len(bm25_results),
        "combined_candidate_count": len(results),
    }
    if not results:
        raise RuntimeError("Hybrid Min-Max 검색 결과가 없습니다.")
    return results


def fuse_query_results(
    plans: list[dict[str, Any]],
    *,
    top_k: int = FINAL_TOP_K,
    rrf_k: int = QUERY_FUSION_RRF_K,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if not plans:
        raise ValueError("검색 계획이 없습니다.")
    if not math.isclose(sum(float(plan["weight"]) for plan in plans), 1.0, abs_tol=1e-9):
        raise ValueError("검색 계획 가중치 합은 1이어야 합니다.")

    per_query = []
    all_hits = []
    for plan_index, plan in enumerate(plans, start=1):
        hits = hybrid_minmax_search(str(plan["query"]), top_k=CANDIDATE_DEPTH)
        trace = dict(_V15_LAST_QUERY_TRACE)
        per_query.append({
            **plan,
            "plan_index": plan_index,
            "latency_ms": trace["query_total_latency_ms"],
            "latency_breakdown_ms": trace,
            "hits": hits,
        })
        all_hits.append((plan_index, plan, hits))

    fusion_started = time.perf_counter()
    fused: dict[str, dict[str, Any]] = {}
    for plan_index, plan, hits in all_hits:
        for hit in hits:
            chunk_id = str(hit["chunk_id"])
            row = fused.setdefault(chunk_id, {
                "chunk_id": chunk_id,
                "query_fusion_score": 0.0,
                "best_minmax_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
                "matched_queries": [],
                "chunk": hit["chunk"],
            })
            row["query_fusion_score"] += float(plan["weight"]) / (rrf_k + int(hit["rank"]))
            row["best_minmax_score"] = max(row["best_minmax_score"], float(hit["minmax_score"]))
            if row["dense_rank"] is None or (hit["dense_rank"] is not None and hit["dense_rank"] < row["dense_rank"]):
                row["dense_rank"] = hit["dense_rank"]
            if row["bm25_rank"] is None or (hit["bm25_rank"] is not None and hit["bm25_rank"] < row["bm25_rank"]):
                row["bm25_rank"] = hit["bm25_rank"]
            row["matched_queries"].append({
                "plan_index": plan_index,
                "query": plan["query"],
                "source": plan["source"],
                "rank": hit["rank"],
                "weight": plan["weight"],
            })

    ordered = sorted(fused.values(), key=lambda row: (
        -row["query_fusion_score"], -row["best_minmax_score"], row["chunk_id"]
    ))
    final = [
        {
            **row,
            "rank": rank,
            "minmax_score": row["best_minmax_score"],
        }
        for rank, row in enumerate(ordered[:top_k], start=1)
    ]
    fusion_latency_ms = (time.perf_counter() - fusion_started) * 1000
    for row in per_query:
        row["query_fusion_latency_ms"] = fusion_latency_ms
    return final, per_query


print({
    "retrieval": "HYBRID_7_3_MINMAX",
    "dense_backend": DENSE_BACKEND,
    "dense_fallback": ALLOW_NUMPY_DENSE_FALLBACK,
    "dense_knn_num_candidates": DENSE_KNN_NUM_CANDIDATES,
    "query_fusion_rrf_k": QUERY_FUSION_RRF_K,
})


### 3-2. Elasticsearch Dense kNN 자체 검증

저장된 문서 벡터 하나를 probe로 사용하므로 HCX API를 추가 호출하지 않습니다. Elasticsearch kNN의 유효성, ID 일치 및 NumPy Exact Top-20과의 겹침을 확인합니다.


In [ ]:
def compare_dense_backends_by_vector(
    query_vector: np.ndarray,
    *,
    depth: int = CANDIDATE_DEPTH,
) -> dict[str, Any]:
    exact = dense_search_numpy_by_vector(query_vector, depth)
    knn = dense_search_by_vector(query_vector, depth)
    exact_ids = [row["chunk_id"] for row in exact]
    knn_ids = [row["chunk_id"] for row in knn]
    overlap_ids = sorted(set(exact_ids) & set(knn_ids))
    return {
        "depth": depth,
        "exact_count": len(exact_ids),
        "knn_count": len(knn_ids),
        "overlap_count": len(overlap_ids),
        "overlap_rate": len(overlap_ids) / max(1, len(exact_ids)),
        "exact_top5": exact_ids[:5],
        "knn_top5": knn_ids[:5],
    }


probe_chunk_id = DENSE_CHUNK_IDS[0]
probe_report = compare_dense_backends_by_vector(
    DENSE_VECTOR_BY_ID[probe_chunk_id],
    depth=min(CANDIDATE_DEPTH, len(CHUNKS)),
)
if probe_report["knn_count"] != min(CANDIDATE_DEPTH, len(CHUNKS)):
    raise RuntimeError(f"Dense kNN 후보 수가 부족합니다: {probe_report}")
if probe_chunk_id not in probe_report["knn_top5"]:
    raise RuntimeError(f"자기 문서 벡터가 ES kNN Top-5에 없습니다: {probe_report}")
if probe_report["overlap_rate"] < 0.80:
    raise RuntimeError(f"NumPy Exact와 ES kNN Top-20 겹침이 지나치게 낮습니다: {probe_report}")

display(probe_report)
print("Elasticsearch Dense kNN 자체 검증 통과")


### 3-2. BGE CrossEncoder Reranker

Hybrid 7:3 Min/Max와 다중질의 결합으로 만든 상위 20개 Child 후보를 같은 평가 실험에서 사용한 `BAAI/bge-reranker-v2-m3`로 재정렬합니다. 답변 D안에는 최종 Top-5만 전달합니다.


In [ ]:
%%writefile kdic_v15_context_rerank_core.py
from __future__ import annotations

import re
import time
from typing import Any, Callable, Iterable, Mapping, Sequence

import numpy as np


BUSINESS_LABELS = (
    "예금자보호제도",
    "예금보험금 안내",
    "고객 미수령금 신청",
    "착오송금 반환 신청",
    "채무조정 안내",
    "은닉재산 신고",
)

BUSINESS_ALIASES: dict[str, tuple[str, ...]] = {
    "예금자보호제도": (
        "예금자보호", "보호한도", "보호 대상", "보호대상",
    ),
    "예금보험금 안내": (
        "예금보험금", "보험금 지급", "보험사고",
    ),
    "고객 미수령금 신청": (
        "미수령금", "파산배당금", "개산지급금 정산금",
    ),
    "착오송금 반환 신청": (
        "착오송금", "잘못 송금", "잘못송금", "반환지원",
    ),
    "채무조정 안내": (
        "채무조정", "신용회복", "채무감면",
    ),
    "은닉재산 신고": (
        "은닉재산", "은닉 재산",
    ),
}

INTENT_ONLY_PATTERN = re.compile(
    r"(?:신청|접수|서류|구비서류|준비물|조건|자격|대상|방법|절차|"
    r"기간|기한|금액|한도|조회|상태|연락처|전화번호)"
)
REFERENCE_PATTERN = re.compile(
    r"(?:^|\s)(?:그거|이거|그것|이것|그\s*신청|해당\s*신청|그\s*경우|"
    r"해당\s*경우|그러면|그럼|거기는|거기서)(?:\s|$|[?!.])"
)
SELECTION_PATTERN = re.compile(r"^\s*(\d{1,2})\s*(?:번)?\s*$")


def _clean_text(value: Any) -> str:
    text = str(value or "").replace("\x00", " ")
    return re.sub(r"\s+", " ", text).strip()


def _ordered_unique(values: Iterable[str]) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for value in values:
        cleaned = _clean_text(value)
        if cleaned and cleaned not in seen:
            seen.add(cleaned)
            output.append(cleaned)
    return output


def detect_businesses(text: str) -> list[str]:
    cleaned = _clean_text(text).lower()
    return [
        business
        for business, aliases in BUSINESS_ALIASES.items()
        if any(alias.lower() in cleaned for alias in aliases)
    ]


def _user_messages(previous_turns: Any) -> list[str]:
    if not isinstance(previous_turns, Sequence) or isinstance(previous_turns, (str, bytes)):
        return []
    output: list[str] = []
    for turn in previous_turns:
        if not isinstance(turn, Mapping):
            continue
        role = _clean_text(turn.get("role")).lower()
        if role:
            if role != "user":
                continue
            content = _clean_text(turn.get("content"))
            if content:
                output.append(content)
            continue
        user = _clean_text(turn.get("user") or turn.get("query"))
        if user:
            output.append(user)
    return output


def latest_context_businesses(
    previous_turns: Any,
    *,
    confirmed_businesses: Sequence[str] | None = None,
    detector: Callable[[str], list[str]] = detect_businesses,
) -> list[str]:
    confirmed = _ordered_unique(confirmed_businesses or [])
    if confirmed:
        return confirmed
    for message in reversed(_user_messages(previous_turns)):
        found = _ordered_unique(detector(message))
        if found:
            return found
    return []


def _is_context_dependent(question: str) -> bool:
    compact_length = len(re.sub(r"\s+", "", question))
    short_intent_only = compact_length <= 30 and bool(INTENT_ONLY_PATTERN.search(question))
    has_reference = bool(REFERENCE_PATTERN.search(question))
    return short_intent_only or has_reference


def _clarification_message(candidates: Sequence[str], *, repeated: bool = False) -> str:
    candidates = _ordered_unique(candidates) or list(BUSINESS_LABELS)
    prefix = (
        "아직 어떤 업무를 말씀하시는지 확인하기 어렵습니다."
        if repeated
        else "어떤 업무에 관한 질문인지 확인이 필요합니다."
    )
    lines = [prefix, "", "아래에서 선택하거나 업무명을 직접 입력해 주세요.", ""]
    lines.extend(f"{index}. {business}" for index, business in enumerate(candidates, start=1))
    return "\n".join(lines)


def _pending_payload(
    *,
    original_question: str,
    candidates: Sequence[str],
    clarification_count: int,
) -> dict[str, Any]:
    return {
        "active": True,
        "original_question": original_question,
        "missing_slots": ["business_function"],
        "business_candidates": _ordered_unique(candidates) or list(BUSINESS_LABELS),
        "clarification_count": int(clarification_count),
    }


def resolve_conversational_question(
    question: str,
    *,
    previous_turns: Any = None,
    pending_clarification: Mapping[str, Any] | None = None,
    confirmed_businesses: Sequence[str] | None = None,
    detector: Callable[[str], list[str]] = detect_businesses,
) -> dict[str, Any]:
    """검색 전 문맥을 보수적으로 복원하거나 CLARIFY를 반환한다.

    업무를 추정할 근거가 하나로 수렴하지 않으면 검색을 허용하지 않는다.
    """
    started = time.perf_counter()
    original = _clean_text(question)
    if not original:
        raise ValueError("사용자 질문이 비어 있습니다.")

    explicit_businesses = _ordered_unique(detector(original))
    pending = dict(pending_clarification or {})
    pending_active = bool(pending.get("active"))

    if pending_active:
        candidates = _ordered_unique(pending.get("business_candidates") or BUSINESS_LABELS)
        selected: list[str] = []
        numeric = SELECTION_PATTERN.fullmatch(original)
        if numeric:
            index = int(numeric.group(1)) - 1
            if 0 <= index < len(candidates):
                selected = [candidates[index]]
        if not selected:
            selected = [item for item in explicit_businesses if item in candidates]
        if not selected and len(explicit_businesses) == 1:
            selected = explicit_businesses

        if len(selected) == 1:
            pending_question = _clean_text(pending.get("original_question"))
            is_short_selection = len(re.sub(r"\s+", "", original)) <= 20
            resolved = (
                f"{selected[0]} {pending_question}"
                if pending_question and is_short_selection
                else original
            )
            return {
                "route": "RETRIEVE",
                "original_question": original,
                "resolved_question": resolved,
                "context_used": True,
                "context_businesses": selected,
                "resolution_reason": "PENDING_CLARIFICATION_RESOLVED",
                "clarification_message": "",
                "pending_clarification": None,
                "latency_ms": (time.perf_counter() - started) * 1000,
            }

        count = int(pending.get("clarification_count") or 1) + 1
        return {
            "route": "CLARIFY",
            "original_question": original,
            "resolved_question": "",
            "context_used": False,
            "context_businesses": candidates,
            "resolution_reason": "PENDING_CLARIFICATION_UNRESOLVED",
            "clarification_message": _clarification_message(candidates, repeated=True),
            "pending_clarification": _pending_payload(
                original_question=_clean_text(pending.get("original_question")) or original,
                candidates=candidates,
                clarification_count=count,
            ),
            "escalation_recommended": count >= 2,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    if explicit_businesses:
        return {
            "route": "CONTINUE",
            "original_question": original,
            "resolved_question": original,
            "context_used": False,
            "context_businesses": explicit_businesses,
            "resolution_reason": "EXPLICIT_BUSINESS_IN_CURRENT_QUESTION",
            "clarification_message": "",
            "pending_clarification": None,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    if not _is_context_dependent(original):
        return {
            "route": "CONTINUE",
            "original_question": original,
            "resolved_question": original,
            "context_used": False,
            "context_businesses": [],
            "resolution_reason": "STANDALONE_OR_BASE_ROUTER_DECISION",
            "clarification_message": "",
            "pending_clarification": None,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    context_businesses = latest_context_businesses(
        previous_turns,
        confirmed_businesses=confirmed_businesses,
        detector=detector,
    )
    if len(context_businesses) == 1:
        return {
            "route": "RETRIEVE",
            "original_question": original,
            "resolved_question": f"{context_businesses[0]} {original}",
            "context_used": True,
            "context_businesses": context_businesses,
            "resolution_reason": "UNIQUE_PREVIOUS_BUSINESS",
            "clarification_message": "",
            "pending_clarification": None,
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    candidates = context_businesses or list(BUSINESS_LABELS)
    reason = "MULTIPLE_PREVIOUS_BUSINESSES" if len(context_businesses) > 1 else "BUSINESS_NOT_SPECIFIED"
    return {
        "route": "CLARIFY",
        "original_question": original,
        "resolved_question": "",
        "context_used": False,
        "context_businesses": context_businesses,
        "resolution_reason": reason,
        "clarification_message": _clarification_message(candidates),
        "pending_clarification": _pending_payload(
            original_question=original,
            candidates=candidates,
            clarification_count=1,
        ),
        "escalation_recommended": False,
        "latency_ms": (time.perf_counter() - started) * 1000,
    }


def _predict_scores(model: Any, pairs: list[list[str]], *, batch_size: int) -> np.ndarray:
    if hasattr(model, "predict"):
        raw = model.predict(
            pairs,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
    elif hasattr(model, "compute_score"):
        raw = model.compute_score(pairs, batch_size=batch_size, normalize=True)
    else:
        raise TypeError("Reranker 모델에 predict 또는 compute_score 메서드가 없습니다.")
    scores = np.atleast_1d(np.asarray(raw, dtype=np.float32)).reshape(-1)
    if len(scores) != len(pairs):
        raise RuntimeError(
            f"Reranker 점수 개수 불일치: pairs={len(pairs)}, scores={len(scores)}"
        )
    return scores


def rerank_candidates(
    question: str,
    candidates: Sequence[Mapping[str, Any]],
    *,
    chunks_by_id: Mapping[str, Mapping[str, Any]],
    model: Any,
    text_builder: Callable[[Mapping[str, Any]], str],
    candidate_depth: int = 20,
    final_top_k: int = 5,
    batch_size: int = 8,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """Hybrid 상위 후보를 CrossEncoder로 재정렬한다."""
    if candidate_depth < final_top_k or final_top_k < 1:
        raise ValueError("candidate_depth는 final_top_k 이상이어야 합니다.")
    started = time.perf_counter()
    prepared: list[dict[str, Any]] = []
    pairs: list[list[str]] = []
    seen: set[str] = set()
    for base_rank, item in enumerate(candidates[:candidate_depth], start=1):
        chunk_id = _clean_text(item.get("chunk_id"))
        if not chunk_id or chunk_id in seen:
            continue
        seen.add(chunk_id)
        chunk = chunks_by_id.get(chunk_id)
        if chunk is None:
            raise KeyError(f"Reranker 후보 청크가 corpus에 없습니다: {chunk_id}")
        passage = _clean_text(text_builder(chunk))
        if not passage:
            continue
        prepared.append({**dict(item), "pre_rerank_rank": base_rank})
        pairs.append([_clean_text(question), passage])

    if not prepared:
        raise RuntimeError("Reranker에 전달할 유효 후보가 없습니다.")
    scores = _predict_scores(model, pairs, batch_size=batch_size)
    scored = [
        {**row, "reranker_score": float(score)}
        for row, score in zip(prepared, scores)
    ]
    ordered = sorted(
        scored,
        key=lambda row: (
            -float(row["reranker_score"]),
            int(row["pre_rerank_rank"]),
            str(row["chunk_id"]),
        ),
    )
    final = [
        {**row, "rank": rank}
        for rank, row in enumerate(ordered[:final_top_k], start=1)
    ]
    return final, {
        "latency_ms": (time.perf_counter() - started) * 1000,
        "candidate_count": len(prepared),
        "returned_count": len(final),
        "batch_size": int(batch_size),
        "question": _clean_text(question),
    }


In [ ]:
import torch
from sentence_transformers import CrossEncoder

from kdic_v15_context_rerank_core import rerank_candidates

RERANKER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANKER_MODEL = CrossEncoder(
    RERANKER_MODEL_NAME,
    device=RERANKER_DEVICE,
    max_length=RERANKER_MAX_LENGTH,
)

_FUSE_QUERY_RESULTS_BEFORE_RERANKER = fuse_query_results
_LAST_RERANK_TRACE: dict[str, Any] = {}


def _reranker_passage(chunk: dict[str, Any]) -> str:
    return build_dense_structured_v2_text(chunk)[:4_000]


def fuse_query_results(
    plans: list[dict[str, Any]],
    *,
    top_k: int = FINAL_TOP_K,
    rrf_k: int = QUERY_FUSION_RRF_K,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    global _LAST_RERANK_TRACE
    # 먼저 Hybrid/다중질의 결합 상위 20개를 확보합니다.
    candidates, per_query = _FUSE_QUERY_RESULTS_BEFORE_RERANKER(
        plans,
        top_k=RERANKER_CANDIDATE_DEPTH,
        rrf_k=rrf_k,
    )
    original_plan = next(
        (plan for plan in plans if "ORIGINAL" in str(plan.get("source") or "")),
        plans[0],
    )
    rerank_question = str(original_plan.get("query") or "").strip()
    reranked, trace = rerank_candidates(
        rerank_question,
        candidates,
        chunks_by_id=CHUNKS_BY_ID,
        model=RERANKER_MODEL,
        text_builder=_reranker_passage,
        candidate_depth=RERANKER_CANDIDATE_DEPTH,
        final_top_k=top_k,
        batch_size=RERANKER_BATCH_SIZE,
    )
    _LAST_RERANK_TRACE = {
        **trace,
        "model": RERANKER_MODEL_NAME,
        "device": RERANKER_DEVICE,
    }
    for row in per_query:
        row["reranker_latency_ms"] = float(trace["latency_ms"])
        row["reranker_candidate_count"] = int(trace["candidate_count"])
    return reranked, per_query


print({
    "reranker": RERANKER_MODEL_NAME,
    "device": RERANKER_DEVICE,
    "candidate_depth": RERANKER_CANDIDATE_DEPTH,
    "final_top_k": FINAL_TOP_K,
})


### 3-3. Parent-Child Retrieval — Child 검색 후 Parent 문맥 확장

현재 검색 순위 자체는 **Child 청크**로 결정합니다.

```text
Hybrid 7:3 Min-Max
    ↓
다중질의 RRF
    ↓
BGE Reranker
    ↓
Top-5 Child 확정
    ↓
parent_doc_id 기준 Parent 확장
    ↓
Parent Evidence Pack
    ↓
Answer Skeleton → 최종 답변
```

이렇게 구성한 이유는 `Child만 vs Parent 확장` 실험에서 **검색 랭킹은 동일하게 유지하고, 답변 생성에 전달되는 문맥만 바꾸기 위해서**입니다.

- 검색 평가: 기존 Top-5 Child 기준 그대로 수행
- 답변 생성: 같은 Parent의 sibling 청크를 함께 전달
- 같은 Parent에서 Child가 여러 개 검색되면 Parent 문맥은 Evidence Pack에 한 번만 넣어 중복 토큰을 줄임
- `PARENT_CONTEXT_MAX_CHARS=None`은 전체 Parent 확장입니다.
- 이후 컨텍스트 길이 제어 실험을 할 때만 `PARENT_CONTEXT_MAX_CHARS`에 숫자를 넣으면 됩니다.


In [ ]:
import time

_LAST_PARENT_CHILD_TRACE: dict[str, Any] = {}


def _parent_id_for_chunk(chunk: dict[str, Any]) -> str:
    chunk_id = _clean_text(chunk.get("chunk_id"))
    return (
        _clean_text(chunk.get("parent_doc_id"))
        or _clean_text(chunk.get("document_id"))
        or chunk_id
    )


def _select_parent_context_chunks(
    parent_id: str,
    matched_child_ids: list[str],
    *,
    max_chars: int | None = PARENT_CONTEXT_MAX_CHARS,
) -> list[dict[str, Any]]:
    """Parent의 sibling 청크를 문서 순서대로 반환합니다.

    max_chars=None이면 전체 Parent를 사용합니다.
    숫자이면 matched child를 반드시 우선 포함하고, 가장 가까운 sibling부터
    예산 안에서 추가한 뒤 최종 출력은 원문 chunk_index 순서로 정렬합니다.
    """
    children = list(PARENT_CHILDREN_BY_ID.get(parent_id) or [])
    if not children:
        return []

    if max_chars is None:
        return children

    if max_chars <= 0:
        raise ValueError("PARENT_CONTEXT_MAX_CHARS는 None 또는 양수여야 합니다.")

    positions = {
        str(chunk.get("chunk_id") or ""): index
        for index, chunk in enumerate(children)
    }
    matched_positions = [
        positions[chunk_id]
        for chunk_id in matched_child_ids
        if chunk_id in positions
    ]
    if not matched_positions:
        matched_positions = [0]

    def distance(index: int) -> tuple[int, int]:
        return (min(abs(index - anchor) for anchor in matched_positions), index)

    priority = sorted(range(len(children)), key=distance)
    selected_indices: list[int] = []
    used_chars = 0

    # matched child는 예산보다 길더라도 최소 1개는 보존합니다.
    matched_set = set(matched_child_ids)
    for index in priority:
        child = children[index]
        chunk_id = str(child.get("chunk_id") or "")
        content_chars = len(_clean_text(child.get("content")))
        must_include = chunk_id in matched_set or not selected_indices
        if must_include or used_chars + content_chars <= max_chars:
            selected_indices.append(index)
            used_chars += content_chars

    selected_indices.sort()
    return [children[index] for index in selected_indices]


def expand_parent_context(
    search_results: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Reranker Top-K Child를 유지하면서 Parent Evidence 단위만 연결합니다."""
    global _LAST_PARENT_CHILD_TRACE
    started = time.perf_counter()

    if not PARENT_CHILD_ENABLED:
        output = []
        for result in search_results:
            row = dict(result)
            row["parent_doc_id"] = _parent_id_for_chunk(result["chunk"])
            row["parent_evidence_ref"] = f"C{int(result['rank'])}"
            row["parent_context_chunk_ids"] = [str(result["chunk_id"])]
            row["parent_context_chunk_count"] = 1
            row["parent_context_char_count"] = len(_clean_text(result["chunk"].get("content")))
            output.append(row)
        _LAST_PARENT_CHILD_TRACE = {
            "enabled": False,
            "latency_ms": (time.perf_counter() - started) * 1000,
            "matched_child_count": len(search_results),
            "unique_parent_count": len(search_results),
            "expanded_chunk_count": len(search_results),
            "expanded_char_count": sum(row["parent_context_char_count"] for row in output),
        }
        return output

    # Top-K 안에서 같은 Parent가 여러 번 검색되면 하나의 Evidence ref를 공유합니다.
    parent_order: list[str] = []
    matched_by_parent: dict[str, list[str]] = {}
    for result in search_results:
        chunk = result["chunk"]
        parent_id = _parent_id_for_chunk(chunk)
        if parent_id not in matched_by_parent:
            parent_order.append(parent_id)
            matched_by_parent[parent_id] = []
        matched_by_parent[parent_id].append(str(result["chunk_id"]))

    evidence_ref_by_parent = {
        parent_id: f"C{index}"
        for index, parent_id in enumerate(parent_order, start=1)
    }
    selected_by_parent: dict[str, list[dict[str, Any]]] = {
        parent_id: _select_parent_context_chunks(
            parent_id,
            matched_by_parent[parent_id],
            max_chars=PARENT_CONTEXT_MAX_CHARS,
        )
        for parent_id in parent_order
    }

    output: list[dict[str, Any]] = []
    for result in search_results:
        row = dict(result)
        parent_id = _parent_id_for_chunk(result["chunk"])
        selected = selected_by_parent[parent_id]
        row["parent_doc_id"] = parent_id
        row["parent_evidence_ref"] = evidence_ref_by_parent[parent_id]
        row["parent_context_chunk_ids"] = [
            str(chunk.get("chunk_id") or "")
            for chunk in selected
        ]
        row["parent_context_chunk_count"] = len(selected)
        row["parent_context_char_count"] = sum(
            len(_clean_text(chunk.get("content")))
            for chunk in selected
        )
        output.append(row)

    unique_selected = {
        (parent_id, str(chunk.get("chunk_id") or ""))
        for parent_id, chunks in selected_by_parent.items()
        for chunk in chunks
    }
    _LAST_PARENT_CHILD_TRACE = {
        "enabled": True,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "matched_child_count": len(search_results),
        "unique_parent_count": len(parent_order),
        "expanded_chunk_count": len(unique_selected),
        "expanded_char_count": sum(
            len(_clean_text(chunk.get("content")))
            for chunks in selected_by_parent.values()
            for chunk in chunks
        ),
        "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
        "parent_refs": evidence_ref_by_parent,
    }
    return output


def parent_child_markdown(search_results: list[dict[str, Any]]) -> str:
    if not search_results:
        return "### Parent-Child 확장\n\n검색 결과가 없습니다."
    seen: set[str] = set()
    lines = [
        "### Parent-Child 확장",
        "",
        "|Evidence|Parent|매칭 Child|확장 청크 수|확장 문자 수|",
        "|---|---|---|---:|---:|",
    ]
    for result in search_results:
        parent_id = str(result.get("parent_doc_id") or "")
        if parent_id in seen:
            continue
        seen.add(parent_id)
        ref = str(result.get("parent_evidence_ref") or "-")
        matched = [
            str(row["chunk_id"])
            for row in search_results
            if str(row.get("parent_doc_id") or "") == parent_id
        ]
        lines.append(
            f"|{ref}|{parent_id}|{', '.join(matched)}|"
            f"{int(result.get('parent_context_chunk_count') or 0)}|"
            f"{int(result.get('parent_context_char_count') or 0):,}|"
        )
    return "\n".join(lines)


print({
    "parent_child_enabled": PARENT_CHILD_ENABLED,
    "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
    "parent_count": len(PARENT_CHILDREN_BY_ID),
})


## 4. 공통 구조화 답변 B

Basic Evidence Pack은 프로그램으로 만들고 HCX-005에는 동일한 Pack만 전달합니다. 기본답변은 결론·요구별 설명·절차·비교·조건을 질문에 맞춰 구조화합니다. 내부 Evidence ID는 검증에 사용하지만 사용자 화면에서는 숨깁니다.

In [ ]:
%%writefile kdic_v15_answer_b_core.py
from __future__ import annotations

"""KDIC 답변 B v2: 기본 답변과 동일 Evidence Pack 기반 근거 상세설명."""

import hashlib
import json
import re
import time
from collections import OrderedDict
from typing import Any, Callable, Mapping, Sequence


ALLOWED_COVERAGE_STATUS = {"SUFFICIENT", "PARTIAL", "INSUFFICIENT"}

BASIC_ANSWER_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "answer", "used_evidence_ids", "used_chunk_ids",
        "coverage_status", "missing_information",
    ],
    "properties": {
        "answer": {"type": "string", "minLength": 1},
        "used_evidence_ids": {
            "type": "array", "minItems": 1, "items": {"type": "string"},
        },
        "used_chunk_ids": {
            "type": "array", "minItems": 1, "items": {"type": "string"},
        },
        "coverage_status": {
            "type": "string", "enum": ["SUFFICIENT", "PARTIAL", "INSUFFICIENT"],
        },
        "missing_information": {"type": "array", "items": {"type": "string"}},
    },
}

EVIDENCE_EXPLANATION_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "explanation_summary", "claim_evidence_map", "conditions",
        "exceptions", "limitations", "additional_information_needed",
    ],
    "properties": {
        "explanation_summary": {"type": "string", "minLength": 1},
        "claim_evidence_map": {
            "type": "array",
            "minItems": 1,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["claim", "evidence_ids", "chunk_ids", "relevance_reason"],
                "properties": {
                    "claim": {"type": "string", "minLength": 1},
                    "evidence_ids": {
                        "type": "array", "minItems": 1, "items": {"type": "string"},
                    },
                    "chunk_ids": {
                        "type": "array", "minItems": 1, "items": {"type": "string"},
                    },
                    "relevance_reason": {"type": "string", "minLength": 1},
                },
            },
        },
        "conditions": {"type": "array", "items": {"type": "string"}},
        "exceptions": {"type": "array", "items": {"type": "string"}},
        "limitations": {"type": "array", "items": {"type": "string"}},
        "additional_information_needed": {
            "type": "array", "items": {"type": "string"},
        },
    },
}

BASIC_ANSWER_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 답변 시스템입니다.

반드시 지킬 규칙:
1. 사용자 질문과 Basic Evidence Pack에 있는 내용만 사용합니다.
2. 질문에 대한 결론을 먼저 제시하는 일반적인 기본 답변을 작성합니다.
3. 필요한 조건·금액·기간·절차·예외를 질문 범위 안에서 포함합니다.
4. 서로 다른 제도나 대상을 임의로 결합하지 않습니다.
5. Evidence에 없는 사실·URL·전화번호·해석을 추가하지 않습니다.
6. 사용한 문장 끝에 [E1] 형식으로 Evidence ID를 표시합니다.
7. 특정 값을 판단할 정보가 부족하면 추측하지 말고 missing_information에 기록합니다.
8. 지정된 JSON 객체 하나만 출력합니다.
9. JSON 문자열 내부의 줄바꿈·탭은 실제 제어문자가 아니라 \\n·\\t로 이스케이프합니다.
""".strip()

EVIDENCE_EXPLANATION_SYSTEM_PROMPT = """
당신은 예금보험공사 답변의 문서 근거를 설명하는 시스템입니다.

반드시 지킬 규칙:
1. 기본 답변 생성에 사용한 동일한 Basic Evidence Pack만 사용합니다.
2. 기본 답변의 핵심 주장과 Evidence ID·Chunk ID의 연결 관계를 설명합니다.
3. 각 Evidence가 질문과 해당 주장에 관련되는 이유를 문서 내용 기준으로 설명합니다.
4. 적용 조건·예외·근거 한계·추가 필요 정보를 구분합니다.
5. 기본 답변과 모순되는 새로운 결론을 만들지 않습니다.
6. 모델의 숨겨진 사고과정이나 내부 추론을 서술하지 않습니다.
7. Evidence에서 사용자가 확인할 수 있는 근거 관계만 설명합니다.
8. Evidence에 없는 사실·URL·전화번호를 추가하지 않습니다.
9. 지정된 JSON 객체 하나만 출력합니다.
10. JSON 문자열 내부의 줄바꿈·탭은 실제 제어문자가 아니라 \\n·\\t로 이스케이프합니다.
""".strip()


def _clean(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _clean_list(value: Any) -> list[str]:
    values = value if isinstance(value, list) else [value] if value not in (None, "") else []
    output: list[str] = []
    for item in values:
        text = _clean(item)
        if text and text not in output:
            output.append(text)
    return output


def _strip_model_urls(text: str) -> str:
    text = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", text)
    return re.sub(r"https?://[^\s)\]}>]+", "", text).strip()


def _escape_control_chars_inside_json_strings(text: str) -> str:
    """JSON 문자열 리터럴 안의 비이스케이프 제어문자만 안전하게 교정한다.

    객체 필드 사이의 정상 줄바꿈은 그대로 두고, 따옴표 안의 LF/CR/TAB 및
    U+0000~U+001F만 JSON 표준 이스케이프 형태로 바꾼다.
    """
    output: list[str] = []
    in_string = False
    escaped = False
    for character in str(text or ""):
        if not in_string:
            output.append(character)
            if character == '"':
                in_string = True
            continue

        if escaped:
            output.append(character)
            escaped = False
            continue
        if character == "\\":
            output.append(character)
            escaped = True
            continue
        if character == '"':
            output.append(character)
            in_string = False
            continue
        if character == "\n":
            output.append("\\n")
        elif character == "\r":
            output.append("\\r")
        elif character == "\t":
            output.append("\\t")
        elif ord(character) < 0x20:
            output.append(f"\\u{ord(character):04x}")
        else:
            output.append(character)
    return "".join(output)


def _json_candidates(text: str) -> list[str]:
    cleaned = str(text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    backslash_repaired = re.sub(
        r'\\(?!["\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\', cleaned
    )
    candidates = [
        cleaned,
        backslash_repaired,
        _escape_control_chars_inside_json_strings(cleaned),
        _escape_control_chars_inside_json_strings(backslash_repaired),
    ]
    return list(dict.fromkeys(candidates))


def _extract_json_object(text: str) -> dict[str, Any]:
    last_error: Exception | None = None
    for candidate in _json_candidates(text):
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError as error:
            last_error = error
            start = candidate.find("{")
            if start < 0:
                continue
            try:
                parsed, _ = json.JSONDecoder().raw_decode(candidate[start:])
            except json.JSONDecodeError as nested:
                last_error = nested
                continue
        if not isinstance(parsed, dict):
            raise TypeError("구조화 답변의 최상위 값은 JSON 객체여야 합니다.")
        return parsed
    raise ValueError(f"구조화 답변 JSON 파싱 실패: {last_error}") from last_error


def _decode_raw_answer_text(text: str) -> str:
    """객체 복구가 불가능할 때 JSON 문자열 또는 일반 본문만 보수적으로 꺼낸다."""
    cleaned = str(text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    for candidate in _json_candidates(cleaned):
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, str):
            return parsed.strip()
        if isinstance(parsed, Mapping) and _clean(parsed.get("answer")):
            return str(parsed.get("answer")).strip()
    if cleaned.startswith("{"):
        return ""
    return cleaned.strip('"').strip()


def build_basic_evidence_pack(
    question: str,
    search_results: Sequence[Mapping[str, Any]],
) -> dict[str, Any]:
    """LLM 없이 검색 근거의 경계와 출처를 결정적으로 정돈한다."""
    evidence: list[dict[str, Any]] = []
    sources_by_url: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for index, result in enumerate(search_results, start=1):
        chunk = result.get("chunk") or result.get("original_chunk") or {}
        if not isinstance(chunk, Mapping):
            raise TypeError(f"검색 결과 {index}의 chunk가 객체가 아닙니다.")
        rank = int(result.get("rank") or index)
        chunk_id = _clean(result.get("chunk_id") or chunk.get("chunk_id"))
        if not chunk_id:
            raise ValueError(f"검색 결과 {index}에 chunk_id가 없습니다.")
        source_url = _clean(chunk.get("source_url"))
        source_id = None
        if source_url:
            if source_url not in sources_by_url:
                sources_by_url[source_url] = {
                    "source_id": f"S{len(sources_by_url) + 1}",
                    "title": _clean(chunk.get("title") or chunk.get("document_title")),
                    "source_url": source_url,
                }
            source_id = sources_by_url[source_url]["source_id"]
        evidence.append({
            "evidence_id": f"E{index}",
            "rank": rank,
            "chunk_id": chunk_id,
            "parent_id": _clean(result.get("parent_id") or chunk.get("parent_doc_id")) or None,
            "context_chunk_ids": list(result.get("context_chunk_ids") or chunk.get("context_chunk_ids") or [chunk_id]),
            "document_title": _clean(chunk.get("title") or chunk.get("document_title")),
            "section_title": _clean(chunk.get("section_title")),
            "content": _clean(chunk.get("content")),
            "source_id": source_id,
            "source_url": source_url,
        })
    if not evidence:
        raise ValueError("Basic Evidence Pack을 만들 검색 결과가 없습니다.")
    return {
        "question": _clean(question),
        "evidence": evidence,
        "sources": list(sources_by_url.values()),
    }


def evidence_pack_sha256(pack: Mapping[str, Any]) -> str:
    raw = json.dumps(pack, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _allowed_evidence(pack: Mapping[str, Any]) -> dict[str, str]:
    return {
        str(row["evidence_id"]): str(row["chunk_id"])
        for row in pack.get("evidence") or []
    }


def _allowed_chunks_by_evidence(pack: Mapping[str, Any]) -> dict[str, set[str]]:
    output: dict[str, set[str]] = {}
    for row in pack.get("evidence") or []:
        evidence_id = str(row["evidence_id"])
        values = {str(row["chunk_id"])}
        values.update(str(item) for item in row.get("context_chunk_ids") or [] if str(item))
        output[evidence_id] = values
    return output


def validate_basic_answer(
    payload: Mapping[str, Any],
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    answer = _strip_model_urls(_clean(payload.get("answer")))
    if not answer:
        raise ValueError("기본 답변 본문이 비어 있습니다.")
    allowed = _allowed_evidence(pack)
    requested_ids = _clean_list(payload.get("used_evidence_ids"))
    invalid_ids = [item for item in requested_ids if item not in allowed]
    if invalid_ids:
        raise ValueError(f"Evidence Pack에 없는 Evidence ID: {invalid_ids}")
    used_ids = list(requested_ids)
    for number in re.findall(r"\[E(\d+)\]", answer):
        evidence_id = f"E{number}"
        if evidence_id in allowed and evidence_id not in used_ids:
            used_ids.append(evidence_id)
    if not used_ids:
        raise ValueError("기본 답변에 유효한 Evidence ID가 없습니다.")
    allowed_chunks = _allowed_chunks_by_evidence(pack)
    permitted_chunks = set().union(*(allowed_chunks[item] for item in used_ids))
    model_chunks = _clean_list(payload.get("used_chunk_ids"))
    if not model_chunks or not set(model_chunks).issubset(permitted_chunks):
        raise ValueError("기본 답변의 Chunk ID가 사용 Evidence와 일치하지 않습니다.")
    coverage = _clean(payload.get("coverage_status")).upper()
    if coverage not in ALLOWED_COVERAGE_STATUS:
        raise ValueError(f"허용되지 않은 coverage_status: {coverage}")
    return {
        "answer": answer,
        "used_evidence_ids": used_ids,
        "used_chunk_ids": model_chunks,
        "coverage_status": coverage,
        "missing_information": _clean_list(payload.get("missing_information")),
    }


def validate_evidence_explanation(
    payload: Mapping[str, Any],
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    summary = _strip_model_urls(_clean(payload.get("explanation_summary")))
    if not summary:
        raise ValueError("근거 상세설명의 요약이 비어 있습니다.")
    allowed = _allowed_evidence(pack)
    allowed_chunks = _allowed_chunks_by_evidence(pack)
    mappings: list[dict[str, Any]] = []
    for index, raw in enumerate(payload.get("claim_evidence_map") or [], start=1):
        if not isinstance(raw, Mapping):
            raise TypeError(f"claim_evidence_map {index}가 객체가 아닙니다.")
        claim = _strip_model_urls(_clean(raw.get("claim")))
        reason = _strip_model_urls(_clean(raw.get("relevance_reason")))
        evidence_ids = _clean_list(raw.get("evidence_ids"))
        if not claim or not reason or not evidence_ids:
            raise ValueError(f"claim_evidence_map {index}의 필수값이 비었습니다.")
        invalid = [item for item in evidence_ids if item not in allowed]
        if invalid:
            raise ValueError(f"상세설명에 Pack 밖의 Evidence ID가 있습니다: {invalid}")
        permitted_chunks = set().union(*(allowed_chunks[item] for item in evidence_ids))
        model_chunks = _clean_list(raw.get("chunk_ids"))
        if not model_chunks or not set(model_chunks).issubset(permitted_chunks):
            raise ValueError("상세설명의 Chunk ID가 Evidence ID와 일치하지 않습니다.")
        mappings.append({
            "claim": claim,
            "evidence_ids": evidence_ids,
            "chunk_ids": model_chunks,
            "relevance_reason": reason,
        })
    if not mappings:
        raise ValueError("유효한 주장-Evidence 연결이 없습니다.")
    return {
        "explanation_summary": summary,
        "claim_evidence_map": mappings,
        "conditions": _clean_list(payload.get("conditions")),
        "exceptions": _clean_list(payload.get("exceptions")),
        "limitations": _clean_list(payload.get("limitations")),
        "additional_information_needed": _clean_list(payload.get("additional_information_needed")),
    }


def _structured_output_is_unsupported(error: Exception) -> bool:
    if type(error).__name__ != "BadRequestError":
        return False
    message = str(error).lower()
    return any(marker in message for marker in (
        "response_format", "json_schema", "json_object", "unsupported",
        "not supported", "convert error",
    ))


def _structured_output_capability_cache(client: Any) -> dict[str, bool]:
    """동일 HCX client에서 확인한 response_format 지원 여부를 보존한다."""
    attribute = "_kdic_structured_output_capability"
    cache = getattr(client, attribute, None)
    if isinstance(cache, dict):
        return cache
    cache = {}
    try:
        setattr(client, attribute, cache)
    except Exception:
        # 일부 client wrapper가 속성 설정을 막아도 정답 생성은 계속한다.
        pass
    return cache


def _call_model(
    *, client: Any, model: str, system_prompt: str, user_prompt: str,
    max_tokens: int, response_format: Mapping[str, Any] | None,
) -> tuple[str, dict[str, int], float]:
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.0,
        "max_tokens": max_tokens,
    }
    if response_format is not None:
        kwargs["response_format"] = dict(response_format)
    started = time.perf_counter()
    response = client.chat.completions.create(**kwargs)
    latency_ms = (time.perf_counter() - started) * 1000
    content = response.choices[0].message.content
    if not content or not str(content).strip():
        raise RuntimeError("HCX 구조화 답변 출력이 비어 있습니다.")
    usage_obj = getattr(response, "usage", None)
    usage = {
        key: int(getattr(usage_obj, key, 0) or 0)
        for key in ("prompt_tokens", "completion_tokens", "total_tokens")
    }
    return str(content), usage, latency_ms


def _call_structured(
    *, client: Any, model: str, system_prompt: str, user_prompt: str,
    schema_name: str, schema: Mapping[str, Any], max_tokens: int,
    validator: Callable[[Mapping[str, Any]], dict[str, Any]],
    raw_recovery: Callable[[str], dict[str, Any]] | None = None,
) -> tuple[dict[str, Any], dict[str, int], float, list[dict[str, Any]]]:
    capability_cache = _structured_output_capability_cache(client)
    if capability_cache.get(model) is False:
        formats: list[tuple[str, Mapping[str, Any] | None]] = [("prompt", None)]
    else:
        formats = [
            ("json_schema", {
                "type": "json_schema",
                "json_schema": {
                    "name": schema_name, "strict": True, "schema": dict(schema),
                },
            }),
            ("json_object", {"type": "json_object"}),
            ("prompt", None),
        ]
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    total_latency_ms = 0.0
    attempts: list[dict[str, Any]] = []
    raw_outputs: list[str] = []
    unsupported_formats: set[str] = set()
    for format_name, response_format in formats:
        previous_output = ""
        for repair_index in range(2):
            prompt = user_prompt if repair_index == 0 else f"""
[작업]
직전 출력은 JSON 파싱 또는 Evidence 검증에 실패했습니다.
원래 Evidence의 사실 범위를 바꾸지 말고 지정된 JSON 객체로 한 번만 교정하세요.

[원래 요청]
{user_prompt}

[직전 출력]
{previous_output[:6000]}
""".strip()
            try:
                raw, usage, latency_ms = _call_model(
                    client=client, model=model, system_prompt=system_prompt,
                    user_prompt=prompt, max_tokens=max_tokens,
                    response_format=response_format,
                )
            except Exception as error:
                if response_format is not None and _structured_output_is_unsupported(error):
                    unsupported_formats.add(format_name)
                    if {"json_schema", "json_object"}.issubset(unsupported_formats):
                        capability_cache[model] = False
                    attempts.append({
                        "format": format_name, "repair_index": repair_index,
                        "valid": False, "fallback_reason": "STRUCTURED_OUTPUT_UNSUPPORTED",
                        "error": f"{type(error).__name__}: {error}",
                    })
                    break
                raise
            raw_outputs.append(raw)
            if response_format is not None:
                capability_cache[model] = True
            total_latency_ms += latency_ms
            for key in total_usage:
                total_usage[key] += int(usage.get(key) or 0)
            try:
                validated = validator(_extract_json_object(raw))
            except (ValueError, TypeError) as error:
                previous_output = raw
                attempts.append({
                    "format": format_name, "repair_index": repair_index,
                    "valid": False, "error": f"{type(error).__name__}: {error}",
                    "raw_output_preview": raw[:2000], "latency_ms": latency_ms,
                })
                continue
            attempts.append({
                "format": format_name, "repair_index": repair_index,
                "valid": True, "latency_ms": latency_ms,
            })
            return validated, total_usage, total_latency_ms, attempts
    if raw_recovery is not None:
        recovery_errors: list[str] = []
        for raw in reversed(list(dict.fromkeys(raw_outputs))):
            try:
                recovered = raw_recovery(raw)
            except (ValueError, TypeError) as error:
                recovery_errors.append(f"{type(error).__name__}: {error}")
                continue
            attempts.append({
                "format": "raw_text_recovery", "repair_index": None,
                "valid": True, "fallback_reason": "STRUCTURED_METADATA_RECOVERED",
                "latency_ms": 0.0,
            })
            return recovered, total_usage, total_latency_ms, attempts
        if recovery_errors:
            attempts.append({
                "format": "raw_text_recovery", "repair_index": None,
                "valid": False, "errors": recovery_errors,
            })
    raise ValueError(
        "HCX 구조화 답변 생성 실패. attempts="
        + json.dumps(attempts, ensure_ascii=False, default=str)
    )


def _recover_basic_answer_from_raw(
    text: str,
    evidence_pack: Mapping[str, Any],
) -> dict[str, Any]:
    """본문과 명시적 [E#]가 있을 때만 기본 답변을 보수적으로 복구한다."""
    answer = _decode_raw_answer_text(text)
    if not answer:
        raise ValueError("복구 가능한 기본 답변 본문이 없습니다.")
    allowed = _allowed_evidence(evidence_pack)
    used_ids: list[str] = []
    for number in re.findall(r"\[E(\d+)\]", answer):
        evidence_id = f"E{int(number)}"
        if evidence_id in allowed and evidence_id not in used_ids:
            used_ids.append(evidence_id)
    if not used_ids:
        raise ValueError("본문에 Evidence Pack과 일치하는 [E#] 인용이 없습니다.")
    payload = {
        "answer": answer,
        "used_evidence_ids": used_ids,
        "used_chunk_ids": [allowed[evidence_id] for evidence_id in used_ids],
        "coverage_status": "PARTIAL",
        "missing_information": [],
    }
    return validate_basic_answer(payload, evidence_pack)


def generate_basic_answer_b_v2(
    *, client: Any, model: str, question: str, evidence_pack: Mapping[str, Any],
) -> dict[str, Any]:
    prompt = f"""
[사용자 질문]
{_clean(question)}

[Basic Evidence Pack JSON]
{json.dumps(evidence_pack, ensure_ascii=False, indent=2)}

[출력 JSON]
{{
  "answer": "사용자에게 바로 제시할 기본 답변. 근거 문장에 [E1] 표시",
  "used_evidence_ids": ["E1"],
  "used_chunk_ids": ["실제 대표 chunk_id"],
  "coverage_status": "SUFFICIENT | PARTIAL | INSUFFICIENT",
  "missing_information": ["근거만으로 확인할 수 없는 필수 정보"]
}}
""".strip()
    validated, usage, latency_ms, attempts = _call_structured(
        client=client, model=model, system_prompt=BASIC_ANSWER_SYSTEM_PROMPT,
        user_prompt=prompt, schema_name="kdic_basic_answer_b_v2",
        schema=BASIC_ANSWER_SCHEMA, max_tokens=1600,
        validator=lambda payload: validate_basic_answer(payload, evidence_pack),
        raw_recovery=lambda raw: _recover_basic_answer_from_raw(raw, evidence_pack),
    )
    return {
        **validated, "mode": "basic",
        "evidence_pack_sha256": evidence_pack_sha256(evidence_pack),
        "latency_ms": latency_ms, "usage": usage, "format_attempts": attempts,
    }


def generate_evidence_explanation_b_v2(
    *, client: Any, model: str, question: str, evidence_pack: Mapping[str, Any],
    basic_answer: Mapping[str, Any],
) -> dict[str, Any]:
    pack_hash = evidence_pack_sha256(evidence_pack)
    if str(basic_answer.get("evidence_pack_sha256")) != pack_hash:
        raise ValueError("기본 답변과 근거 상세설명의 Evidence Pack이 다릅니다.")
    basic_view = {
        key: basic_answer.get(key)
        for key in (
            "answer", "used_evidence_ids", "used_chunk_ids",
            "coverage_status", "missing_information",
        )
    }
    prompt = f"""
[사용자 질문]
{_clean(question)}

[이미 생성된 기본 답변]
{json.dumps(basic_view, ensure_ascii=False, indent=2)}

[동일 Basic Evidence Pack JSON]
{json.dumps(evidence_pack, ensure_ascii=False, indent=2)}

[출력 JSON]
{{
  "explanation_summary": "기본 답변이 어떤 문서 근거로 구성됐는지 요약",
  "claim_evidence_map": [
    {{
      "claim": "기본 답변의 핵심 주장",
      "evidence_ids": ["E1"],
      "chunk_ids": ["실제 대표 chunk_id"],
      "relevance_reason": "해당 Evidence가 질문과 주장에 관련되는 문서상 이유"
    }}
  ],
  "conditions": ["근거에 명시된 적용 조건"],
  "exceptions": ["근거에 명시된 예외"],
  "limitations": ["현재 근거로 단정할 수 없는 범위"],
  "additional_information_needed": ["개별 판단에 추가로 필요한 정보"]
}}
""".strip()
    validated, usage, latency_ms, attempts = _call_structured(
        client=client, model=model, system_prompt=EVIDENCE_EXPLANATION_SYSTEM_PROMPT,
        user_prompt=prompt, schema_name="kdic_evidence_explanation_b_v2",
        schema=EVIDENCE_EXPLANATION_SCHEMA, max_tokens=2200,
        validator=lambda payload: validate_evidence_explanation(payload, evidence_pack),
    )
    return {
        **validated, "mode": "evidence_explanation",
        "evidence_pack_sha256": pack_hash,
        "latency_ms": latency_ms, "usage": usage, "format_attempts": attempts,
    }


def build_used_sources(
    evidence_pack: Mapping[str, Any],
    payload: Mapping[str, Any],
) -> list[dict[str, Any]]:
    used = set(payload.get("used_evidence_ids") or [])
    if not used:
        for item in payload.get("claim_evidence_map") or []:
            used.update(item.get("evidence_ids") or [])
    sources: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for row in evidence_pack.get("evidence") or []:
        if row.get("evidence_id") not in used:
            continue
        url = _clean(row.get("source_url"))
        if not url:
            continue
        target = sources.setdefault(url, {
            "url": url,
            "title": _clean(row.get("document_title")) or "공식 출처",
            "evidence_ids": [],
            "chunk_ids": [],
        })
        target["evidence_ids"].append(row["evidence_id"])
        target["chunk_ids"].append(row["chunk_id"])
    return list(sources.values())


def evidence_explanation_to_markdown(payload: Mapping[str, Any]) -> str:
    lines = ["#### 답변 근거 설명", "", _clean(payload.get("explanation_summary")), ""]
    lines.extend(["##### 답변 주장과 문서 근거", ""])
    for index, item in enumerate(payload.get("claim_evidence_map") or [], start=1):
        evidence = ", ".join(item.get("evidence_ids") or [])
        chunks = ", ".join(item.get("chunk_ids") or [])
        lines.extend([
            f"{index}. **{_clean(item.get('claim'))}**",
            f"   - 사용 근거: {evidence} · `{chunks}`",
            f"   - 관련 이유: {_clean(item.get('relevance_reason'))}",
            "",
        ])
    for title, key in (
        ("적용 조건", "conditions"),
        ("예외", "exceptions"),
        ("현재 근거의 한계", "limitations"),
        ("추가로 필요한 정보", "additional_information_needed"),
    ):
        values = _clean_list(payload.get(key))
        if values:
            lines.extend([f"##### {title}", ""])
            lines.extend(f"- {value}" for value in values)
            lines.append("")
    return "\n".join(lines).strip()

In [ ]:

from __future__ import annotations

import random
from collections import OrderedDict
from types import SimpleNamespace

import kdic_v15_answer_b_core as answer_b_core
from kdic_v15_answer_b_core import (
    build_used_sources,
    evidence_explanation_to_markdown,
    evidence_pack_sha256,
    generate_basic_answer_b_v2,
    generate_evidence_explanation_b_v2,
)


# D안 최종답변 프롬프트의 근거 제한과 설명 원칙을 B안 입력 구조에 맞게 이전합니다.
# Answer Skeleton은 생성하지 않으며, Basic Evidence Pack에서 기본답변을 직접 만듭니다.
answer_b_core.BASIC_ANSWER_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 질의응답 시스템입니다.

반드시 지킬 규칙:
1. 사용자 질문과 Basic Evidence Pack에 적힌 사실만 사용하여 한국어로 답하세요.
2. 질문에 먼저 직접 답한 뒤 필요한 대상·조건·예외·금액·기간·절차를 설명하세요.
3. Basic Evidence Pack에 없는 사실을 추정하거나 일반상식으로 보완하지 마세요.
4. 서로 다른 대상이나 제도의 내용을 임의로 결합하지 마세요.
5. 청크 사이에 차이가 있으면 한쪽을 임의로 선택하지 말고 확인 가능한 차이를 설명하세요.
6. 근거가 부족한 필수 내용은 추측하지 말고 missing_information에 기록하세요.
7. URL, 전화번호, 추천 질문, 추천 키워드를 답변 본문에 작성하지 마세요.
8. 답변을 짧게 줄이는 것보다 사용자가 이해할 수 있게 충분히 설명하는 것을 우선하세요.
9. 전문용어는 공식 용어를 사용하되 같은 문장이나 다음 문장에서 쉽게 풀어 설명하세요.
10. 일반 조건과 예외를 분리하고, 절차는 Evidence에 순서가 있을 때 번호로 설명하세요.
11. 근거를 사용한 문장 끝에 [E1] 형식으로 실제 evidence_id를 표시하세요.
12. 검색 점수, Basic Evidence Pack, JSON, 내부 구현을 답변 본문에서 언급하지 마세요.
13. 지정된 JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()

answer_b_core.EVIDENCE_EXPLANATION_SYSTEM_PROMPT = """
당신은 예금보험공사 기본답변의 공식 문서 근거를 설명하는 시스템입니다.

반드시 지킬 규칙:
1. 기본답변 생성에 사용한 것과 SHA-256이 동일한 Basic Evidence Pack만 사용하세요.
2. 기본답변의 핵심 주장과 Evidence ID·Chunk ID 연결을 문서 내용 기준으로 설명하세요.
3. 왜 해당 Evidence가 질문과 주장에 관련되는지 사용자가 확인 가능한 문장으로 설명하세요.
4. 적용 조건·예외·근거 한계·추가 필요 정보를 구분하세요.
5. 기본답변과 모순되는 새 결론이나 Evidence에 없는 사실을 추가하지 마세요.
6. 모델의 숨겨진 사고과정이나 내부 추론을 출력하지 마세요.
7. URL과 전화번호를 생성하지 마세요. 출처는 프로그램이 별도로 표시합니다.
8. 검색 점수나 내부 구현을 설명하지 마세요.
9. 지정된 JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()


ANSWER_API_MIN_INTERVAL_SECONDS = 2.0
ANSWER_API_MAX_ATTEMPTS = 5
_LAST_ANSWER_API_TRACE: dict[str, Any] = {}


def _header_seconds(value: Any) -> float | None:
    text = str(value or "").strip().lower()
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", text)
    return float(match.group(1)) if match else None


def _answer_retry_delay(error: Exception, attempt: int) -> float:
    response = getattr(error, "response", None)
    headers = getattr(response, "headers", None) or {}
    values = []
    for key in (
        "retry-after", "Retry-After",
        "x-ratelimit-reset-requests", "x-ratelimit-reset-tokens",
    ):
        seconds = _header_seconds(headers.get(key))
        if seconds is not None:
            values.append(seconds)
    if values:
        return min(120.0, max(1.0, max(values)) + random.uniform(0.1, 0.9))
    return min(60.0, 5.0 * (2 ** max(0, attempt - 1)) + random.uniform(0.1, 1.0))


class _RateLimitedAnswerCompletions:
    def __init__(self, base_completions: Any):
        self.base_completions = base_completions
        self.last_call_at = 0.0

    def create(self, **kwargs: Any) -> Any:
        global _LAST_ANSWER_API_TRACE
        started = time.perf_counter()
        attempts = []
        total_wait_seconds = 0.0
        for attempt in range(1, ANSWER_API_MAX_ATTEMPTS + 1):
            elapsed = time.perf_counter() - self.last_call_at
            pacing_wait = max(0.0, ANSWER_API_MIN_INTERVAL_SECONDS - elapsed)
            if pacing_wait:
                time.sleep(pacing_wait)
                total_wait_seconds += pacing_wait
            self.last_call_at = time.perf_counter()
            try:
                response = self.base_completions.create(**kwargs)
                attempts.append({"attempt": attempt, "status": "SUCCESS"})
                _LAST_ANSWER_API_TRACE = {
                    "attempts": attempts,
                    "total_wait_ms": total_wait_seconds * 1000,
                    "wall_latency_ms": (time.perf_counter() - started) * 1000,
                }
                return response
            except Exception as error:
                if type(error).__name__ != "RateLimitError":
                    raise
                delay = _answer_retry_delay(error, attempt)
                attempts.append({
                    "attempt": attempt, "status": "RATE_LIMIT_429",
                    "delay_seconds": delay,
                })
                if attempt >= ANSWER_API_MAX_ATTEMPTS:
                    _LAST_ANSWER_API_TRACE = {
                        "attempts": attempts,
                        "total_wait_ms": total_wait_seconds * 1000,
                        "wall_latency_ms": (time.perf_counter() - started) * 1000,
                    }
                    raise RuntimeError(
                        "HCX-005 답변 생성 단계에서 429 재시도를 모두 소진했습니다. "
                        "잠시 후 다시 시도하세요."
                    ) from error
                time.sleep(delay)
                total_wait_seconds += delay
        raise AssertionError("도달할 수 없는 답변 재시도 상태")


class _RateLimitedAnswerClient:
    def __init__(self, base_client: Any):
        self.chat = SimpleNamespace(
            completions=_RateLimitedAnswerCompletions(base_client.chat.completions)
        )
        # 현재 HCX OpenAI 호환 API에서 두 response_format이 거부된 것이 확인됐으므로
        # 첫 질문부터 prompt JSON 방식만 사용합니다.
        self._kdic_structured_output_capability = {HCX_CHAT_MODEL: False}


_ANSWER_BASE_CLIENT = (
    HCX_CLIENT.with_options(max_retries=0)
    if hasattr(HCX_CLIENT, "with_options")
    else HCX_CLIENT
)
ANSWER_HCX_CLIENT = _RateLimitedAnswerClient(_ANSWER_BASE_CLIENT)


def build_parent_basic_evidence_pack(
    question: str,
    search_results: list[dict[str, Any]],
) -> dict[str, Any]:
    """Reranker Top-5를 동일 Parent별로 묶어 결정적인 B안 Pack을 만든다."""
    by_parent: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for result in search_results:
        child = result["chunk"]
        parent_id = str(result.get("parent_doc_id") or _parent_id_for_chunk(child))
        target = by_parent.setdefault(parent_id, {
            "rank": int(result["rank"]),
            "parent_id": parent_id,
            "representative_chunk_id": str(result["chunk_id"]),
            "matched_child_ids": [],
            "matched_child_ranks": [],
            "context_chunk_ids": list(result.get("parent_context_chunk_ids") or [str(result["chunk_id"])]),
            "document_title": _clean_text(child.get("title") or child.get("document_title")),
            "source_url": _clean_text(child.get("source_url")),
        })
        target["matched_child_ids"].append(str(result["chunk_id"]))
        target["matched_child_ranks"].append(int(result["rank"]))

    evidence = []
    sources: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for index, row in enumerate(by_parent.values(), start=1):
        context_parts = []
        section_titles = []
        valid_context_ids = []
        for chunk_id in row["context_chunk_ids"]:
            chunk = CHUNKS_BY_ID.get(str(chunk_id))
            if chunk is None:
                raise KeyError(f"Parent context 청크가 corpus에 없습니다: {chunk_id}")
            valid_context_ids.append(str(chunk_id))
            title = _clean_text(chunk.get("title"))
            section = _clean_text(chunk.get("section_title"))
            if section and section not in section_titles:
                section_titles.append(section)
            label = " / ".join(value for value in (title, section) if value)
            content = _clean_text(chunk.get("content"))
            context_parts.append(f"[{chunk_id}] {label}\n{content}".strip())

        joined_content = "\n\n".join(context_parts)
        context_truncated = len(joined_content) > PARENT_CONTEXT_MAX_CHARS
        if context_truncated:
            limited = joined_content[:PARENT_CONTEXT_MAX_CHARS]
            boundary = max(limited.rfind("\n"), limited.rfind(" "))
            if boundary >= int(PARENT_CONTEXT_MAX_CHARS * 0.8):
                limited = limited[:boundary]
            joined_content = limited.rstrip()

        evidence_id = f"E{index}"
        evidence.append({
            "evidence_id": evidence_id,
            "rank": int(row["rank"]),
            "chunk_id": row["representative_chunk_id"],
            "parent_id": row["parent_id"],
            "context_chunk_ids": valid_context_ids,
            "matched_child_ids": list(dict.fromkeys(row["matched_child_ids"])),
            "matched_child_ranks": sorted(set(row["matched_child_ranks"])),
            "document_title": row["document_title"],
            "section_title": " · ".join(section_titles),
            "content": joined_content,
            "context_char_count": len(joined_content),
            "context_truncated": context_truncated,
            "source_url": row["source_url"],
        })
        url = row["source_url"]
        if url:
            source = sources.setdefault(url, {
                "source_id": f"S{len(sources) + 1}",
                "title": row["document_title"] or "공식 출처",
                "source_url": url,
                "evidence_ids": [],
            })
            source["evidence_ids"].append(evidence_id)

    if not evidence:
        raise ValueError("Basic Evidence Pack을 만들 검색 결과가 없습니다.")
    return {
        "question": _clean_text(question),
        "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
        "evidence": evidence,
        "sources": list(sources.values()),
    }


def format_basic_evidence_pack(pack: dict[str, Any]) -> str:
    return json.dumps(pack, ensure_ascii=False, indent=2, default=str)


def sources_to_markdown_b(sources: list[dict[str, Any]]) -> str:
    if not sources:
        return ""
    lines = ["### 출처", ""]
    for source in sources:
        title = str(source.get("title") or "공식 출처")
        url = str(source.get("url") or source.get("source_url") or "")
        evidence_ids = ", ".join(source.get("evidence_ids") or [])
        if url:
            lines.append(f"- [{title}]({url}) — {evidence_ids}")
    return "\n".join(lines) if len(lines) > 2 else ""


print({
    "answer_system": "B_BASIC_EVIDENCE_PACK",
    "initial_hcx005_calls": 1,
    "detail_policy": "SAME_EVIDENCE_PACK_ON_DEMAND",
    "answer_skeleton": False,
    "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
})

# 사용자에게 보이는 답변은 구조화하되, 내부 Evidence ID는 검증용으로만 유지합니다.
answer_b_core.BASIC_ANSWER_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 답변 시스템입니다.

근거 제한:
1. 사용자 질문과 Basic Evidence Pack에 포함된 사실만 사용하세요.
2. Evidence에 없는 사실·URL·전화번호·조건을 추정하거나 일반상식으로 보완하지 마세요.
3. 서로 다른 업무·대상·신청인의 조건을 임의로 결합하지 마세요.
4. 근거가 부족한 필수 내용은 추측하지 말고 missing_information에 기록하세요.

답변 구조:
5. 첫 문단에서 질문에 대한 결론을 직접 제시하세요.
6. 질문에 독립 요구가 둘 이상이면 요구별 Markdown 소제목으로 나누세요.
7. 절차 질문은 Evidence에 순서가 있을 때 번호 목록으로 작성하세요.
8. 비교 질문은 비교 기준이 둘 이상이면 간결한 Markdown 표를 사용하세요.
9. 조건·기한·금액·필요서류·예외는 질문과 관련된 항목만 별도로 구분하세요.
10. 해당 내용이 없는데 형식만 맞추기 위한 빈 소제목을 만들지 마세요.
11. 전문용어는 공식 명칭을 사용하고 바로 이해할 수 있게 풀어 설명하세요.

출력·내부 검증:
12. answer 문자열에는 근거 문장 끝에 [E1] 형식의 실제 evidence_id를 표시하세요.
    이 표시는 프로그램이 검증 후 사용자 화면에서 숨깁니다.
13. used_evidence_ids와 used_chunk_ids에는 실제 사용한 값만 넣으세요.
14. 검색 점수, JSON, Evidence Pack, 내부 구현은 answer 본문에서 언급하지 마세요.
15. 지정된 JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()

answer_b_core.EVIDENCE_EXPLANATION_SYSTEM_PROMPT = """
당신은 예금보험공사 기본답변이 공식 문서의 어떤 내용에 근거했는지 설명하는 시스템입니다.

반드시 지킬 규칙:
1. 기본답변 생성에 사용한 것과 SHA-256이 동일한 Basic Evidence Pack만 사용하세요.
2. claim에는 기본답변의 핵심 안내 내용을 적으세요.
3. relevance_reason에는 반드시 다음 두 내용을 한 문단으로 적으세요.
   - 공식 문서에서 확인되는 근거 내용을 구체적으로 요약
   - 그 문서 내용 때문에 기본답변의 해당 안내를 할 수 있었던 연결 이유
4. relevance_reason을 'E1이 근거다'처럼 ID만 나열하는 문장으로 작성하지 마세요.
5. 적용 조건·예외·근거 한계·추가 필요 정보를 구분하세요.
6. 기본답변과 모순되는 새 결론이나 Evidence에 없는 사실을 추가하지 마세요.
7. 숨겨진 사고과정이나 내부 추론은 출력하지 말고, 사용자가 문서에서 확인할 수 있는 근거 관계만 설명하세요.
8. URL과 전화번호는 생성하지 마세요. 공식 출처는 프로그램이 별도로 표시합니다.
9. 지정된 JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()

# 비교 실행에서 연속 HCX 호출로 429가 발생할 때를 위한 공통 답변 호출 간격입니다.
ANSWER_API_MIN_INTERVAL_SECONDS = 3.0
ANSWER_API_MAX_ATTEMPTS = 6


def user_visible_answer(text: Any) -> str:
    """내부 검증용 [E#] 표지만 사용자 화면에서 제거합니다."""
    value = str(text or "")
    value = re.sub(r"\s*\[(?:E\d+)(?:\s*,\s*E\d+)*\]", "", value)
    value = re.sub(r"\bE\d+\b", "해당 공식 문서", value)
    value = re.sub(
        r"`?[A-Z]{2,}-[A-Za-z0-9_-]+_chunk_[A-Za-z0-9_-]+`?",
        "관련 문서 구간",
        value,
    )
    value = re.sub(r"[ \t]+\n", "\n", value)
    return value.strip()


def sources_to_markdown_user(sources: list[dict[str, Any]]) -> str:
    lines = ["### 공식 출처", ""]
    seen: set[str] = set()
    for source in sources:
        title = str(source.get("title") or "공식 출처").strip()
        url = str(source.get("url") or source.get("source_url") or "").strip()
        if not url or url in seen:
            continue
        seen.add(url)
        lines.append(f"- [{title}]({url})")
    return "\n".join(lines) if len(lines) > 2 else ""


def evidence_explanation_to_user_markdown(payload: Mapping[str, Any]) -> str:
    """Evidence/Chunk ID 대신 문서 내용과 답변의 연결을 사용자에게 설명합니다."""
    lines = [
        "### 이 답변을 낸 근거", "",
        user_visible_answer(answer_b_core._clean(payload.get("explanation_summary"))), "",
    ]
    for index, item in enumerate(payload.get("claim_evidence_map") or [], start=1):
        claim = user_visible_answer(answer_b_core._clean(item.get("claim")))
        reason = user_visible_answer(answer_b_core._clean(item.get("relevance_reason")))
        reason = re.sub(r"\bE\d+\b", "해당 공식 문서", reason)
        lines.extend([
            f"#### {index}. 답변에서 안내한 내용",
            "",
            claim,
            "",
            "**문서에서 확인한 내용과 답변의 연결**",
            "",
            reason,
            "",
        ])
    for title, key in (
        ("적용 조건", "conditions"),
        ("예외", "exceptions"),
        ("현재 문서 근거의 한계", "limitations"),
        ("정확한 판단에 추가로 필요한 정보", "additional_information_needed"),
    ):
        values = answer_b_core._clean_list(payload.get(key))
        if values:
            lines.extend([f"#### {title}", ""])
            lines.extend(f"- {user_visible_answer(value)}" for value in values)
            lines.append("")
    return "\n".join(lines).strip()


print({
    "answer_system": "B_STRUCTURED_BASIC_EVIDENCE_PACK",
    "user_visible_evidence_ids": False,
    "evidence_detail": "DOCUMENT_CONTENT_TO_ANSWER_CONNECTION",
})

## 5. 공통 개선 문맥 정책

In [ ]:
%%writefile kdic_context_policy_v2.py
from __future__ import annotations

"""V1.5 문맥 개선 정책: 현재 질문 우선, 규칙 우선, LLM은 경계 사례만."""

import copy
import re
import time
from typing import Any, Callable, Mapping, Sequence


BUSINESS_PATTERNS: dict[str, tuple[str, ...]] = {
    "예금자보호": ("예금자보호", "보호한도", "보호 대상", "보호대상"),
    "예금보험금": ("예금보험금", "보험금 지급", "보험금 신청"),
    "고객 미수령금": ("고객 미수령금", "미수령금", "미수령 예금"),
    "착오송금 반환지원": ("착오송금 반환지원", "착오송금 반환", "착오송금", "잘못 송금", "잘못 보낸 돈", "잘못 받은 돈"),
    "채무조정": ("채무조정", "신용회복 지원", "채무 감면", "상환 유예"),
    "은닉재산 신고": ("은닉재산 신고", "은닉재산", "숨긴 재산 신고"),
}

INTENT_TERMS = (
    "한도", "대상", "자격", "신청", "서류", "절차", "방법", "기간", "기한",
    "얼마", "언제", "비용", "수수료", "왜", "이유", "종류", "조건", "예외",
)
EXCLUSION_PATTERN = re.compile(r"(?:말고|제외(?:하고|한|해|해서)?|빼고)")
CORRECTION_PATTERN = re.compile(r"(?:아니고|아니라|정정)")
CANCEL_PATTERN = re.compile(r"^(?:그만|취소|됐어|괜찮아|필요\s*없어)[.!?\s]*$")
STRONG_FOLLOWUP_PATTERN = re.compile(
    r"^(?:(?:그럼|그러면|그건|그거|그 경우|이건|이거|여기서)\s*)?"
    r"(?:얼마나\s*걸리나요?|기간(?:은|이)?(?:요)?|언제(?:까지)?(?:인가요|예요)?|"
    r"(?:서류|준비물|신청|절차|방법|대상|자격|금액|한도|이유|수취인|송금인)"
    r"(?:은|는|이)?(?:요)?|왜(?:요)?)\s*[?.!]*$"
)
AMBIGUOUS_REFERENCE_PATTERN = re.compile(
    r"(?:그때|아까|이전에|그쪽|그 부분|그 내용|그거 말고|신청하는 쪽|처리하는 쪽)"
)


def _clean(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def detect_businesses(text: str) -> list[str]:
    cleaned = _clean(text)
    found: list[tuple[int, str]] = []
    for business, terms in BUSINESS_PATTERNS.items():
        positions = [cleaned.find(term) for term in terms if term in cleaned]
        if positions:
            found.append((min(positions), business))
    return [business for _, business in sorted(found)]


def _current_question_complete(question: str, explicit_businesses: Sequence[str]) -> bool:
    if not explicit_businesses:
        return False
    has_intent = any(term in question for term in INTENT_TERMS)
    has_explanation = bool(re.search(r"(?:알려|설명|궁금|무엇|뭔가|어떤)", question))
    unresolved_reference = bool(re.search(r"(?:그거|그건|그것|그 경우|저거|그쪽)(?!\s*말고)", question))
    return bool((has_intent or has_explanation or len(question) >= 10) and not unresolved_reference)


def _selected_pending(question: str, pending: Mapping[str, Any]) -> str | None:
    options = [_clean(value) for value in pending.get("options") or []]
    match = re.fullmatch(r"(?:선택지\s*)?(\d+)(?:번)?[.!?\s]*", re.sub(r"\s+", "", question))
    if match:
        index = int(match.group(1)) - 1
        if 0 <= index < len(options):
            return options[index]
    for option in options:
        if option and (option in question or question in option):
            return option
    return None


def _excluded_explicit_businesses(question: str, businesses: Sequence[str]) -> list[str]:
    if not EXCLUSION_PATTERN.search(question):
        return []
    output: list[str] = []
    for business in businesses:
        positions = [question.find(term) for term in BUSINESS_PATTERNS[business] if term in question]
        if not positions:
            continue
        start = min(positions)
        if EXCLUSION_PATTERN.search(question[start:start + 40]):
            output.append(business)
    return output


def _clarify(
    *,
    question: str,
    state: dict[str, Any],
    reason: str,
    message: str,
    options: Sequence[str],
    missing_slots: Sequence[str],
    started: float,
    llm_trace: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    pending = {
        "original_question": question, "reason": reason,
        "options": list(options), "missing_slots": list(missing_slots),
    }
    state["pending_clarification"] = pending
    return {
        "route": "CLARIFY", "dialogue_act": "CLARIFY",
        "original_question": question, "resolved_question": "",
        "current_question_complete": False, "context_used": False,
        "reason": reason, "clarification_message": message,
        "active_businesses": list(state.get("active_businesses") or []),
        "excluded_businesses": list(state.get("excluded_businesses") or []),
        "actor_role": state.get("actor_role"), "missing_slots": list(missing_slots),
        "pending_clarification": pending, "llm_judgment": dict(llm_trace or {}),
        "latency_ms": (time.perf_counter() - started) * 1000,
    }


def _validated_llm_decision(
    raw: Mapping[str, Any],
    *,
    explicit_businesses: Sequence[str],
    active_businesses: Sequence[str],
) -> dict[str, Any] | None:
    allowed_acts = {"NEW_TOPIC", "FOLLOW_UP", "CORRECTION", "EXCLUSION", "AMBIGUOUS"}
    act = _clean(raw.get("dialogue_act")).upper()
    confidence = float(raw.get("confidence") or 0.0)
    selected = [_clean(value) for value in raw.get("selected_businesses") or [] if _clean(value)]
    if act not in allowed_acts or confidence < 0.85:
        return None
    allowed_businesses = set(explicit_businesses) | set(active_businesses)
    if selected and not set(selected).issubset(allowed_businesses):
        return None
    if explicit_businesses and selected and set(selected) != set(explicit_businesses):
        return None
    return {
        "dialogue_act": act,
        "current_question_complete": bool(raw.get("current_question_complete")),
        "context_required": bool(raw.get("context_required")),
        "selected_businesses": selected,
        "excluded_businesses": [_clean(value) for value in raw.get("excluded_businesses") or [] if _clean(value)],
        "actor_role": _clean(raw.get("actor_role")) or None,
        "missing_slots": [_clean(value) for value in raw.get("missing_slots") or [] if _clean(value)],
        "confidence": confidence,
        "reason_code": _clean(raw.get("reason_code")),
    }


def new_context_state() -> dict[str, Any]:
    return {
        "turns": [], "active_businesses": [], "excluded_businesses": [],
        "actor_role": None, "pending_clarification": None,
        "last_resolved_question": "",
    }


def resolve_context_v2(
    question: str,
    *,
    state: dict[str, Any],
    llm_classifier: Callable[[str, Mapping[str, Any]], Mapping[str, Any]] | None = None,
) -> dict[str, Any]:
    started = time.perf_counter()
    original = _clean(question)
    if not original:
        raise ValueError("질문이 비어 있습니다.")
    for key, default in new_context_state().items():
        state.setdefault(key, copy.deepcopy(default))

    if CANCEL_PATTERN.fullmatch(original):
        state["pending_clarification"] = None
        return {
            "route": "DIRECT_RESPONSE", "dialogue_act": "CANCEL",
            "original_question": original, "resolved_question": "",
            "current_question_complete": True, "context_used": False,
            "reason": "EXPLICIT_CANCEL", "direct_response": "알겠습니다. 현재 요청을 중단했습니다.",
            "active_businesses": list(state["active_businesses"]),
            "excluded_businesses": list(state["excluded_businesses"]),
            "actor_role": state.get("actor_role"), "missing_slots": [],
            "pending_clarification": None, "llm_judgment": {},
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    explicit = detect_businesses(original)
    complete = _current_question_complete(original, explicit)
    pending = state.get("pending_clarification") or {}
    selected = _selected_pending(original, pending) if pending else None

    if selected:
        base = _clean(pending.get("original_question"))
        if selected in {"송금인", "보낸 사람"}:
            state["actor_role"] = "SENDER"
            prefix = f"{state['active_businesses'][0]}에 관하여 " if len(state["active_businesses"]) == 1 else ""
            resolved = f"{prefix}{base} 송금인 기준"
        elif selected in {"수취인", "받은 사람"}:
            state["actor_role"] = "RECIPIENT"
            prefix = f"{state['active_businesses'][0]}에 관하여 " if len(state["active_businesses"]) == 1 else ""
            resolved = f"{prefix}{base} 수취인 기준"
        else:
            state["active_businesses"] = [selected]
            resolved = f"{selected}에 관하여 {base}"
        state["pending_clarification"] = None
        state["last_resolved_question"] = resolved
        return {
            "route": "CONTINUE", "dialogue_act": "SELECT_OPTION",
            "original_question": original, "resolved_question": resolved,
            "current_question_complete": False, "context_used": True,
            "reason": "PENDING_OPTION_MATCH", "clarification_message": "",
            "active_businesses": list(state["active_businesses"]),
            "excluded_businesses": list(state["excluded_businesses"]),
            "actor_role": state.get("actor_role"), "missing_slots": [],
            "pending_clarification": None, "llm_judgment": {},
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    excluded_now = _excluded_explicit_businesses(original, explicit)
    if excluded_now:
        state["excluded_businesses"] = list(dict.fromkeys(state["excluded_businesses"] + excluded_now))
        state["active_businesses"] = [
            business for business in state["active_businesses"] if business not in excluded_now
        ]
        remaining = [business for business in explicit if business not in excluded_now]
        state["pending_clarification"] = None
        if not remaining:
            return _clarify(
                question=original, state=state, reason="EXCLUSION_WITHOUT_REPLACEMENT",
                message=f"{', '.join(excluded_now)} 업무는 제외하겠습니다. 대신 어떤 업무를 안내할까요?",
                options=[business for business in BUSINESS_PATTERNS if business not in state["excluded_businesses"]],
                missing_slots=["business_function"], started=started,
            )
        state["active_businesses"] = remaining
        state["last_resolved_question"] = original
        return {
            "route": "CONTINUE", "dialogue_act": "CORRECTION",
            "original_question": original, "resolved_question": original,
            "current_question_complete": True, "context_used": False,
            "reason": "EXPLICIT_EXCLUSION_WITH_REPLACEMENT", "clarification_message": "",
            "active_businesses": list(state["active_businesses"]),
            "excluded_businesses": list(state["excluded_businesses"]),
            "actor_role": state.get("actor_role"), "missing_slots": [],
            "pending_clarification": None, "llm_judgment": {},
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    # 현재 질문이 독립적으로 완결되면 이전 pending과 업무를 무조건 덮어쓴다.
    if complete:
        state["active_businesses"] = explicit
        state["pending_clarification"] = None
        state["last_resolved_question"] = original
        return {
            "route": "CONTINUE", "dialogue_act": "CORRECTION" if CORRECTION_PATTERN.search(original) else "NEW_TOPIC",
            "original_question": original, "resolved_question": original,
            "current_question_complete": True, "context_used": False,
            "reason": "CURRENT_QUESTION_COMPLETE", "clarification_message": "",
            "active_businesses": list(state["active_businesses"]),
            "excluded_businesses": list(state["excluded_businesses"]),
            "actor_role": state.get("actor_role"), "missing_slots": [],
            "pending_clarification": None, "llm_judgment": {},
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    if STRONG_FOLLOWUP_PATTERN.fullmatch(original) and not explicit:
        active = list(state.get("active_businesses") or [])
        if len(active) == 1:
            if active[0] == "착오송금 반환지원" and re.search(r"(?:얼마나\s*걸|기간|언제)", original):
                return _clarify(
                    question=original, state=state, reason="MISTAKEN_TRANSFER_TIME_SCOPE_AMBIGUOUS",
                    message="착오송금의 어느 기간을 묻는지 확인이 필요합니다. 송금인의 반환지원 처리기간과 수취인의 자진반환 관련 기간 중 선택해 주세요.",
                    options=["송금인", "수취인"], missing_slots=["actor_role", "process_stage"], started=started,
                )
            resolved = f"{active[0]}에 관하여 {original}"
            state["pending_clarification"] = None
            state["last_resolved_question"] = resolved
            return {
                "route": "CONTINUE", "dialogue_act": "FOLLOW_UP",
                "original_question": original, "resolved_question": resolved,
                "current_question_complete": False, "context_used": True,
                "reason": "UNIQUE_ACTIVE_BUSINESS", "clarification_message": "",
                "active_businesses": active, "excluded_businesses": list(state["excluded_businesses"]),
                "actor_role": state.get("actor_role"), "missing_slots": [],
                "pending_clarification": None, "llm_judgment": {},
                "latency_ms": (time.perf_counter() - started) * 1000,
            }
        return _clarify(
            question=original, state=state,
            reason="FOLLOW_UP_WITHOUT_UNIQUE_BUSINESS",
            message="어떤 업무에 관한 후속 질문인지 알려주세요.",
            options=active or list(BUSINESS_PATTERNS), missing_slots=["business_function"], started=started,
        )

    # 규칙 경계 사례에서만 LLM을 호출한다.
    if AMBIGUOUS_REFERENCE_PATTERN.search(original) and llm_classifier is not None:
        raw = dict(llm_classifier(original, state) or {})
        decision = _validated_llm_decision(raw, explicit_businesses=explicit, active_businesses=state["active_businesses"])
        trace = {"called": True, "raw": raw, "accepted": bool(decision), "decision": decision}
        if decision and decision["dialogue_act"] == "FOLLOW_UP" and len(decision["selected_businesses"] or state["active_businesses"]) == 1:
            business = (decision["selected_businesses"] or state["active_businesses"])[0]
            resolved = f"{business}에 관하여 {original}"
            state["active_businesses"] = [business]
            state["pending_clarification"] = None
            state["last_resolved_question"] = resolved
            return {
                "route": "CONTINUE", "dialogue_act": "FOLLOW_UP",
                "original_question": original, "resolved_question": resolved,
                "current_question_complete": False, "context_used": True,
                "reason": "LLM_STRUCTURED_FOLLOW_UP", "clarification_message": "",
                "active_businesses": [business], "excluded_businesses": list(state["excluded_businesses"]),
                "actor_role": decision.get("actor_role"), "missing_slots": [],
                "pending_clarification": None, "llm_judgment": trace,
                "latency_ms": (time.perf_counter() - started) * 1000,
            }
        if decision and decision["current_question_complete"]:
            state["pending_clarification"] = None
            state["last_resolved_question"] = original
            return {
                "route": "CONTINUE", "dialogue_act": decision["dialogue_act"],
                "original_question": original, "resolved_question": original,
                "current_question_complete": True, "context_used": False,
                "reason": "LLM_STRUCTURED_NEW_TOPIC", "clarification_message": "",
                "active_businesses": list(state["active_businesses"]),
                "excluded_businesses": list(state["excluded_businesses"]),
                "actor_role": decision.get("actor_role"), "missing_slots": [],
                "pending_clarification": None, "llm_judgment": trace,
                "latency_ms": (time.perf_counter() - started) * 1000,
            }
        return _clarify(
            question=original, state=state, reason="AMBIGUOUS_AFTER_STRUCTURED_JUDGMENT",
            message="현재 질문이 이전 내용의 후속 질문인지 새로운 질문인지 확인해 주세요.",
            options=list(state["active_businesses"]), missing_slots=["dialogue_target"], started=started, llm_trace=trace,
        )

    # 문맥 필요성이 명확하지 않으면 이전 상태를 섞지 않고 원문을 V1.5에 전달한다.
    state["pending_clarification"] = None if pending else state["pending_clarification"]
    state["last_resolved_question"] = original
    return {
        "route": "CONTINUE", "dialogue_act": "NEW_QUESTION_UNCHANGED",
        "original_question": original, "resolved_question": original,
        "current_question_complete": False, "context_used": False,
        "reason": "CONTEXT_NOT_PROVEN_USE_ORIGINAL", "clarification_message": "",
        "active_businesses": list(state["active_businesses"]),
        "excluded_businesses": list(state["excluded_businesses"]),
        "actor_role": state.get("actor_role"), "missing_slots": [],
        "pending_clarification": state.get("pending_clarification"), "llm_judgment": {},
        "latency_ms": (time.perf_counter() - started) * 1000,
    }

## 6. V1.5 개선 분석기와 V3.1 교차업무 선택적 재작성 분석기

In [ ]:
%%writefile kdic_query_analyzer_v31.py
from __future__ import annotations

import json
import math
import os
import random
import re
import time
import unicodedata
import uuid
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Sequence

import pandas as pd
import requests

PIPELINE_VERSION = "KDIC_LIGHTWEIGHT_RAG_QUERY_ANALYZER_V1_2026_08_11"

BUSINESS_FUNCTIONS = [
    "예금자보호제도",
    "예금보험금 안내",
    "고객 미수령금 신청",
    "착오송금 반환 신청",
    "채무조정 안내",
    "은닉재산 신고",
]
INTENTS = ["AMOUNT", "ELIGIBILITY", "TIME", "APPLICATION", "OVERVIEW", "STATUS", "DOCUMENTS", "CONTACT"]
ROUTES = ["RETRIEVE", "CLARIFY", "DIRECT", "OUT_OF_SCOPE"]
APPLICANT_TYPES = ["SELF", "PROXY", "HEIR", "LEGAL_REPRESENTATIVE", "CORPORATION"]
USER_ROLES = ["DEPOSITOR", "SENDER", "RECIPIENT", "DEBTOR", "REPORTER", "CLAIMANT", "GENERAL_USER"]
MISSING_FIELDS = ["business_function", "applicant_type", "target_type", "case_details"]

INTENT_ALIASES = {
    "신청 방법": "APPLICATION", "신청 절차": "APPLICATION", "접수 방법": "APPLICATION",
    "필요 서류": "DOCUMENTS", "구비서류": "DOCUMENTS", "준비 서류": "DOCUMENTS",
    "자격": "ELIGIBILITY", "대상": "ELIGIBILITY", "조건": "ELIGIBILITY",
    "금액": "AMOUNT", "한도": "AMOUNT", "보호한도": "AMOUNT",
    "기간": "TIME", "기한": "TIME", "처리 기간": "TIME",
    "조회": "STATUS", "진행 상태": "STATUS", "처리 상태": "STATUS",
    "문의": "CONTACT", "연락처": "CONTACT",
    "안내": "OVERVIEW", "개요": "OVERVIEW", "무엇": "OVERVIEW",
}

BUSINESS_KEYWORDS = {
    "예금자보호제도": ["예금자보호", "보호한도", "보호대상", "예금 보호"],
    "예금보험금 안내": ["예금보험금", "보험금 지급", "보험사고", "가지급금"],
    "고객 미수령금 신청": ["미수령금", "파산배당금", "개산지급금 정산금", "상속인 금융거래 조회"],
    "착오송금 반환 신청": ["착오송금", "잘못 보낸 돈", "잘못 송금", "착오 송금"],
    "채무조정 안내": ["채무조정", "신용회복지원", "파산선고", "면책", "채무감면"],
    "은닉재산 신고": ["은닉재산", "은닉 재산"],
}

EXPLICIT_TYPO_MAP = (
    ("예금보헝금", "예금보험금"),
    ("예금보혐금", "예금보험금"),
    ("착오송금반한", "착오송금 반환"),
    ("미수령금신정", "미수령금 신청"),
)

@dataclass(frozen=True)
class PipelineConfig:
    model: str = "HCX-007"
    base_url: str = "https://clovastudio.stream.ntruss.com"
    timeout_seconds: float = 90.0
    max_api_attempts: int = 2
    max_completion_tokens: int = 1400
    temperature: float = 0.1
    top_p: float = 0.8
    top_k: int = 0
    request_interval_seconds: float = 0.3
    max_context_turns: int = 2
    intent_soft_boost: float = 0.15

class HCXAPIError(RuntimeError):
    def __init__(self, message: str, *, error_type: str, telemetry: dict[str, Any]):
        super().__init__(message)
        self.error_type = error_type
        self.telemetry = telemetry

def get_hcx_api_key(secret_name: str = "HCX_API_KEY") -> str:
    try:
        from google.colab import userdata
        value = str(userdata.get(secret_name) or "").strip()
    except ImportError:
        value = os.getenv(secret_name, "").strip()
    if not value:
        raise ValueError(f"Colab Secrets 또는 환경변수에 {secret_name}가 없습니다.")
    if value.lower().startswith("bearer ") or any(ch.isspace() for ch in value):
        raise ValueError(f"{secret_name}에는 Bearer 접두사 없이 API 키 값만 저장하세요.")
    return value


FOLLOW_UP_PATTERN = re.compile(r"(?:그럼|그러면|그거|그건|그 경우|이거|이건|이 경우|앞서|방금|그때|그것|그 서류|그 신청)")

def normalize_query(text: str) -> dict[str, Any]:
    original = str(text or "")
    value = unicodedata.normalize("NFKC", original)
    changes = []
    cleaned = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", value)
    if cleaned != value:
        changes.append("CONTROL_CHARACTER")
        value = cleaned
    cleaned = re.sub(r"([!?ㅋㅎㅠㅜ])\1{2,}", r"\1\1", value)
    if cleaned != value:
        changes.append("REPEATED_CHARACTER")
        value = cleaned
    for wrong, correct in EXPLICIT_TYPO_MAP:
        if wrong in value:
            value = value.replace(wrong, correct)
            changes.append("EXPLICIT_TYPO")
    cleaned = re.sub(r"\s+", " ", value).strip()
    if cleaned != value:
        changes.append("WHITESPACE")
    if not cleaned:
        raise ValueError("사용자 질의가 비어 있습니다.")
    return {"original_query": original, "normalized_query": cleaned, "changes": changes}

def build_context(query: str, conversation_state: dict[str, Any] | None, max_turns: int) -> dict[str, Any]:
    state = dict(conversation_state or {})
    if not FOLLOW_UP_PATTERN.search(query):
        return {"used": False, "confirmed": {}, "recent_turns": []}
    confirmed = state.get("confirmed") if isinstance(state.get("confirmed"), dict) else {}
    turns = state.get("recent_turns") if isinstance(state.get("recent_turns"), list) else []
    return {"used": True, "confirmed": confirmed, "recent_turns": turns[-max_turns:]}

def exact_fullmatch(pattern: str, query: str) -> bool:
    return re.fullmatch(pattern, query.strip(), flags=re.I) is not None

def detect_fast_path(query: str, conversation_state: dict[str, Any] | None = None) -> dict[str, Any] | None:
    # 문장 전체가 규칙에 일치할 때만 처리해 실제 질문을 잘라내지 않는다.
    detectors = []
    if exact_fullmatch(r"(?:안녕|안녕하세요|반갑습니다|반가워요)[.!?]*", query):
        detectors.append({"route": "DIRECT", "action": "GREETING"})
    if exact_fullmatch(r"(?:고마워요|고맙습니다|감사합니다|도움이 됐어요|알겠습니다)[.!?]*", query):
        detectors.append({"route": "DIRECT", "action": "ACKNOWLEDGEMENT"})
    if exact_fullmatch(r"(?:무슨 질문을 할 수 있나요|지원하는 업무를 (?:알려주세요|목록으로 보여주세요)|이 챗봇은 어떻게 사용하면 되나요)[.!?]*", query):
        detectors.append({"route": "DIRECT", "action": "CAPABILITY_GUIDE"})
    has_previous = bool((conversation_state or {}).get("has_previous_answer"))
    if has_previous and exact_fullmatch(r"(?:쉽게 설명해 주세요|핵심만 알려주세요|표로 정리해 주세요|더 자세히 설명해 주세요)[.!?]*", query):
        detectors.append({"route": "DIRECT", "action": "REFORMAT_PREVIOUS_ANSWER"})
    if exact_fullmatch(r"(?:오늘|내일|이번 주말)?\s*(?:서울 )?(?:날씨|기온|미세먼지)(?:를|가|은|는)?.*", query):
        detectors.append({"route": "OUT_OF_SCOPE", "action": "EXPLICIT_WEATHER"})
    if exact_fullmatch(r"(?:주식 종목을 추천해 주세요|로또 번호를 알려주세요)[.!?]*", query):
        detectors.append({"route": "OUT_OF_SCOPE", "action": "EXPLICIT_NON_KDIC"})
    return detectors[0] if len(detectors) == 1 else None

def find_businesses(text: str) -> list[str]:
    compact = re.sub(r"\s+", "", text)
    found = []
    for business, keywords in BUSINESS_KEYWORDS.items():
        if any(re.sub(r"\s+", "", keyword) in compact for keyword in keywords):
            found.append(business)
    return found

def normalize_intent(value: Any) -> str | None:
    if value is None:
        return None
    text = re.sub(r"\s+", " ", str(value)).strip()
    if text in INTENTS:
        return text
    if text.upper() in INTENTS:
        return text.upper()
    return INTENT_ALIASES.get(text)


def query_analysis_schema() -> dict[str, Any]:
    # HCX-007 공식 지원 타입에 null이 없으므로 UNKNOWN/빈 문자열을 sentinel로 쓴다.
    business_value = {"type": "string", "enum": [*BUSINESS_FUNCTIONS, "UNKNOWN"]}
    intent_value = {"type": "string", "enum": [*INTENTS, "UNKNOWN"]}
    applicant_value = {"type": "string", "enum": [*APPLICANT_TYPES, "UNKNOWN"]}
    user_role_value = {"type": "string", "enum": [*USER_ROLES, "UNKNOWN"]}
    return {
        "type": "object",
        "properties": {
            "route": {"type": "string", "enum": ROUTES},
            "needs": {
                "type": "array",
                "maxItems": 6,
                "items": {
                    "type": "object",
                    "properties": {
                        "need_id": {"type": "string"},
                        "query": {"type": "string"},
                        "business_function": business_value,
                        "business_confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                        "intent": intent_value,
                        "intent_confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                        "user_role": user_role_value,
                        "applicant_type": applicant_value,
                        "target_type": {"type": "string"},
                        "case_details": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": [
                        "need_id", "query", "business_function", "business_confidence",
                        "intent", "intent_confidence", "user_role",
                        "applicant_type", "target_type", "case_details",
                    ],
                },
            },
            "missing_information": {"type": "array", "items": {"type": "string", "enum": MISSING_FIELDS}},
        },
        "required": ["route", "needs", "missing_information"],
    }

SYSTEM_PROMPT = f"""
역할: 예금보험공사 RAG 시스템의 경량 질의 분석기.
목적: 사용자 요구를 검색 가능한 최소 Need로 나누고 각 Need의 검색어·업무·의도·핵심 조건을 반환한다.

규칙:
1. 서로 다른 정보 요구는 N1, N2, ...로 분리한다.
2. query는 해당 Need를 독립적으로 검색할 수 있는 한국어 문장으로 쓴다.
3. 원문이 이미 독립적이면 불필요하게 바꾸지 않는다.
4. 확정 context는 후속 질문일 때만 사용한다.
5. 근거가 없는 범주형 값은 UNKNOWN, target_type은 빈 문자열, 목록은 []로 두고 추측하지 않는다.
   business_confidence와 intent_confidence는 각각 해당 분류가 맞을 확률을 0~1로 쓴다.
6. 업무가 불확실해도 원문으로 검색 가능하면 RETRIEVE다.
7. 사용자만 제공할 수 있는 필수 정보가 없어 정답 대상이 바뀐 때만 CLARIFY다.
8. 인사·감사·사용법·이전 답변 재구성은 DIRECT다.
9. 예금보험공사 업무와 명확히 무관한 요청은 OUT_OF_SCOPE다.

허용 business_function: {BUSINESS_FUNCTIONS}
허용 intent: {INTENTS}
허용 user_role: {USER_ROLES}
허용 applicant_type: {APPLICANT_TYPES}
"""

class HCX007StructuredClient:
    def __init__(self, api_key: str, config: PipelineConfig | None = None):
        self.config = config or PipelineConfig()
        self.api_key = api_key
        self.session = requests.Session()

    @property
    def url(self) -> str:
        return f"{self.config.base_url}/v3/chat-completions/{self.config.model}"

    def analyze(self, payload: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any]]:
        started = time.perf_counter()
        attempts = []
        total_tokens = 0
        last_error = None
        body = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
            ],
            "topP": self.config.top_p,
            "topK": self.config.top_k,
            "maxCompletionTokens": self.config.max_completion_tokens,
            "temperature": self.config.temperature,
            "repetitionPenalty": 1.05,
            "thinking": {"effort": "none"},
            "stop": [],
            "responseFormat": {"type": "json", "schema": query_analysis_schema()},
        }
        for attempt in range(1, self.config.max_api_attempts + 1):
            attempt_started = time.perf_counter()
            request_id = str(uuid.uuid4())
            try:
                response = self.session.post(
                    self.url,
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "X-NCP-CLOVASTUDIO-REQUEST-ID": request_id,
                        "Content-Type": "application/json",
                        "Accept": "application/json",
                    },
                    json=body,
                    timeout=self.config.timeout_seconds,
                )
                if response.status_code >= 400:
                    try:
                        error_body = response.json()
                    except Exception:
                        error_body = {"text": response.text[:1000]}
                    retryable = response.status_code in {408, 429, 500, 502, 503, 504}
                    attempts.append({
                        "attempt": attempt, "request_id": request_id, "http_status": response.status_code,
                        "retryable": retryable, "error": error_body,
                        "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                    })
                    last_error = f"HTTP {response.status_code}: {error_body}"
                    if retryable and attempt < self.config.max_api_attempts:
                        retry_after = response.headers.get("Retry-After")
                        delay = float(retry_after) if retry_after and retry_after.replace(".", "", 1).isdigit() else 1.5 + random.random()
                        time.sleep(min(delay, 10.0))
                        continue
                    raise HCXAPIError(last_error, error_type="API_RETRYABLE" if retryable else "API_FATAL", telemetry={
                        "api_request_count": len(attempts), "attempts": attempts, "total_tokens": total_tokens,
                        "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                    })

                envelope = response.json()
                result = envelope.get("result", envelope)
                usage = result.get("usage") or {}
                used = int(usage.get("totalTokens") or 0)
                total_tokens += used
                content = str((result.get("message") or {}).get("content") or "")
                parsed = json.loads(content)
                if not isinstance(parsed, dict):
                    raise TypeError("모델 결과의 최상위가 object가 아닙니다.")
                attempts.append({
                    "attempt": attempt, "request_id": request_id, "http_status": response.status_code,
                    "tokens": used, "finish_reason": result.get("finishReason"),
                    "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                })
                return parsed, {
                    "api_request_count": len(attempts), "attempts": attempts, "total_tokens": total_tokens,
                    "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                }
            except HCXAPIError:
                raise
            except (requests.Timeout, requests.ConnectionError) as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                attempts.append({
                    "attempt": attempt, "request_id": request_id, "retryable": True, "error": last_error,
                    "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                })
                if attempt < self.config.max_api_attempts:
                    time.sleep(1.5 + random.random())
                    continue
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                attempts.append({
                    "attempt": attempt, "request_id": request_id, "retryable": False, "error": last_error,
                    "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                })
                break
        raise HCXAPIError(last_error or "HCX-007 호출 실패", error_type="API_OR_PARSE_ERROR", telemetry={
            "api_request_count": len(attempts), "attempts": attempts, "total_tokens": total_tokens,
            "latency_ms": round((time.perf_counter() - started) * 1000, 3),
        })


def validate_analysis(raw: dict[str, Any], normalized_query: str) -> tuple[dict[str, Any], list[str]]:
    warnings = []
    route = raw.get("route")
    if route not in ROUTES:
        route = "RETRIEVE"
        warnings.append("INVALID_ROUTE_TO_RETRIEVE")
    rows = raw.get("needs")
    if not isinstance(rows, list):
        rows = []
        warnings.append("NEEDS_NOT_LIST")
    needs = []
    for index, row in enumerate(rows, 1):
        if not isinstance(row, dict):
            warnings.append(f"N{index}_NOT_OBJECT")
            continue
        query = str(row.get("query") or "").strip()
        if not query:
            query = normalized_query
            warnings.append(f"N{index}_EMPTY_QUERY_USED_ORIGINAL")
        business = row.get("business_function")
        if business not in BUSINESS_FUNCTIONS:
            if business not in (None, "", "UNKNOWN"):
                warnings.append(f"N{index}_INVALID_BUSINESS_TO_NULL")
            business = None
        intent = normalize_intent(row.get("intent"))
        if row.get("intent") not in (None, "", "UNKNOWN") and intent is None:
            warnings.append(f"N{index}_INVALID_INTENT_TO_NULL")
        applicant = row.get("applicant_type")
        if applicant not in APPLICANT_TYPES:
            applicant = None
        user_role = row.get("user_role")
        if user_role not in USER_ROLES:
            user_role = None
        try:
            business_confidence = min(1.0, max(0.0, float(row.get("business_confidence") or 0.0)))
        except (TypeError, ValueError):
            business_confidence = 0.0
            warnings.append(f"N{index}_INVALID_BUSINESS_CONFIDENCE_TO_ZERO")
        try:
            intent_confidence = min(1.0, max(0.0, float(row.get("intent_confidence") or 0.0)))
        except (TypeError, ValueError):
            intent_confidence = 0.0
            warnings.append(f"N{index}_INVALID_INTENT_CONFIDENCE_TO_ZERO")
        target = row.get("target_type")
        target = str(target).strip() if target not in (None, "") else None
        cases = row.get("case_details")
        cases = [str(x).strip() for x in cases if str(x).strip()] if isinstance(cases, list) else []
        needs.append({
            "need_id": f"N{index}", "query": query, "business_function": business,
            "business_confidence": business_confidence,
            "intent": intent, "intent_confidence": intent_confidence,
            "user_role": user_role, "applicant_type": applicant, "target_type": target,
            "case_details": cases,
        })
    if route == "RETRIEVE" and not needs:
        needs = [{
            "need_id": "N1", "query": normalized_query, "business_function": None,
            "business_confidence": 0.0, "intent": None, "intent_confidence": 0.0,
            "user_role": None, "applicant_type": None, "target_type": None, "case_details": [],
        }]
        warnings.append("EMPTY_RETRIEVE_NEEDS_USED_ORIGINAL")
    if route in {"DIRECT", "OUT_OF_SCOPE"}:
        needs = []
    missing = raw.get("missing_information")
    missing = [x for x in missing if x in MISSING_FIELDS] if isinstance(missing, list) else []
    return {"route": route, "needs": needs, "missing_information": missing}, warnings

def build_keyword_query(need: dict[str, Any]) -> str:
    values = [
        need.get("business_function"), need.get("intent"), need.get("user_role"),
        need.get("applicant_type"), need.get("target_type"),
    ]
    values.extend(need.get("case_details") or [])
    return " ".join(str(x) for x in values if x)

def determine_filter_policy(need: dict[str, Any], original: str, context: dict[str, Any], manual: dict[str, Any]) -> dict[str, Any]:
    business = need.get("business_function")
    if business not in BUSINESS_FUNCTIONS:
        return {"mode": "NONE", "value": None, "soft_hint": None, "evidence": "UNKNOWN"}
    if manual.get("business_function") == business:
        return {"mode": "HARD", "value": business, "soft_hint": None, "evidence": "MANUAL"}
    explicit = business in find_businesses(original)
    if explicit:
        return {"mode": "HARD", "value": business, "soft_hint": None, "evidence": "ORIGINAL"}
    if context.get("confirmed", {}).get("business_function") == business:
        return {"mode": "HARD", "value": business, "soft_hint": None, "evidence": "CONTEXT"}
    if float(need.get("business_confidence") or 0.0) >= 0.65:
        return {"mode": "SOFT", "value": None, "soft_hint": business, "evidence": "MODEL"}
    return {"mode": "NONE", "value": None, "soft_hint": None, "evidence": "LOW_CONFIDENCE_MODEL"}

def build_query_plans(analysis: dict[str, Any], original: str, context: dict[str, Any], manual: dict[str, Any], config: PipelineConfig) -> list[dict[str, Any]]:
    if analysis["route"] != "RETRIEVE":
        return []
    plans = []
    for need in analysis["needs"]:
        policy = determine_filter_policy(need, original, context, manual)
        plans.append({
            "need_id": need["need_id"],
            "semantic_query": need["query"],
            "keyword_query": build_keyword_query(need) or need["query"],
            "business_filter": policy,
            "intent_boost": {
                "mode": "SOFT" if need.get("intent") in INTENTS else "NONE",
                "value": need.get("intent"),
                "weight": config.intent_soft_boost if need.get("intent") in INTENTS else 0.0,
            },
            "entities": {
                "user_role": need.get("user_role"),
                "applicant_type": need.get("applicant_type"),
                "target_type": need.get("target_type"),
                "case_details": need.get("case_details") or [],
            },
        })
    return plans

GENERIC_CLARIFY_PATTERN = re.compile(r"^(?:신청 방법|필요한 서류|제출해야 하는 서류|신청 기한|처리 기간|문의처)(?:을|가|은|는|이|가)?(?: 어떻게 되나요| 언제까지인가요| 알려주세요| 무엇인가요)?[.!?]*$")

def fallback_analysis(normalized_query: str, context: dict[str, Any], reason: str) -> dict[str, Any]:
    businesses = find_businesses(normalized_query)
    confirmed_business = context.get("confirmed", {}).get("business_function")
    if not businesses and not confirmed_business and GENERIC_CLARIFY_PATTERN.fullmatch(normalized_query):
        return {
            "route": "CLARIFY", "needs": [], "missing_information": ["business_function"],
            "fallback_reason": reason,
        }
    return {
        "route": "RETRIEVE",
        "needs": [{
            "need_id": "N1", "query": normalized_query,
            "business_function": businesses[0] if len(businesses) == 1 else confirmed_business,
            "business_confidence": 1.0 if len(businesses) == 1 else 0.0,
            "intent": None, "intent_confidence": 0.0,
            "user_role": None, "applicant_type": None, "target_type": None, "case_details": [],
        }],
        "missing_information": [], "fallback_reason": reason,
    }

class KDICLightweightRAGAnalyzer:
    def __init__(self, client: HCX007StructuredClient, config: PipelineConfig | None = None):
        self.client = client
        self.config = config or client.config

    def run(self, query: str, *, conversation_state=None, manual_selection=None) -> dict[str, Any]:
        started = time.perf_counter()
        normalized = normalize_query(query)
        original = normalized["original_query"]
        text = normalized["normalized_query"]
        manual = dict(manual_selection or {})
        context = build_context(text, conversation_state, self.config.max_context_turns)

        fast = detect_fast_path(text, conversation_state)
        if fast:
            analysis = {"route": fast["route"], "needs": [], "missing_information": []}
            return {
                "pipeline_version": PIPELINE_VERSION, "analysis_status": "FAST_PATH",
                "original_query": original, "normalized_query": text, "context": context,
                "analysis": analysis, "fast_path": fast, "query_plans": [],
                "runtime": {
                    "api_request_count": 0, "total_tokens": 0,
                    "latency_ms": round((time.perf_counter() - started) * 1000, 3), "attempts": [],
                },
            }

        payload = {
            "query": text,
            "confirmed_context": context["confirmed"],
            "recent_turns": context["recent_turns"],
            "manual_selection": manual,
        }
        try:
            raw, telemetry = self.client.analyze(payload)
            analysis, warnings = validate_analysis(raw, text)
            status = "REPAIRED" if warnings else "OK"
        except HCXAPIError as exc:
            telemetry = exc.telemetry
            analysis = fallback_analysis(text, context, f"{exc.error_type}: {exc}")
            warnings = ["MODEL_ANALYSIS_FAILED_USED_FALLBACK"]
            status = "FALLBACK"

        plans = build_query_plans(analysis, original, context, manual, self.config)
        return {
            "pipeline_version": PIPELINE_VERSION, "analysis_status": status,
            "original_query": original, "normalized_query": text, "context": context,
            "analysis": analysis, "validation_warnings": warnings, "query_plans": plans,
            "runtime": {
                **telemetry,
                "latency_ms": round((time.perf_counter() - started) * 1000, 3),
            },
        }


PIPELINE_VERSION = "KDIC_LIGHTWEIGHT_RAG_QUERY_ANALYZER_V3_1_2026_08_12"
HARD_BUSINESS_CONFIDENCE_THRESHOLD = 0.98
HARD_BUSINESS_MARGIN_THRESHOLD = 0.20
V3_ROUTES = ["RETRIEVE", "RETRIEVE_RELAXED", "CLARIFY", "DIRECT", "OUT_OF_SCOPE"]

BUSINESS_KEYWORDS = {
    "예금자보호제도": ["예금자보호", "보호한도", "보호대상", "예금 보호", "보호 한도"],
    "예금보험금 안내": ["예금보험금", "보험금 지급", "보험사고", "가지급금", "개산지급금", "1종 보험사고", "2종 보험사고"],
    "고객 미수령금 신청": ["미수령금", "파산배당금", "개산지급금 정산금", "지급대행점", "상속인 금융거래 조회", "상속인 금융거래 조회서비스"],
    "착오송금 반환 신청": ["착오송금", "잘못 보낸 돈", "잘못 송금", "착오 송금", "반환지원", "매입계약", "지급명령", "강제집행", "송금인", "수취인"],
    "채무조정 안내": ["채무조정", "신용회복지원", "파산선고", "면책", "채무감면", "개인회생", "개인파산", "워크아웃", "변제기간", "부채증명원", "채무정보"],
    "은닉재산 신고": ["은닉재산", "은닉 재산", "금융부실관련자", "부실관련자", "차명 재산", "차명재산", "신고 포상금"],
}

DIRECT_META_PATTERN = re.compile(
    r"^(?:안녕|안녕하세요|반갑습니다|반가워요|고마워요|고맙습니다|감사합니다|도움이 됐어요|"
    r"알겠습니다|무슨 질문을 할 수 있나요|지원하는 업무를 (?:알려주세요|목록으로 보여주세요)|"
    r"이 챗봇은 어떻게 사용하면 되나요|답변을 쉽게 설명해 줄 수 있나요|"
    r"전문가 수준으로 자세히 설명해 주세요|긴 설명보다 핵심 내용만 먼저 알려주세요|"
    r"질문을 잘못 입력했어요[.]? 다시 물어볼게요)[.!?]*$", re.I,
)
OOS_EXACT_PATTERN = re.compile(
    r".*(?:코스피|비트코인|주택담보대출 금리|신용점수|실손보험|국민연금|보이스피싱|"
    r"은행 계좌를 새로|해외송금 수수료|카드 결제|전세대출|세금 환급|퇴직금|"
    r"개인정보 유출|상속세|환율|주식 투자|신용카드 연회비|사업자등록|서울 날씨|"
    r"이번 주말.*날씨).*", re.I,
)
USER_ROLE_PATTERNS = [
    ("SENDER", r"송금인|돈을 보낸 사람"), ("RECIPIENT", r"수취인|돈을 받은 사람"),
    ("DEBTOR", r"채무자"), ("REPORTER", r"신고자|신고인"),
    ("CLAIMANT", r"청구인"), ("DEPOSITOR", r"예금자(?!보호)"),
]
APPLICANT_PATTERNS = [
    ("LEGAL_REPRESENTATIVE", r"법정대리인|후견인"),
    ("PROXY", r"대리인|대신해 신청|대리 신청|위임받"),
    ("HEIR", r"상속인"), ("CORPORATION", r"법인(?: 명의|이|으로| 신청)"),
    ("SELF", r"본인이 직접|본인 신청|제가 직접|직접 신청"),
]

def extract_explicit_value(text: str, patterns: list[tuple[str, str]]) -> str | None:
    for value, pattern in patterns:
        if re.search(pattern, text, flags=re.I):
            return value
    return None

def detect_fast_path_v3(query: str, conversation_state: dict[str, Any] | None = None) -> dict[str, Any] | None:
    text = query.strip()
    if DIRECT_META_PATTERN.fullmatch(text):
        return {"route": "DIRECT", "action": "META_OR_SOCIAL"}
    has_previous = bool((conversation_state or {}).get("has_previous_answer"))
    if has_previous and exact_fullmatch(
        r"(?:쉽게 설명해 주세요|핵심만 알려주세요|표로 정리해 주세요|더 자세히 설명해 주세요)[.!?]*", text
    ):
        return {"route": "DIRECT", "action": "REFORMAT_PREVIOUS_ANSWER"}
    if OOS_EXACT_PATTERN.fullmatch(text) and not find_businesses(text):
        return {"route": "OUT_OF_SCOPE", "action": "EXPLICIT_NON_KDIC"}
    return None

# Intent 규칙은 단어가 아니라 완성 구문을 우선한다.
HIGH_PRECISION_INTENT_RULES = [
    ("DOCUMENTS", [r"필요한?\s*(?:서류|증빙)", r"제출(?:해야 하는|할)?\s*서류", r"(?:서류|증빙)(?:가|는|은)?\s*무엇", r"구비\s*서류", r"신분증", r"위임장", r"준비\s*서류"]),
    ("STATUS", [r"어디(?:서|에서)\s*(?:확인|조회)", r"조회\s*(?:방법|결과)", r"처리\s*결과", r"진행\s*상황", r"지급\s*정보.*보는\s*방법"]),
    ("APPLICATION", [r"신청\s*(?:방법|절차)", r"접수\s*(?:방법|절차)", r"제출\s*방법", r"신고\s*채널", r"어떻게\s*(?:신청|접수)", r"(?:온라인|방문|직접).*신청.*(?:가능|할\s*수)", r"취소.*방법", r"철회.*방법"]),
    ("TIME", [r"언제(?:부터|까지)", r"신청.*(?:기한|기간)", r"처리\s*기간", r"소요\s*(?:기간|시간)", r"얼마나\s*걸"]),
    ("AMOUNT", [r"보호\s*한도", r"지급\s*금액", r"금액\s*계산", r"금액.*얼마(?:여야|이어야)", r"계산\s*(?:기준|방법)", r"수수료\s*(?:금액|비용)", r"얼마나\s*(?:감면|지급|보상|돌려|받)"]),
    ("CONTACT", [r"연락처", r"전화번호", r"문의처", r"어디로\s*연락", r"어느\s*기관.*문의"]),
    ("ELIGIBILITY", [r"신청\s*(?:대상|자격|요건)", r"가능한\s*대상", r"제외되는?\s*경우", r"받을\s*수\s*있", r"신청할\s*수\s*있", r"포함되", r"어떤\s*경우.*(?:지급|지원|보호)", r"(?:대상|자격)에\s*해당", r"(?:예금|계좌|금융상품|상품|원금|이자).{0,20}보호(?:가)?\s*되", r"지원\s*대상"]),
    ("OVERVIEW", [r"무엇(?:인가요|인지)", r"의미", r"정의", r"차이", r"종류", r"개요", r"설명"]),
]
WEAK_INTENT_RULES = [
    ("DOCUMENTS", [r"서류", r"증빙"]), ("STATUS", [r"조회", r"확인"]),
    ("APPLICATION", [r"신청", r"접수", r"절차", r"제출"]),
    ("TIME", [r"기간", r"기한", r"시점"]), ("AMOUNT", [r"한도", r"금액", r"계산", r"비용", r"포상금"]),
    ("CONTACT", [r"연락", r"문의", r"전화"]), ("ELIGIBILITY", [r"대상", r"자격", r"요건", r"조건", r"가능"]),
    ("OVERVIEW", [r"설명", r"관계", r"방식"]),
]

def match_intent(text: str, rules: list[tuple[str, list[str]]]) -> tuple[str | None, str | None]:
    for intent, patterns in rules:
        for pattern in patterns:
            if re.search(pattern, text, flags=re.I):
                return intent, pattern
    return None, None

def match_all_intents(text: str, rules: list[tuple[str, list[str]]]) -> list[tuple[str, str]]:
    matches = []
    for intent, patterns in rules:
        for pattern in patterns:
            if re.search(pattern, text, flags=re.I):
                matches.append((intent, pattern))
                break
    return matches

def resolve_intent_v3(text: str, model_intent: str | None) -> tuple[str | None, str, str | None]:
    high_matches = match_all_intents(text, HIGH_PRECISION_INTENT_RULES)
    high_intents = list(dict.fromkeys(intent for intent, _ in high_matches))
    if len(high_intents) > 1:
        if model_intent in INTENTS:
            pattern = next((p for i, p in high_matches if i == model_intent), high_matches[0][1])
            return model_intent, "RULE_AMBIGUOUS_MODEL_KEPT", pattern
        return None, "RULE_AMBIGUOUS_UNKNOWN", high_matches[0][1]
    if len(high_intents) == 1:
        high, pattern = high_matches[0]
        if model_intent == high:
            return high, "RULE_CONFIRMED", pattern
        return high, "RULE_OVERRIDE", pattern
    if model_intent in INTENTS:
        weak, weak_pattern = match_intent(text, WEAK_INTENT_RULES)
        if weak and weak != model_intent:
            return model_intent, "RULE_CONFLICT_MODEL_KEPT", weak_pattern
        return model_intent, "MODEL", None
    weak, weak_pattern = match_intent(text, WEAK_INTENT_RULES)
    return weak, "RULE_FILLED_UNKNOWN" if weak else "UNKNOWN", weak_pattern

def query_analysis_schema_v3() -> dict[str, Any]:
    business_value = {"type": "string", "enum": [*BUSINESS_FUNCTIONS, "UNKNOWN"]}
    intent_value = {"type": "string", "enum": [*INTENTS, "UNKNOWN"]}
    return {
        "type": "object",
        "properties": {
            "needs": {
                "type": "array", "maxItems": 6,
                "items": {
                    "type": "object",
                    "properties": {
                        "need_id": {"type": "string"}, "query": {"type": "string"},
                        "business_function": business_value, "intent": intent_value,
                        "target_type": {"type": "string"},
                        "case_details": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": ["need_id", "query", "business_function", "intent", "target_type", "case_details"],
                },
            },
        },
        "required": ["needs"],
    }

SYSTEM_PROMPT_V3 = f"""
역할: 예금보험공사 RAG의 Atomic Need 분석기.
출력: JSON Schema만 준수한다. Route와 확인 질문은 결정하지 않는다.

업무 사전:
- 예금자보호제도: 보호대상, 보호한도, 금융상품 보호 여부
- 예금보험금 안내: 보험사고, 예금보험금, 가지급금, 개산지급금
- 고객 미수령금 신청: 미수령금, 파산배당금, 정산금, 지급대행점, 상속인 금융거래 조회
- 착오송금 반환 신청: 반환지원, 매입계약, 지급명령, 강제집행
- 채무조정 안내: 개인회생, 개인파산, 워크아웃, 신용회복지원, 면책, 부채증명원
- 은닉재산 신고: 금융부실관련자, 차명재산, 신고, 포상금

규칙:
1. 사용자에게 다시 질문하지 말고 현재 입력에서 검색할 Need를 최대한 생성한다.
2. 서로 다른 정보 요구는 N1, N2로 분리한다. 서류와 제출방법은 서로 다른 Need다.
3. 같은 Intent여도 검색 대상·상황이 다르면 분리한다.
4. 비교 질문은 대상별 Cartesian 분리보다 정보 차원별로 분리한다.
   예: 개인회생과 워크아웃의 조건과 변제방식 → 조건 비교, 변제방식 비교의 두 Need.
5. query는 해당 Need만 독립 검색 가능한 문장으로 쓰고 원문 의미를 보존한다.
6. 원문이 이미 독립적이면 불필요하게 재작성하지 않는다.
7. 업무나 Intent가 불확실하면 UNKNOWN을 사용하되 Need를 삭제하지 않는다.
8. target_type과 case_details는 원문에 명시된 검색 조건만 기록한다.

허용 business_function: {BUSINESS_FUNCTIONS}
허용 intent: {INTENTS}
"""

class HCX007AtomicNeedClientV3(HCX007StructuredClient):
    def analyze(self, payload: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any]]:
        started = time.perf_counter()
        attempts = []
        totals = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        last_error = None
        body = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_V3},
                {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
            ],
            "topP": self.config.top_p, "topK": self.config.top_k,
            "maxCompletionTokens": self.config.max_completion_tokens,
            "temperature": self.config.temperature, "repetitionPenalty": 1.05,
            "thinking": {"effort": "none"}, "stop": [],
            "responseFormat": {"type": "json", "schema": query_analysis_schema_v3()},
        }
        for attempt in range(1, self.config.max_api_attempts + 1):
            attempt_started = time.perf_counter()
            request_id = str(uuid.uuid4())
            try:
                response = self.session.post(
                    self.url,
                    headers={
                        "Authorization": f"Bearer {self.api_key}",
                        "X-NCP-CLOVASTUDIO-REQUEST-ID": request_id,
                        "Content-Type": "application/json", "Accept": "application/json",
                    },
                    json=body, timeout=self.config.timeout_seconds,
                )
                if response.status_code >= 400:
                    try:
                        error_body = response.json()
                    except Exception:
                        error_body = {"text": response.text[:1000]}
                    retryable = response.status_code in {408, 429, 500, 502, 503, 504}
                    attempts.append({
                        "attempt": attempt, "request_id": request_id, "http_status": response.status_code,
                        "retryable": retryable, "error": error_body,
                        "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                    })
                    last_error = f"HTTP {response.status_code}: {error_body}"
                    if retryable and attempt < self.config.max_api_attempts:
                        retry_after = response.headers.get("Retry-After")
                        delay = float(retry_after) if retry_after and retry_after.replace(".", "", 1).isdigit() else 1.5 + random.random()
                        time.sleep(min(delay, 10.0))
                        continue
                    raise HCXAPIError(last_error, error_type="API_RETRYABLE" if retryable else "API_FATAL", telemetry={
                        "api_request_count": len(attempts), "attempts": attempts, **totals,
                        "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                    })
                envelope = response.json()
                result = envelope.get("result", envelope)
                usage = result.get("usage") or {}
                used = {
                    "prompt_tokens": int(usage.get("promptTokens") or 0),
                    "completion_tokens": int(usage.get("completionTokens") or 0),
                    "total_tokens": int(usage.get("totalTokens") or 0),
                }
                for key in totals:
                    totals[key] += used[key]
                content = str((result.get("message") or {}).get("content") or "")
                parsed = json.loads(content)
                if not isinstance(parsed, dict):
                    raise TypeError("모델 결과의 최상위가 object가 아닙니다.")
                attempts.append({
                    "attempt": attempt, "request_id": request_id, "http_status": response.status_code,
                    **used, "finish_reason": result.get("finishReason"),
                    "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                })
                return parsed, {
                    "api_request_count": len(attempts), "attempts": attempts, **totals,
                    "latency_ms": round((time.perf_counter() - started) * 1000, 3),
                }
            except HCXAPIError:
                raise
            except (requests.Timeout, requests.ConnectionError) as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                attempts.append({
                    "attempt": attempt, "request_id": request_id, "retryable": True, "error": last_error,
                    "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                })
                if attempt < self.config.max_api_attempts:
                    time.sleep(1.5 + random.random())
                    continue
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                attempts.append({
                    "attempt": attempt, "request_id": request_id, "retryable": False, "error": last_error,
                    "latency_ms": round((time.perf_counter() - attempt_started) * 1000, 3),
                })
                break
        raise HCXAPIError(last_error or "HCX-007 호출 실패", error_type="API_OR_PARSE_ERROR", telemetry={
            "api_request_count": len(attempts), "attempts": attempts, **totals,
            "latency_ms": round((time.perf_counter() - started) * 1000, 3),
        })

def validate_atomic_needs_v3(raw: dict[str, Any], normalized_query: str) -> tuple[list[dict[str, Any]], list[str]]:
    warnings = []
    rows = raw.get("needs") if isinstance(raw.get("needs"), list) else []
    needs = []
    for index, row in enumerate(rows[:6], 1):
        if not isinstance(row, dict):
            warnings.append(f"N{index}_NOT_OBJECT")
            continue
        query = str(row.get("query") or normalized_query).strip() or normalized_query
        business = row.get("business_function")
        business = business if business in BUSINESS_FUNCTIONS else None
        model_intent = normalize_intent(row.get("intent"))
        target = str(row.get("target_type") or "").strip() or None
        cases = row.get("case_details")
        cases = [str(x).strip() for x in cases if str(x).strip()] if isinstance(cases, list) else []
        needs.append({
            "need_id": f"N{index}", "query": query,
            "business_function": business, "business_source": "MODEL" if business else None,
            "model_intent": model_intent, "intent": model_intent, "intent_source": "MODEL" if model_intent else "UNKNOWN",
            "intent_rule_pattern": None, "target_type": target, "case_details": cases,
            "user_role": None, "user_role_source": None,
            "applicant_type": None, "applicant_type_source": None,
        })
    if not needs:
        warnings.append("EMPTY_MODEL_NEEDS_RECOVERED")
        needs = [create_retrieve_need_v3(normalized_query)]
    return needs, warnings

def create_retrieve_need_v3(query: str, business: str | None = None) -> dict[str, Any]:
    intent, source, pattern = resolve_intent_v3(query, None)
    return {
        "need_id": "N1", "query": query,
        "business_function": business, "business_source": "ORIGINAL" if business else None,
        "model_intent": None, "intent": intent, "intent_source": source, "intent_rule_pattern": pattern,
        "target_type": None, "case_details": [],
        "user_role": None, "user_role_source": None,
        "applicant_type": None, "applicant_type_source": None,
    }

GENERIC_BUSINESS_CLARIFY_PATTERN = re.compile(
    r"^(?:신청\s*(?:방법|자격|기한|과정)|접수\s*(?:방법|절차)|제출해야 하는 서류|필요한 서류|"
    r"조회는 어디에서|온라인으로 신청|방문해서 접수|접수 후 처리 기간|신청 대상|신청 자격과 제외 조건|"
    r"신청 과정에서 수수료나 비용|처리 결과는 어디에서 확인|이미 접수한 신청을 취소|"
    r"문의하거나 접수하려면 어느 기관|제가 신청 대상에 해당|본인 대신 대리인이 신청|"
    r"상속인이 신청하거나 받을 수).*$", re.I,
)
TARGET_DEMONSTRATIVE_PATTERN_V3 = re.compile(
    r"(?:^|[\s,.(])(?:제가\s*(?:말한|가입한|가진|본)\s*)?이\s*(?:금융상품|계좌|상품|돈|송금|거래|금액|채무|재산)(?:을|를|이|가|도|은|는|의|이나|과|와|\s|[,.!?]|$)"
)
TARGET_REFERENCE_PHRASES_V3 = [
    "제가 가입한 상품", "어떤 송금 건", "어떤 예금에 대해", "어떤 돈을 신청", "어떤 예금이나 금융상품",
]
APPLICANT_REFERENCE_PHRASES_V3 = [
    "제 신청 유형", "제 신청 자격", "제 경우", "누구를 신청인", "누가 방문", "신고 주체 유형", "신청인란",
]
CASE_REFERENCE_PHRASES_V3 = [
    "제 상황", "현재 상황", "제 채무 상태", "제 신고 상황", "반려", "거절", "보완 요청", "진행되지 않",
    "여러 금융회사에 예금", "여러 계좌에 나뉘",
]

def has_resolved_target(query: str, needs: list[dict[str, Any]]) -> bool:
    if any(n.get("target_type") for n in needs):
        return True
    demonstrative = TARGET_DEMONSTRATIVE_PATTERN_V3.search(query)
    if not demonstrative:
        return False
    phrase = demonstrative.group(0)
    # 지시 표현 자체만 있을 뿐 구체 상품명·거래정보는 없으므로 unresolved로 본다.
    return False if phrase else True

def detect_blocking_slot_v3(
    query: str, needs: list[dict[str, Any]], context: dict[str, Any]
) -> tuple[str | None, str | None]:
    confirmed = context.get("confirmed", {}) if context.get("used") else {}
    if GENERIC_BUSINESS_CLARIFY_PATTERN.fullmatch(query.strip()) and not find_businesses(query) and not confirmed.get("business_function"):
        return "business_function", "GENERIC_BUSINESS_EXACT_MATCH"
    if any(x in query for x in APPLICANT_REFERENCE_PHRASES_V3):
        if not confirmed.get("applicant_type") and not extract_explicit_value(query, APPLICANT_PATTERNS):
            return "applicant_type", "UNRESOLVED_APPLICANT_REFERENCE"
    if any(x in query for x in TARGET_REFERENCE_PHRASES_V3) or TARGET_DEMONSTRATIVE_PATTERN_V3.search(query):
        if not confirmed.get("target_type") and not has_resolved_target(query, needs):
            return "target_type", "UNRESOLVED_TARGET_REFERENCE"
    if any(x in query for x in CASE_REFERENCE_PHRASES_V3):
        return "case_details", "PERSONAL_CASE_REQUIRES_DETAILS"
    return None, None

STRONG_BUSINESS_KEYWORDS_V31 = {
    "예금자보호제도": ["예금자보호", "보호한도", "예금 보호", "보호 한도"],
    "예금보험금 안내": ["예금보험금", "보험금 지급", "1종 보험사고", "2종 보험사고"],
    "고객 미수령금 신청": ["미수령금", "파산배당금", "지급대행점", "상속인 금융거래 조회"],
    "착오송금 반환 신청": ["착오송금", "착오 송금", "반환지원", "매입계약", "지급명령", "강제집행"],
    "채무조정 안내": ["채무조정", "신용회복지원", "개인회생", "개인파산", "워크아웃", "부채증명원"],
    "은닉재산 신고": ["은닉재산", "은닉 재산", "금융부실관련자", "차명재산", "차명 재산", "신고 포상금"],
}
WEAK_OR_CROSS_BUSINESS_TERMS_V31 = {
    "예금보험금 안내": ["보험사고", "가지급금", "개산지급금"],
    "고객 미수령금 신청": ["개산지급금 정산금"],
}
MULTI_DECOMPOSITION_SIGNAL_V31 = re.compile(
    r"각각|뿐만\s*아니라|함께\s*알려|동시에\s*알려|"
    r"(?:와|과).{0,30}(?:관계|차이|비교)|"
    r"(?:대상|기준|방법|절차|서류|금액|시점).{0,25}(?:와|과|그리고).{0,25}(?:대상|기준|방법|절차|서류|금액|시점)",
    re.I,
)

def matched_terms_v31(text: str, mapping: dict[str, list[str]]) -> dict[str, list[str]]:
    lowered = (text or "").lower()
    return {
        business: [term for term in terms if term.lower() in lowered]
        for business, terms in mapping.items()
        if any(term.lower() in lowered for term in terms)
    }

def decomposition_status_v31(original: str, needs: list[dict[str, Any]]) -> str:
    if not needs or any(not (need.get("query") or "").strip() for need in needs):
        return "PARTIAL"
    if len(needs) == 1 and MULTI_DECOMPOSITION_SIGNAL_V31.search(original):
        return "PARTIAL"
    return "COMPLETE"

def apply_business_and_intent_rules_v31(
    needs: list[dict[str, Any]], original: str
) -> tuple[list[dict[str, Any]], list[str]]:
    actions = []
    need_count = len(needs)
    for need in needs:
        query = need.get("query") or original
        explicit_businesses = find_businesses(query)
        # V3의 오류 원인이었던 복합 질문 원문 업무의 전체 need 전파를 금지한다.
        if not explicit_businesses and need_count == 1:
            explicit_businesses = find_businesses(original)
        if len(explicit_businesses) == 1 and explicit_businesses[0] != need.get("business_function"):
            need["model_business_function"] = need.get("business_function")
            need["business_function"] = explicit_businesses[0]
            need["business_source"] = "ORIGINAL"
            actions.append(f"{need['need_id']}_BUSINESS_ORIGINAL_OVERRIDE")

        model_intent = need.get("model_intent")
        final_intent, source, pattern = resolve_intent_v3(query, model_intent)
        # V3 FULL에서 N2 override는 개선 0, 회귀 3이었다. 알려진 모델 intent는 보존한다.
        if need.get("need_id") != "N1" and source == "RULE_OVERRIDE" and model_intent in INTENTS:
            final_intent = model_intent
            source = "RULE_CONFLICT_MODEL_KEPT_V31"
            actions.append(f"{need['need_id']}_RULE_OVERRIDE_BLOCKED_V31")
        need["intent"] = final_intent
        need["intent_source"] = source
        need["intent_rule_pattern"] = pattern
        if source in {
            "RULE_OVERRIDE", "RULE_FILLED_UNKNOWN", "RULE_CONFLICT_MODEL_KEPT",
            "RULE_CONFLICT_MODEL_KEPT_V31", "RULE_AMBIGUOUS_MODEL_KEPT", "RULE_AMBIGUOUS_UNKNOWN",
        }:
            actions.append(f"{need['need_id']}_{source}")

    deduped, seen = [], set()
    for need in needs:
        key = (
            need.get("business_function"), need.get("intent"),
            normalize_query(need.get("query") or "")["normalized_query"],
            need.get("target_type"), tuple(sorted(need.get("case_details") or [])),
        )
        if key in seen:
            actions.append("EXACT_DUPLICATE_NEED_COLLAPSED")
            continue
        seen.add(key)
        deduped.append(need)
    for index, need in enumerate(deduped, 1):
        need["need_id"] = f"N{index}"
    return deduped, list(dict.fromkeys(actions))

def annotate_business_safety_v31(
    needs: list[dict[str, Any]], original: str
) -> list[dict[str, Any]]:
    decomposition = decomposition_status_v31(original, needs)
    need_count = len(needs)
    original_candidates = find_businesses(original)
    for need in needs:
        query = need.get("query") or original
        business = need.get("business_function")
        source = need.get("business_source") or "UNKNOWN"
        query_candidates = find_businesses(query)
        candidates = query_candidates or (original_candidates if need_count == 1 else [])
        candidates = list(dict.fromkeys(candidates))
        strong_matches = matched_terms_v31(query, STRONG_BUSINESS_KEYWORDS_V31)
        weak_matches = matched_terms_v31(query, WEAK_OR_CROSS_BUSINESS_TERMS_V31)
        if not strong_matches and need_count == 1:
            strong_matches = matched_terms_v31(original, STRONG_BUSINESS_KEYWORDS_V31)
        if not weak_matches and need_count == 1:
            weak_matches = matched_terms_v31(original, WEAK_OR_CROSS_BUSINESS_TERMS_V31)

        model_business = need.get("model_business_function")
        model_rule_conflict = bool(model_business and business and model_business != business)
        cross_business_ambiguity = len(candidates) > 1
        strong_for_business = business in strong_matches

        if source == "MANUAL":
            confidence = 1.0
        elif source == "CONTEXT":
            confidence = 0.995
        elif strong_for_business and len(candidates) == 1:
            confidence = 0.99
        elif business in candidates and len(candidates) == 1:
            confidence = 0.90
        elif business:
            confidence = 0.70
        else:
            confidence = 0.0
        candidate_margin = 1.0 if len(candidates) <= 1 else 0.0

        denial_reasons = []
        if decomposition != "COMPLETE": denial_reasons.append("INCOMPLETE_DECOMPOSITION")
        if not business: denial_reasons.append("NO_BUSINESS_CANDIDATE")
        if len(candidates) > 1: denial_reasons.append("MULTIPLE_BUSINESS_CANDIDATES")
        if model_rule_conflict: denial_reasons.append("MODEL_RULE_CONFLICT")
        if cross_business_ambiguity: denial_reasons.append("CROSS_BUSINESS_AMBIGUITY")
        if source not in {"MANUAL", "CONTEXT"} and not strong_for_business:
            denial_reasons.append("NO_STRONG_EXPLICIT_EVIDENCE")
        if confidence < HARD_BUSINESS_CONFIDENCE_THRESHOLD:
            denial_reasons.append("LOW_BUSINESS_CONFIDENCE")
        if candidate_margin < HARD_BUSINESS_MARGIN_THRESHOLD:
            denial_reasons.append("LOW_CANDIDATE_MARGIN")

        need["business_candidates"] = [
            {
                "value": candidate,
                "confidence": 0.99 if candidate in strong_matches else 0.80,
                "strong_evidence": strong_matches.get(candidate, []),
                "weak_evidence": weak_matches.get(candidate, []),
            }
            for candidate in candidates
        ]
        need["decomposition_status"] = decomposition
        need["model_rule_conflict"] = model_rule_conflict
        need["cross_business_ambiguity"] = cross_business_ambiguity
        need["business_confidence"] = confidence
        need["business_candidate_margin"] = candidate_margin
        need["hard_filter_eligible"] = not denial_reasons
        need["hard_filter_denial_reasons"] = list(dict.fromkeys(denial_reasons))
    return needs

def decide_route_v3(
    query: str, needs: list[dict[str, Any]], context: dict[str, Any]
) -> tuple[str, list[str], str | None]:
    blocking_slot, reason = detect_blocking_slot_v3(query, needs, context)
    if blocking_slot:
        return "CLARIFY", [reason], blocking_slot
    businesses = [n.get("business_function") for n in needs if n.get("business_function") in BUSINESS_FUNCTIONS]
    if not businesses or any(n.get("business_function") not in BUSINESS_FUNCTIONS for n in needs):
        return "RETRIEVE_RELAXED", ["BUSINESS_UNCERTAIN_BROAD_RETRIEVAL"], None
    return "RETRIEVE", ["SEARCH_CONDITIONS_AVAILABLE"], None

def enrich_evidence_v3(
    needs: list[dict[str, Any]], original: str, context: dict[str, Any], manual: dict[str, Any]
) -> list[dict[str, Any]]:
    explicit_businesses = find_businesses(original)
    explicit_role = extract_explicit_value(original, USER_ROLE_PATTERNS)
    explicit_applicant = extract_explicit_value(original, APPLICANT_PATTERNS)
    confirmed = context.get("confirmed", {}) if context.get("used") else {}
    for need in needs:
        business = need.get("business_function")
        if manual.get("business_function") == business:
            need["business_source"] = "MANUAL"
        elif business in explicit_businesses:
            need["business_source"] = "ORIGINAL"
        elif confirmed.get("business_function") == business:
            need["business_source"] = "CONTEXT"
        elif business:
            need["business_source"] = need.get("business_source") or "MODEL"
        if manual.get("user_role") in USER_ROLES:
            need["user_role"], need["user_role_source"] = manual["user_role"], "MANUAL"
        elif explicit_role:
            need["user_role"], need["user_role_source"] = explicit_role, "ORIGINAL"
        elif confirmed.get("user_role") in USER_ROLES:
            need["user_role"], need["user_role_source"] = confirmed["user_role"], "CONTEXT"
        if manual.get("applicant_type") in APPLICANT_TYPES:
            need["applicant_type"], need["applicant_type_source"] = manual["applicant_type"], "MANUAL"
        elif explicit_applicant:
            need["applicant_type"], need["applicant_type_source"] = explicit_applicant, "ORIGINAL"
        elif confirmed.get("applicant_type") in APPLICANT_TYPES:
            need["applicant_type"], need["applicant_type_source"] = confirmed["applicant_type"], "CONTEXT"
    return needs

def build_keyword_query_v3(need: dict[str, Any]) -> str:
    values = [need.get("business_function"), need.get("intent"), need.get("target_type")]
    values.extend(need.get("case_details") or [])
    if need.get("user_role_source") in {"ORIGINAL", "CONTEXT", "MANUAL"}:
        values.append(need.get("user_role"))
    if need.get("applicant_type_source") in {"ORIGINAL", "CONTEXT", "MANUAL"}:
        values.append(need.get("applicant_type"))
    return " ".join(str(x) for x in values if x and x != "UNKNOWN")

def build_query_plans_v31(analysis: dict[str, Any], config: PipelineConfig) -> list[dict[str, Any]]:
    if analysis["route"] not in {"RETRIEVE", "RETRIEVE_RELAXED"}:
        return []
    relaxed = analysis["route"] == "RETRIEVE_RELAXED"
    plans = []
    for need in analysis["needs"]:
        business, source = need.get("business_function"), need.get("business_source")
        eligible = bool(need.get("hard_filter_eligible"))
        denial_reasons = need.get("hard_filter_denial_reasons") or []
        if relaxed or not business:
            business_filter = {
                "mode": "NONE", "value": None, "soft_hint": business,
                "candidates": need.get("business_candidates") or [],
                "evidence": source or "UNKNOWN", "denial_reasons": denial_reasons,
            }
            fallback_chain = []
        elif eligible:
            business_filter = {
                "mode": "HARD", "value": business, "soft_hint": None,
                "candidates": need.get("business_candidates") or [],
                "evidence": source, "denial_reasons": [],
            }
            fallback_chain = ["SOFT", "NONE"]
        else:
            business_filter = {
                "mode": "SOFT", "value": None, "soft_hint": business,
                "candidates": need.get("business_candidates") or [],
                "evidence": source or "UNKNOWN", "denial_reasons": denial_reasons,
            }
            fallback_chain = ["NONE"]
        intent = need.get("intent")
        intent_weight = 0.20 if need.get("intent_source") in {
            "RULE_OVERRIDE", "RULE_CONFIRMED", "RULE_FILLED_UNKNOWN"
        } else config.intent_soft_boost
        plans.append({
            "need_id": need["need_id"],
            "retrieval_mode": "RELAXED" if relaxed else "STANDARD",
            "semantic_query": need.get("query"),
            "keyword_query": build_keyword_query_v3(need) or need.get("query"),
            "business_filter": business_filter,
            "filter_safety": {
                "hard_filter_eligible": eligible,
                "decomposition_status": need.get("decomposition_status"),
                "model_rule_conflict": need.get("model_rule_conflict"),
                "cross_business_ambiguity": need.get("cross_business_ambiguity"),
                "business_confidence": need.get("business_confidence"),
                "candidate_margin": need.get("business_candidate_margin"),
                "denial_reasons": denial_reasons,
            },
            "fallback_policy": {
                "enabled": bool(fallback_chain),
                "on": ["NO_RESULTS", "LOW_TOP_SCORE", "LOW_COVERAGE"],
                "next_filter_modes": fallback_chain,
                "fail_open": True,
            },
            "intent_boost": {
                "mode": "SOFT" if intent in INTENTS else "NONE", "value": intent,
                "weight": intent_weight if intent in INTENTS else 0.0,
                "evidence": need.get("intent_source"),
            },
            "entities": {
                "user_role": need.get("user_role"), "user_role_source": need.get("user_role_source"),
                "applicant_type": need.get("applicant_type"), "applicant_type_source": need.get("applicant_type_source"),
                "target_type": need.get("target_type"), "case_details": need.get("case_details") or [],
            },
        })
    return plans

def relax_query_plan_v31(plan: dict[str, Any], reason: str) -> dict[str, Any]:
    """검색기가 결과 부족 시 호출할 수 있는 fail-open helper."""
    relaxed_plan = json.loads(json.dumps(plan, ensure_ascii=False))
    current_mode = relaxed_plan.get("business_filter", {}).get("mode")
    if current_mode == "HARD":
        relaxed_plan["business_filter"]["mode"] = "SOFT"
        relaxed_plan["business_filter"]["soft_hint"] = relaxed_plan["business_filter"].get("value")
        relaxed_plan["business_filter"]["value"] = None
    elif current_mode == "SOFT":
        relaxed_plan["business_filter"]["mode"] = "NONE"
        relaxed_plan["retrieval_mode"] = "RELAXED"
    relaxed_plan.setdefault("fallback_history", []).append({"from": current_mode, "reason": reason})
    return relaxed_plan

class KDICLightweightRAGAnalyzerV31:
    def __init__(self, client: HCX007AtomicNeedClientV3, config: PipelineConfig | None = None):
        self.client = client
        self.config = config or client.config

    def run(self, query: str, *, conversation_state=None, manual_selection=None) -> dict[str, Any]:
        started = time.perf_counter()
        normalized = normalize_query(query)
        original, text = normalized["original_query"], normalized["normalized_query"]
        manual = dict(manual_selection or {})
        context = build_context(text, conversation_state, self.config.max_context_turns)
        fast = detect_fast_path_v3(text, conversation_state)
        if fast:
            analysis = {"route": fast["route"], "needs": [], "missing_information": []}
            return {
                "pipeline_version": PIPELINE_VERSION, "analysis_status": "FAST_PATH",
                "original_query": original, "normalized_query": text, "context": context,
                "gate_reasons": ["FAST_PATH"], "rule_actions": [], "blocking_slot": None,
                "analysis": analysis, "fast_path": fast, "validation_warnings": [], "query_plans": [],
                "runtime": {"api_request_count": 0, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
                            "latency_ms": round((time.perf_counter() - started) * 1000, 3), "attempts": []},
            }
        payload = {"query": text, "confirmed_context": context["confirmed"], "recent_turns": context["recent_turns"], "manual_selection": manual}
        try:
            raw, telemetry = self.client.analyze(payload)
            needs, warnings = validate_atomic_needs_v3(raw, text)
            status = "REPAIRED" if warnings else "OK"
        except HCXAPIError as exc:
            telemetry = exc.telemetry
            needs = [create_retrieve_need_v3(text, find_businesses(text)[0] if len(find_businesses(text)) == 1 else None)]
            warnings = ["MODEL_ANALYSIS_FAILED_USED_FALLBACK"]
            status = "FALLBACK"
        model_needs = json.loads(json.dumps(needs, ensure_ascii=False))
        needs, rule_actions = apply_business_and_intent_rules_v31(needs, original)
        needs = enrich_evidence_v3(needs, original, context, manual)
        needs = annotate_business_safety_v31(needs, original)
        route, gate_reasons, blocking_slot = decide_route_v3(text, needs, context)
        analysis_needs = [] if route == "CLARIFY" else needs
        analysis = {"route": route, "needs": analysis_needs, "missing_information": [blocking_slot] if blocking_slot else []}
        plans = build_query_plans_v31(analysis, self.config)
        return {
            "pipeline_version": PIPELINE_VERSION, "analysis_status": status,
            "original_query": original, "normalized_query": text, "context": context,
            "model_needs": model_needs, "gate_reasons": gate_reasons,
            "rule_actions": rule_actions, "blocking_slot": blocking_slot,
            "analysis": analysis, "validation_warnings": warnings, "query_plans": plans,
            "runtime": {**telemetry, "latency_ms": round((time.perf_counter() - started) * 1000, 3)},
        }

In [ ]:
%%writefile kdic_v31_v15_cross_rewrite.py
from __future__ import annotations

import copy
import math
import re
import time
from dataclasses import asdict, dataclass
from typing import Any, Mapping, Protocol, Sequence


PIPELINE_VERSION = "KDIC_V31_ANALYSIS_V15_CROSS_REWRITE_2026_08_18"
NUMBER_PATTERN = re.compile(r"\d+(?:[.,]\d+)?")
TOKEN_PATTERN = re.compile(r"[가-힣A-Za-z0-9]+")
NEGATION_TERMS = ("아닌", "아니", "않", "못", "제외", "불가", "없이", "없", "미해당")


class V31AnalyzerProtocol(Protocol):
    def run(
        self,
        query: str,
        *,
        conversation_state: Mapping[str, Any] | None = None,
        manual_selection: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]: ...


@dataclass(frozen=True)
class CrossRewritePolicy:
    original_weight: float = 0.40
    rewritten_total_weight: float = 0.60
    max_rewritten_queries: int = 4
    minimum_business_confidence: float = 0.70
    minimum_token_overlap: float = 0.25
    allow_soft_business_hint: bool = True

    def __post_init__(self) -> None:
        if not math.isclose(
            self.original_weight + self.rewritten_total_weight,
            1.0,
            abs_tol=1e-9,
        ):
            raise ValueError("원문과 재작성 질의의 가중치 합은 1이어야 합니다.")
        if self.max_rewritten_queries < 2:
            raise ValueError("교차업무 재작성 최대 개수는 2 이상이어야 합니다.")
        if not 0.0 <= self.minimum_business_confidence <= 1.0:
            raise ValueError("minimum_business_confidence는 0~1이어야 합니다.")
        if not 0.0 <= self.minimum_token_overlap <= 1.0:
            raise ValueError("minimum_token_overlap은 0~1이어야 합니다.")


def _clean_text(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _compact(value: Any) -> str:
    return re.sub(r"\s+", "", str(value or "")).lower()


def _ordered_unique(values: Sequence[Any]) -> list[str]:
    output: list[str] = []
    seen: set[str] = set()
    for value in values:
        text = _clean_text(value)
        key = _compact(text)
        if text and key not in seen:
            seen.add(key)
            output.append(text)
    return output


def _normalize_route(value: Any) -> str:
    text = str(value or "").strip().upper()
    mapping = {
        "RETRIEVE": "RETRIEVE",
        "RETRIEVE_RELAXED": "RETRIEVE",
        "CLARIFY": "CLARIFY",
        "DIRECT": "DIRECT_RESPONSE",
        "DIRECT_RESPONSE": "DIRECT_RESPONSE",
        "OUT_OF_SCOPE": "OUT_OF_SCOPE",
        "OOS": "OUT_OF_SCOPE",
    }
    return mapping.get(text, text)


def _businesses_from_needs(needs: Sequence[Mapping[str, Any]]) -> list[str]:
    return _ordered_unique(
        [need.get("business_function") for need in needs if need.get("business_function")]
    )


def _explicit_businesses(v31_module: Any, original: str) -> list[str]:
    finder = getattr(v31_module, "find_businesses", None)
    if not callable(finder):
        return []
    return _ordered_unique(list(finder(original) or []))


def detect_cross_business(
    *,
    original: str,
    needs: Sequence[Mapping[str, Any]],
    v31_module: Any,
) -> dict[str, Any]:
    explicit_businesses = _explicit_businesses(v31_module, original)
    need_businesses = _businesses_from_needs(needs)
    if len(explicit_businesses) >= 2:
        businesses = explicit_businesses
        source = "EXPLICIT_ORIGINAL_TERMS"
    elif len(need_businesses) >= 2:
        businesses = need_businesses
        source = "V31_STRUCTURED_NEEDS"
    else:
        businesses = _ordered_unique(explicit_businesses + need_businesses)
        source = "SINGLE_OR_UNKNOWN_BUSINESS"
    return {
        "is_cross_business": len(businesses) >= 2,
        "businesses": businesses,
        "explicit_businesses": explicit_businesses,
        "need_businesses": need_businesses,
        "evidence_source": source,
    }


def _token_overlap(original: str, rewritten: str) -> float:
    original_tokens = set(TOKEN_PATTERN.findall(original.lower()))
    rewritten_tokens = set(TOKEN_PATTERN.findall(rewritten.lower()))
    if not original_tokens:
        return 1.0
    return len(original_tokens & rewritten_tokens) / len(original_tokens)


def validate_cross_rewrite(
    *,
    original: str,
    needs: Sequence[Mapping[str, Any]],
    expected_businesses: Sequence[str],
    policy: CrossRewritePolicy,
) -> dict[str, Any]:
    issues: list[str] = []
    accepted_needs: list[dict[str, Any]] = []
    seen_queries: set[str] = set()

    for raw_need in needs[: policy.max_rewritten_queries]:
        need = copy.deepcopy(dict(raw_need))
        query = _clean_text(need.get("query"))
        business = _clean_text(need.get("business_function"))
        confidence = float(need.get("business_confidence") or 0.0)
        if not query:
            issues.append("EMPTY_REWRITTEN_QUERY")
            continue
        key = _compact(query)
        if key in seen_queries:
            issues.append("DUPLICATE_REWRITTEN_QUERY")
            continue
        seen_queries.add(key)
        if len(query) < 5:
            issues.append("REWRITTEN_QUERY_TOO_SHORT")
        if key == _compact(original):
            issues.append("REWRITTEN_QUERY_EQUALS_ORIGINAL")
        if not business:
            issues.append("REWRITTEN_QUERY_WITHOUT_BUSINESS")
        if confidence < policy.minimum_business_confidence:
            issues.append("LOW_BUSINESS_CONFIDENCE")
        need["query"] = query
        need["business_function"] = business or None
        need["business_confidence"] = confidence
        need["token_overlap"] = _token_overlap(original, query)
        accepted_needs.append(need)

    if len(accepted_needs) < 2:
        issues.append("TOO_FEW_REWRITTEN_QUERIES")

    covered_businesses = _businesses_from_needs(accepted_needs)
    missing_businesses = [
        business for business in expected_businesses if business not in covered_businesses
    ]
    if missing_businesses:
        issues.append("MISSING_CROSS_BUSINESS_COVERAGE")

    reconstructed = " ".join(str(need.get("query") or "") for need in accepted_needs)
    original_numbers = NUMBER_PATTERN.findall(original)
    missing_numbers = [number for number in original_numbers if number not in reconstructed]
    if missing_numbers:
        issues.append("MISSING_NUMERIC_CONSTRAINT")

    original_negations = [term for term in NEGATION_TERMS if term in original]
    missing_negations = [term for term in original_negations if term not in reconstructed]
    if missing_negations:
        issues.append("MISSING_NEGATION_CONSTRAINT")

    combined_overlap = _token_overlap(original, reconstructed)
    if combined_overlap < policy.minimum_token_overlap:
        issues.append("LOW_ORIGINAL_TOKEN_COVERAGE")

    safety_issues = []
    for need in accepted_needs:
        safety_issues.extend(need.get("hard_filter_denial_reasons") or [])
        if need.get("model_rule_conflict"):
            issues.append("MODEL_RULE_CONFLICT")
        if str(need.get("decomposition_status") or "") not in {"", "COMPLETE"}:
            issues.append("INCOMPLETE_V31_DECOMPOSITION")

    issues = _ordered_unique(issues)
    return {
        "accepted": not issues,
        "issues": issues,
        "rewritten_needs": accepted_needs,
        "expected_businesses": list(expected_businesses),
        "covered_businesses": covered_businesses,
        "missing_businesses": missing_businesses,
        "missing_numbers": missing_numbers,
        "missing_negations": missing_negations,
        "combined_token_overlap": combined_overlap,
        "v31_filter_safety_notes": _ordered_unique(safety_issues),
    }


def _original_plan(original: str) -> dict[str, Any]:
    return {
        "need_id": "FUSED",
        "variant_id": "ORIGINAL",
        "query_source": "ORIGINAL",
        "semantic_query": original,
        "keyword_query": original,
        "query_weight": 1.0,
        "business_function": None,
        "business_filter": {
            "mode": "NONE",
            "value": None,
            "soft_hint": None,
            "denial_reasons": ["HARD_DISABLED_BY_CROSS_REWRITE_POLICY"],
        },
    }


def build_search_plans(
    *,
    original: str,
    rewrite_validation: Mapping[str, Any],
    policy: CrossRewritePolicy,
) -> list[dict[str, Any]]:
    if not rewrite_validation.get("accepted"):
        return [_original_plan(original)]
    rewritten_needs = list(rewrite_validation.get("rewritten_needs") or [])
    sub_weight = policy.rewritten_total_weight / len(rewritten_needs)
    plans = [{
        **_original_plan(original),
        "variant_id": "ORIGINAL_ANCHOR",
        "query_source": "ORIGINAL_ANCHOR",
        "query_weight": policy.original_weight,
    }]
    for index, need in enumerate(rewritten_needs, start=1):
        business = _clean_text(need.get("business_function")) or None
        query = _clean_text(need.get("query"))
        plans.append({
            "need_id": str(need.get("need_id") or f"N{index}"),
            "variant_id": f"REWRITTEN_{index:02d}",
            "query_source": "V31_REWRITTEN_SUBQUERY",
            "semantic_query": query,
            "keyword_query": query,
            "query_weight": sub_weight,
            "business_function": business,
            "intent": need.get("intent"),
            "business_filter": {
                "mode": "SOFT" if policy.allow_soft_business_hint and business else "NONE",
                "value": None,
                "soft_hint": business if policy.allow_soft_business_hint else None,
                "denial_reasons": ["HARD_DISABLED_BY_CROSS_REWRITE_POLICY"],
            },
        })
    return plans


def query_plans_are_valid(route: str, plans: Sequence[Mapping[str, Any]]) -> bool:
    if route != "RETRIEVE":
        return len(plans) == 0
    if not plans:
        return False
    if not math.isclose(
        sum(float(plan.get("query_weight") or 0.0) for plan in plans),
        1.0,
        abs_tol=1e-9,
    ):
        return False
    for plan in plans:
        if not _clean_text(plan.get("semantic_query")):
            return False
        if not _clean_text(plan.get("keyword_query")):
            return False
        if float(plan.get("query_weight") or 0.0) <= 0:
            return False
        if str((plan.get("business_filter") or {}).get("mode") or "") == "HARD":
            return False
    return True


class KDICV31V15CrossRewriteAnalyzer:
    def __init__(
        self,
        v31_analyzer: V31AnalyzerProtocol,
        v31_module: Any,
        policy: CrossRewritePolicy | None = None,
    ) -> None:
        self.v31_analyzer = v31_analyzer
        self.v31_module = v31_module
        self.policy = policy or CrossRewritePolicy()

    def run(
        self,
        query: str,
        *,
        conversation_state: Mapping[str, Any] | None = None,
        manual_selection: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        started = time.perf_counter()
        base_started = time.perf_counter()
        base = self.v31_analyzer.run(
            query,
            conversation_state=conversation_state,
            manual_selection=manual_selection,
        )
        base_latency_ms = (time.perf_counter() - base_started) * 1000
        original = _clean_text(base.get("original_query") or query)
        analysis = dict(base.get("analysis") or {})
        v31_route = str(analysis.get("route") or "")
        route = _normalize_route(v31_route)
        needs = list(analysis.get("needs") or [])

        cross = detect_cross_business(
            original=original,
            needs=needs,
            v31_module=self.v31_module,
        )
        rewrite_called = bool(route == "RETRIEVE" and cross["is_cross_business"])
        if rewrite_called:
            validation = validate_cross_rewrite(
                original=original,
                needs=needs,
                expected_businesses=cross["businesses"],
                policy=self.policy,
            )
        else:
            validation = {
                "accepted": False,
                "issues": ["NOT_CROSS_BUSINESS"] if route == "RETRIEVE" else ["NO_RETRIEVAL_ROUTE"],
                "rewritten_needs": [],
                "expected_businesses": cross["businesses"],
                "covered_businesses": [],
                "missing_businesses": [],
                "missing_numbers": [],
                "missing_negations": [],
                "combined_token_overlap": 0.0,
                "v31_filter_safety_notes": [],
            }

        rewrite_accepted = bool(rewrite_called and validation["accepted"])
        if route == "RETRIEVE":
            plans = build_search_plans(
                original=original,
                rewrite_validation=validation if rewrite_accepted else {"accepted": False},
                policy=self.policy,
            )
        else:
            plans = []

        query_type = (
            "CROSS_BUSINESS"
            if cross["is_cross_business"]
            else ("SAME_BUSINESS_MULTI" if len(needs) >= 2 else "SINGLE")
        ) if route == "RETRIEVE" else "NO_RETRIEVAL"
        plan_valid = query_plans_are_valid(route, plans)
        warnings = list(base.get("validation_warnings") or [])
        if not plan_valid:
            warnings.append("COMBINED_QUERY_PLAN_INVALID")

        return {
            "pipeline_version": PIPELINE_VERSION,
            "analysis_status": "OK" if plan_valid else "INVALID_PLAN",
            "original_query": original,
            "normalized_query": base.get("normalized_query") or original,
            "route": route,
            "v31_route": v31_route,
            "query_type": query_type,
            "v31_analysis": analysis,
            "v31_model_needs": base.get("model_needs") or [],
            "v31_rule_actions": base.get("rule_actions") or [],
            "v31_gate_reasons": base.get("gate_reasons") or [],
            "cross_business": cross,
            "rewrite_called": rewrite_called,
            "rewrite_accepted": rewrite_accepted,
            "rewrite_validation": validation,
            "fallback_to_original": bool(route == "RETRIEVE" and rewrite_called and not rewrite_accepted),
            "search_plans": plans,
            "query_plan_valid": plan_valid,
            "hard_filter_count": sum(
                1 for plan in plans
                if str((plan.get("business_filter") or {}).get("mode") or "") == "HARD"
            ),
            "validation_warnings": _ordered_unique(warnings),
            "runtime": {
                "v31_analysis_latency_ms": base_latency_ms,
                "wrapper_latency_ms": (time.perf_counter() - started) * 1000 - base_latency_ms,
                "total_latency_ms": (time.perf_counter() - started) * 1000,
                "v31_runtime": base.get("runtime") or {},
            },
            "policy": asdict(self.policy),
        }


def analyze_query(
    query: str,
    *,
    analyzer: KDICV31V15CrossRewriteAnalyzer,
    conversation_state: Mapping[str, Any] | None = None,
    manual_selection: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    return analyzer.run(
        query,
        conversation_state=conversation_state,
        manual_selection=manual_selection,
    )

In [ ]:

from typing import Mapping, Sequence

import pandas as pd


LEGACY_HARD_GATES = {
    "execution_success_rate": {"operator": ">=", "threshold": 0.995},
    "retrieve_query_valid_rate": {"operator": ">=", "threshold": 0.99},
    "wrong_oos_count": {"operator": "==", "threshold": 0},
    "wrong_direct_count": {"operator": "==", "threshold": 0},
    "wrong_hard_filter_count": {"operator": "==", "threshold": 0},
    "clarify_precision": {"operator": ">=", "threshold": 0.95},
}

ANALYZER_LABELS = {
    "v15": "V1.5 추가질의 개선",
    "v31": "V3.1 교차업무만 재작성",
}
COMPARISON_BRANCH_INTERVAL_SECONDS = 3.0


def _comparison_plans_valid(route: str, plans: Sequence[Mapping[str, Any]]) -> bool:
    if route != "RETRIEVE":
        return len(plans) == 0
    if not plans:
        return False
    total = sum(float(plan.get("weight") or 0) for plan in plans)
    if not math.isclose(total, 1.0, abs_tol=1e-9):
        return False
    for plan in plans:
        if not str(plan.get("query") or "").strip():
            return False
        if float(plan.get("weight") or 0) <= 0:
            return False
        if str((plan.get("business_filter") or {}).get("mode") or "").upper() == "HARD":
            return False
    return True

In [ ]:
import time
import uuid

from kdic_decomposition_quality_core import BASELINE
from kdic_hcx007_resumable_decomposition_core import (
    HCX_DECOMPOSITION_ENDPOINT,
    ResumableBaselineDecomposer,
    TransportPolicy,
    _condition_record,
    _normalize_baseline_record,
)
from kdic_lightweight_query_ablation_core import AblationConfig, analyze_common


v15_transport_policy = TransportPolicy(
    request_delay_seconds=0.0,
    max_transport_retries=4,
    base_backoff_seconds=5.0,
    max_backoff_seconds=120.0,
    jitter_seconds=1.0,
    consecutive_429_cooldown_threshold=1,
    cooldown_seconds=65.0,
    timeout_seconds=HCX_REQUEST_TIMEOUT,
)
v15_decomposition_config = AblationConfig(
    llm_model=HCX_DECOMPOSITION_MODEL,
    llm_endpoint=HCX_DECOMPOSITION_ENDPOINT,
    llm_timeout_seconds=HCX_REQUEST_TIMEOUT,
    llm_min_confidence=V15_MIN_CONFIDENCE,
    max_subqueries=V15_MAX_SUBQUERIES,
)
V15_DECOMPOSER = ResumableBaselineDecomposer(
    HCX_API_KEY,
    cache_path=V15_CACHE_PATH,
    seed_cache_paths=[],
    config=v15_decomposition_config,
    transport_policy=v15_transport_policy,
)


def _route_response(route: str, common: dict[str, Any]) -> str:
    if route == "DIRECT_RESPONSE":
        return "안녕하세요. 예금보험공사 관련 제도와 신청 절차에 관해 질문해 주세요."
    if route == "OUT_OF_SCOPE":
        return "이 챗봇은 예금자보호, 예금보험금, 고객 미수령금, 착오송금 반환지원, 채무조정, 은닉재산 신고 관련 질문에 답변합니다."
    missing = common.get("missing_information") or []
    detail = " / ".join(str(value) for value in missing if str(value).strip())
    if detail:
        return f"정확한 안내를 위해 정보가 더 필요합니다: {detail}"
    return "어떤 업무에 관한 질문인지 선택해 주세요: 예금자보호, 예금보험금, 고객 미수령금, 착오송금 반환지원, 채무조정, 은닉재산 신고."


def analyze_v15_chat_query(question: str, previous_turns: Any = None) -> dict[str, Any]:
    started = time.perf_counter()
    common = analyze_common(
        f"CHAT_{uuid.uuid4().hex[:12]}",
        question,
        previous_turns=previous_turns,
    )
    route = str(common["route"])
    businesses = list(dict.fromkeys((common.get("complexity") or {}).get("businesses") or []))
    cross_candidate = bool(
        route == "RETRIEVE"
        and common.get("complex_candidate")
        and len(businesses) >= 2
    )
    record = None
    decomposition_latency_ms = 0.0
    if cross_candidate:
        decomposition_started = time.perf_counter()
        first = _normalize_baseline_record(
            common["normalized_question"],
            businesses,
            V15_DECOMPOSER.decompose(common["normalized_question"], businesses),
        )
        record = _condition_record(BASELINE, first, first, retry_called=False)
        decomposition_latency_ms = (time.perf_counter() - decomposition_started) * 1000

    accepted = bool((record or {}).get("final_accepted"))
    subqueries = list((record or {}).get("final_subqueries") or []) if accepted else []
    if route != "RETRIEVE":
        plans = []
    elif subqueries:
        sub_weight = V15_SUBQUERY_TOTAL_WEIGHT / len(subqueries)
        plans = [{
            "query": common["original_question"],
            "weight": V15_ORIGINAL_WEIGHT,
            "source": "ORIGINAL_ANCHOR",
        }]
        plans.extend({
            "query": query,
            "weight": sub_weight,
            "source": "DECOMPOSED",
        } for query in subqueries)
    else:
        plans = [{
            "query": common["original_question"],
            "weight": 1.0,
            "source": "ORIGINAL",
        }]

    return {
        "route": route,
        "route_reasons": common.get("route_reasons") or [],
        "businesses": businesses,
        "complexity": (common.get("complexity") or {}).get("question_type", "NONE"),
        "cross_business_candidate": cross_candidate,
        "decomposition_called": cross_candidate,
        "decomposition_accepted": accepted,
        "decomposition_status": (record or {}).get("final_status", "NOT_CALLED"),
        "decomposition_issues": (record or {}).get("final_issues") or [],
        "decomposition_cache_hit": bool((record or {}).get("first_cache_hit")),
        "subqueries": subqueries,
        "fallback_to_original": bool(cross_candidate and not accepted),
        "plans": plans,
        "route_response": _route_response(route, common) if route != "RETRIEVE" else "",
        "routing_latency_ms": float(common.get("common_latency_ms") or 0.0),
        "rule_latency_ms": float(common.get("rule_latency_ms") or 0.0),
        "decomposition_latency_ms": decomposition_latency_ms,
        "analysis_wall_latency_ms": (time.perf_counter() - started) * 1000,
    }


def retrieval_table_markdown(results: list[dict[str, Any]]) -> str:
    lines = [
        "### 검색 결과",
        "",
        "|순위|청크 ID|Dense 순위|BM25 순위|Min-Max 최고점|질의결합점수|제목 / 소제목|",
        "|---:|---|---:|---:|---:|---:|---|",
    ]
    for result in results:
        chunk = result["chunk"]
        dense_rank = result.get("dense_rank") or "-"
        bm25_rank = result.get("bm25_rank") or "-"
        title = " / ".join(
            part for part in [
                str(chunk.get("title") or "").replace("|", "\\|"),
                str(chunk.get("section_title") or "").replace("|", "\\|"),
            ] if part
        )
        lines.append(
            f"|{result['rank']}|{result['chunk_id']}|{dense_rank}|{bm25_rank}|"
            f"{float(result.get('minmax_score') or 0):.6f}|"
            f"{float(result.get('query_fusion_score') or 0):.6f}|{title}|"
        )
    return "\n".join(lines)


def analysis_markdown(analysis: dict[str, Any]) -> str:
    plans = analysis.get("plans") or []
    plan_text = "<br>".join(
        f"{index}. {plan['source']} · {plan['weight']:.3f} · {plan['query']}"
        for index, plan in enumerate(plans, start=1)
    ) or "검색 계획 없음"
    return (
        "### V1.5 질의분석\n\n"
        "|항목|결과|\n|---|---|\n"
        f"|최종 경로|{analysis['route']}|\n"
        f"|탐지 업무|{', '.join(analysis['businesses']) or '-'}|\n"
        f"|복합 후보|{analysis['complexity']}|\n"
        f"|교차업무 분해 호출|{analysis['decomposition_called']}|\n"
        f"|분해 승인|{analysis['decomposition_accepted']}|\n"
        f"|원문 fallback|{analysis['fallback_to_original']}|\n"
        f"|검색 계획|{plan_text}|"
    )


def latency_markdown(latency: dict[str, float]) -> str:
    return (
        "### 단계별 지연시간\n\n"
        "|단계|지연시간|\n|---|---:|\n"
        + "\n".join(f"|{key}|{value:,.1f}ms|" for key, value in latency.items())
    )

In [ ]:

from __future__ import annotations

import copy
import hashlib
import html
import time
from typing import Mapping, Sequence

import kdic_query_analyzer_v31 as v31
from kdic_context_policy_v2 import new_context_state, resolve_context_v2
from kdic_v31_v15_cross_rewrite import CrossRewritePolicy, KDICV31V15CrossRewriteAnalyzer


# ---------- V3.1: 단일·동일업무는 원문, 교차업무만 Need 재작성 ----------
V31_CONFIG = v31.PipelineConfig(
    model="HCX-007",
    max_completion_tokens=700,
    temperature=0.1,
    top_p=0.8,
    request_interval_seconds=1.05,
)
V31_CLIENT = v31.HCX007AtomicNeedClientV3(HCX_API_KEY, V31_CONFIG)
V31_BASE_ANALYZER = v31.KDICLightweightRAGAnalyzerV31(V31_CLIENT, V31_CONFIG)
V31_CROSS_POLICY = CrossRewritePolicy(
    original_weight=0.40,
    rewritten_total_weight=0.60,
    max_rewritten_queries=4,
    minimum_business_confidence=0.70,
    minimum_token_overlap=0.25,
    allow_soft_business_hint=True,
)
V31_CROSS_ANALYZER = KDICV31V15CrossRewriteAnalyzer(
    V31_BASE_ANALYZER,
    v31,
    V31_CROSS_POLICY,
)
V31_ANALYSIS_CACHE: dict[str, dict[str, Any]] = {}


# ---------- 두 분석기에 공통 적용하는 개선 문맥 게이트 ----------
CONTEXT_CLASSIFIER_SYSTEM_PROMPT = """
당신은 대화 문맥 적용 여부만 판정하는 구조화 분류기입니다.
질의를 재작성하거나 사용자 질문에 답하지 마세요.
현재 질문이 독립적으로 완결되면 이전 대화 상태를 사용하지 마세요.
현재 질문에 명시된 업무는 이전 업무보다 우선합니다.
확신할 수 없으면 AMBIGUOUS로 판정하세요.
JSON 객체 하나만 출력하세요.
""".strip()

_CONTEXT_CLASSIFIER_CACHE: dict[str, dict[str, Any]] = {}


def _extract_context_json(text: str) -> dict[str, Any]:
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(text or "").strip(), flags=re.I)
    try:
        value = json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        if start < 0:
            raise ValueError("문맥 판단 JSON 객체가 없습니다.")
        value, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(value, dict):
        raise TypeError("문맥 판단 결과의 최상위 값은 객체여야 합니다.")
    return value


def classify_ambiguous_context(question: str, state: Mapping[str, Any]) -> dict[str, Any]:
    cache_payload = {
        "question": question,
        "active_businesses": state.get("active_businesses") or [],
        "excluded_businesses": state.get("excluded_businesses") or [],
        "actor_role": state.get("actor_role"),
        "pending_clarification": state.get("pending_clarification"),
        "last_resolved_question": state.get("last_resolved_question"),
        "turns": (state.get("turns") or [])[-4:],
    }
    cache_key = hashlib.sha256(
        json.dumps(cache_payload, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()
    cached = _CONTEXT_CLASSIFIER_CACHE.get(cache_key)
    if cached is not None:
        return {**copy.deepcopy(cached), "_cache_hit": True, "_latency_ms": 0.0}

    prompt = f"""
[현재 질문]
{question}

[현재 대화 상태]
{json.dumps(cache_payload, ensure_ascii=False, indent=2)}

[출력 JSON]
{{
  "dialogue_act": "NEW_TOPIC | FOLLOW_UP | CORRECTION | EXCLUSION | AMBIGUOUS",
  "current_question_complete": true,
  "context_required": false,
  "selected_businesses": [],
  "excluded_businesses": [],
  "actor_role": "",
  "missing_slots": [],
  "confidence": 0.0,
  "reason_code": ""
}}
""".strip()
    started = time.perf_counter()
    response = ANSWER_HCX_CLIENT.chat.completions.create(
        model="HCX-007",
        messages=[
            {"role": "system", "content": CONTEXT_CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=500,
    )
    value = _extract_context_json(response.choices[0].message.content)
    value["_cache_hit"] = False
    value["_latency_ms"] = (time.perf_counter() - started) * 1000
    _CONTEXT_CLASSIFIER_CACHE[cache_key] = copy.deepcopy(value)
    return value


BUSINESS_TO_CONTEXT = {
    "예금자보호제도": "예금자보호",
    "예금자보호": "예금자보호",
    "예금보험금 안내": "예금보험금",
    "예금보험금": "예금보험금",
    "고객 미수령금 신청": "고객 미수령금",
    "고객 미수령금": "고객 미수령금",
    "착오송금 반환 신청": "착오송금 반환지원",
    "착오송금 반환지원": "착오송금 반환지원",
    "채무조정 안내": "채무조정",
    "채무조정": "채무조정",
    "은닉재산 신고": "은닉재산 신고",
}


def _context_businesses(values: Sequence[Any]) -> list[str]:
    output = []
    for value in values:
        mapped = BUSINESS_TO_CONTEXT.get(str(value).strip(), str(value).strip())
        if mapped and mapped not in output:
            output.append(mapped)
    return output


def new_analyzer_state() -> dict[str, Any]:
    return new_context_state()


def _route_only_analysis(
    analyzer_key: str,
    question: str,
    resolution: Mapping[str, Any],
    started: float,
) -> dict[str, Any]:
    route = str(resolution.get("route") or "CLARIFY")
    return {
        "analyzer": analyzer_key,
        "route": route,
        "original_question": question,
        "resolved_question": "",
        "businesses": list(resolution.get("active_businesses") or []),
        "query_type": "NO_RETRIEVAL",
        "plans": [],
        "context_resolution": dict(resolution),
        "context_used": bool(resolution.get("context_used")),
        "context_reason": resolution.get("reason"),
        "route_response": resolution.get("clarification_message") or resolution.get("direct_response") or "추가 정보가 필요합니다.",
        "decomposition_or_rewrite_called": False,
        "decomposition_or_rewrite_accepted": False,
        "fallback_to_original": False,
        "issues": [],
        "query_plan_valid": route != "RETRIEVE",
        "hard_filter_count": 0,
        "context_latency_ms": float(resolution.get("latency_ms") or 0),
        "core_analysis_latency_ms": 0.0,
        "analysis_wall_latency_ms": (time.perf_counter() - started) * 1000,
        "raw_analysis": {},
    }


def analyze_v15_improved(question: str, state: dict[str, Any]) -> dict[str, Any]:
    started = time.perf_counter()
    resolution = resolve_context_v2(
        question,
        state=state,
        llm_classifier=classify_ambiguous_context,
    )
    if resolution["route"] != "CONTINUE":
        return _route_only_analysis("V1.5_IMPROVED", question, resolution, started)

    resolved = str(resolution["resolved_question"])
    core_started = time.perf_counter()
    base = analyze_v15_chat_query(resolved, previous_turns=[])
    core_latency_ms = (time.perf_counter() - core_started) * 1000
    businesses = list(base.get("businesses") or [])
    if base.get("cross_business_candidate"):
        query_type = "CROSS_BUSINESS"
    elif str(base.get("complexity") or "") == "MULTI":
        query_type = "SAME_BUSINESS_MULTI"
    else:
        query_type = "SINGLE"
    plans = [
        {
            "query": str(plan.get("query") or "").strip(),
            "weight": float(plan.get("weight") or 0),
            "source": str(plan.get("source") or "V15"),
            "business_filter": {"mode": "NONE"},
        }
        for plan in base.get("plans") or []
    ]
    if base.get("route") == "RETRIEVE" and businesses:
        state["active_businesses"] = _context_businesses(businesses)
    return {
        "analyzer": "V1.5_IMPROVED",
        "route": str(base.get("route") or ""),
        "original_question": question,
        "resolved_question": resolved,
        "businesses": businesses,
        "query_type": query_type,
        "plans": plans,
        "context_resolution": dict(resolution),
        "context_used": bool(resolution.get("context_used")),
        "context_reason": resolution.get("reason"),
        "route_response": str(base.get("route_response") or ""),
        "decomposition_or_rewrite_called": bool(base.get("decomposition_called")),
        "decomposition_or_rewrite_accepted": bool(base.get("decomposition_accepted")),
        "fallback_to_original": bool(base.get("fallback_to_original")),
        "issues": list(base.get("decomposition_issues") or []),
        "query_plan_valid": _comparison_plans_valid(str(base.get("route") or ""), plans),
        "hard_filter_count": 0,
        "context_latency_ms": float(resolution.get("latency_ms") or 0),
        "core_analysis_latency_ms": core_latency_ms,
        "analysis_wall_latency_ms": (time.perf_counter() - started) * 1000,
        "analysis_cache_hit": bool(base.get("decomposition_cache_hit")),
        "raw_analysis": base,
    }


def _v31_cache_key(question: str) -> str:
    return hashlib.sha256(question.strip().encode("utf-8")).hexdigest()


def analyze_v31_cross_only(question: str, state: dict[str, Any]) -> dict[str, Any]:
    started = time.perf_counter()
    resolution = resolve_context_v2(
        question,
        state=state,
        llm_classifier=classify_ambiguous_context,
    )
    if resolution["route"] != "CONTINUE":
        return _route_only_analysis("V3.1_CROSS_ONLY", question, resolution, started)

    resolved = str(resolution["resolved_question"])
    cache_key = _v31_cache_key(resolved)
    cached = V31_ANALYSIS_CACHE.get(cache_key)
    core_started = time.perf_counter()
    if cached is None:
        base = V31_CROSS_ANALYZER.run(resolved, conversation_state=None)
        V31_ANALYSIS_CACHE[cache_key] = copy.deepcopy(base)
        cache_hit = False
    else:
        base = copy.deepcopy(cached)
        cache_hit = True
    core_latency_ms = (time.perf_counter() - core_started) * 1000

    raw_plans = list(base.get("search_plans") or [])
    plans = [
        {
            "query": str(plan.get("semantic_query") or "").strip(),
            "weight": float(plan.get("query_weight") or 0),
            "source": str(plan.get("query_source") or "V31"),
            "business_filter": dict(plan.get("business_filter") or {"mode": "NONE"}),
        }
        for plan in raw_plans
    ]
    cross = dict(base.get("cross_business") or {})
    needs = list((base.get("v31_analysis") or {}).get("needs") or [])
    businesses = list(cross.get("businesses") or [])
    if not businesses:
        businesses = list(dict.fromkeys(
            str(need.get("business_function") or "").strip()
            for need in needs if str(need.get("business_function") or "").strip()
        ))
    if base.get("route") == "RETRIEVE" and businesses:
        state["active_businesses"] = _context_businesses(businesses)
    return {
        "analyzer": "V3.1_CROSS_ONLY",
        "route": str(base.get("route") or ""),
        "original_question": question,
        "resolved_question": resolved,
        "businesses": businesses,
        "query_type": str(base.get("query_type") or "SINGLE"),
        "plans": plans,
        "context_resolution": dict(resolution),
        "context_used": bool(resolution.get("context_used")),
        "context_reason": resolution.get("reason"),
        "route_response": "",
        "decomposition_or_rewrite_called": bool(base.get("rewrite_called")),
        "decomposition_or_rewrite_accepted": bool(base.get("rewrite_accepted")),
        "fallback_to_original": bool(base.get("fallback_to_original")),
        "issues": list((base.get("rewrite_validation") or {}).get("issues") or []),
        "query_plan_valid": bool(base.get("query_plan_valid")) and _comparison_plans_valid(str(base.get("route") or ""), plans),
        "hard_filter_count": int(base.get("hard_filter_count") or 0),
        "context_latency_ms": float(resolution.get("latency_ms") or 0),
        "core_analysis_latency_ms": core_latency_ms,
        "analysis_wall_latency_ms": (time.perf_counter() - started) * 1000,
        "analysis_cache_hit": cache_hit,
        "raw_analysis": base,
    }


print({
    "analyzers": ["V1.5_IMPROVED", "V3.1_CROSS_ONLY"],
    "shared_context_policy": "CONTEXT_POLICY_V2",
    "single_and_same_business_policy": "ORIGINAL_1.0",
    "cross_business_policy": "ORIGINAL_0.4_REWRITTEN_0.6",
})

## 7. 고정 검색·Parent-Child·구조화 답변 실행 계층

In [ ]:
def new_comparison_state() -> dict[str, Any]:
    return {
        "v15": new_analyzer_state(),
        "v31": new_analyzer_state(),
        "results": {"v15": [], "v31": []},
        "compare_run_count": 0,
        "busy": False,
    }


ANALYZER_FUNCTIONS = {
    "v15": analyze_v15_improved,
    "v31": analyze_v31_cross_only,
}


def _route_message(analysis: Mapping[str, Any]) -> str:
    if analysis.get("route_response"):
        return str(analysis["route_response"])
    route = str(analysis.get("route") or "")
    if route == "DIRECT_RESPONSE":
        return "안녕하세요. 예금보험공사 관련 제도와 신청 절차에 관해 질문해 주세요."
    if route == "OUT_OF_SCOPE":
        return "예금보험공사의 예금자보호·예금보험금·미수령금·착오송금 반환지원·채무조정·은닉재산 신고 범위에서 질문해 주세요."
    if route == "CLARIFY":
        return "정확한 안내를 위해 어떤 업무와 대상에 관한 질문인지 조금 더 알려주세요."
    return "답변할 수 없는 경로입니다."


def _query_latency_rows(per_query: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for row in per_query:
        trace = dict(row.get("latency_breakdown_ms") or {})
        rows.append({
            "plan_index": row.get("plan_index"),
            "source": row.get("source"),
            "weight": float(row.get("weight") or 0),
            "query": row.get("query"),
            "embedding_latency_ms": float(trace.get("embedding_latency_ms") or 0),
            "dense_compute_latency_ms": float(trace.get("dense_compute_latency_ms") or 0),
            "bm25_latency_ms": float(trace.get("bm25_latency_ms") or 0),
            "minmax_latency_ms": float(trace.get("minmax_latency_ms") or 0),
            "query_total_latency_ms": float(trace.get("query_total_latency_ms") or 0),
        })
    return rows


def run_fixed_pipeline(
    analyzer_key: str,
    question: str,
    *,
    state: dict[str, Any],
) -> dict[str, Any]:
    global _LAST_RERANK_TRACE, _LAST_PARENT_CHILD_TRACE
    if analyzer_key not in ANALYZER_FUNCTIONS:
        raise KeyError(f"알 수 없는 분석기: {analyzer_key}")
    total_started = time.perf_counter()
    _LAST_RERANK_TRACE = {}
    _LAST_PARENT_CHILD_TRACE = {}

    analysis = ANALYZER_FUNCTIONS[analyzer_key](question, state)
    if analysis["route"] != "RETRIEVE":
        result = {
            "analyzer_key": analyzer_key,
            "analyzer_label": ANALYZER_LABELS[analyzer_key],
            "question": question,
            "resolved_question": analysis.get("resolved_question") or "",
            "route": analysis["route"],
            "analysis": analysis,
            "display_answer": _route_message(analysis),
            "basic_answer": _route_message(analysis),
            "basic_answer_payload": None,
            "evidence_explanation_payload": None,
            "search_results": [],
            "evidence_pack": None,
            "sources": [],
            "latency_ms": {
                "문맥 처리": float(analysis.get("context_latency_ms") or 0),
                "질의분석": float(analysis.get("core_analysis_latency_ms") or 0),
                "전체": (time.perf_counter() - total_started) * 1000,
            },
            "success": True,
        }
        return result

    plans = list(analysis.get("plans") or [])
    if not analysis.get("query_plan_valid") or not _comparison_plans_valid("RETRIEVE", plans):
        raise RuntimeError("RETRIEVE 검색 계획이 유효하지 않습니다.")

    search_started = time.perf_counter()
    search_results, per_query = fuse_query_results(plans)
    child_search_ms = (time.perf_counter() - search_started) * 1000
    reranker_trace = dict(_LAST_RERANK_TRACE)

    parent_started = time.perf_counter()
    search_results = expand_parent_context(search_results)
    parent_ms = (time.perf_counter() - parent_started) * 1000
    parent_trace = dict(_LAST_PARENT_CHILD_TRACE)

    answer_question = str(analysis.get("resolved_question") or question)
    evidence_started = time.perf_counter()
    evidence_pack = build_parent_basic_evidence_pack(answer_question, search_results)
    evidence_ms = (time.perf_counter() - evidence_started) * 1000

    answer_started = time.perf_counter()
    basic_payload = generate_basic_answer_b_v2(
        client=ANSWER_HCX_CLIENT,
        model=HCX_CHAT_MODEL,
        question=answer_question,
        evidence_pack=evidence_pack,
    )
    answer_ms = (time.perf_counter() - answer_started) * 1000
    display_answer = user_visible_answer(basic_payload["answer"])
    sources = build_used_sources(evidence_pack, basic_payload)
    per_query_latency = _query_latency_rows(per_query)
    search_wall_ms = child_search_ms + parent_ms
    result = {
        "analyzer_key": analyzer_key,
        "analyzer_label": ANALYZER_LABELS[analyzer_key],
        "question": question,
        "resolved_question": answer_question,
        "route": analysis["route"],
        "analysis": analysis,
        "display_answer": display_answer,
        "basic_answer": basic_payload["answer"],
        "basic_answer_payload": basic_payload,
        "evidence_explanation_payload": None,
        "search_plans": plans,
        "per_query_search": per_query,
        "per_query_latency": per_query_latency,
        "search_results": search_results,
        "reranker": reranker_trace,
        "parent_child": parent_trace,
        "evidence_pack": evidence_pack,
        "evidence_pack_sha256": evidence_pack_sha256(evidence_pack),
        "sources": sources,
        "detail_sources": [],
        "answer_api_trace": dict(_LAST_ANSWER_API_TRACE),
        "latency_ms": {
            "문맥 처리": float(analysis.get("context_latency_ms") or 0),
            "질의분석": float(analysis.get("core_analysis_latency_ms") or 0),
            "검색": search_wall_ms,
            "질문 임베딩": sum(row["embedding_latency_ms"] for row in per_query_latency),
            "Dense 계산": sum(row["dense_compute_latency_ms"] for row in per_query_latency),
            "BM25": sum(row["bm25_latency_ms"] for row in per_query_latency),
            "BAAI Reranker": float(reranker_trace.get("latency_ms") or 0),
            "Parent-Child8192": parent_ms,
            "Evidence Pack": evidence_ms,
            "구조화 기본답변": answer_ms,
            "전체": (time.perf_counter() - total_started) * 1000,
        },
        "success": True,
    }
    state.setdefault("turns", []).extend([
        {"role": "user", "content": question},
        {"role": "assistant", "content": display_answer},
    ])
    return result


def generate_answer_basis(result: dict[str, Any]) -> dict[str, Any]:
    if result.get("route") != "RETRIEVE" or not result.get("evidence_pack"):
        raise ValueError("답변 근거를 생성할 RETRIEVE 결과가 없습니다.")
    if result.get("evidence_explanation_payload"):
        return result["evidence_explanation_payload"]
    started = time.perf_counter()
    payload = generate_evidence_explanation_b_v2(
        client=ANSWER_HCX_CLIENT,
        model=HCX_CHAT_MODEL,
        question=result["resolved_question"],
        evidence_pack=result["evidence_pack"],
        basic_answer=result["basic_answer_payload"],
    )
    latency_ms = (time.perf_counter() - started) * 1000
    result["evidence_explanation_payload"] = payload
    result["detail_sources"] = build_used_sources(result["evidence_pack"], payload)
    result["latency_ms"]["답변 근거 설명"] = latency_ms
    result["latency_ms"]["기본답변+근거 누적"] = float(result["latency_ms"]["전체"]) + latency_ms
    return payload


def analysis_markdown_compare(result: Mapping[str, Any]) -> str:
    analysis = dict(result.get("analysis") or {})
    plans = analysis.get("plans") or []
    plan_text = "<br>".join(
        f"{index}. {plan.get('source')} · {float(plan.get('weight') or 0):.3f} · {str(plan.get('query') or '').replace('|', chr(92) + '|')}"
        for index, plan in enumerate(plans, start=1)
    ) or "검색 계획 없음"
    businesses = ", ".join(analysis.get("businesses") or []) or "-"
    issues = ", ".join(analysis.get("issues") or []) or "-"
    return (
        "### 질의분석 결과\n\n|항목|값|\n|---|---|\n"
        f"|분석기|{result.get('analyzer_label')}|\n"
        f"|최종 경로|{analysis.get('route')}|\n"
        f"|검색용 독립질의|{analysis.get('resolved_question') or '-'}|\n"
        f"|문맥 판단|{analysis.get('context_reason') or '-'}|\n"
        f"|문맥 사용|{bool(analysis.get('context_used'))}|\n"
        f"|탐지 업무|{businesses}|\n"
        f"|질문 유형|{analysis.get('query_type')}|\n"
        f"|분해·재작성 호출|{bool(analysis.get('decomposition_or_rewrite_called'))}|\n"
        f"|분해·재작성 승인|{bool(analysis.get('decomposition_or_rewrite_accepted'))}|\n"
        f"|원문 fallback|{bool(analysis.get('fallback_to_original'))}|\n"
        f"|분석 캐시 적중|{bool(analysis.get('analysis_cache_hit'))}|\n"
        f"|검증 이슈|{issues}|\n"
        f"|검색 계획|{plan_text}|"
    )


def latency_markdown_compare(result: Mapping[str, Any]) -> str:
    lines = ["### 단계별 레이턴시", "", "|단계|지연시간|", "|---|---:|"]
    for key, value in (result.get("latency_ms") or {}).items():
        lines.append(f"|{key}|{float(value):,.1f}ms|")
    return "\n".join(lines)


def retrieval_markdown_compare(result: Mapping[str, Any]) -> str:
    rows = result.get("search_results") or []
    if not rows:
        return "### 검색 결과\n\n검색을 실행하지 않았습니다."
    lines = [
        "### 공통 BAAI Reranker + Parent-Child8192 검색 결과", "",
        "|순위|Child|Parent|Reranker|제목 / 섹션|",
        "|---:|---|---|---:|---|",
    ]
    for row in rows:
        chunk = row["chunk"]
        title = " / ".join(str(value).replace("|", "\\|") for value in (chunk.get("title"), chunk.get("section_title")) if value)
        lines.append(
            f"|{row.get('rank')}|{row.get('chunk_id')}|{row.get('parent_doc_id', '-')}|"
            f"{float(row.get('reranker_score') or 0):.6f}|{title}|"
        )
    return "\n".join(lines)


def render_result(result: dict[str, Any], *, show_pack: bool = False) -> None:
    display(Markdown(f"## {result['analyzer_label']}\n\n### 기본답변\n\n{result['display_answer']}"))
    source_text = sources_to_markdown_user(result.get("sources") or [])
    if source_text:
        display(Markdown(source_text))
    display(Markdown(analysis_markdown_compare(result)))
    display(Markdown(retrieval_markdown_compare(result)))
    if show_pack and result.get("evidence_pack"):
        display(JSON(result["evidence_pack"], expanded=False))
    display(Markdown(latency_markdown_compare(result)))


def comparison_summary_frame(results: Mapping[str, Mapping[str, Any]]) -> pd.DataFrame:
    rows = []
    for key in ("v15", "v31"):
        result = results.get(key)
        if not result:
            continue
        analysis = dict(result.get("analysis") or {})
        rows.append({
            "질의분석기": result.get("analyzer_label"),
            "최종 경로": result.get("route"),
            "문맥 판단": analysis.get("context_reason"),
            "질문 유형": analysis.get("query_type"),
            "분해·재작성 승인": bool(analysis.get("decomposition_or_rewrite_accepted")),
            "원문 fallback": bool(analysis.get("fallback_to_original")),
            "검색계획 수": len(analysis.get("plans") or []),
            "Top-5": ", ".join(str(row.get("chunk_id")) for row in result.get("search_results") or []),
            "질의분석(ms)": float((result.get("latency_ms") or {}).get("질의분석") or 0),
            "검색(ms)": float((result.get("latency_ms") or {}).get("검색") or 0),
            "답변(ms)": float((result.get("latency_ms") or {}).get("구조화 기본답변") or 0),
            "전체(ms)": float((result.get("latency_ms") or {}).get("전체") or 0),
        })
    return pd.DataFrame(rows)


print({
    "retrieval": "HYBRID_7_3_MINMAX",
    "reranker": RERANKER_MODEL_NAME,
    "parent_child": PARENT_CHILD_ENABLED,
    "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
    "answer": "B_STRUCTURED_BASIC_AND_DOCUMENT_BASIS",
    "legacy_hard_gates": LEGACY_HARD_GATES,
})

## 8. 설정·안전성 검사

In [ ]:

assert DENSE_WEIGHT == 0.7
assert BM25_WEIGHT == 0.3
assert CANDIDATE_DEPTH == 20
assert FINAL_TOP_K == 5
assert RERANKER_MODEL_NAME == "BAAI/bge-reranker-v2-m3"
assert PARENT_CHILD_ENABLED is True
assert PARENT_CONTEXT_MAX_CHARS == 8192
assert math.isclose(V15_ORIGINAL_WEIGHT, 0.40)
assert math.isclose(V15_SUBQUERY_TOTAL_WEIGHT, 0.60)
assert math.isclose(V31_CROSS_POLICY.original_weight, 0.40)
assert math.isclose(V31_CROSS_POLICY.rewritten_total_weight, 0.60)

# 네트워크 없이 확인 가능한 검색 계획 안전성 검사
assert _comparison_plans_valid("RETRIEVE", [{
    "query": "예금자보호 한도는 얼마인가요?",
    "weight": 1.0,
    "source": "ORIGINAL",
    "business_filter": {"mode": "NONE"},
}])
assert not _comparison_plans_valid("RETRIEVE", [{
    "query": "예금자보호 한도는 얼마인가요?",
    "weight": 1.0,
    "source": "INVALID_HARD",
    "business_filter": {"mode": "HARD"},
}])

print("공통 검색·답변 조건과 질의계획 안전성 검사 통과")

## 10. 결과 해석 주의사항

- `V1.5` 분해 캐시와 `V3.1` 분석 캐시 적중 여부를 질의분석 표에서 확인하세요.
- `[답변 근거 보기]`는 추가 HCX-005 호출이므로 기본답변 레이턴시와 분리됩니다.
- Parent-Child 확장 자체의 로컬 시간과, 확장 문맥을 HCX-005가 읽는 답변 시간은 다른 지표입니다.
- `CLARIFY Precision`, 잘못된 OOS·DIRECT 건수는 정답 라벨이 있는 평가셋을 실행해야 확정할 수 있습니다.

In [ ]:
def elasticsearch_diagnostics() -> dict[str, Any]:
    info = ES.info()
    nodes = ES.nodes.info(metric="plugins")
    plugins = sorted({
        str(plugin.get("name") or "")
        for node in nodes["nodes"].values()
        for plugin in node.get("plugins", [])
    })
    index_exists = bool(ES.indices.exists(index=ES_INDEX_NAME))
    document_count = int(ES.count(index=ES_INDEX_NAME)["count"]) if index_exists else None
    tokens = []
    if index_exists:
        tokens = [
            token["token"]
            for token in ES.indices.analyze(
                index=ES_INDEX_NAME,
                analyzer=ES_ANALYZER_NAME,
                text="착오송금 반환지원",
            )["tokens"]
        ]
    return {
        "connected": True,
        "url": ES_URL,
        "version": info["version"]["number"],
        "cluster_name": info["cluster_name"],
        "analysis_nori_installed": "analysis-nori" in plugins,
        "plugins": plugins,
        "index_name": ES_INDEX_NAME,
        "index_exists": index_exists,
        "document_count": document_count,
        "expected_document_count": len(CHUNKS),
        "nori_none_tokens": tokens,
    }


display(JSON(elasticsearch_diagnostics(), expanded=True))


## V1.5 멀티턴 의도 결합 개선

이 셀부터는 V1.5의 후속질의 판정을 완성 문장 `fullmatch` 방식에서 구성요소 기반 방식으로 보완합니다.

- 현재 질문에 명시적 업무가 없고 이전 활성 업무가 정확히 하나이며 신청·서류·자격·기간 등의 의도가 있으면 활성 업무를 결합합니다.
- 활성 업무가 없거나 둘 이상이면 전체 문서를 검색하지 않고 `CLARIFY`합니다.
- 명시적 새 업무와 OOS 주제는 이전 문맥보다 우선합니다.
- 착오송금 기간처럼 역할·단계가 필요한 질문은 기존처럼 `CLARIFY`합니다.
- 기존 V1.5 라우팅, HCX-007 교차업무 Structured Output, BAAI Reranker, Parent-Child8192, 답변 B는 유지합니다.

In [ ]:
from __future__ import annotations

import copy
import re
import time
from typing import Any, Callable, Mapping, Sequence

from kdic_context_policy_v2 import (
    AMBIGUOUS_REFERENCE_PATTERN,
    BUSINESS_PATTERNS,
    CANCEL_PATTERN,
    CORRECTION_PATTERN,
    EXCLUSION_PATTERN,
    _clean,
    _clarify,
    _selected_pending,
    detect_businesses,
)

FOLLOWUP_INTENT_RULES_V21: dict[str, tuple[str, ...]] = {
    "APPLICATION": ("신청", "접수", "신청하려", "접수하려"),
    "DOCUMENTS": ("서류", "필요서류", "필요 서류", "구비서류", "구비 서류", "준비물", "뭘 준비", "무엇을 준비"),
    "ELIGIBILITY": ("자격", "대상", "해당", "신청 가능", "가능한 사람"),
    "PROCEDURE": ("절차", "방법", "순서", "과정", "어떻게"),
    "TIME": ("기간", "기한", "언제", "얼마나 걸", "며칠", "몇 일"),
    "COST": ("비용", "수수료", "돈이 드", "얼마가 드"),
    "LOOKUP": ("조회", "확인", "진행상태", "진행 상태"),
    "LIMIT": ("금액", "한도", "얼마까지"),
    "EXCEPTION": ("예외", "제외", "안 되는", "불가능"),
    "CHANGE_CANCEL": ("취소", "철회", "변경", "수정"),
    "ACTOR": ("본인", "대리인", "송금인", "수취인", "상속인"),
    "REASON": ("왜", "이유"),
}

EXPLICIT_OOS_CONTEXT_BLOCK_V21 = re.compile(
    r"(?:비트코인|가상자산|코스피|주식\s*(?:투자|매수|매도|포트폴리오)|"
    r"날씨|기온|미세먼지|환율|환전|주택담보대출|전세대출|신용카드|상속세|세금\s*환급)",
    re.I,
)


def detect_followup_intents_v21(question: str) -> list[str]:
    text = _clean(question).lower()
    return [
        intent
        for intent, terms in FOLLOWUP_INTENT_RULES_V21.items()
        if any(term.lower() in text for term in terms)
    ]


def _explicit_oos_before_context_v21(question: str) -> bool:
    if EXPLICIT_OOS_CONTEXT_BLOCK_V21.search(question):
        return True
    try:
        return bool(light_router.OOS_PATTERN.search(question))
    except Exception:
        return False


def _context_resolution_payload_v21(
    *,
    original: str,
    resolved: str,
    state: dict[str, Any],
    intents: Sequence[str],
    started: float,
) -> dict[str, Any]:
    state["pending_clarification"] = None
    state["last_resolved_question"] = resolved
    return {
        "route": "CONTINUE",
        "dialogue_act": "FOLLOW_UP",
        "original_question": original,
        "resolved_question": resolved,
        "current_question_complete": False,
        "context_used": True,
        "reason": "INTENT_BASED_UNIQUE_ACTIVE_BUSINESS",
        "clarification_message": "",
        "active_businesses": list(state.get("active_businesses") or []),
        "excluded_businesses": list(state.get("excluded_businesses") or []),
        "actor_role": state.get("actor_role"),
        "missing_slots": [],
        "followup_intents": list(intents),
        "pending_clarification": None,
        "llm_judgment": {"called": False, "reason": "RULE_INTENT_FOLLOWUP"},
        "latency_ms": (time.perf_counter() - started) * 1000,
    }


if "_ORIGINAL_RESOLVE_CONTEXT_V2" not in globals():
    _ORIGINAL_RESOLVE_CONTEXT_V2 = resolve_context_v2


def resolve_context_v21(
    question: str,
    *,
    state: dict[str, Any],
    llm_classifier: Callable[[str, Mapping[str, Any]], Mapping[str, Any]] | None = None,
) -> dict[str, Any]:
    started = time.perf_counter()
    original = _clean(question)
    if not original:
        raise ValueError("질문이 비어 있습니다.")
    for key, default in new_context_state().items():
        state.setdefault(key, copy.deepcopy(default))

    explicit_businesses = detect_businesses(original)
    active_businesses = list(state.get("active_businesses") or [])
    intents = detect_followup_intents_v21(original)
    pending = state.get("pending_clarification") or {}
    pending_selection = _selected_pending(original, pending) if pending else None

    must_use_existing_policy = bool(
        CANCEL_PATTERN.fullmatch(original)
        or pending_selection
        or explicit_businesses
        or EXCLUSION_PATTERN.search(original)
        or CORRECTION_PATTERN.search(original)
        or AMBIGUOUS_REFERENCE_PATTERN.search(original)
        or _explicit_oos_before_context_v21(original)
    )
    if must_use_existing_policy or not intents:
        return _ORIGINAL_RESOLVE_CONTEXT_V2(
            original,
            state=state,
            llm_classifier=llm_classifier,
        )

    if len(active_businesses) == 1:
        business = active_businesses[0]
        if business == "착오송금 반환지원" and "TIME" in intents:
            return _clarify(
                question=original,
                state=state,
                reason="MISTAKEN_TRANSFER_TIME_SCOPE_AMBIGUOUS",
                message=(
                    "착오송금의 어느 기간을 묻는지 확인이 필요합니다. "
                    "송금인의 반환지원 처리기간과 수취인의 자진반환 관련 기간 중 선택해 주세요."
                ),
                options=["송금인", "수취인"],
                missing_slots=["actor_role", "process_stage"],
                started=started,
            )
        resolved = f"{business} 관련 {original}"
        return _context_resolution_payload_v21(
            original=original,
            resolved=resolved,
            state=state,
            intents=intents,
            started=started,
        )

    if len(active_businesses) > 1:
        return _clarify(
            question=original,
            state=state,
            reason="INTENT_FOLLOWUP_MULTIPLE_ACTIVE_BUSINESSES",
            message="어느 업무에 관한 후속 질문인지 선택해 주세요.",
            options=active_businesses,
            missing_slots=["business_function"],
            started=started,
        )

    return _clarify(
        question=original,
        state=state,
        reason="INTENT_FOLLOWUP_WITHOUT_ACTIVE_BUSINESS",
        message="어떤 업무에 관한 질문인지 알려주세요.",
        options=list(BUSINESS_PATTERNS),
        missing_slots=["business_function"],
        started=started,
    )


resolve_context_v2 = resolve_context_v21


if "_ORIGINAL_ANALYZE_V15_IMPROVED" not in globals():
    _ORIGINAL_ANALYZE_V15_IMPROVED = analyze_v15_improved


def analyze_v15_improved_v21(question: str, state: dict[str, Any]) -> dict[str, Any]:
    result = _ORIGINAL_ANALYZE_V15_IMPROVED(question, state)
    if result.get("route") != "RETRIEVE" or not result.get("context_used"):
        result["context_business_preserved"] = True
        return result

    expected = set(_context_businesses((result.get("context_resolution") or {}).get("active_businesses") or []))
    detected = set(_context_businesses(result.get("businesses") or []))
    preserved = not expected or bool(expected & detected)
    result["context_business_preserved"] = preserved
    if preserved:
        return result

    options = sorted(expected) or list(BUSINESS_PATTERNS)
    state["pending_clarification"] = {
        "original_question": question,
        "reason": "CONTEXT_BUSINESS_NOT_PRESERVED",
        "options": options,
        "missing_slots": ["business_function"],
    }
    result.update({
        "route": "CLARIFY",
        "query_type": "NO_RETRIEVAL",
        "plans": [],
        "query_plan_valid": True,
        "route_response": "검색 질의에 이전 업무가 안전하게 반영되지 않았습니다. 어떤 업무인지 다시 선택해 주세요.",
        "issues": list(result.get("issues") or []) + ["CONTEXT_BUSINESS_NOT_PRESERVED"],
    })
    return result


analyze_v15_improved = analyze_v15_improved_v21
ANALYZER_FUNCTIONS["v15"] = analyze_v15_improved_v21
ANALYZER_LABELS["v15"] = "V1.5 멀티턴 의도결합 개선"

print({
    "analyzer": ANALYZER_LABELS["v15"],
    "followup_policy": "UNIQUE_ACTIVE_BUSINESS_PLUS_INTENT",
    "missing_business_policy": "CLARIFY",
    "context_business_guard": True,
    "cross_business_decomposer": "HCX-007_STRUCTURED_OUTPUT",
})


## 멀티턴 규칙 회귀 테스트

아래 테스트는 API를 호출하지 않고 문맥 판정 규칙만 검증합니다. 하나라도 실패하면 노트북 실행을 중단합니다.

In [ ]:
def run_multiturn_regression_tests_v21() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    def run_case(
        name: str,
        question: str,
        active: Sequence[str],
        expected_route: str,
        *,
        expected_context: bool | None = None,
        resolved_contains: str | None = None,
        expected_reason: str | None = None,
    ) -> None:
        state = new_context_state()
        state["active_businesses"] = list(active)
        result = resolve_context_v2(question, state=state, llm_classifier=None)
        passed = result["route"] == expected_route
        if expected_context is not None:
            passed = passed and bool(result.get("context_used")) is expected_context
        if resolved_contains is not None:
            passed = passed and resolved_contains in str(result.get("resolved_question") or "")
        if expected_reason is not None:
            passed = passed and result.get("reason") == expected_reason
        rows.append({
            "case": name,
            "question": question,
            "active_businesses": " | ".join(active) or "-",
            "route": result.get("route"),
            "context_used": bool(result.get("context_used")),
            "reason": result.get("reason"),
            "resolved_question": result.get("resolved_question"),
            "passed": passed,
        })

    run_case(
        "자연어 신청서류 후속질의",
        "신청 서류는 어떻게 돼?",
        ["채무조정"],
        "CONTINUE",
        expected_context=True,
        resolved_contains="채무조정",
        expected_reason="INTENT_BASED_UNIQUE_ACTIVE_BUSINESS",
    )
    run_case(
        "준비물 표현 후속질의",
        "필요한 준비물이 뭐야?",
        ["채무조정"],
        "CONTINUE",
        expected_context=True,
        resolved_contains="채무조정",
    )
    run_case(
        "활성업무 없는 신청서류",
        "신청 서류는 어떻게 돼?",
        [],
        "CLARIFY",
        expected_context=False,
        expected_reason="INTENT_FOLLOWUP_WITHOUT_ACTIVE_BUSINESS",
    )
    run_case(
        "활성업무 둘인 후속질의",
        "신청 서류는 어떻게 돼?",
        ["고객 미수령금", "착오송금 반환지원"],
        "CLARIFY",
        expected_context=False,
        expected_reason="INTENT_FOLLOWUP_MULTIPLE_ACTIVE_BUSINESSES",
    )
    run_case(
        "명시적 새 업무 우선",
        "착오송금 신청 서류는 무엇인가요?",
        ["채무조정"],
        "CONTINUE",
        expected_context=False,
        resolved_contains="착오송금",
    )
    run_case(
        "착오송금 기간 역할 확인",
        "얼마나 걸리나요?",
        ["착오송금 반환지원"],
        "CLARIFY",
        expected_context=False,
        expected_reason="MISTAKEN_TRANSFER_TIME_SCOPE_AMBIGUOUS",
    )
    run_case(
        "명시적 OOS는 문맥결합 금지",
        "비트코인 투자 방법은 어떻게 돼?",
        ["채무조정"],
        "CONTINUE",
        expected_context=False,
    )

    frame = pd.DataFrame(rows)
    failed = frame.loc[~frame["passed"]]
    if not failed.empty:
        raise AssertionError("멀티턴 회귀 테스트 실패:\n" + failed.to_string(index=False))
    return frame


multiturn_regression_v21 = run_multiturn_regression_tests_v21()
display(multiturn_regression_v21)
print(f"멀티턴 회귀 테스트: {len(multiturn_regression_v21)}/{len(multiturn_regression_v21)} 통과")


## 11. 관계형 교차업무 멀티턴 보정

명시적 새 업무가 있더라도 `도·같이·함께·동시에·병행·둘 다`가 있으면 이전 업무를 버리지 않습니다. 제외·정정 표현은 기존 정책을 우선합니다.


In [ ]:
from __future__ import annotations

import copy
import re
import time
from typing import Any, Mapping, Sequence


RELATIONAL_CROSS_BUSINESS_PATTERN_V22 = re.compile(
    r"(?:도\s*(?:같이|함께|동시에)|같이\s*(?:신청|이용|진행)|"
    r"함께|동시에|동시\s*신청|병행|둘\s*다|두\s*(?:제도|업무)\s*모두)",
    re.I,
)


def _is_relational_cross_business_followup_v22(
    question: str,
    explicit_businesses: Sequence[str],
    active_businesses: Sequence[str],
) -> bool:
    if not explicit_businesses or not active_businesses:
        return False
    if EXCLUSION_PATTERN.search(question) or CORRECTION_PATTERN.search(question):
        return False
    if not RELATIONAL_CROSS_BUSINESS_PATTERN_V22.search(question):
        return False
    return bool(set(explicit_businesses) - set(active_businesses))


_RESOLVE_CONTEXT_V21_BEFORE_RELATIONAL = resolve_context_v2


def resolve_context_v22(
    question: str,
    *,
    state: dict[str, Any],
    llm_classifier=None,
) -> dict[str, Any]:
    started = time.perf_counter()
    original = _clean(question)
    if not original:
        raise ValueError("질문이 비어 있습니다.")
    for key, default in new_context_state().items():
        state.setdefault(key, copy.deepcopy(default))

    explicit = detect_businesses(original)
    active = list(state.get("active_businesses") or [])
    if _is_relational_cross_business_followup_v22(original, explicit, active):
        combined = list(dict.fromkeys(active + explicit))
        if len(combined) < 2:
            return _RESOLVE_CONTEXT_V21_BEFORE_RELATIONAL(
                original, state=state, llm_classifier=llm_classifier
            )
        relation_subject = "과 ".join(combined)
        resolved = f"{relation_subject}의 동시·병행 신청 가능 여부에 관한 질문: {original}"
        state["active_businesses"] = combined
        state["pending_clarification"] = None
        state["last_resolved_question"] = resolved
        return {
            "route": "CONTINUE",
            "dialogue_act": "RELATIONAL_FOLLOW_UP",
            "original_question": original,
            "resolved_question": resolved,
            "current_question_complete": False,
            "context_used": True,
            "reason": "RELATIONAL_CROSS_BUSINESS_FOLLOWUP",
            "clarification_message": "",
            "active_businesses": combined,
            "excluded_businesses": list(state.get("excluded_businesses") or []),
            "actor_role": state.get("actor_role"),
            "missing_slots": [],
            "followup_intents": detect_followup_intents_v21(original),
            "pending_clarification": None,
            "llm_judgment": {"called": False, "reason": "RULE_RELATIONAL_CROSS_BUSINESS"},
            "latency_ms": (time.perf_counter() - started) * 1000,
        }

    return _RESOLVE_CONTEXT_V21_BEFORE_RELATIONAL(
        original, state=state, llm_classifier=llm_classifier
    )


resolve_context_v2 = resolve_context_v22


def run_relational_multiturn_regression_v22() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    def check(
        name: str,
        question: str,
        active: Sequence[str],
        expected_reason: str,
        expected_context: bool,
        expected_businesses: Sequence[str],
    ) -> None:
        state = new_context_state()
        state["active_businesses"] = list(active)
        result = resolve_context_v2(question, state=state, llm_classifier=None)
        passed = (
            result.get("reason") == expected_reason
            and bool(result.get("context_used")) is expected_context
            and set(result.get("active_businesses") or []) == set(expected_businesses)
        )
        rows.append({
            "case": name,
            "question": question,
            "reason": result.get("reason"),
            "context_used": bool(result.get("context_used")),
            "active_businesses": " | ".join(result.get("active_businesses") or []),
            "resolved_question": result.get("resolved_question"),
            "passed": passed,
        })

    check(
        "관계형 교차업무 보존",
        "그럼 채무조정도 같이 신청할 수 있어요?",
        ["착오송금 반환지원"],
        "RELATIONAL_CROSS_BUSINESS_FOLLOWUP",
        True,
        ["착오송금 반환지원", "채무조정"],
    )
    check(
        "명시적 독립 새 질문",
        "채무조정 신청 자격은 무엇인가요?",
        ["착오송금 반환지원"],
        "CURRENT_QUESTION_COMPLETE",
        False,
        ["채무조정"],
    )
    check(
        "제외 표현은 기존 정책 우선",
        "착오송금 말고 채무조정 신청을 알려주세요",
        ["착오송금 반환지원"],
        "EXPLICIT_EXCLUSION_WITH_REPLACEMENT",
        False,
        ["채무조정"],
    )
    frame = pd.DataFrame(rows)
    if not bool(frame["passed"].all()):
        raise AssertionError("관계형 멀티턴 회귀 테스트 실패\n" + frame.to_string(index=False))
    return frame


relational_multiturn_regression_v22 = run_relational_multiturn_regression_v22()
display(relational_multiturn_regression_v22)
print("관계형 멀티턴 회귀 테스트 통과")


## 12. 답변 입력 예산과 공통 B/D 생성기

검색 Top-5와 Parent-Child8192는 바꾸지 않습니다. 답변용 Pack만 Reranker 적중 Child→인접 Parent 청크 순서로 구성하며, 순위별 문자 예산 합계는 최대 14,000자입니다.

B와 D 모두 같은 Pack을 사용합니다. 문장 수 제한은 적용하지 않습니다.


In [ ]:
from __future__ import annotations

import json
import random
import re
import time
from collections import OrderedDict
from typing import Any, Mapping, Sequence


ANSWER_EVIDENCE_TOTAL_MAX_CHARS = 14_000
ANSWER_EVIDENCE_RANK_BUDGETS = (4_000, 3_500, 3_000, 2_000, 1_500)
ANSWER_CACHE_ENABLED_FOR_COMPARISON = False
ANSWER_PROMPT_VERSION = "bd-low-latency-v1"


def _truncate_at_boundary_v1(text: str, limit: int) -> str:
    if limit <= 0:
        return ""
    if len(text) <= limit:
        return text
    limited = text[:limit]
    boundary = max(limited.rfind("\n"), limited.rfind(" "))
    if boundary >= int(limit * 0.75):
        limited = limited[:boundary]
    return limited.rstrip()


def _proximity_order_v1(context_ids: Sequence[str], matched_ids: Sequence[str]) -> list[str]:
    context = list(dict.fromkeys(str(value) for value in context_ids if str(value)))
    matched = list(dict.fromkeys(str(value) for value in matched_ids if str(value)))
    output = [value for value in matched if value in context]
    for matched_id in matched:
        if matched_id not in context:
            output.append(matched_id)
            continue
        center = context.index(matched_id)
        for distance in range(1, len(context) + 1):
            for index in (center - distance, center + distance):
                if 0 <= index < len(context) and context[index] not in output:
                    output.append(context[index])
    output.extend(value for value in context if value not in output)
    return output


def build_compact_parent_evidence_pack_v1(
    question: str,
    search_results: Sequence[Mapping[str, Any]],
) -> dict[str, Any]:
    by_parent: OrderedDict[str, dict[str, Any]] = OrderedDict()
    for result in search_results:
        child = dict(result.get("chunk") or {})
        child_id = str(result.get("chunk_id") or child.get("chunk_id") or "")
        parent_id = str(result.get("parent_doc_id") or _parent_id_for_chunk(child))
        row = by_parent.setdefault(parent_id, {
            "rank": int(result.get("rank") or len(by_parent) + 1),
            "parent_id": parent_id,
            "representative_chunk_id": child_id,
            "matched_child_ids": [],
            "matched_child_ranks": [],
            "context_chunk_ids": list(result.get("parent_context_chunk_ids") or [child_id]),
            "document_title": _clean_text(child.get("title") or child.get("document_title")),
            "source_url": _clean_text(child.get("source_url")),
        })
        row["matched_child_ids"].append(child_id)
        row["matched_child_ranks"].append(int(result.get("rank") or 0))

    evidence: list[dict[str, Any]] = []
    sources: OrderedDict[str, dict[str, Any]] = OrderedDict()
    total_remaining = ANSWER_EVIDENCE_TOTAL_MAX_CHARS
    for parent_index, row in enumerate(by_parent.values()):
        if total_remaining <= 0 or parent_index >= len(ANSWER_EVIDENCE_RANK_BUDGETS):
            break
        parent_budget = min(ANSWER_EVIDENCE_RANK_BUDGETS[parent_index], total_remaining)
        ordered_ids = _proximity_order_v1(
            row["context_chunk_ids"], row["matched_child_ids"]
        )
        parts: list[str] = []
        included_ids: list[str] = []
        section_titles: list[str] = []
        remaining = parent_budget
        for chunk_id in ordered_ids:
            chunk = CHUNKS_BY_ID.get(str(chunk_id))
            if chunk is None:
                continue
            title = _clean_text(chunk.get("title"))
            section = _clean_text(chunk.get("section_title"))
            if section and section not in section_titles:
                section_titles.append(section)
            label = " / ".join(value for value in (title, section) if value)
            part = f"[{chunk_id}] {label}\n{_clean_text(chunk.get('content'))}".strip()
            separator_cost = 2 if parts else 0
            if remaining <= separator_cost:
                break
            part = _truncate_at_boundary_v1(part, remaining - separator_cost)
            if not part:
                break
            parts.append(part)
            included_ids.append(str(chunk_id))
            remaining -= len(part) + separator_cost
            if remaining < 120:
                break

        content = "\n\n".join(parts)
        if not content:
            continue
        evidence_id = f"E{len(evidence) + 1}"
        evidence.append({
            "evidence_id": evidence_id,
            "rank": int(row["rank"]),
            "chunk_id": row["representative_chunk_id"],
            "parent_id": row["parent_id"],
            "context_chunk_ids": included_ids,
            "matched_child_ids": list(dict.fromkeys(row["matched_child_ids"])),
            "matched_child_ranks": sorted(set(row["matched_child_ranks"])),
            "document_title": row["document_title"],
            "section_title": " · ".join(section_titles),
            "content": content,
            "context_char_count": len(content),
            "context_truncated": len(included_ids) < len(ordered_ids),
            "source_url": row["source_url"],
        })
        total_remaining -= len(content)
        url = row["source_url"]
        if url:
            source = sources.setdefault(url, {
                "source_id": f"S{len(sources) + 1}",
                "title": row["document_title"] or "공식 출처",
                "source_url": url,
                "evidence_ids": [],
            })
            source["evidence_ids"].append(evidence_id)

    if not evidence:
        raise ValueError("저지연 Evidence Pack을 만들 근거가 없습니다.")
    return {
        "question": _clean_text(question),
        "search_parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
        "answer_evidence_total_max_chars": ANSWER_EVIDENCE_TOTAL_MAX_CHARS,
        "answer_prompt_version": ANSWER_PROMPT_VERSION,
        "evidence": evidence,
        "sources": list(sources.values()),
    }


RELATION_QUESTION_PATTERN_V1 = re.compile(
    r"(?:같이|함께|동시에|동시\s*신청|병행|둘\s*다|두\s*(?:제도|업무)\s*모두)", re.I
)
RELATION_POSITIVE_ANSWER_PATTERN_V1 = re.compile(
    r"(?:같이|함께|동시에|병행).{0,12}(?:신청|이용).{0,8}(?:가능|할\s*수\s*있)", re.I
)
RELATION_EVIDENCE_PATTERN_V1 = re.compile(
    r"(?:동시\s*신청|함께\s*신청|같이\s*신청|병행\s*(?:신청|이용)|중복\s*신청)", re.I
)


def relation_constraint_v1(question: str, pack: Mapping[str, Any]) -> dict[str, Any]:
    relation_question = bool(RELATION_QUESTION_PATTERN_V1.search(question))
    direct_rows = [
        str(row.get("evidence_id"))
        for row in pack.get("evidence") or []
        if RELATION_EVIDENCE_PATTERN_V1.search(str(row.get("content") or ""))
    ]
    return {
        "relation_question": relation_question,
        "direct_relation_evidence_ids": direct_rows,
        "may_affirm_joint_application": bool(direct_rows),
        "rule": (
            "두 제도의 동시·병행 가능성을 직접 명시한 동일 Evidence가 없으면 "
            "가능하다고 단정하지 않고 확인되지 않는다고 답한다."
        ),
    }


B_LOW_LATENCY_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 답변 시스템입니다.

1. 사용자 질문과 제공된 Evidence Pack의 사실만 사용하세요.
2. 질문에 직접 답하고 필요한 대상·조건·예외·금액·기간·절차를 설명하세요.
3. Evidence에 없는 사실이나 서로 다른 제도의 조건을 임의로 결합하지 마세요.
4. 동시·병행 신청 질문은 같은 Evidence가 그 관계를 직접 명시할 때만 가능하다고 답하세요.
5. 별도 문서가 각 제도의 자격을 각각 설명한다는 사실만으로 동시 신청 가능성을 추론하지 마세요.
6. 직접 관계 근거가 없으면 확인되지 않는다고 답하고 coverage_status를 PARTIAL로 두세요.
7. 문장 수는 제한하지 않되 질문하지 않은 배경 설명과 중복은 넣지 마세요.
8. 근거 문장 끝에 [E1] 형식으로 실제 Evidence ID를 표시하세요.
9. 지정된 JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()


D_SKELETON_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서에서 답변에 필요한 사실 구조만 추출하는 분석기입니다.

1. 사용자 질문과 동일 Evidence Pack에 명시된 사실만 사용하세요.
2. 최종 사용자 문장이 아니라 Answer Skeleton JSON만 작성하세요.
3. 각 answer_item에는 질문이 요구한 항목 하나와 실제 evidence_ids를 연결하세요.
4. 서로 다른 제도의 조건을 임의로 결합하지 마세요.
5. 동시·병행 신청 관계는 같은 Evidence가 그 관계를 직접 명시할 때만 claim으로 채택하세요.
6. 직접 관계 근거가 없으면 uncertainties에 기록하고 가능하다고 추론하지 마세요.
7. 문서 충돌은 conflicts, 확인 불가는 uncertainties에 기록하세요.
8. JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()


D_FINAL_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 질의응답 시스템입니다.

1. 제공된 Answer Skeleton과 동일 Evidence Pack의 사실만 사용하세요.
2. 질문에 직접 답하고 Skeleton의 항목에 필요한 조건·예외·금액·기간·절차를 설명하세요.
3. Skeleton 또는 Evidence에 없는 사실을 추정하지 마세요.
4. 동시·병행 가능성에 직접 근거가 없으면 가능하다고 단정하지 마세요.
5. 문장 수는 제한하지 않되 질문하지 않은 배경 설명과 중복은 넣지 마세요.
6. 각 주장에는 Skeleton이 허용한 [E1] 형식의 Evidence ID만 표시하세요.
7. JSON, Skeleton, 내부 구현, 검색 점수는 답변에 언급하지 마세요.
""".strip()


def _compact_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"), default=str)


def _usage_dict(response: Any) -> dict[str, int]:
    usage = getattr(response, "usage", None)
    return {
        key: int(getattr(usage, key, 0) or 0)
        for key in ("prompt_tokens", "completion_tokens", "total_tokens")
    }


def _call_answer_api_v1(
    *, system_prompt: str, user_prompt: str, max_tokens: int
) -> tuple[str, dict[str, int], float, dict[str, Any]]:
    global _LAST_ANSWER_API_TRACE
    started = time.perf_counter()
    response = ANSWER_HCX_CLIENT.chat.completions.create(
        model=HCX_CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    wall_ms = (time.perf_counter() - started) * 1000
    content = response.choices[0].message.content
    if not content or not str(content).strip():
        raise RuntimeError("HCX 답변 출력이 비어 있습니다.")
    return str(content), _usage_dict(response), wall_ms, dict(_LAST_ANSWER_API_TRACE)


def _merge_usage_v1(*values: Mapping[str, Any]) -> dict[str, int]:
    return {
        key: sum(int(value.get(key) or 0) for value in values)
        for key in ("prompt_tokens", "completion_tokens", "total_tokens")
    }


def _relation_safe_answer_v1(
    answer: str,
    constraint: Mapping[str, Any],
) -> tuple[str, bool]:
    if (
        constraint.get("relation_question")
        and not constraint.get("may_affirm_joint_application")
        and RELATION_POSITIVE_ANSWER_PATTERN_V1.search(answer)
    ):
        return (
            "현재 검색된 공식 문서 근거만으로 두 제도를 동시에 또는 병행하여 "
            "신청할 수 있는지는 확인되지 않습니다. 각 제도의 개별 신청 요건은 "
            "확인할 수 있지만, 그것만으로 동시 신청 가능성을 단정할 수는 없습니다.",
            True,
        )
    return answer, False


def generate_answer_b_low_latency_v1(
    question: str,
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    constraint = relation_constraint_v1(question, pack)
    prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[관계 주장 안전조건]\n{_compact_json(constraint)}\n\n[Basic Evidence Pack]\n{_compact_json(pack)}\n\n[출력 JSON]\n{{\"answer\":\"근거 문장에 [E1] 표시\",\"used_evidence_ids\":[\"E1\"],\"coverage_status\":\"SUFFICIENT|PARTIAL|INSUFFICIENT\",\"missing_information\":[]}}"""
    raw, usage, first_ms, first_trace = _call_answer_api_v1(
        system_prompt=B_LOW_LATENCY_SYSTEM_PROMPT,
        user_prompt=prompt,
        max_tokens=1600,
    )
    attempts = [{"stage": "initial", "latency_ms": first_ms, "trace": first_trace}]
    total_usage = dict(usage)
    total_ms = first_ms
    try:
        raw_payload = answer_b_core._extract_json_object(raw)
        requested_ids = answer_b_core._clean_list(raw_payload.get("used_evidence_ids"))
        allowed = answer_b_core._allowed_evidence(pack)
        valid_ids = [value for value in requested_ids if value in allowed]
        answer = answer_b_core._strip_model_urls(answer_b_core._clean(raw_payload.get("answer")))
        if not answer:
            raise ValueError("answer가 비어 있습니다.")
        if not valid_ids:
            valid_ids = [f"E{n}" for n in re.findall(r"\[E(\d+)\]", answer) if f"E{n}" in allowed]
        if not valid_ids:
            raise ValueError("유효 Evidence ID가 없습니다.")
        payload = {
            "answer": answer,
            "used_evidence_ids": list(dict.fromkeys(valid_ids)),
            "used_chunk_ids": [allowed[value] for value in dict.fromkeys(valid_ids)],
            "coverage_status": str(raw_payload.get("coverage_status") or "PARTIAL").upper(),
            "missing_information": answer_b_core._clean_list(raw_payload.get("missing_information")),
        }
        payload = answer_b_core.validate_basic_answer(payload, pack)
    except (ValueError, TypeError):
        try:
            payload = answer_b_core._recover_basic_answer_from_raw(raw, pack)
            attempts.append({"stage": "local_raw_recovery", "latency_ms": 0.0})
        except (ValueError, TypeError):
            repair_prompt = f"""다음 출력을 사실 변경 없이 올바른 JSON 객체로만 고치세요. Evidence Pack 밖의 ID를 만들지 마세요.\n\n[원래 요청]\n{prompt}\n\n[교정 대상]\n{raw[:6000]}"""
            repaired, repair_usage, repair_ms, repair_trace = _call_answer_api_v1(
                system_prompt=B_LOW_LATENCY_SYSTEM_PROMPT,
                user_prompt=repair_prompt,
                max_tokens=1600,
            )
            total_usage = _merge_usage_v1(total_usage, repair_usage)
            total_ms += repair_ms
            attempts.append({"stage": "repair", "latency_ms": repair_ms, "trace": repair_trace})
            parsed = answer_b_core._extract_json_object(repaired)
            allowed = answer_b_core._allowed_evidence(pack)
            ids = [value for value in answer_b_core._clean_list(parsed.get("used_evidence_ids")) if value in allowed]
            payload = answer_b_core.validate_basic_answer({
                "answer": parsed.get("answer"),
                "used_evidence_ids": ids,
                "used_chunk_ids": [allowed[value] for value in ids],
                "coverage_status": parsed.get("coverage_status") or "PARTIAL",
                "missing_information": parsed.get("missing_information") or [],
            }, pack)

    safe_answer, guard_applied = _relation_safe_answer_v1(payload["answer"], constraint)
    payload["answer"] = safe_answer
    if guard_applied:
        payload["coverage_status"] = "PARTIAL"
        payload["missing_information"] = list(dict.fromkeys(
            list(payload.get("missing_information") or [])
            + ["두 제도의 동시·병행 신청 가능 여부를 직접 명시한 공식 근거"]
        ))
    return {
        **payload,
        "system": "B",
        "latency_ms": total_ms,
        "usage": total_usage,
        "api_calls": sum(1 for row in attempts if row["stage"] in {"initial", "repair"}),
        "attempts": attempts,
        "relation_constraint": constraint,
        "relation_guard_applied": guard_applied,
    }


def _validate_d_skeleton_v1(
    raw: Mapping[str, Any], pack: Mapping[str, Any]
) -> dict[str, Any]:
    allowed = answer_b_core._allowed_evidence(pack)
    items: list[dict[str, Any]] = []
    for index, item in enumerate(raw.get("answer_items") or [], start=1):
        if not isinstance(item, Mapping):
            continue
        claim = answer_b_core._clean(item.get("claim"))
        ids = [value for value in answer_b_core._clean_list(item.get("evidence_ids")) if value in allowed]
        if claim and ids:
            items.append({
                "item_id": f"A{len(items) + 1}",
                "topic": answer_b_core._clean(item.get("topic")) or f"답변 항목 {index}",
                "claim": claim,
                "conditions": answer_b_core._clean_list(item.get("conditions")),
                "details": answer_b_core._clean_list(item.get("details")),
                "evidence_ids": list(dict.fromkeys(ids)),
            })
    core = answer_b_core._clean(raw.get("core_answer"))
    if not core or not items:
        raise ValueError("D안 Skeleton의 핵심 답변 또는 유효 항목이 없습니다.")
    coverage = answer_b_core._clean(raw.get("coverage_status")).upper()
    if coverage not in answer_b_core.ALLOWED_COVERAGE_STATUS:
        coverage = "PARTIAL"
    return {
        "core_answer": core,
        "answer_items": items,
        "uncertainties": answer_b_core._clean_list(raw.get("uncertainties")),
        "conflicts": answer_b_core._clean_list(raw.get("conflicts")),
        "coverage_status": coverage,
    }


def generate_answer_d_v1(
    question: str,
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    constraint = relation_constraint_v1(question, pack)
    skeleton_prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[관계 주장 안전조건]\n{_compact_json(constraint)}\n\n[Evidence Pack]\n{_compact_json(pack)}\n\n[출력 JSON]\n{{\"core_answer\":\"핵심 결론\",\"answer_items\":[{{\"topic\":\"항목\",\"claim\":\"근거 사실\",\"conditions\":[],\"details\":[],\"evidence_ids\":[\"E1\"]}}],\"uncertainties\":[],\"conflicts\":[],\"coverage_status\":\"SUFFICIENT|PARTIAL|INSUFFICIENT\"}}"""
    raw_skeleton, usage1, skeleton_ms, trace1 = _call_answer_api_v1(
        system_prompt=D_SKELETON_SYSTEM_PROMPT,
        user_prompt=skeleton_prompt,
        max_tokens=2000,
    )
    skeleton = _validate_d_skeleton_v1(
        answer_b_core._extract_json_object(raw_skeleton), pack
    )
    final_prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[관계 주장 안전조건]\n{_compact_json(constraint)}\n\n[Answer Skeleton]\n{_compact_json(skeleton)}\n\n[동일 Evidence Pack]\n{_compact_json(pack)}\n\n위 근거 범위에서 사용자용 최종 답변을 작성하세요."""
    raw_answer, usage2, final_ms, trace2 = _call_answer_api_v1(
        system_prompt=D_FINAL_SYSTEM_PROMPT,
        user_prompt=final_prompt,
        max_tokens=1600,
    )
    answer = answer_b_core._strip_model_urls(str(raw_answer).strip())
    if not answer:
        raise ValueError("D안 최종 답변이 비어 있습니다.")
    safe_answer, guard_applied = _relation_safe_answer_v1(answer, constraint)
    return {
        "system": "D",
        "answer": safe_answer,
        "skeleton": skeleton,
        "coverage_status": "PARTIAL" if guard_applied else skeleton["coverage_status"],
        "latency_ms": skeleton_ms + final_ms,
        "skeleton_latency_ms": skeleton_ms,
        "final_latency_ms": final_ms,
        "usage": _merge_usage_v1(usage1, usage2),
        "api_calls": 2,
        "attempts": [
            {"stage": "skeleton", "latency_ms": skeleton_ms, "trace": trace1},
            {"stage": "final", "latency_ms": final_ms, "trace": trace2},
        ],
        "relation_constraint": constraint,
        "relation_guard_applied": guard_applied,
    }


print({
    "search_parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
    "answer_evidence_total_max_chars": ANSWER_EVIDENCE_TOTAL_MAX_CHARS,
    "answer_evidence_rank_budgets": ANSWER_EVIDENCE_RANK_BUDGETS,
    "sentence_limit": None,
    "comparison_cache": ANSWER_CACHE_ENABLED_FOR_COMPARISON,
})


## 13. 공통 검색 1회와 B/D 공정 비교

질의분석과 검색은 한 번만 실행합니다. B/D 실행 순서는 질문마다 교대할 수 있으며, 각 시스템의 API 호출 수·토큰·429 대기·답변 Wall·가상 End-to-End latency를 분리합니다.


In [ ]:
from __future__ import annotations

import copy
import html
import time
from typing import Any, Mapping, Sequence


def new_bd_comparison_state() -> dict[str, Any]:
    return new_context_state()


def prepare_common_retrieval_v1(
    question: str,
    *,
    state: dict[str, Any],
) -> dict[str, Any]:
    global _LAST_RERANK_TRACE, _LAST_PARENT_CHILD_TRACE
    total_started = time.perf_counter()
    _LAST_RERANK_TRACE = {}
    _LAST_PARENT_CHILD_TRACE = {}

    analysis = ANALYZER_FUNCTIONS["v15"](question, state)
    analysis_ms = (time.perf_counter() - total_started) * 1000
    if analysis["route"] != "RETRIEVE":
        return {
            "question": question,
            "resolved_question": analysis.get("resolved_question") or "",
            "route": analysis["route"],
            "analysis": analysis,
            "route_message": _route_message(analysis),
            "latency_ms": {
                "질의분석": analysis_ms,
                "공통 준비 전체": (time.perf_counter() - total_started) * 1000,
            },
        }

    plans = list(analysis.get("plans") or [])
    if not analysis.get("query_plan_valid") or not _comparison_plans_valid("RETRIEVE", plans):
        raise RuntimeError("RETRIEVE 검색 계획이 유효하지 않습니다.")

    search_started = time.perf_counter()
    search_results, per_query = fuse_query_results(plans)
    child_search_ms = (time.perf_counter() - search_started) * 1000
    reranker_trace = dict(_LAST_RERANK_TRACE)

    parent_started = time.perf_counter()
    search_results = expand_parent_context(search_results)
    parent_ms = (time.perf_counter() - parent_started) * 1000
    parent_trace = dict(_LAST_PARENT_CHILD_TRACE)

    answer_question = str(analysis.get("resolved_question") or question)
    pack_started = time.perf_counter()
    pack = build_compact_parent_evidence_pack_v1(answer_question, search_results)
    pack_ms = (time.perf_counter() - pack_started) * 1000
    per_query_latency = _query_latency_rows(per_query)
    common_ms = (time.perf_counter() - total_started) * 1000
    return {
        "question": question,
        "resolved_question": answer_question,
        "route": analysis["route"],
        "analysis": analysis,
        "plans": plans,
        "search_results": search_results,
        "per_query": per_query,
        "per_query_latency": per_query_latency,
        "reranker": reranker_trace,
        "parent_child": parent_trace,
        "evidence_pack": pack,
        "evidence_pack_sha256": evidence_pack_sha256(pack),
        "evidence_chars": sum(int(row.get("context_char_count") or 0) for row in pack["evidence"]),
        "latency_ms": {
            "문맥 처리": float(analysis.get("context_latency_ms") or 0),
            "질의분석": analysis_ms,
            "검색": child_search_ms + parent_ms,
            "질문 임베딩": sum(row["embedding_latency_ms"] for row in per_query_latency),
            "Dense 계산": sum(row["dense_compute_latency_ms"] for row in per_query_latency),
            "BM25": sum(row["bm25_latency_ms"] for row in per_query_latency),
            "BAAI Reranker": float(reranker_trace.get("latency_ms") or 0),
            "Parent-Child8192": parent_ms,
            "저지연 Evidence Pack": pack_ms,
            "공통 준비 전체": common_ms,
        },
    }


def _trace_totals_v1(payload: Mapping[str, Any]) -> dict[str, float | int]:
    waits = 0.0
    rate_limits = 0
    for stage in payload.get("attempts") or []:
        trace = stage.get("trace") or {}
        waits += float(trace.get("total_wait_ms") or 0)
        rate_limits += sum(
            1 for attempt in trace.get("attempts") or []
            if attempt.get("status") == "RATE_LIMIT_429"
        )
    return {"wait_ms": waits, "rate_limit_429_count": rate_limits}


_BD_ORDER_COUNTER = 0


def compare_answer_systems_v1(
    question: str,
    *,
    state: dict[str, Any],
    order: str = "ALTERNATE",
) -> dict[str, Any]:
    global _BD_ORDER_COUNTER
    common = prepare_common_retrieval_v1(question, state=state)
    if common["route"] != "RETRIEVE":
        return {"common": common, "answers": {}, "summary": pd.DataFrame()}

    normalized_order = str(order or "ALTERNATE").upper()
    if normalized_order == "ALTERNATE":
        normalized_order = "B_FIRST" if _BD_ORDER_COUNTER % 2 == 0 else "D_FIRST"
        _BD_ORDER_COUNTER += 1
    sequence = ("B", "D") if normalized_order == "B_FIRST" else ("D", "B")
    answers: dict[str, dict[str, Any]] = {}
    for system in sequence:
        if system == "B":
            answers[system] = generate_answer_b_low_latency_v1(
                common["resolved_question"], common["evidence_pack"]
            )
        else:
            answers[system] = generate_answer_d_v1(
                common["resolved_question"], common["evidence_pack"]
            )

    common_ms = float(common["latency_ms"]["공통 준비 전체"])
    rows = []
    for system in ("B", "D"):
        payload = answers[system]
        trace = _trace_totals_v1(payload)
        usage = payload.get("usage") or {}
        rows.append({
            "답변안": system,
            "실행순서": sequence.index(system) + 1,
            "Evidence문자": common["evidence_chars"],
            "API호출": int(payload.get("api_calls") or 0),
            "429횟수": int(trace["rate_limit_429_count"]),
            "호출간격·429대기(ms)": float(trace["wait_ms"]),
            "입력토큰": int(usage.get("prompt_tokens") or 0),
            "출력토큰": int(usage.get("completion_tokens") or 0),
            "답변지연(ms)": float(payload.get("latency_ms") or 0),
            "가상E2E(ms)": common_ms + float(payload.get("latency_ms") or 0),
            "답변글자": len(str(payload.get("answer") or "")),
            "Coverage": payload.get("coverage_status"),
            "관계안전가드": bool(payload.get("relation_guard_applied")),
        })
    summary = pd.DataFrame(rows)
    state.setdefault("turns", []).extend([
        {"role": "user", "content": question},
        {"role": "assistant", "content": user_visible_answer(answers["B"]["answer"])},
    ])
    return {
        "common": common,
        "answers": answers,
        "execution_order": list(sequence),
        "summary": summary,
    }


def _sources_from_pack_v1(pack: Mapping[str, Any]) -> str:
    lines = ["### 공통 공식 출처", ""]
    for source in pack.get("sources") or []:
        title = str(source.get("title") or "공식 출처")
        url = str(source.get("source_url") or "")
        if url:
            lines.append(f"- [{title}]({url})")
    return "\n".join(lines) if len(lines) > 2 else ""


def render_bd_comparison_v1(result: Mapping[str, Any], *, show_pack: bool = False) -> None:
    common = result["common"]
    if common["route"] != "RETRIEVE":
        display(Markdown(f"### {common['route']}\n\n{common.get('route_message', '')}"))
        return
    answers = result["answers"]
    display(Markdown("## 답변 B안\n\n" + user_visible_answer(answers["B"]["answer"])))
    display(Markdown("## 답변 D안\n\n" + user_visible_answer(answers["D"]["answer"])))
    source_md = _sources_from_pack_v1(common["evidence_pack"])
    if source_md:
        display(Markdown(source_md))
    display(Markdown("### B/D 레이턴시·토큰 비교"))
    display(result["summary"])
    display(Markdown(analysis_markdown_compare({
        "analyzer_label": "V1.5 관계형 멀티턴 개선",
        "analysis": common["analysis"],
    })))
    display(Markdown(retrieval_markdown_compare({"search_results": common["search_results"]})))
    display(Markdown(latency_markdown_compare({"latency_ms": common["latency_ms"]})))
    if show_pack:
        display(JSON(common["evidence_pack"], expanded=False))


print("공통 검색 1회 기반 B/D 비교 실행기 준비 완료")


## 14. 규칙 기반 Answer Need 추출

검색 질의를 다시 작성하지 않습니다. 사용자가 최종 답변에서 요구한 `주체·자격·금액·절차·서류·기간·비용·예외·비교` 항목만 규칙으로 표시합니다. 추가 LLM 호출은 없습니다.


In [ ]:
from __future__ import annotations

import json
import re
from typing import Any, Mapping, Sequence


ANSWER_NEED_RULES_V2: tuple[tuple[str, str, re.Pattern[str]], ...] = (
    ("ACTOR", "신청·신고 주체", re.compile(r"(?:누가|누구|어떤\s*사람|신청자|신고자)")),
    ("ELIGIBILITY", "신청·지원 자격", re.compile(r"(?:자격|지원\s*대상|신청\s*대상|신고\s*대상|대상자|가능\s*여부)")),
    ("AMOUNT", "금액·한도", re.compile(r"(?:얼마|금액|한도|포상금|보호액|비율|퍼센트|%)")),
    ("DOCUMENTS", "필요서류", re.compile(r"(?:서류|구비서류|필요\s*서류|준비물|증빙)")),
    ("PROCEDURE", "신청·처리 절차", re.compile(r"(?:어떻게|절차|방법|순서|과정|신청하려면|신고하려면|접수하려면)")),
    ("TIME", "기간·기한", re.compile(r"(?:기간|기한|언제|얼마나\s*걸|며칠|몇\s*일)")),
    ("COST", "비용·수수료", re.compile(r"(?:비용|수수료|돈이\s*드|차감)")),
    ("EXCEPTION", "예외·제외", re.compile(r"(?:예외|제외|안\s*되는|불가능|받지\s*못|해당하지\s*않)")),
    ("COMPARISON", "차이·비교", re.compile(r"(?:차이|다른가|비교|무엇이\s*다|뭐가\s*다)")),
)

NEED_STATUS_VALUES_V2 = {"ANSWERED", "PARTIAL", "UNSUPPORTED"}


def _question_clauses_v2(question: str) -> list[str]:
    cleaned = _clean_text(question)
    parts = re.split(
        r"\s*(?:,|;|\?|그리고|또한|또|및|그러면|그럼)\s*",
        cleaned,
    )
    return [part.strip(" .?!") for part in parts if part.strip(" .?!")]


def extract_answer_needs_v2(question: str) -> list[dict[str, Any]]:
    original = _clean_text(question)
    clauses = _question_clauses_v2(original) or [original]
    needs: list[dict[str, Any]] = []
    for need_type, label, pattern in ANSWER_NEED_RULES_V2:
        if not pattern.search(original):
            continue
        matching = [clause for clause in clauses if pattern.search(clause)]
        needs.append({
            "need_id": f"N{len(needs) + 1}",
            "need_type": need_type,
            "label": label,
            "question_part": matching[0] if matching else original,
        })
    if not needs:
        needs.append({
            "need_id": "N1",
            "need_type": "GENERAL",
            "label": "질문의 핵심 요청",
            "question_part": original,
        })
    return needs


def normalize_need_coverage_v2(
    raw_rows: Any,
    answer_needs: Sequence[Mapping[str, Any]],
    allowed_evidence_ids: set[str],
) -> list[dict[str, Any]]:
    rows = raw_rows if isinstance(raw_rows, list) else []
    by_id = {
        str(row.get("need_id") or ""): row
        for row in rows
        if isinstance(row, Mapping)
    }
    normalized: list[dict[str, Any]] = []
    for need in answer_needs:
        need_id = str(need["need_id"])
        row = by_id.get(need_id) or {}
        status = str(row.get("status") or "UNSUPPORTED").upper()
        evidence_ids = [
            value
            for value in answer_b_core._clean_list(row.get("evidence_ids"))
            if value in allowed_evidence_ids
        ]
        if status not in NEED_STATUS_VALUES_V2:
            status = "UNSUPPORTED"
        if status == "ANSWERED" and not evidence_ids:
            status = "PARTIAL"
        normalized.append({
            "need_id": need_id,
            "need_type": str(need.get("need_type") or "GENERAL"),
            "label": str(need.get("label") or need_id),
            "status": status,
            "evidence_ids": list(dict.fromkeys(evidence_ids)),
            "missing_reason": answer_b_core._clean(row.get("missing_reason")),
        })
    return normalized


def calculate_program_coverage_v2(
    need_coverage: Sequence[Mapping[str, Any]],
) -> dict[str, Any]:
    statuses = [str(row.get("status") or "UNSUPPORTED") for row in need_coverage]
    total = len(statuses)
    answered = statuses.count("ANSWERED")
    partial = statuses.count("PARTIAL")
    unsupported = statuses.count("UNSUPPORTED")
    if total and answered == total:
        overall = "SUFFICIENT"
    elif answered or partial:
        overall = "PARTIAL"
    else:
        overall = "INSUFFICIENT"
    return {
        "coverage_status": overall,
        "need_count": total,
        "answered_need_count": answered,
        "partial_need_count": partial,
        "unsupported_need_count": unsupported,
        "strict_need_coverage_rate": answered / total if total else 0.0,
        "answerable_need_coverage_rate": (answered + partial) / total if total else 0.0,
    }


NUMERIC_FACT_PATTERN_V2 = re.compile(
    r"\d+(?:[.,]\d+)?\s*(?:%|퍼센트|억원|백만원|만원|원|년|개월|일|시간)",
    re.I,
)


def _normalize_numeric_fact_v2(value: str) -> str:
    return re.sub(r"[\s,]", "", str(value or "")).lower()


def audit_numeric_support_v2(
    answer: str,
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    evidence_text = " ".join(str(row.get("content") or "") for row in pack.get("evidence") or [])
    evidence_numbers = {
        _normalize_numeric_fact_v2(value)
        for value in NUMERIC_FACT_PATTERN_V2.findall(evidence_text)
    }
    answer_numbers = list(dict.fromkeys(
        _normalize_numeric_fact_v2(value)
        for value in NUMERIC_FACT_PATTERN_V2.findall(str(answer or ""))
    ))
    unsupported = [value for value in answer_numbers if value not in evidence_numbers]
    return {
        "answer_numeric_facts": answer_numbers,
        "unsupported_numeric_facts": unsupported,
        "numeric_support_passed": not unsupported,
    }


answer_need_regression_v2 = pd.DataFrame([
    {
        "question": question,
        "types": ",".join(row["need_type"] for row in extract_answer_needs_v2(question)),
        "expected": expected,
    }
    for question, expected in (
        ("은닉재산 신고는 누가 할 수 있고, 포상금은 얼마나 받나요?", "ACTOR,AMOUNT"),
        ("채무조정 신청 자격과 필요서류를 알려주세요", "ELIGIBILITY,DOCUMENTS"),
        ("착오송금 반환 신청은 어떻게 하나요?", "PROCEDURE"),
        ("미수령금과 착오송금은 무엇이 다른가요?", "COMPARISON"),
    )
])
answer_need_regression_v2["passed"] = (
    answer_need_regression_v2["types"] == answer_need_regression_v2["expected"]
)
display(answer_need_regression_v2)
if not bool(answer_need_regression_v2["passed"].all()):
    raise AssertionError("Answer Need 규칙 회귀 테스트 실패")
print("Answer Need 규칙 회귀 테스트 통과")


## 15. B2 · Answer Need와 프로그램 Coverage

B0와 동일하게 답변 API는 원칙적으로 한 번만 호출합니다. 모델이 전체 Coverage를 결정하지 않으며, 모든 Need의 상태와 Evidence 연결을 프로그램이 검증해 최종 Coverage를 계산합니다.


In [ ]:
B2_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 답변 시스템입니다.

근거 규칙:
1. 사용자 질문, Answer Need, Evidence Pack에 포함된 사실만 사용하세요.
2. Evidence에 없는 사실이나 서로 다른 제도의 조건을 임의로 결합하지 마세요.
3. 동시·병행 가능성은 같은 Evidence가 그 관계를 직접 명시할 때만 단정하세요.

완전성 규칙:
4. 모든 need_id를 한 번씩 처리하세요.
5. ANSWERED에는 실제로 해당 요구를 뒷받침하는 evidence_ids가 있어야 합니다.
6. 일부만 확인되면 PARTIAL, 확인할 수 없으면 UNSUPPORTED로 표시하세요.
7. 금액 구간·최고 한도·선행 조건이 질문과 직접 관련되고 Evidence에 있으면 생략하지 마세요.
8. 질문하지 않은 배경·회수 이후 절차·중복 설명은 추가하지 마세요.

출력 규칙:
9. 문장 수는 제한하지 않습니다. 결론을 먼저 쓰고 질문 유형에 맞게 목록·표·소제목을 사용하세요.
10. 근거 문장 끝에는 [E1] 형식의 실제 Evidence ID를 표시하세요.
11. JSON 객체 하나만 출력하고 문자열 내부 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()


def _validate_b2_payload_v2(
    raw: Mapping[str, Any],
    answer_needs: Sequence[Mapping[str, Any]],
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    answer = answer_b_core._strip_model_urls(answer_b_core._clean(raw.get("answer")))
    if not answer:
        raise ValueError("B2 answer가 비어 있습니다.")
    allowed = set(answer_b_core._allowed_evidence(pack))
    need_coverage = normalize_need_coverage_v2(
        raw.get("need_coverage"), answer_needs, allowed
    )
    used_ids = list(dict.fromkeys(
        evidence_id
        for row in need_coverage
        for evidence_id in row["evidence_ids"]
    ))
    for number in re.findall(r"\[E(\d+)\]", answer):
        evidence_id = f"E{number}"
        if evidence_id in allowed and evidence_id not in used_ids:
            used_ids.append(evidence_id)
    if not used_ids:
        raise ValueError("B2 출력에 유효한 Evidence ID가 없습니다.")
    program = calculate_program_coverage_v2(need_coverage)
    numeric = audit_numeric_support_v2(answer, pack)
    return {
        "answer": answer,
        "answer_needs": list(answer_needs),
        "need_coverage": need_coverage,
        **program,
        "used_evidence_ids": used_ids,
        "used_chunk_ids": [answer_b_core._allowed_evidence(pack)[value] for value in used_ids],
        "missing_information": answer_b_core._clean_list(raw.get("missing_information")),
        "numeric_audit": numeric,
    }


def generate_answer_b2_v2(
    question: str,
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    answer_needs = extract_answer_needs_v2(question)
    relation_constraint = relation_constraint_v1(question, pack)
    prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[Answer Needs]\n{_compact_json(answer_needs)}\n\n[관계 주장 안전조건]\n{_compact_json(relation_constraint)}\n\n[Evidence Pack]\n{_compact_json(pack)}\n\n[출력 JSON]\n{{\"answer\":\"근거 문장에 [E1] 표시\",\"need_coverage\":[{{\"need_id\":\"N1\",\"status\":\"ANSWERED|PARTIAL|UNSUPPORTED\",\"evidence_ids\":[\"E1\"],\"missing_reason\":\"\"}}],\"missing_information\":[]}}"""
    raw, usage, first_ms, first_trace = _call_answer_api_v1(
        system_prompt=B2_SYSTEM_PROMPT,
        user_prompt=prompt,
        max_tokens=1600,
    )
    attempts = [{"stage": "initial", "latency_ms": first_ms, "trace": first_trace}]
    total_usage = dict(usage)
    total_ms = first_ms
    try:
        payload = _validate_b2_payload_v2(
            answer_b_core._extract_json_object(raw), answer_needs, pack
        )
    except (ValueError, TypeError):
        try:
            recovered = answer_b_core._recover_basic_answer_from_raw(raw, pack)
            allowed_ids = list(recovered["used_evidence_ids"])
            fallback_rows = [
                {
                    "need_id": need["need_id"],
                    "status": "PARTIAL",
                    "evidence_ids": allowed_ids,
                    "missing_reason": "구조화 Need 메타데이터를 로컬 복구함",
                }
                for need in answer_needs
            ]
            payload = _validate_b2_payload_v2({
                "answer": recovered["answer"],
                "need_coverage": fallback_rows,
                "missing_information": ["Need별 구조화 메타데이터 로컬 복구"],
            }, answer_needs, pack)
            attempts.append({"stage": "local_raw_recovery", "latency_ms": 0.0})
        except (ValueError, TypeError):
            repair_prompt = f"""직전 출력을 사실 변경 없이 요청한 JSON 객체로만 교정하세요. 모든 need_id를 보존하세요.\n\n[원래 요청]\n{prompt}\n\n[직전 출력]\n{raw[:6000]}"""
            repaired, repair_usage, repair_ms, repair_trace = _call_answer_api_v1(
                system_prompt=B2_SYSTEM_PROMPT,
                user_prompt=repair_prompt,
                max_tokens=1600,
            )
            total_usage = _merge_usage_v1(total_usage, repair_usage)
            total_ms += repair_ms
            attempts.append({"stage": "repair", "latency_ms": repair_ms, "trace": repair_trace})
            payload = _validate_b2_payload_v2(
                answer_b_core._extract_json_object(repaired), answer_needs, pack
            )

    safe_answer, guard_applied = _relation_safe_answer_v1(
        payload["answer"], relation_constraint
    )
    payload["answer"] = safe_answer
    if guard_applied:
        payload["coverage_status"] = "PARTIAL"
    return {
        **payload,
        "system": "B2",
        "latency_ms": total_ms,
        "usage": total_usage,
        "api_calls": sum(1 for row in attempts if row["stage"] in {"initial", "repair"}),
        "attempts": attempts,
        "relation_constraint": relation_constraint,
        "relation_guard_applied": guard_applied,
    }


print("B2 Answer Need + 프로그램 Coverage 생성기 준비 완료")


## 16. D2 · Need 기반 Skeleton과 선택 Evidence

Skeleton이 모든 Need를 보존하도록 검증합니다. 최종답변 호출에는 Skeleton이 실제 참조한 Evidence만 전달하여 D안의 두 번째 입력 토큰을 줄입니다.


In [ ]:
D2_SKELETON_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서에서 답변에 필요한 사실 구조를 추출하는 분석기입니다.

1. 사용자 질문, Answer Need, Evidence Pack에 명시된 사실만 사용하세요.
2. 모든 need_id에 정확히 하나의 answer_item을 작성하세요.
3. Evidence로 충분히 답하면 ANSWERED, 일부만 확인되면 PARTIAL, 없으면 UNSUPPORTED로 표시하세요.
4. ANSWERED와 PARTIAL에는 실제 evidence_ids를 연결하세요.
5. 신청 주체 질문에서도 선행 자격 조건이 Evidence에 있으면 conditions에 보존하세요.
6. 금액 질문에서는 구간, 최고 한도, 산정 기준을 Evidence 범위에서 details에 보존하세요.
7. 서로 다른 제도의 조건을 결합하지 말고, 직접 관계 근거가 없는 동시 신청은 uncertainties에 기록하세요.
8. 최종 사용자 문장이 아니라 JSON Skeleton 하나만 출력하세요.
""".strip()

D2_FINAL_SYSTEM_PROMPT = """
당신은 예금보험공사 공식 문서 기반 질의응답 시스템입니다.

1. Answer Needs, Answer Skeleton, 선택 Evidence에 있는 사실만 사용하세요.
2. 모든 answer_item을 최종답변에 반영하세요.
3. ANSWERED의 claim·conditions·details에서 질문 판단에 필요한 내용을 생략하지 마세요.
4. PARTIAL과 UNSUPPORTED는 확인 가능한 범위와 부족한 정보를 구분하세요.
5. core_answer 한 문장만 복사하고 종료하지 마세요.
6. 문장 수는 제한하지 않되 질문하지 않은 배경과 반복은 추가하지 마세요.
7. 근거 문장에는 Skeleton이 허용한 [E1] 형식의 Evidence ID만 표시하세요.
8. Skeleton, JSON, 내부 구현, 검색 점수는 답변에 언급하지 마세요.
""".strip()


def validate_d2_skeleton_v2(
    raw: Mapping[str, Any],
    answer_needs: Sequence[Mapping[str, Any]],
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    allowed = set(answer_b_core._allowed_evidence(pack))
    raw_items = raw.get("answer_items") if isinstance(raw.get("answer_items"), list) else []
    by_need = {
        str(item.get("need_id") or ""): item
        for item in raw_items
        if isinstance(item, Mapping)
    }
    items: list[dict[str, Any]] = []
    coverage_rows: list[dict[str, Any]] = []
    for need in answer_needs:
        need_id = str(need["need_id"])
        source = by_need.get(need_id) or {}
        status = str(source.get("status") or "UNSUPPORTED").upper()
        if status not in NEED_STATUS_VALUES_V2:
            status = "UNSUPPORTED"
        evidence_ids = [
            value
            for value in answer_b_core._clean_list(source.get("evidence_ids"))
            if value in allowed
        ]
        if status == "ANSWERED" and not evidence_ids:
            status = "PARTIAL" if answer_b_core._clean(source.get("claim")) else "UNSUPPORTED"
        claim = answer_b_core._clean(source.get("claim"))
        if status == "UNSUPPORTED" and not claim:
            claim = f"{need['label']}은 현재 Evidence로 확인되지 않습니다."
        items.append({
            "need_id": need_id,
            "need_type": need["need_type"],
            "topic": answer_b_core._clean(source.get("topic")) or need["label"],
            "status": status,
            "claim": claim,
            "conditions": answer_b_core._clean_list(source.get("conditions")),
            "details": answer_b_core._clean_list(source.get("details")),
            "evidence_ids": list(dict.fromkeys(evidence_ids)),
            "missing_reason": answer_b_core._clean(source.get("missing_reason")),
        })
        coverage_rows.append({
            "need_id": need_id,
            "need_type": need["need_type"],
            "label": need["label"],
            "status": status,
            "evidence_ids": list(dict.fromkeys(evidence_ids)),
            "missing_reason": answer_b_core._clean(source.get("missing_reason")),
        })
    program = calculate_program_coverage_v2(coverage_rows)
    return {
        "core_answer": answer_b_core._clean(raw.get("core_answer")),
        "answer_items": items,
        "need_coverage": coverage_rows,
        "uncertainties": answer_b_core._clean_list(raw.get("uncertainties")),
        "conflicts": answer_b_core._clean_list(raw.get("conflicts")),
        **program,
    }


def filter_pack_for_d2_v2(
    pack: Mapping[str, Any],
    skeleton: Mapping[str, Any],
) -> dict[str, Any]:
    used_ids = {
        evidence_id
        for item in skeleton.get("answer_items") or []
        for evidence_id in item.get("evidence_ids") or []
    }
    evidence = [
        dict(row)
        for row in pack.get("evidence") or []
        if row.get("evidence_id") in used_ids
    ]
    source_urls = {str(row.get("source_url") or "") for row in evidence}
    return {
        **dict(pack),
        "evidence": evidence,
        "sources": [
            dict(row)
            for row in pack.get("sources") or []
            if str(row.get("source_url") or "") in source_urls
        ],
        "filtered_for_d2": True,
        "original_evidence_count": len(pack.get("evidence") or []),
        "selected_evidence_count": len(evidence),
    }


def generate_answer_d2_v2(
    question: str,
    pack: Mapping[str, Any],
) -> dict[str, Any]:
    answer_needs = extract_answer_needs_v2(question)
    relation_constraint = relation_constraint_v1(question, pack)
    skeleton_prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[Answer Needs]\n{_compact_json(answer_needs)}\n\n[관계 주장 안전조건]\n{_compact_json(relation_constraint)}\n\n[Evidence Pack]\n{_compact_json(pack)}\n\n[출력 JSON]\n{{\"core_answer\":\"핵심 결론\",\"answer_items\":[{{\"need_id\":\"N1\",\"topic\":\"항목\",\"status\":\"ANSWERED|PARTIAL|UNSUPPORTED\",\"claim\":\"근거 사실\",\"conditions\":[],\"details\":[],\"evidence_ids\":[\"E1\"],\"missing_reason\":\"\"}}],\"uncertainties\":[],\"conflicts\":[]}}"""
    raw_skeleton, usage1, skeleton_ms, trace1 = _call_answer_api_v1(
        system_prompt=D2_SKELETON_SYSTEM_PROMPT,
        user_prompt=skeleton_prompt,
        max_tokens=2000,
    )
    skeleton = validate_d2_skeleton_v2(
        answer_b_core._extract_json_object(raw_skeleton), answer_needs, pack
    )
    selected_pack = filter_pack_for_d2_v2(pack, skeleton)
    final_prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[Answer Needs]\n{_compact_json(answer_needs)}\n\n[관계 주장 안전조건]\n{_compact_json(relation_constraint)}\n\n[Answer Skeleton]\n{_compact_json(skeleton)}\n\n[Skeleton 참조 Evidence]\n{_compact_json(selected_pack)}\n\n위 범위에서 최종 사용자 답변을 작성하세요."""
    raw_answer, usage2, final_ms, trace2 = _call_answer_api_v1(
        system_prompt=D2_FINAL_SYSTEM_PROMPT,
        user_prompt=final_prompt,
        max_tokens=1600,
    )
    answer = answer_b_core._strip_model_urls(str(raw_answer).strip())
    if not answer:
        raise ValueError("D2 최종 답변이 비어 있습니다.")
    safe_answer, guard_applied = _relation_safe_answer_v1(answer, relation_constraint)
    numeric = audit_numeric_support_v2(safe_answer, selected_pack)
    return {
        "system": "D2",
        "answer": safe_answer,
        "answer_needs": answer_needs,
        "skeleton": skeleton,
        "need_coverage": skeleton["need_coverage"],
        "coverage_status": "PARTIAL" if guard_applied else skeleton["coverage_status"],
        "strict_need_coverage_rate": skeleton["strict_need_coverage_rate"],
        "answerable_need_coverage_rate": skeleton["answerable_need_coverage_rate"],
        "numeric_audit": numeric,
        "selected_evidence_pack": selected_pack,
        "selected_evidence_count": len(selected_pack["evidence"]),
        "latency_ms": skeleton_ms + final_ms,
        "skeleton_latency_ms": skeleton_ms,
        "final_latency_ms": final_ms,
        "usage": _merge_usage_v1(usage1, usage2),
        "api_calls": 2,
        "attempts": [
            {"stage": "skeleton", "latency_ms": skeleton_ms, "trace": trace1},
            {"stage": "final", "latency_ms": final_ms, "trace": trace2},
        ],
        "relation_constraint": relation_constraint,
        "relation_guard_applied": guard_applied,
    }


print("D2 Need Skeleton + 선택 Evidence 생성기 준비 완료")


## 18. HCX 공통 호출 게이트와 질문 임베딩 캐시

질문 임베딩과 B0/B2/D2 답변 호출이 동시에 실행되지 않도록 하나의 잠금과 호출 간격을 공유합니다. 429은 `Retry-After`를 우선하며, 없으면 지수 백오프를 사용합니다.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import random
import re
import threading
import time
from collections import OrderedDict
from types import SimpleNamespace
from typing import Any, Callable, Mapping


HCX_GLOBAL_MIN_INTERVAL_SECONDS_V3 = 3.0
HCX_GLOBAL_MAX_ATTEMPTS_V3 = 6
HCX_GLOBAL_BACKOFF_BASE_SECONDS_V3 = 2.0
HCX_GLOBAL_BACKOFF_MAX_SECONDS_V3 = 60.0
QUERY_EMBEDDING_CACHE_ENABLED_V3 = True
QUERY_EMBEDDING_CACHE_MAX_SIZE_V3 = 512


def _rate_limit_headers_v3(error: Exception) -> dict[str, str]:
    response = getattr(error, "response", None)
    headers = getattr(response, "headers", None) or {}
    return {
        str(key): str(value)
        for key, value in dict(headers).items()
        if any(marker in str(key).lower() for marker in (
            "rate", "limit", "remaining", "reset", "retry"
        ))
    }


def _header_delay_seconds_v3(error: Exception) -> float | None:
    headers = _rate_limit_headers_v3(error)
    values: list[float] = []
    for key, value in headers.items():
        if key.lower() not in {
            "retry-after",
            "x-ratelimit-reset-requests",
            "x-ratelimit-reset-tokens",
        }:
            continue
        match = re.search(r"([0-9]+(?:\.[0-9]+)?)", value)
        if match:
            values.append(float(match.group(1)))
    return min(120.0, max(values)) if values else None


class HCXSharedRequestGateV3:
    def __init__(
        self,
        *,
        min_interval_seconds: float,
        max_attempts: int,
    ) -> None:
        self.min_interval_seconds = float(min_interval_seconds)
        self.max_attempts = int(max_attempts)
        self.lock = threading.RLock()
        self.last_call_started_at = 0.0
        self.last_trace: dict[str, Any] = {}
        self.history: list[dict[str, Any]] = []

    def _backoff(self, error: Exception, attempt: int) -> float:
        header_delay = _header_delay_seconds_v3(error)
        if header_delay is not None:
            return min(120.0, max(1.0, header_delay) + random.uniform(0.1, 0.8))
        exponential = HCX_GLOBAL_BACKOFF_BASE_SECONDS_V3 * (2 ** max(0, attempt - 1))
        return min(HCX_GLOBAL_BACKOFF_MAX_SECONDS_V3, exponential) + random.uniform(0.1, 0.8)

    def call(
        self,
        stage: str,
        operation: Callable[[], Any],
        *,
        max_attempts: int | None = None,
    ) -> tuple[Any, dict[str, Any]]:
        attempts_limit = int(max_attempts or self.max_attempts)
        started = time.perf_counter()
        attempts: list[dict[str, Any]] = []
        pacing_wait_seconds = 0.0
        retry_wait_seconds = 0.0
        with self.lock:
            for attempt in range(1, attempts_limit + 1):
                elapsed = time.perf_counter() - self.last_call_started_at
                pacing = max(0.0, self.min_interval_seconds - elapsed)
                if pacing:
                    time.sleep(pacing)
                    pacing_wait_seconds += pacing
                self.last_call_started_at = time.perf_counter()
                try:
                    result = operation()
                except Exception as error:
                    if type(error).__name__ != "RateLimitError":
                        trace = {
                            "stage": stage,
                            "success": False,
                            "attempts": attempts + [{
                                "attempt": attempt,
                                "status": type(error).__name__,
                                "message": str(error)[:1000],
                            }],
                            "pacing_wait_ms": pacing_wait_seconds * 1000,
                            "retry_wait_ms": retry_wait_seconds * 1000,
                            "wall_latency_ms": (time.perf_counter() - started) * 1000,
                        }
                        self.last_trace = trace
                        self.history.append(copy.deepcopy(trace))
                        raise
                    delay = self._backoff(error, attempt)
                    attempts.append({
                        "attempt": attempt,
                        "status": "RATE_LIMIT_429",
                        "delay_seconds": delay,
                        "headers": _rate_limit_headers_v3(error),
                    })
                    if attempt >= attempts_limit:
                        trace = {
                            "stage": stage,
                            "success": False,
                            "attempts": attempts,
                            "pacing_wait_ms": pacing_wait_seconds * 1000,
                            "retry_wait_ms": retry_wait_seconds * 1000,
                            "wall_latency_ms": (time.perf_counter() - started) * 1000,
                        }
                        self.last_trace = trace
                        self.history.append(copy.deepcopy(trace))
                        raise RuntimeError(
                            f"{stage} 단계에서 429 재시도 {attempts_limit}회를 모두 소진했습니다. "
                            "Trace의 Retry-After와 Rate Limit 헤더를 확인하세요."
                        ) from error
                    time.sleep(delay)
                    retry_wait_seconds += delay
                    continue
                attempts.append({"attempt": attempt, "status": "SUCCESS"})
                trace = {
                    "stage": stage,
                    "success": True,
                    "attempts": attempts,
                    "pacing_wait_ms": pacing_wait_seconds * 1000,
                    "retry_wait_ms": retry_wait_seconds * 1000,
                    "wall_latency_ms": (time.perf_counter() - started) * 1000,
                }
                self.last_trace = trace
                self.history.append(copy.deepcopy(trace))
                return result, trace
        raise AssertionError("도달할 수 없는 HCX 공통 게이트 상태")


HCX_SHARED_GATE_V3 = HCXSharedRequestGateV3(
    min_interval_seconds=HCX_GLOBAL_MIN_INTERVAL_SECONDS_V3,
    max_attempts=HCX_GLOBAL_MAX_ATTEMPTS_V3,
)

_HCX_RAW_CLIENT_V3 = (
    HCX_CLIENT.with_options(max_retries=0)
    if hasattr(HCX_CLIENT, "with_options")
    else HCX_CLIENT
)

QUERY_EMBEDDING_CACHE_V3: OrderedDict[str, np.ndarray] = OrderedDict()
_LAST_QUERY_EMBEDDING_TRACE_V3: dict[str, Any] = {}


def _query_embedding_cache_key_v3(text: str) -> str:
    raw = f"{HCX_EMBEDDING_MODEL}\n{HCX_ENCODING_FORMAT}\n{_clean_text(text)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def clear_query_embedding_cache_v3() -> None:
    QUERY_EMBEDDING_CACHE_V3.clear()


def embed_hcx_single(text: str) -> np.ndarray:
    global _LAST_QUERY_EMBEDDING_TRACE_V3
    cleaned = _clean_text(text)
    if not cleaned:
        raise ValueError("임베딩 입력이 비어 있습니다.")
    cache_key = _query_embedding_cache_key_v3(cleaned)
    if QUERY_EMBEDDING_CACHE_ENABLED_V3 and cache_key in QUERY_EMBEDDING_CACHE_V3:
        vector = QUERY_EMBEDDING_CACHE_V3.pop(cache_key)
        QUERY_EMBEDDING_CACHE_V3[cache_key] = vector
        _LAST_QUERY_EMBEDDING_TRACE_V3 = {
            "stage": "query_embedding",
            "success": True,
            "cache_hit": True,
            "attempts": [],
            "pacing_wait_ms": 0.0,
            "retry_wait_ms": 0.0,
            "wall_latency_ms": 0.0,
        }
        return vector.copy()

    def operation() -> Any:
        return _HCX_RAW_CLIENT_V3.embeddings.create(
            model=HCX_EMBEDDING_MODEL,
            input=cleaned,
            encoding_format=HCX_ENCODING_FORMAT,
        )

    response, trace = HCX_SHARED_GATE_V3.call("query_embedding", operation)
    if len(response.data) != 1:
        raise RuntimeError(f"단일 임베딩 응답 개수가 1이 아닙니다: {len(response.data)}")
    vector = np.asarray(response.data[0].embedding, dtype=np.float32)
    if vector.ndim != 1 or vector.size == 0 or not np.all(np.isfinite(vector)):
        raise RuntimeError(f"잘못된 질문 임베딩 벡터입니다: shape={vector.shape}")
    _LAST_QUERY_EMBEDDING_TRACE_V3 = {**trace, "cache_hit": False}
    if QUERY_EMBEDDING_CACHE_ENABLED_V3:
        QUERY_EMBEDDING_CACHE_V3[cache_key] = vector.copy()
        while len(QUERY_EMBEDDING_CACHE_V3) > QUERY_EMBEDDING_CACHE_MAX_SIZE_V3:
            QUERY_EMBEDDING_CACHE_V3.popitem(last=False)
    return vector


class _SharedGateAnswerCompletionsV3:
    def __init__(self, raw_completions: Any) -> None:
        self.raw_completions = raw_completions

    def create(self, **kwargs: Any) -> Any:
        global _LAST_ANSWER_API_TRACE
        response, trace = HCX_SHARED_GATE_V3.call(
            "answer_generation",
            lambda: self.raw_completions.create(**kwargs),
        )
        _LAST_ANSWER_API_TRACE = {
            "attempts": trace.get("attempts") or [],
            "total_wait_ms": float(trace.get("pacing_wait_ms") or 0)
            + float(trace.get("retry_wait_ms") or 0),
            "pacing_wait_ms": trace.get("pacing_wait_ms"),
            "retry_wait_ms": trace.get("retry_wait_ms"),
            "wall_latency_ms": trace.get("wall_latency_ms"),
            "stage": trace.get("stage"),
        }
        return response


class _SharedGateAnswerClientV3:
    def __init__(self, raw_client: Any) -> None:
        self.chat = SimpleNamespace(
            completions=_SharedGateAnswerCompletionsV3(raw_client.chat.completions)
        )
        self._kdic_structured_output_capability = {HCX_CHAT_MODEL: False}


ANSWER_HCX_CLIENT = _SharedGateAnswerClientV3(_HCX_RAW_CLIENT_V3)


def hcx_gate_diagnostics_v3() -> pd.DataFrame:
    rows = []
    for index, trace in enumerate(HCX_SHARED_GATE_V3.history, start=1):
        attempts = trace.get("attempts") or []
        rows.append({
            "call": index,
            "stage": trace.get("stage"),
            "success": bool(trace.get("success")),
            "attempt_count": len(attempts),
            "rate_limit_429_count": sum(
                1 for row in attempts if row.get("status") == "RATE_LIMIT_429"
            ),
            "pacing_wait_ms": float(trace.get("pacing_wait_ms") or 0),
            "retry_wait_ms": float(trace.get("retry_wait_ms") or 0),
            "wall_latency_ms": float(trace.get("wall_latency_ms") or 0),
        })
    return pd.DataFrame(rows)


print({
    "global_hcx_min_interval_seconds": HCX_GLOBAL_MIN_INTERVAL_SECONDS_V3,
    "global_hcx_max_attempts": HCX_GLOBAL_MAX_ATTEMPTS_V3,
    "query_embedding_cache": QUERY_EMBEDDING_CACHE_ENABLED_V3,
    "query_embedding_cache_max_size": QUERY_EMBEDDING_CACHE_MAX_SIZE_V3,
})


## 17. 답변 C용 Fact Index 로딩과 검증

원본 JSON을 수정하지 않습니다. Colab에 업로드된 27개 레코드의 스키마, 상태, 트리거 중복을 검사합니다.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
from pathlib import Path
from typing import Any, Mapping, Sequence


FACT_INDEX_UPLOAD_NAME_C1 = "KDIC_Fact_Index_C_Ver3_Reviewable.json"
FACT_INDEX_EXPECTED_SCHEMA_C1 = "kdic-fact-index-c-v3-reviewable"
FACT_INDEX_ALLOWED_REVIEW_STATUS_C1 = {
    "HUMAN_APPROVED_EVAL",
    "CANDIDATE_EVIDENCE_REVIEWED",
}


def locate_or_upload_fact_index_c1() -> Path:
    candidates = [
        Path("/content") / FACT_INDEX_UPLOAD_NAME_C1,
        Path.cwd() / FACT_INDEX_UPLOAD_NAME_C1,
    ]
    for candidate in candidates:
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate

    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError(
            f"{FACT_INDEX_UPLOAD_NAME_C1} 파일을 현재 작업 폴더에 두세요."
        ) from error

    print(f"Fact Index 파일을 업로드하세요: {FACT_INDEX_UPLOAD_NAME_C1}")
    uploaded = files.upload()
    if FACT_INDEX_UPLOAD_NAME_C1 not in uploaded:
        raise FileNotFoundError(
            f"업로드 파일명이 다릅니다. 필요한 파일: {FACT_INDEX_UPLOAD_NAME_C1}"
        )
    path = Path("/content") / FACT_INDEX_UPLOAD_NAME_C1
    if not path.is_file():
        path = Path(FACT_INDEX_UPLOAD_NAME_C1)
    return path


def load_fact_index_c1(path: Path) -> dict[str, Any]:
    raw = path.read_bytes()
    document = json.loads(raw.decode("utf-8-sig"))
    if not isinstance(document, dict):
        raise TypeError("Fact Index 최상위 값은 JSON 객체여야 합니다.")
    if document.get("schema_version") != FACT_INDEX_EXPECTED_SCHEMA_C1:
        raise ValueError(
            "Fact Index schema_version 불일치: "
            f"{document.get('schema_version')}"
        )
    records = document.get("records")
    if not isinstance(records, list) or not records:
        raise ValueError("Fact Index records가 비어 있습니다.")

    trigger_owner: dict[str, str] = {}
    errors: list[str] = []
    for record in records:
        fact_id = str(record.get("fact_index_id") or "")
        triggers = [str(value) for value in record.get("trigger_chunk_ids") or []]
        if not fact_id or not triggers:
            errors.append(f"식별자 또는 trigger 누락: {fact_id or '<empty>'}")
            continue
        if record.get("activation_policy") != "TRIGGER_CHUNK_AND_BUSINESS_AND_KEYWORD":
            errors.append(f"지원하지 않는 activation_policy: {fact_id}")
        if record.get("review_status") not in FACT_INDEX_ALLOWED_REVIEW_STATUS_C1:
            errors.append(f"평가 허용 review_status 아님: {fact_id}")
        for chunk_id in triggers:
            previous = trigger_owner.get(chunk_id)
            if previous and previous != fact_id:
                errors.append(f"trigger 중복: {chunk_id} -> {previous}, {fact_id}")
            trigger_owner[chunk_id] = fact_id
    if errors:
        raise ValueError("Fact Index 검증 실패: " + " | ".join(errors[:10]))

    return {
        **document,
        "_path": str(path),
        "_sha256": hashlib.sha256(raw).hexdigest(),
        "_record_count": len(records),
    }


FACT_INDEX_PATH_C1 = locate_or_upload_fact_index_c1()
FACT_INDEX_DOCUMENT_C1 = load_fact_index_c1(FACT_INDEX_PATH_C1)
FACT_INDEX_RECORDS_C1 = tuple(FACT_INDEX_DOCUMENT_C1["records"])

print({
    "fact_index_path": str(FACT_INDEX_PATH_C1),
    "schema_version": FACT_INDEX_DOCUMENT_C1["schema_version"],
    "status": FACT_INDEX_DOCUMENT_C1.get("status"),
    "prototype_use_allowed": FACT_INDEX_DOCUMENT_C1.get("prototype_use_allowed"),
    "record_count": len(FACT_INDEX_RECORDS_C1),
    "sha256": FACT_INDEX_DOCUMENT_C1["_sha256"],
})


## 18. Fact Index 안전 매칭과 보강 Evidence Pack

`Top-5 trigger_chunk_ids 일치 AND 탐지 업무 일치 AND activation keyword 일치`를 모두 만족할 때만 연결합니다. 의미 유사도 확장은 사용하지 않습니다.


In [ ]:
BUSINESS_CODE_BY_LABEL_C1 = {
    "예금자보호제도": "deposit_protection",
    "예금자보호": "deposit_protection",
    "예금보험금 안내": "deposit_insurance_payout",
    "예금보험금": "deposit_insurance_payout",
    "고객 미수령금 신청": "unclaimed_funds",
    "고객 미수령금": "unclaimed_funds",
    "착오송금 반환 신청": "mistaken_transfer",
    "착오송금 반환지원": "mistaken_transfer",
    "채무조정 안내": "debt_adjustment",
    "채무조정": "debt_adjustment",
    "은닉재산 신고": "hidden_assets_report",
}
FACT_PRIORITY_ORDER_C1 = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}


def _detected_business_codes_c1(common: Mapping[str, Any]) -> set[str]:
    analysis = common.get("analysis") or {}
    values = list(analysis.get("businesses") or [])
    values.extend(analysis.get("detected_businesses") or [])
    output: set[str] = set()
    for value in values:
        label = str(value).strip()
        code = BUSINESS_CODE_BY_LABEL_C1.get(label)
        if code:
            output.add(code)
    return output


def match_fact_index_c1(
    common: Mapping[str, Any],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if common.get("route") != "RETRIEVE":
        return [], []

    top5_ids = {
        str(row.get("chunk_id") or (row.get("chunk") or {}).get("chunk_id") or "")
        for row in list(common.get("search_results") or [])[:5]
    }
    top5_ids.discard("")
    detected_codes = _detected_business_codes_c1(common)
    question = _clean_text(
        " ".join([
            str(common.get("question") or ""),
            str(common.get("resolved_question") or ""),
        ])
    ).lower()

    matched: list[dict[str, Any]] = []
    audit: list[dict[str, Any]] = []
    for record in FACT_INDEX_RECORDS_C1:
        triggers = {str(value) for value in record.get("trigger_chunk_ids") or []}
        trigger_hits = sorted(top5_ids.intersection(triggers))
        record_code = str(record.get("business_function_code") or "")
        business_match = bool(record_code and record_code in detected_codes)
        keyword_hits = [
            str(keyword)
            for keyword in record.get("activation_keywords") or []
            if str(keyword).strip() and str(keyword).strip().lower() in question
        ]
        accepted = bool(trigger_hits and business_match and keyword_hits)
        row = {
            "fact_index_id": str(record.get("fact_index_id") or ""),
            "accepted": accepted,
            "trigger_hits": trigger_hits,
            "business_function": record.get("business_function"),
            "business_function_code": record_code,
            "business_match": business_match,
            "keyword_hits": list(dict.fromkeys(keyword_hits)),
            "confusion_type": record.get("confusion_type"),
            "priority": record.get("priority"),
            "review_status": record.get("review_status"),
            "reason": (
                "TRIGGER_BUSINESS_KEYWORD_MATCH"
                if accepted
                else "NO_TRIGGER" if not trigger_hits
                else "BUSINESS_MISMATCH" if not business_match
                else "NO_ACTIVATION_KEYWORD"
            ),
        }
        audit.append(row)
        if accepted:
            matched.append({**copy.deepcopy(record), "_match": row})

    matched.sort(key=lambda record: (
        FACT_PRIORITY_ORDER_C1.get(str(record.get("priority") or ""), 9),
        str(record.get("fact_index_id") or ""),
    ))
    return matched, audit


def _claim_source_ids_c1(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value if str(item)]
    return [item for item in str(value or "").split() if item]


def build_fact_augmented_pack_c1(
    base_pack: Mapping[str, Any],
    matched_records: Sequence[Mapping[str, Any]],
) -> dict[str, Any]:
    supplements: list[dict[str, Any]] = []
    for record in matched_records:
        verified_claims = []
        for claim in record.get("verified_claims") or []:
            verified_claims.append({
                "claim_id": str(claim.get("claim_id") or ""),
                "statement": _clean_text(claim.get("statement")),
                "source_chunk_ids": _claim_source_ids_c1(claim.get("source_chunk_ids")),
                "origin": str(claim.get("origin") or ""),
            })
        supplements.append({
            "fact_index_id": str(record.get("fact_index_id") or ""),
            "review_status": str(record.get("review_status") or ""),
            "business_function": str(record.get("business_function") or ""),
            "business_function_code": str(record.get("business_function_code") or ""),
            "priority": str(record.get("priority") or ""),
            "confusion_type": _clean_text(record.get("confusion_type")),
            "confusion_point": _clean_text(record.get("confusion_point")),
            "verified_claims": verified_claims,
            "forbidden_claims": [
                _clean_text(value) for value in record.get("forbidden_claims") or []
                if _clean_text(value)
            ],
            "related_chunk_ids": [
                str(value) for value in record.get("related_chunk_ids") or [] if str(value)
            ],
            "source_chunk_ids": [
                str(value) for value in record.get("source_chunk_ids") or [] if str(value)
            ],
            "source_urls": [
                str(value) for value in record.get("source_urls") or [] if str(value)
            ],
            "activation_audit": copy.deepcopy(record.get("_match") or {}),
        })

    return {
        **copy.deepcopy(dict(base_pack)),
        "fact_index": {
            "schema_version": FACT_INDEX_DOCUMENT_C1["schema_version"],
            "source_sha256": FACT_INDEX_DOCUMENT_C1["_sha256"],
            "matching_policy": FACT_INDEX_DOCUMENT_C1.get("matching_policy"),
            "review_required": FACT_INDEX_DOCUMENT_C1.get("status") == "REVIEW_REQUIRED",
            "supplement_count": len(supplements),
            "supplements": supplements,
            "precedence_rule": (
                "원본 Evidence를 보존한다. Fact Index는 검증 보조정보이며 원본을 덮어쓰지 않는다. "
                "충돌이 의심되면 단정하지 않고 검토 필요로 표시한다."
            ),
        },
    }


print({
    "matching_policy": "trigger_chunk_ids AND business_function_code AND activation_keywords",
    "semantic_similarity_matching": False,
    "fact_record_count": len(FACT_INDEX_RECORDS_C1),
})


## 19. B/C/D 구조화 Markdown 답변과 세부 레이턴시

B와 C는 결론·번호 목록·글머리표 구조를 프롬프트에서 요구하며, 보수적 로컬 후처리로 한 문단 번호 나열과 빈 괄호를 정리합니다. D는 Need·Skeleton·Evidence 선택·최종답변 시간을 분리합니다. 표시 개선을 위한 추가 LLM 호출은 없습니다.


In [ ]:
from __future__ import annotations

import json
import re
import time
from typing import Any, Mapping


MARKDOWN_FORMAT_RULES_V3 = """
[답변 표시 형식]
- 먼저 질문에 대한 결론을 한 문단으로 작성하세요.
- 절차는 각 단계가 한 줄에 하나씩 보이도록 Markdown 번호 목록을 사용하세요.
- 서류·조건·예외·대상은 각 항목이 한 줄에 하나씩 보이도록 Markdown 글머리표를 사용하세요.
- 목록 앞뒤에는 빈 줄을 넣으세요.
- `1. 내용 2. 내용 3. 내용`처럼 번호를 한 문단에 이어 쓰지 마세요.
- 짧은 단답형 질문에는 불필요한 제목이나 목록을 만들지 마세요.
- JSON answer 문자열 내부의 줄바꿈은 \\n으로 이스케이프하세요.
""".strip()

B_STRUCTURED_SYSTEM_PROMPT_V3 = (
    B_LOW_LATENCY_SYSTEM_PROMPT + "\n\n" + MARKDOWN_FORMAT_RULES_V3
)
C_BASE_SYSTEM_PROMPT_V3 = """
당신은 예금보험공사 공식 문서 기반 답변 시스템입니다.

1. 사용자 질문, 원본 Evidence, 활성화된 Fact Index 보강정보만 사용하세요.
2. 원본 Evidence를 기본 근거로 사용하고 Fact Index는 혼동 방지와 사실 검증에 사용하세요.
3. verified_claims는 해당 Fact Index의 source_chunk_ids와 source_urls 범위에서 검수된 주장입니다.
4. forbidden_claims에 해당하는 내용은 답변에서 주장하지 마세요.
5. Fact Index와 원본 Evidence가 충돌하면 어느 쪽도 임의로 선택하지 말고 확인이 필요하다고 답하세요.
6. Fact Index가 연결되지 않은 질문은 B안과 같은 원본 Evidence 범위에서 답하세요.
7. 서로 다른 대상·제도·금액·기간·조건을 임의로 결합하지 마세요.
8. 동시·병행 신청 관계는 동일 근거가 직접 명시한 경우에만 가능하다고 답하세요.
9. 근거 문장에는 원본 Evidence를 [E1], Fact claim을 [FI-CAND-001:F1] 형식으로 표시하세요.
10. 지정된 JSON 객체 하나만 출력하세요.
""".strip()
C_STRUCTURED_SYSTEM_PROMPT_V3 = (
    C_BASE_SYSTEM_PROMPT_V3 + "\n\n" + MARKDOWN_FORMAT_RULES_V3
)
D_STRUCTURED_FINAL_SYSTEM_PROMPT_V3 = (
    D2_FINAL_SYSTEM_PROMPT + "\n\n" + MARKDOWN_FORMAT_RULES_V3
)


def normalize_answer_markdown_v3(text: Any) -> str:
    value = user_visible_answer(text).replace("\r\n", "\n").replace("\r", "\n")
    value = re.sub(r"\s*\[FI-CAND-\d+:F\d+\]", "", value)
    value = re.sub(r"\(\s*\)", "", value)
    numbered_markers = re.findall(r"(?<!\d)(?:[1-9]|1[0-9])\.\s+\S", value)
    if len(numbered_markers) >= 2:
        value = re.sub(
            r"\s+(?=(?:[1-9]|1[0-9])\.\s+\S)",
            "\n",
            value,
        )
        value = re.sub(r"([^\n])\n(?=1\.\s+)", r"\1\n\n", value)
    value = re.sub(r"[ \t]+\n", "\n", value)
    value = re.sub(r"\n{3,}", "\n\n", value)
    value = re.sub(r"\s+([,.!?])", r"\1", value)
    return value.strip()


def _trace_parts_v3(trace: Mapping[str, Any]) -> dict[str, float]:
    wall = float(trace.get("wall_latency_ms") or 0)
    pacing = float(trace.get("pacing_wait_ms") or 0)
    retry = float(trace.get("retry_wait_ms") or 0)
    return {
        "api_wall_ms": wall,
        "pacing_wait_ms": pacing,
        "retry_wait_ms": retry,
        "estimated_service_ms": max(0.0, wall - pacing - retry),
    }


def _allowed_fact_claims_v3(pack: Mapping[str, Any]) -> dict[str, dict[str, Any]]:
    output: dict[str, dict[str, Any]] = {}
    for supplement in (pack.get("fact_index") or {}).get("supplements") or []:
        fact_id = str(supplement.get("fact_index_id") or "")
        for claim in supplement.get("verified_claims") or []:
            claim_id = str(claim.get("claim_id") or "")
            if fact_id and claim_id:
                output[f"{fact_id}:{claim_id}"] = dict(claim)
    return output


def _clean_fact_claim_keys_v3(value: Any) -> list[str]:
    if not isinstance(value, list):
        return []
    return list(dict.fromkeys(
        str(item).strip().strip("[]") for item in value if str(item).strip()
    ))


def _direct_answer_payload_v3(
    raw: str,
    pack: Mapping[str, Any],
) -> tuple[dict[str, Any], bool]:
    allowed = answer_b_core._allowed_evidence(pack)
    local_recovery = False
    try:
        parsed = answer_b_core._extract_json_object(raw)
        answer = answer_b_core._strip_model_urls(answer_b_core._clean(parsed.get("answer")))
        if not answer:
            raise ValueError("answer가 비어 있습니다.")
        used_ids = [
            value for value in answer_b_core._clean_list(parsed.get("used_evidence_ids"))
            if value in allowed
        ]
        if not used_ids:
            used_ids = [
                f"E{number}" for number in re.findall(r"\[E(\d+)\]", answer)
                if f"E{number}" in allowed
            ]
        coverage = str(parsed.get("coverage_status") or "PARTIAL").upper()
        if coverage not in answer_b_core.ALLOWED_COVERAGE_STATUS:
            coverage = "PARTIAL"
        missing = answer_b_core._clean_list(parsed.get("missing_information"))
    except (ValueError, TypeError, json.JSONDecodeError):
        local_recovery = True
        recovered = answer_b_core._recover_basic_answer_from_raw(raw, pack)
        answer = recovered["answer"]
        used_ids = list(recovered.get("used_evidence_ids") or [])
        coverage = "PARTIAL"
        missing = ["구조화 메타데이터를 로컬 복구함"]
    if not used_ids and allowed:
        used_ids = [next(iter(allowed))]
        coverage = "PARTIAL"
        missing = list(dict.fromkeys(missing + ["근거 ID를 로컬 보완함"]))
    return {
        "answer": answer,
        "used_evidence_ids": list(dict.fromkeys(used_ids)),
        "used_chunk_ids": [allowed[value] for value in dict.fromkeys(used_ids)],
        "coverage_status": coverage,
        "missing_information": missing,
    }, local_recovery


def generate_answer_b_v3(question: str, pack: Mapping[str, Any]) -> dict[str, Any]:
    total_started = time.perf_counter()
    prompt_started = time.perf_counter()
    constraint = relation_constraint_v1(question, pack)
    prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[관계 주장 안전조건]\n{_compact_json(constraint)}\n\n[Basic Evidence Pack]\n{_compact_json(pack)}\n\n[출력 JSON]\n{{\"answer\":\"Markdown 형식 답변과 [E1] 근거표기\",\"used_evidence_ids\":[\"E1\"],\"coverage_status\":\"SUFFICIENT|PARTIAL|INSUFFICIENT\",\"missing_information\":[]}}"""
    prompt_ms = (time.perf_counter() - prompt_started) * 1000

    raw, usage, api_ms, trace = _call_answer_api_v1(
        system_prompt=B_STRUCTURED_SYSTEM_PROMPT_V3,
        user_prompt=prompt,
        max_tokens=1600,
    )
    parse_started = time.perf_counter()
    payload, local_recovery = _direct_answer_payload_v3(raw, pack)
    parse_ms = (time.perf_counter() - parse_started) * 1000

    post_started = time.perf_counter()
    safe_answer, guard_applied = _relation_safe_answer_v1(payload["answer"], constraint)
    payload["answer"] = normalize_answer_markdown_v3(safe_answer)
    if guard_applied:
        payload["coverage_status"] = "PARTIAL"
    numeric = audit_numeric_support_v2(payload["answer"], pack)
    post_ms = (time.perf_counter() - post_started) * 1000
    trace_parts = _trace_parts_v3(trace)
    total_ms = (time.perf_counter() - total_started) * 1000
    return {
        **payload,
        "system": "B",
        "latency_ms": total_ms,
        "usage": usage,
        "api_calls": 1,
        "attempts": [{"stage": "answer_b", "latency_ms": api_ms, "trace": trace}],
        "local_recovery": local_recovery,
        "relation_constraint": constraint,
        "relation_guard_applied": guard_applied,
        "numeric_audit": numeric,
        "stage_latency_ms": {
            "prompt_build_ms": prompt_ms,
            **trace_parts,
            "parse_validation_ms": parse_ms,
            "postprocess_ms": post_ms,
            "answer_total_ms": total_ms,
        },
    }


def generate_answer_c_v3(
    question: str,
    augmented_pack: Mapping[str, Any],
) -> dict[str, Any]:
    total_started = time.perf_counter()
    prompt_started = time.perf_counter()
    constraint = relation_constraint_v1(question, augmented_pack)
    prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[관계 주장 안전조건]\n{_compact_json(constraint)}\n\n[Fact Index 보강 Evidence Pack]\n{_compact_json(augmented_pack)}\n\n[출력 JSON]\n{{\"answer\":\"Markdown 형식 답변과 [E1] 또는 [FI-CAND-001:F1] 근거표기\",\"used_evidence_ids\":[\"E1\"],\"used_fact_claim_ids\":[\"FI-CAND-001:F1\"],\"coverage_status\":\"SUFFICIENT|PARTIAL|INSUFFICIENT\",\"missing_information\":[]}}"""
    prompt_ms = (time.perf_counter() - prompt_started) * 1000

    raw, usage, api_ms, trace = _call_answer_api_v1(
        system_prompt=C_STRUCTURED_SYSTEM_PROMPT_V3,
        user_prompt=prompt,
        max_tokens=1600,
    )
    parse_started = time.perf_counter()
    allowed_evidence = answer_b_core._allowed_evidence(augmented_pack)
    allowed_facts = _allowed_fact_claims_v3(augmented_pack)
    local_recovery = False
    try:
        parsed = answer_b_core._extract_json_object(raw)
        answer = answer_b_core._strip_model_urls(answer_b_core._clean(parsed.get("answer")))
        if not answer:
            raise ValueError("C안 answer가 비어 있습니다.")
        used_evidence = [
            value for value in answer_b_core._clean_list(parsed.get("used_evidence_ids"))
            if value in allowed_evidence
        ]
        used_facts = [
            value for value in _clean_fact_claim_keys_v3(parsed.get("used_fact_claim_ids"))
            if value in allowed_facts
        ]
        coverage = str(parsed.get("coverage_status") or "PARTIAL").upper()
        if coverage not in answer_b_core.ALLOWED_COVERAGE_STATUS:
            coverage = "PARTIAL"
        missing = answer_b_core._clean_list(parsed.get("missing_information"))
    except (ValueError, TypeError, json.JSONDecodeError):
        local_recovery = True
        answer = answer_b_core._strip_model_urls(str(raw).strip())
        if not answer:
            raise ValueError("C안 출력이 비어 있습니다.")
        used_evidence = [
            f"E{number}" for number in re.findall(r"\[E(\d+)\]", answer)
            if f"E{number}" in allowed_evidence
        ]
        used_facts = [
            value for value in re.findall(r"\[(FI-CAND-\d+:F\d+)\]", answer)
            if value in allowed_facts
        ]
        coverage = "PARTIAL"
        missing = ["구조화 메타데이터를 로컬 복구함"]
    parse_ms = (time.perf_counter() - parse_started) * 1000

    post_started = time.perf_counter()
    safe_answer, guard_applied = _relation_safe_answer_v1(answer, constraint)
    answer = normalize_answer_markdown_v3(safe_answer)
    if guard_applied:
        coverage = "PARTIAL"
    post_ms = (time.perf_counter() - post_started) * 1000
    trace_parts = _trace_parts_v3(trace)
    total_ms = (time.perf_counter() - total_started) * 1000
    return {
        "system": "C",
        "answer": answer,
        "used_evidence_ids": list(dict.fromkeys(used_evidence)),
        "used_chunk_ids": [
            allowed_evidence[value] for value in dict.fromkeys(used_evidence)
        ],
        "used_fact_claim_ids": list(dict.fromkeys(used_facts)),
        "used_fact_index_ids": list(dict.fromkeys(
            value.split(":", 1)[0] for value in used_facts
        )),
        "coverage_status": coverage,
        "missing_information": missing,
        "latency_ms": total_ms,
        "usage": usage,
        "api_calls": 1,
        "attempts": [{"stage": "answer_c", "latency_ms": api_ms, "trace": trace}],
        "local_recovery": local_recovery,
        "relation_constraint": constraint,
        "relation_guard_applied": guard_applied,
        "stage_latency_ms": {
            "prompt_build_ms": prompt_ms,
            **trace_parts,
            "parse_validation_ms": parse_ms,
            "postprocess_ms": post_ms,
            "answer_total_ms": total_ms,
        },
    }


def generate_answer_d_v3(question: str, pack: Mapping[str, Any]) -> dict[str, Any]:
    total_started = time.perf_counter()
    need_started = time.perf_counter()
    answer_needs = extract_answer_needs_v2(question)
    relation_constraint = relation_constraint_v1(question, pack)
    need_ms = (time.perf_counter() - need_started) * 1000

    skeleton_prompt_started = time.perf_counter()
    skeleton_prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[Answer Needs]\n{_compact_json(answer_needs)}\n\n[관계 주장 안전조건]\n{_compact_json(relation_constraint)}\n\n[Evidence Pack]\n{_compact_json(pack)}\n\n[출력 JSON]\n{{\"core_answer\":\"핵심 결론\",\"answer_items\":[{{\"need_id\":\"N1\",\"topic\":\"항목\",\"status\":\"ANSWERED|PARTIAL|UNSUPPORTED\",\"claim\":\"근거 사실\",\"conditions\":[],\"details\":[],\"evidence_ids\":[\"E1\"],\"missing_reason\":\"\"}}],\"uncertainties\":[],\"conflicts\":[]}}"""
    skeleton_prompt_ms = (time.perf_counter() - skeleton_prompt_started) * 1000
    raw_skeleton, usage1, skeleton_api_ms, trace1 = _call_answer_api_v1(
        system_prompt=D2_SKELETON_SYSTEM_PROMPT,
        user_prompt=skeleton_prompt,
        max_tokens=2000,
    )

    validation_started = time.perf_counter()
    skeleton = validate_d2_skeleton_v2(
        answer_b_core._extract_json_object(raw_skeleton), answer_needs, pack
    )
    validation_ms = (time.perf_counter() - validation_started) * 1000

    selection_started = time.perf_counter()
    selected_pack = filter_pack_for_d2_v2(pack, skeleton)
    selection_ms = (time.perf_counter() - selection_started) * 1000

    final_prompt_started = time.perf_counter()
    final_prompt = f"""[사용자 질문]\n{_clean_text(question)}\n\n[Answer Needs]\n{_compact_json(answer_needs)}\n\n[관계 주장 안전조건]\n{_compact_json(relation_constraint)}\n\n[Answer Skeleton]\n{_compact_json(skeleton)}\n\n[Skeleton 참조 Evidence]\n{_compact_json(selected_pack)}\n\n위 범위에서 Markdown 구조를 지켜 최종 사용자 답변을 작성하세요."""
    final_prompt_ms = (time.perf_counter() - final_prompt_started) * 1000
    raw_answer, usage2, final_api_ms, trace2 = _call_answer_api_v1(
        system_prompt=D_STRUCTURED_FINAL_SYSTEM_PROMPT_V3,
        user_prompt=final_prompt,
        max_tokens=1600,
    )

    post_started = time.perf_counter()
    answer = answer_b_core._strip_model_urls(str(raw_answer).strip())
    if not answer:
        raise ValueError("D안 최종 답변이 비어 있습니다.")
    safe_answer, guard_applied = _relation_safe_answer_v1(answer, relation_constraint)
    safe_answer = normalize_answer_markdown_v3(safe_answer)
    numeric = audit_numeric_support_v2(safe_answer, selected_pack)
    post_ms = (time.perf_counter() - post_started) * 1000
    skeleton_trace = _trace_parts_v3(trace1)
    final_trace = _trace_parts_v3(trace2)
    total_ms = (time.perf_counter() - total_started) * 1000
    return {
        "system": "D",
        "answer": safe_answer,
        "answer_needs": answer_needs,
        "skeleton": skeleton,
        "need_coverage": skeleton["need_coverage"],
        "coverage_status": "PARTIAL" if guard_applied else skeleton["coverage_status"],
        "strict_need_coverage_rate": skeleton["strict_need_coverage_rate"],
        "answerable_need_coverage_rate": skeleton["answerable_need_coverage_rate"],
        "numeric_audit": numeric,
        "selected_evidence_pack": selected_pack,
        "selected_evidence_count": len(selected_pack["evidence"]),
        "latency_ms": total_ms,
        "skeleton_latency_ms": skeleton_api_ms,
        "final_latency_ms": final_api_ms,
        "usage": _merge_usage_v1(usage1, usage2),
        "api_calls": 2,
        "attempts": [
            {"stage": "skeleton", "latency_ms": skeleton_api_ms, "trace": trace1},
            {"stage": "final", "latency_ms": final_api_ms, "trace": trace2},
        ],
        "relation_constraint": relation_constraint,
        "relation_guard_applied": guard_applied,
        "stage_latency_ms": {
            "need_extraction_ms": need_ms,
            "skeleton_prompt_build_ms": skeleton_prompt_ms,
            "skeleton_api_wall_ms": skeleton_trace["api_wall_ms"],
            "skeleton_pacing_wait_ms": skeleton_trace["pacing_wait_ms"],
            "skeleton_retry_wait_ms": skeleton_trace["retry_wait_ms"],
            "skeleton_estimated_service_ms": skeleton_trace["estimated_service_ms"],
            "skeleton_validation_ms": validation_ms,
            "evidence_selection_ms": selection_ms,
            "final_prompt_build_ms": final_prompt_ms,
            "final_api_wall_ms": final_trace["api_wall_ms"],
            "final_pacing_wait_ms": final_trace["pacing_wait_ms"],
            "final_retry_wait_ms": final_trace["retry_wait_ms"],
            "final_estimated_service_ms": final_trace["estimated_service_ms"],
            "postprocess_ms": post_ms,
            "answer_total_ms": total_ms,
        },
    }


print({
    "answer_b": "1-call + Markdown + detailed latency",
    "answer_c": "1-call + Fact Index + Markdown + detailed latency",
    "answer_d": "Skeleton + final + detailed latency",
    "additional_llm_calls_for_formatting": 0,
})


## 20. 검증된 Action Link Registry

이 셀은 답변 모델과 독립적으로 공식 신청·조회·서류 페이지를 선택합니다. `RETRIEVE 경로 + 업무 + 행동 의도 + 대상 역할`을 모두 확인하며, 허용 도메인·HTTPS·승인 상태 하드 게이트와 회귀 테스트를 통과해야 링크를 표시합니다. 런타임 네트워크 호출과 추가 LLM 호출은 없습니다.


In [ ]:
from __future__ import annotations

import copy
import html
import json
import re
import time
from typing import Any, Mapping, Sequence
from urllib.parse import urlsplit


ACTION_LINK_REGISTRY_DOCUMENT_V1 = json.loads('{"schema_version":"kdic-action-link-registry-v1.0","registry_version":"2026-08-20","policy":{"allowed_schemes":["https"],"allowed_hosts":["www.kdic.or.kr","fins.kdic.or.kr","mkcs.kdic.or.kr"],"max_buttons":3,"selection_mode":"ROUTE_AND_BUSINESS_AND_ACTION_AND_ROLE","network_check_at_runtime":false,"llm_may_generate_or_select_urls":false,"source_links_and_action_links_are_separate":true},"records":[{"link_id":"DP-INSTITUTION-SEARCH-001","business_function_code":"deposit_protection","action_type":"INSTITUTION_SEARCH","actor_roles":["ANY"],"channel":"WEB","button_label":"보호대상 금융회사 검색","description":"금융회사명이 예금자보호 대상인지 공식 검색 화면에서 확인합니다.","url":"https://www.kdic.or.kr/sp/dpstrprot/selectProtSystProtSrch.do","activation_keywords":["금융회사","은행","저축은행","보호대상","보호되나요","가입기관"],"exclusion_keywords":[],"source_chunk_ids":[],"priority":10,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"DP-PRODUCT-SEARCH-001","business_function_code":"deposit_protection","action_type":"PRODUCT_SEARCH","actor_roles":["ANY"],"channel":"WEB","button_label":"보호대상 금융상품 검색","description":"예금·적금·금융상품이 보호대상인지 공식 검색 화면에서 확인합니다.","url":"https://www.kdic.or.kr/sp/dpstrprot/selectProtSystProtTrgtPrdctSrchList.do","activation_keywords":["금융상품","예금","적금","상품","보호대상","보호되나요"],"exclusion_keywords":[],"source_chunk_ids":[],"priority":10,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"DI-APPLICATION-PROCEDURE-001","business_function_code":"deposit_insurance_payout","action_type":"APPLY_GUIDE","actor_roles":["SELF","PROXY","HEIR","ANY"],"channel":"WEB","button_label":"예금보험금 신청 절차","description":"방문·인터넷 신청 절차와 지급 흐름을 확인합니다.","url":"https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyProc/selectScrn.do","activation_keywords":["신청","지급","받으려면","절차","방법","수령"],"exclusion_keywords":["신청할 수 없는"],"source_chunk_ids":["BI-002_chunk_002"],"priority":20,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"DI-DOCUMENT-GUIDE-001","business_function_code":"deposit_insurance_payout","action_type":"DOCUMENT_GUIDE","actor_roles":["SELF","PROXY","HEIR","ANY"],"channel":"WEB","button_label":"예금보험금 구비서류·양식","description":"본인·대리인·상속인 등 신청 상황별 서류와 공식 양식을 확인합니다.","url":"https://www.kdic.or.kr/sp/dpstrprot/DpsmIbamtAplyPossDcmnt/selectScrn.do","activation_keywords":["서류","구비","준비물","위임장","양식","다운로드"],"exclusion_keywords":[],"source_chunk_ids":["BI-001_chunk_000","BI-001_chunk_001","BI-001_chunk_004","BI-001_chunk_006"],"priority":5,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"DI-PAYMENT-AGENT-SEARCH-001","business_function_code":"deposit_insurance_payout","action_type":"OFFLINE_LOCATION_SEARCH","actor_roles":["ANY"],"channel":"WEB","button_label":"예금보험금 지급대행점 조회","description":"방문 신청이 가능한 지급대행점을 조회합니다.","url":"https://www.kdic.or.kr/sp/dpstrprot/selectProtSystBamtGiveInq.do","activation_keywords":["방문","지급대행점","어디","지점","오프라인"],"exclusion_keywords":[],"source_chunk_ids":["BI-002_chunk_002"],"priority":8,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"UN-INTEGRATED-APPLICATION-001","business_function_code":"unclaimed_funds","action_type":"APPLY","actor_roles":["SELF","PROXY","HEIR","ANY"],"channel":"WEB_AUTH","button_label":"미수령금 통합신청","description":"본인인증 후 미수령금 통합 확인·신청을 진행합니다.","url":"https://fins.kdic.or.kr/ua/itgraply/selectItgrInqDsctn.do","activation_keywords":["신청","찾기","받기","수령","통합신청"],"exclusion_keywords":["신청할 수 없는","제외"],"source_chunk_ids":["UN-001_chunk_000"],"priority":5,"approved_for_display":true,"verification_status":"OFFICIAL_AUTH_REDIRECT_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"UN-APPLICATION-STATUS-001","business_function_code":"unclaimed_funds","action_type":"STATUS_CHECK","actor_roles":["SELF","PROXY","HEIR","ANY"],"channel":"WEB_AUTH","button_label":"미수령금 진행·지급내역 조회","description":"본인인증 후 신청 진행상태와 지급내역을 확인합니다.","url":"https://fins.kdic.or.kr/ua/dsctninq/selectItgrInq.do","activation_keywords":["조회","확인","진행","상태","지급내역","신청내역"],"exclusion_keywords":[],"source_chunk_ids":["UN-001_chunk_000"],"priority":4,"approved_for_display":true,"verification_status":"OFFICIAL_AUTH_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"UN-APPLICATION-GUIDE-001","business_function_code":"unclaimed_funds","action_type":"APPLY_GUIDE","actor_roles":["SELF","PROXY","HEIR","ANY"],"channel":"WEB","button_label":"미수령금 통합신청 안내","description":"미수령금 종류와 온라인·오프라인 신청방법을 먼저 확인합니다.","url":"https://fins.kdic.or.kr/ua/aplygudn/NramtItgrAplyItrdMthdGudn/selectScrn.do","activation_keywords":["미수령금","신청","방법","절차","어떻게"],"exclusion_keywords":[],"source_chunk_ids":["UN-003_chunk_000"],"priority":15,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"UN-HEIR-INQUIRY-GUIDE-001","business_function_code":"unclaimed_funds","action_type":"HEIR_INQUIRY","actor_roles":["HEIR"],"channel":"WEB","button_label":"상속인 금융거래조회 안내","description":"상속인의 금융거래·미수령금 조회 절차를 확인합니다.","url":"https://www.kdic.or.kr/sp/dpstrprot/ProtSystHrpeHistInq/selectScrn.do","activation_keywords":["상속","상속인","사망","피상속인"],"exclusion_keywords":[],"source_chunk_ids":["UN-004_chunk_000"],"priority":3,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-SITUATION-SELECT-001","business_function_code":"mistaken_transfer","action_type":"SITUATION_SELECT","actor_roles":["ANY"],"channel":"WEB","button_label":"착오송금 상황 선택","description":"송금인인지 수취인인지에 따라 이용할 절차를 선택합니다.","url":"https://fins.kdic.or.kr/ir/aplygudn/MtrsStutChc/selectScrn.do","activation_keywords":["착오송금","잘못 보냈","모르는 돈","송금인","수취인","받았"],"exclusion_keywords":[],"source_chunk_ids":["MT-002_chunk_000"],"priority":30,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-SENDER-ELIGIBILITY-001","business_function_code":"mistaken_transfer","action_type":"ELIGIBILITY_CHECK","actor_roles":["SENDER"],"channel":"WEB","button_label":"착오송금 신청자격 확인","description":"반환지원 신청 전에 공식 자가진단 항목으로 대상 여부를 확인합니다.","url":"https://fins.kdic.or.kr/ir/msdrpr/selectAplyQlfcIdntyRslt.do","activation_keywords":["자격","대상","신청할 수","가능","조건","제외","누가"],"exclusion_keywords":[],"source_chunk_ids":["MT-004_chunk_000","MT-013_chunk_000"],"priority":3,"approved_for_display":true,"verification_status":"OFFICIAL_INTERACTIVE_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-SENDER-APPLICATION-001","business_function_code":"mistaken_transfer","action_type":"APPLY","actor_roles":["SENDER"],"channel":"WEB_AUTH","button_label":"착오송금 반환지원 신청","description":"신청자격 확인 후 본인인증을 거쳐 반환지원을 신청합니다.","url":"https://fins.kdic.or.kr/ir/msdrpr/selectAplyQlfcIdntyChc.do","activation_keywords":["신청","접수","반환지원","어떻게","방법"],"exclusion_keywords":["신청할 수 없는","제외","수취인"],"source_chunk_ids":["MT-002_chunk_000","MT-013_chunk_006"],"priority":5,"approved_for_display":true,"verification_status":"OFFICIAL_AUTH_REDIRECT_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-SENDER-STATUS-001","business_function_code":"mistaken_transfer","action_type":"STATUS_CHECK","actor_roles":["SENDER"],"channel":"WEB_AUTH","button_label":"착오송금 신청내역 확인","description":"본인인증 후 반환지원 신청 진행·지급내역을 확인합니다.","url":"https://fins.kdic.or.kr/ir/msdrpr/selectAplyDsctnInqList.do","activation_keywords":["신청내역","진행","상태","조회","지급내역","확인"],"exclusion_keywords":[],"source_chunk_ids":["MT-013_chunk_006"],"priority":3,"approved_for_display":true,"verification_status":"OFFICIAL_AUTH_REDIRECT_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-SENDER-DOCUMENTS-001","business_function_code":"mistaken_transfer","action_type":"DOCUMENT_GUIDE","actor_roles":["SENDER","PROXY"],"channel":"WEB","button_label":"착오송금인 구비서류","description":"착오송금인 본인·대리 신청에 필요한 공식 서류와 양식을 확인합니다.","url":"https://fins.kdic.or.kr/ir/aplygudn/MsdrprPossDcmntGudn/selectScrn.do","activation_keywords":["서류","구비","준비물","위임장","양식","대리인"],"exclusion_keywords":["수취인"],"source_chunk_ids":["MT-010_operation_layer01_chunk_002"],"priority":3,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-RECIPIENT-DOCUMENTS-001","business_function_code":"mistaken_transfer","action_type":"DOCUMENT_GUIDE","actor_roles":["RECIPIENT","PROXY"],"channel":"WEB","button_label":"착오송금 수취인 구비서류","description":"착오송금 수취인 관련 반환·이의 절차의 공식 서류를 확인합니다.","url":"https://fins.kdic.or.kr/ir/aplygudn/MsdrAddrsePossDcmntGudn/selectScrn.do","activation_keywords":["서류","구비","준비물","위임장","양식","수취인","받은 사람"],"exclusion_keywords":["송금인"],"source_chunk_ids":["MT-010_operation_layer01_chunk_002"],"priority":3,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-RECIPIENT-BALANCE-001","business_function_code":"mistaken_transfer","action_type":"BALANCE_CHECK","actor_roles":["RECIPIENT"],"channel":"WEB_AUTH","button_label":"수취인 채무잔액 확인","description":"본인인증 후 착오송금 수취인의 채무잔액 관련 내역을 확인합니다.","url":"https://fins.kdic.or.kr/ir/addrse/selectLbltBlncIdntyList.do","activation_keywords":["채무잔액","잔액","수취인","받은 사람"],"exclusion_keywords":["송금인"],"source_chunk_ids":[],"priority":2,"approved_for_display":true,"verification_status":"OFFICIAL_AUTH_REDIRECT_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"MT-RECIPIENT-RETURN-CHECK-001","business_function_code":"mistaken_transfer","action_type":"RETURN_CHECK","actor_roles":["RECIPIENT"],"channel":"WEB_AUTH","button_label":"수취인 반환 확인","description":"본인인증 후 착오송금 수취인의 반환 처리 결과를 확인합니다.","url":"https://fins.kdic.or.kr/ir/addrse/selectGvbkIdntyList.do","activation_keywords":["반환 확인","반환했","처리 결과","수취인","받은 사람"],"exclusion_keywords":["송금인"],"source_chunk_ids":[],"priority":2,"approved_for_display":true,"verification_status":"OFFICIAL_AUTH_REDIRECT_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"DA-DEBT-INQUIRY-001","business_function_code":"debt_adjustment","action_type":"DEBT_INQUIRY","actor_roles":["DEBTOR","SELF","ANY"],"channel":"WEB_AUTH","button_label":"채무정보 조회·상담 신청","description":"본인인증 후 채무정보를 조회하고 대상이면 채무조정 상담을 신청합니다.","url":"https://fins.kdic.or.kr/lb/lbltinfo/selectLbltInfoInq.do","activation_keywords":["채무정보","조회","상담","신청","채무조정","확인"],"exclusion_keywords":[],"source_chunk_ids":["DA-002_chunk_000","DA-002_chunk_002"],"priority":5,"approved_for_display":true,"verification_status":"OFFICIAL_ACTION_LINK_FROM_CORPUS","last_verified_date":"2026-08-20"},{"link_id":"DA-ELIGIBILITY-DOCUMENTS-001","business_function_code":"debt_adjustment","action_type":"DOCUMENT_GUIDE","actor_roles":["DEBTOR","SELF","ANY"],"channel":"WEB","button_label":"채무조정 자격·구비서류","description":"채무조정 신청자격과 필요한 서류를 공식 안내에서 확인합니다.","url":"https://www.kdic.or.kr/rb/lbltajmt/LbltAjmtSprtLbltAjmtSyst/selectScrn.do","activation_keywords":["자격","대상","서류","구비","준비물","조건"],"exclusion_keywords":[],"source_chunk_ids":["DA-001_chunk_002","DA-001_chunk_003"],"priority":3,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"HP-REPORT-GUIDE-001","business_function_code":"hidden_assets_report","action_type":"REPORT_GUIDE","actor_roles":["REPORTER","ANY"],"channel":"WEB","button_label":"은닉재산 신고 안내","description":"신고 대상, 포상금, 신고방법과 보호조치를 공식 안내에서 확인합니다.","url":"https://www.kdic.or.kr/sp/sprtfund/SprtFndCncmDclrGudn/selectScrn.do","activation_keywords":["신고","제보","포상금","은닉재산","방법"],"exclusion_keywords":[],"source_chunk_ids":["HP-001_chunk_000","HP-001_chunk_002","HP-001_chunk_005"],"priority":10,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"},{"link_id":"HP-REPORT-AND-INQUIRY-001","business_function_code":"hidden_assets_report","action_type":"REPORT","actor_roles":["REPORTER","ANY"],"channel":"WEB","button_label":"은닉재산 신고·조회","description":"은닉재산을 신고하거나 기존 신고 관련 조회 화면으로 이동합니다.","url":"https://www.kdic.or.kr/sp/sprtfund/SprtCncmDclrInqGudn/selectScrn.do","activation_keywords":["신고","제보","접수","조회","진행","신고내역"],"exclusion_keywords":[],"source_chunk_ids":["HP-001_chunk_005"],"priority":5,"approved_for_display":true,"verification_status":"OFFICIAL_PAGE_VERIFIED","last_verified_date":"2026-08-20"}]}')
ACTION_LINK_RECORDS_V1 = copy.deepcopy(
    ACTION_LINK_REGISTRY_DOCUMENT_V1.get("records") or []
)
ACTION_LINK_POLICY_V1 = copy.deepcopy(
    ACTION_LINK_REGISTRY_DOCUMENT_V1.get("policy") or {}
)
ACTION_LINK_ALLOWED_HOSTS_V1 = {
    str(value).lower() for value in ACTION_LINK_POLICY_V1.get("allowed_hosts") or []
}
ACTION_LINK_ALLOWED_SCHEMES_V1 = {
    str(value).lower() for value in ACTION_LINK_POLICY_V1.get("allowed_schemes") or []
}
ACTION_LINK_MAX_BUTTONS_V1 = int(ACTION_LINK_POLICY_V1.get("max_buttons") or 3)


ACTION_LINK_BUSINESS_BY_LABEL_V1 = {
    "예금자보호제도": "deposit_protection",
    "예금자보호": "deposit_protection",
    "예금보험금 안내": "deposit_insurance_payout",
    "예금보험금": "deposit_insurance_payout",
    "고객 미수령금 신청": "unclaimed_funds",
    "고객 미수령금": "unclaimed_funds",
    "미수령금": "unclaimed_funds",
    "착오송금 반환 신청": "mistaken_transfer",
    "착오송금 반환지원": "mistaken_transfer",
    "착오송금": "mistaken_transfer",
    "채무조정 안내": "debt_adjustment",
    "채무조정": "debt_adjustment",
    "은닉재산 신고": "hidden_assets_report",
}


def validate_action_url_v1(url: Any) -> tuple[bool, str]:
    value = str(url or "").strip()
    if not value:
        return False, "EMPTY_URL"
    try:
        parsed = urlsplit(value)
    except Exception:
        return False, "URL_PARSE_ERROR"
    if parsed.scheme.lower() not in ACTION_LINK_ALLOWED_SCHEMES_V1:
        return False, "SCHEME_NOT_ALLOWED"
    if (parsed.hostname or "").lower() not in ACTION_LINK_ALLOWED_HOSTS_V1:
        return False, "HOST_NOT_ALLOWED"
    if parsed.username or parsed.password:
        return False, "URL_CREDENTIALS_NOT_ALLOWED"
    if not parsed.path or parsed.path == "/":
        return False, "EMPTY_ACTION_PATH"
    return True, "VALID"


def validate_action_link_registry_v1() -> list[dict[str, Any]]:
    required = {
        "link_id", "business_function_code", "action_type", "actor_roles",
        "button_label", "description", "url", "activation_keywords",
        "priority", "approved_for_display", "verification_status",
        "last_verified_date",
    }
    seen: set[str] = set()
    rows: list[dict[str, Any]] = []
    for record in ACTION_LINK_RECORDS_V1:
        link_id = str(record.get("link_id") or "")
        missing = sorted(required.difference(record))
        duplicate = bool(link_id and link_id in seen)
        seen.add(link_id)
        url_valid, url_reason = validate_action_url_v1(record.get("url"))
        roles_valid = bool(record.get("actor_roles"))
        approved = record.get("approved_for_display") is True
        passed = bool(
            link_id and not missing and not duplicate and url_valid and roles_valid and approved
        )
        rows.append({
            "link_id": link_id,
            "passed": passed,
            "missing_fields": ", ".join(missing),
            "duplicate": duplicate,
            "url_valid": url_valid,
            "url_reason": url_reason,
            "actor_roles_valid": roles_valid,
            "approved_for_display": approved,
        })
    return rows


def _action_clean_v1(value: Any) -> str:
    cleaner = globals().get("_clean_text") or globals().get("_clean")
    if callable(cleaner):
        try:
            return str(cleaner(value))
        except Exception:
            pass
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _action_business_codes_v1(common: Mapping[str, Any]) -> set[str]:
    existing = globals().get("_detected_business_codes_c1")
    if callable(existing):
        try:
            values = set(existing(common))
            if values:
                return values
        except Exception:
            pass
    analysis = common.get("analysis") or {}
    labels = list(analysis.get("businesses") or [])
    labels.extend(analysis.get("detected_businesses") or [])
    labels.extend(analysis.get("active_businesses") or [])
    return {
        ACTION_LINK_BUSINESS_BY_LABEL_V1[str(label).strip()]
        for label in labels
        if str(label).strip() in ACTION_LINK_BUSINESS_BY_LABEL_V1
    }


def detect_action_actor_roles_v1(
    common: Mapping[str, Any],
    question_text: str,
    business_codes: set[str],
) -> set[str]:
    roles: set[str] = set()
    analysis = common.get("analysis") or {}
    raw_context = analysis.get("context")
    context = raw_context if isinstance(raw_context, Mapping) else {}
    raw_roles = [
        analysis.get("actor_role"),
        context.get("actor_role"),
        (analysis.get("context_result") or {}).get("actor_role")
        if isinstance(analysis.get("context_result"), Mapping) else None,
    ]
    role_aliases = {
        "SENDER": "SENDER", "송금인": "SENDER", "착오송금인": "SENDER",
        "RECIPIENT": "RECIPIENT", "수취인": "RECIPIENT", "착오송금수취인": "RECIPIENT",
        "PROXY": "PROXY", "대리인": "PROXY",
        "HEIR": "HEIR", "상속인": "HEIR",
        "DEBTOR": "DEBTOR", "채무자": "DEBTOR",
        "REPORTER": "REPORTER", "신고자": "REPORTER", "제보자": "REPORTER",
        "SELF": "SELF", "본인": "SELF",
    }
    for raw in raw_roles:
        mapped = role_aliases.get(str(raw or "").strip())
        if mapped:
            roles.add(mapped)

    text = question_text.lower()
    if re.search(r"송금인|착오송금인|잘못\s*보냈|돈을\s*보낸|보낸\s*사람", text):
        roles.add("SENDER")
    if re.search(r"수취인|받은\s*사람|모르는\s*돈|잘못\s*받았|입금\s*받", text):
        roles.add("RECIPIENT")
    if re.search(r"대리인|대리\s*신청|위임", text):
        roles.add("PROXY")
    if re.search(r"상속인|피상속인|사망한|사망자", text):
        roles.add("HEIR")
    if re.search(r"채무자|내\s*채무|본인\s*채무", text):
        roles.add("DEBTOR")
    if re.search(r"신고자|제보자|신고하려|제보하려", text):
        roles.add("REPORTER")
    if re.search(r"본인|제가|내가|저는", text):
        roles.add("SELF")

    if "mistaken_transfer" in business_codes and not {"SENDER", "RECIPIENT"}.intersection(roles):
        if re.search(r"착오송금.*(?:반환지원\s*)?신청|반환지원.*신청", text):
            roles.add("SENDER")
    if "debt_adjustment" in business_codes and re.search(r"채무조정|채무정보|상담", text):
        roles.add("DEBTOR")
    if "hidden_assets_report" in business_codes and re.search(r"신고|제보", text):
        roles.add("REPORTER")
    if not roles:
        roles.add("ANY")
    return roles


def detect_action_types_v1(
    question_text: str,
    business_codes: set[str],
    actor_roles: set[str],
) -> set[str]:
    text = question_text.lower()
    output: set[str] = set()

    has_application = bool(re.search(r"신청|접수|신고|제보|받으려|수령", text))
    has_status = bool(re.search(r"신청\s*내역|진행\s*(?:상태|상황)?|지급\s*내역|처리\s*결과|조회", text))
    has_documents = bool(re.search(r"서류|구비|준비물|위임장|양식|다운로드|첨부", text))
    has_eligibility = bool(re.search(r"자격|대상|누가|신청할\s*수|가능한가|가능해|조건|제외|안\s*되", text))
    has_method = bool(re.search(r"어떻게|방법|절차|하려면", text))

    if "deposit_protection" in business_codes:
        protection_check_intent = bool(
            re.search(
                r"검색|조회|확인|보호\s*대상|보호되|가입\s*(?:여부|기관)|"
                r"금융회사\s*(?:인가|인지|여부)|상품\s*(?:인가|인지|여부)|"
                r"(?:은행|금융회사|상품|적금).*보호",
                text,
            )
        )
        if protection_check_intent:
            if re.search(r"금융회사|은행|저축은행|가입기관", text):
                output.add("INSTITUTION_SEARCH")
            if re.search(r"금융상품|적금|상품|예금\s*(?:상품|계좌|통장)", text):
                output.add("PRODUCT_SEARCH")

    if "deposit_insurance_payout" in business_codes:
        if has_documents:
            output.add("DOCUMENT_GUIDE")
        elif re.search(r"방문|지급대행점|지점|오프라인", text):
            output.add("OFFLINE_LOCATION_SEARCH")
        elif has_application or has_method:
            output.add("APPLY_GUIDE")

    if "unclaimed_funds" in business_codes:
        if re.search(r"상속|상속인|사망", text):
            output.add("HEIR_INQUIRY")
        elif has_status:
            output.add("STATUS_CHECK")
        elif has_application or has_method:
            output.update({"APPLY", "APPLY_GUIDE"})

    if "mistaken_transfer" in business_codes:
        if re.search(r"채무\s*잔액|잔액", text) and "RECIPIENT" in actor_roles:
            output.add("BALANCE_CHECK")
        elif re.search(r"반환\s*확인|반환했|처리\s*결과", text) and "RECIPIENT" in actor_roles:
            output.add("RETURN_CHECK")
        elif has_documents:
            output.add("DOCUMENT_GUIDE")
        elif has_status and "SENDER" in actor_roles:
            output.add("STATUS_CHECK")
        elif has_eligibility and "SENDER" in actor_roles:
            output.add("ELIGIBILITY_CHECK")
        elif (has_application or has_method) and "SENDER" in actor_roles:
            output.add("APPLY")
        elif not {"SENDER", "RECIPIENT"}.intersection(actor_roles) and (
            has_application or has_status or has_documents or has_eligibility or has_method
        ):
            output.add("SITUATION_SELECT")

    if "debt_adjustment" in business_codes:
        if has_documents or has_eligibility:
            output.add("DOCUMENT_GUIDE")
        elif has_application or has_status or has_method or re.search(r"상담|문의|채무정보", text):
            output.add("DEBT_INQUIRY")

    if "hidden_assets_report" in business_codes:
        if re.search(r"신고|제보|접수|조회|신고내역", text):
            output.update({"REPORT", "REPORT_GUIDE"})
        elif re.search(r"포상금.*(?:어디|방법)|어디.*포상금", text):
            output.add("REPORT_GUIDE")

    if re.search(r"철회|취소", text):
        return {"WITHDRAW"}
    if re.search(r"정보\s*변경|신청\s*변경|수정", text):
        return {"MODIFY"}
    return output


def resolve_action_links_v1(
    common: Mapping[str, Any],
    *,
    max_buttons: int | None = None,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if str(common.get("route") or "") != "RETRIEVE":
        return [], [{"accepted": False, "reason": "NON_RETRIEVE_ROUTE"}]

    question_text = _action_clean_v1(" ".join([
        str(common.get("question") or ""),
        str(common.get("resolved_question") or ""),
    ]))
    text_lower = question_text.lower()
    business_codes = _action_business_codes_v1(common)
    actor_roles = detect_action_actor_roles_v1(common, question_text, business_codes)
    action_types = detect_action_types_v1(question_text, business_codes, actor_roles)
    limit = max(0, int(max_buttons or ACTION_LINK_MAX_BUTTONS_V1))
    if not business_codes or not action_types or limit == 0:
        return [], [{
            "accepted": False,
            "reason": "NO_BUSINESS" if not business_codes else "NO_ACTION_INTENT",
            "business_codes": sorted(business_codes),
            "actor_roles": sorted(actor_roles),
            "action_types": sorted(action_types),
        }]

    accepted: list[dict[str, Any]] = []
    audit: list[dict[str, Any]] = []
    for record in ACTION_LINK_RECORDS_V1:
        link_id = str(record.get("link_id") or "")
        business_match = str(record.get("business_function_code") or "") in business_codes
        action_match = str(record.get("action_type") or "") in action_types
        record_roles = {str(value) for value in record.get("actor_roles") or []}
        role_match = "ANY" in record_roles or bool(record_roles.intersection(actor_roles))
        keyword_hits = [
            str(value) for value in record.get("activation_keywords") or []
            if str(value).strip() and str(value).strip().lower() in text_lower
        ]
        exclusion_hits = [
            str(value) for value in record.get("exclusion_keywords") or []
            if str(value).strip() and str(value).strip().lower() in text_lower
        ]
        url_valid, url_reason = validate_action_url_v1(record.get("url"))
        approved = record.get("approved_for_display") is True
        is_accepted = bool(
            business_match and action_match and role_match and keyword_hits
            and not exclusion_hits and url_valid and approved
        )
        score = (
            100 * int(action_match)
            + 30 * int(role_match and "ANY" not in record_roles)
            + 10 * len(set(keyword_hits))
            - int(record.get("priority") or 99)
        )
        reason = (
            "ACCEPTED"
            if is_accepted else
            "BUSINESS_MISMATCH" if not business_match else
            "ACTION_MISMATCH" if not action_match else
            "ROLE_MISMATCH" if not role_match else
            "NO_ACTIVATION_KEYWORD" if not keyword_hits else
            "EXCLUSION_KEYWORD" if exclusion_hits else
            url_reason if not url_valid else
            "NOT_APPROVED"
        )
        row = {
            "link_id": link_id,
            "accepted": is_accepted,
            "reason": reason,
            "business_match": business_match,
            "action_match": action_match,
            "role_match": role_match,
            "keyword_hits": list(dict.fromkeys(keyword_hits)),
            "exclusion_hits": list(dict.fromkeys(exclusion_hits)),
            "url_valid": url_valid,
            "url_reason": url_reason,
            "score": score,
            "detected_business_codes": sorted(business_codes),
            "detected_actor_roles": sorted(actor_roles),
            "detected_action_types": sorted(action_types),
        }
        audit.append(row)
        if is_accepted:
            accepted.append({**copy.deepcopy(record), "_selection_audit": row})

    accepted.sort(key=lambda value: (
        -int((value.get("_selection_audit") or {}).get("score") or 0),
        int(value.get("priority") or 99),
        str(value.get("link_id") or ""),
    ))
    selected: list[dict[str, Any]] = []
    seen_urls: set[str] = set()
    for record in accepted:
        url = str(record.get("url") or "")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        selected.append(record)
        if len(selected) >= limit:
            break
    selected_ids = {str(value.get("link_id") or "") for value in selected}
    for row in audit:
        if row.get("accepted") and row.get("link_id") not in selected_ids:
            row["accepted"] = False
            row["reason"] = "LOWER_PRIORITY_OR_BUTTON_LIMIT"
    return selected, audit


def sanitize_answer_urls_v1(value: Any) -> str:
    text = str(value or "")
    text = re.sub(r"<a\b[^>]*>(.*?)</a>", r"\1", text, flags=re.I | re.S)
    text = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", text, flags=re.I)
    text = re.sub(r"https?://[^\s<>)\]]+", "", text, flags=re.I)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def action_links_markdown_v1(action_links: Sequence[Mapping[str, Any]]) -> str:
    lines = ["### 관련 공식 서비스", ""]
    for record in action_links:
        url_valid, _ = validate_action_url_v1(record.get("url"))
        if not url_valid or record.get("approved_for_display") is not True:
            continue
        label = str(record.get("button_label") or "공식 서비스 열기").replace("|", "\\|")
        description = _action_clean_v1(record.get("description")).replace("|", "\\|")
        url = str(record.get("url") or "")
        auth_note = " · 본인인증 필요" if str(record.get("channel") or "") == "WEB_AUTH" else ""
        lines.append(
            f"- [{label}]({url}){auth_note}  \n  {description}"
        )
    if len(lines) == 2:
        return ""
    lines.extend([
        "",
        "> 위 링크는 답변 모델이 생성한 주소가 아니라 Action Link Registry에서 검증된 예금보험공사 공식 페이지입니다.",
    ])
    return "\n".join(lines)


def render_action_links_colab_v1(action_links: Sequence[Mapping[str, Any]]) -> None:
    markdown = action_links_markdown_v1(action_links)
    if not markdown:
        return
    from IPython.display import Markdown, display
    display(Markdown(markdown))


def action_links_for_streamlit_v1(
    action_links: Sequence[Mapping[str, Any]],
) -> list[dict[str, Any]]:
    """추후 Streamlit의 st.link_button()에 바로 전달할 안전한 표시 데이터입니다."""
    output: list[dict[str, Any]] = []
    for record in action_links:
        url_valid, _ = validate_action_url_v1(record.get("url"))
        if url_valid and record.get("approved_for_display") is True:
            output.append({
                "link_id": str(record.get("link_id") or ""),
                "business_function_code": str(record.get("business_function_code") or ""),
                "action_type": str(record.get("action_type") or ""),
                "label": str(record.get("button_label") or "공식 서비스 열기"),
                "url": str(record.get("url") or ""),
                "description": str(record.get("description") or ""),
                "requires_auth": str(record.get("channel") or "") == "WEB_AUTH",
            })
    return output


ACTION_LINK_PROMPT_RULE_V1 = """
[후속 행동 링크 안전 규칙]
- 답변 본문에 URL, 링크 주소, Markdown 링크를 직접 만들지 마세요.
- 신청·조회·서류·상담 페이지는 프로그램의 검증된 Action Link Registry가 답변 뒤에 별도로 제공합니다.
- Evidence에 URL이 있어도 답변 문장 안에 복사하지 마세요.
""".strip()
for _prompt_name in (
    "B_STRUCTURED_SYSTEM_PROMPT_V3",
    "C_STRUCTURED_SYSTEM_PROMPT_V3",
    "D_STRUCTURED_FINAL_SYSTEM_PROMPT_V3",
):
    if _prompt_name in globals():
        globals()[_prompt_name] = globals()[_prompt_name] + "\n\n" + ACTION_LINK_PROMPT_RULE_V1


def run_action_link_regression_v1() -> list[dict[str, Any]]:
    cases = [
        ("보호 금융회사", "이 은행은 예금자보호 금융회사인가요?", ["예금자보호제도"], {"DP-INSTITUTION-SEARCH-001"}, set()),
        ("보호 금융상품", "이 적금 상품도 보호대상인가요?", ["예금자보호제도"], {"DP-PRODUCT-SEARCH-001"}, set()),
        ("보험금 서류", "예금보험금 신청 서류와 양식은 어디서 받나요?", ["예금보험금 안내"], {"DI-DOCUMENT-GUIDE-001"}, set()),
        ("보험금 방문", "예금보험금을 방문 신청할 지급대행점은 어디인가요?", ["예금보험금 안내"], {"DI-PAYMENT-AGENT-SEARCH-001"}, set()),
        ("미수령금 신청", "미수령금 통합신청은 어떻게 하나요?", ["고객 미수령금 신청"], {"UN-APPLICATION-GUIDE-001", "UN-INTEGRATED-APPLICATION-001"}, set()),
        ("미수령금 상태", "미수령금 신청 진행상태를 조회하고 싶어요", ["고객 미수령금 신청"], {"UN-APPLICATION-STATUS-001"}, set()),
        ("상속인 조회", "사망한 가족의 미수령금을 상속인이 조회하려면?", ["고객 미수령금 신청"], {"UN-HEIR-INQUIRY-GUIDE-001"}, set()),
        ("착오송금 신청", "착오송금 반환지원 신청은 어떻게 하나요?", ["착오송금 반환 신청"], {"MT-SENDER-APPLICATION-001"}, set()),
        ("착오송금 자격", "착오송금인은 누가 반환지원을 신청할 수 있나요?", ["착오송금 반환 신청"], {"MT-SENDER-ELIGIBILITY-001"}, set()),
        ("송금인 서류", "착오송금인 본인 신청 서류는 무엇인가요?", ["착오송금 반환 신청"], {"MT-SENDER-DOCUMENTS-001"}, set()),
        ("수취인 서류", "착오송금 수취인이 준비할 서류는 무엇인가요?", ["착오송금 반환 신청"], {"MT-RECIPIENT-DOCUMENTS-001"}, set()),
        ("수취인 잔액", "착오송금 수취인이 채무잔액을 확인하려면?", ["착오송금 반환 신청"], {"MT-RECIPIENT-BALANCE-001"}, set()),
        ("채무 조회", "채무조정 신청 전에 채무정보를 조회하고 상담받고 싶어요", ["채무조정 안내"], {"DA-DEBT-INQUIRY-001"}, set()),
        ("채무 서류", "채무조정 신청 자격과 구비서류가 궁금합니다", ["채무조정 안내"], {"DA-ELIGIBILITY-DOCUMENTS-001"}, set()),
        ("은닉재산 신고", "은닉재산을 신고하려면 어디에서 접수하나요?", ["은닉재산 신고"], {"HP-REPORT-AND-INQUIRY-001", "HP-REPORT-GUIDE-001"}, set()),
        ("정보형 무버튼", "예금자보호 한도는 얼마인가요?", ["예금자보호제도"], set(), set()),
        ("비검색 무버튼", "채무조정 신청 방법", ["채무조정 안내"], set(), set()),
    ]
    rows = []
    for name, question, businesses, required, forbidden in cases:
        route = "OUT_OF_SCOPE" if name == "비검색 무버튼" else "RETRIEVE"
        common = {
            "route": route,
            "question": question,
            "resolved_question": question,
            "analysis": {"businesses": businesses},
        }
        selected, _ = resolve_action_links_v1(common)
        selected_ids = {str(value.get("link_id") or "") for value in selected}
        passed = selected_ids == required and not forbidden.intersection(selected_ids)
        rows.append({
            "case": name,
            "question": question,
            "route": route,
            "businesses": " | ".join(businesses),
            "selected_link_ids": " | ".join(sorted(selected_ids)),
            "required_link_ids": " | ".join(sorted(required)),
            "forbidden_link_ids": " | ".join(sorted(forbidden)),
            "passed": bool(passed),
        })
    return rows


ACTION_LINK_REGISTRY_GATE_ROWS_V1 = validate_action_link_registry_v1()
ACTION_LINK_REGRESSION_ROWS_V1 = run_action_link_regression_v1()
ACTION_LINK_HARD_GATE_V1 = {
    "registry_record_count": len(ACTION_LINK_RECORDS_V1),
    "invalid_registry_record_count": sum(
        1 for row in ACTION_LINK_REGISTRY_GATE_ROWS_V1 if not row.get("passed")
    ),
    "regression_case_count": len(ACTION_LINK_REGRESSION_ROWS_V1),
    "regression_failure_count": sum(
        1 for row in ACTION_LINK_REGRESSION_ROWS_V1 if not row.get("passed")
    ),
    "llm_url_selection_enabled": False,
    "runtime_network_check_enabled": False,
}
if ACTION_LINK_HARD_GATE_V1["invalid_registry_record_count"]:
    raise RuntimeError("Action Link Registry 구조·도메인 하드 게이트 실패")
if ACTION_LINK_HARD_GATE_V1["regression_failure_count"]:
    failed = [row for row in ACTION_LINK_REGRESSION_ROWS_V1 if not row.get("passed")]
    raise RuntimeError("Action Link 회귀 테스트 실패: " + json.dumps(failed, ensure_ascii=False))

display(pd.DataFrame(ACTION_LINK_REGISTRY_GATE_ROWS_V1))
display(pd.DataFrame(ACTION_LINK_REGRESSION_ROWS_V1))
print(ACTION_LINK_HARD_GATE_V1)


## 20. 429 Circuit Breaker와 검색·답변 캐시

429 재시도는 논리 호출당 최대 2회입니다. 반복 실패하면 60초 동안 실제 API 호출을 차단하고 시간이 지나면 자동으로 해제합니다.


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import threading
import time
from collections import OrderedDict
from typing import Any, Callable, Mapping


HCX_CIRCUIT_MAX_ATTEMPTS_C1 = 2
HCX_CIRCUIT_COOLDOWN_SECONDS_C1 = 60.0


class HCXCircuitOpenErrorC1(RuntimeError):
    pass


class HCXCircuitBreakerGateC1(HCXSharedRequestGateV3):
    def __init__(self, *, min_interval_seconds: float) -> None:
        super().__init__(
            min_interval_seconds=min_interval_seconds,
            max_attempts=HCX_CIRCUIT_MAX_ATTEMPTS_C1,
        )
        self.cooldown_until_monotonic = 0.0
        self.circuit_lock = threading.RLock()

    def cooldown_remaining_seconds(self) -> float:
        return max(0.0, self.cooldown_until_monotonic - time.monotonic())

    def call(
        self,
        stage: str,
        operation: Callable[[], Any],
        *,
        max_attempts: int | None = None,
    ) -> tuple[Any, dict[str, Any]]:
        with self.circuit_lock:
            remaining = self.cooldown_remaining_seconds()
            if remaining > 0:
                trace = {
                    "stage": stage,
                    "success": False,
                    "circuit_open": True,
                    "cooldown_remaining_seconds": remaining,
                    "attempts": [],
                    "pacing_wait_ms": 0.0,
                    "retry_wait_ms": 0.0,
                    "wall_latency_ms": 0.0,
                }
                self.last_trace = trace
                self.history.append(copy.deepcopy(trace))
                raise HCXCircuitOpenErrorC1(
                    f"HCX 429 보호 대기 중입니다. 약 {remaining:.1f}초 후 다시 시도하세요."
                )

        try:
            return super().call(
                stage,
                operation,
                max_attempts=HCX_CIRCUIT_MAX_ATTEMPTS_C1,
            )
        except Exception as error:
            cause = getattr(error, "__cause__", None)
            is_rate_limit = (
                type(error).__name__ == "RateLimitError"
                or type(cause).__name__ == "RateLimitError"
                or "429" in str(error)
            )
            if is_rate_limit:
                with self.circuit_lock:
                    self.cooldown_until_monotonic = max(
                        self.cooldown_until_monotonic,
                        time.monotonic() + HCX_CIRCUIT_COOLDOWN_SECONDS_C1,
                    )
                raise HCXCircuitOpenErrorC1(
                    "HCX 429가 반복되어 이번 실행을 중단했습니다. "
                    f"{HCX_CIRCUIT_COOLDOWN_SECONDS_C1:.0f}초 후 자동으로 다시 허용됩니다."
                ) from error
            raise


# 기존 질문 임베딩·문맥 판정·답변 래퍼가 참조하는 전역 게이트만 교체합니다.
HCX_SHARED_GATE_V3 = HCXCircuitBreakerGateC1(
    min_interval_seconds=HCX_GLOBAL_MIN_INTERVAL_SECONDS_V3,
)


# HCX-007 분해기는 별도 requests 전송기를 사용하므로 재시도 횟수와 429 상태를
# 공통 Circuit Breaker에 연결합니다. 정상 분해·캐시 동작은 그대로 유지합니다.
if hasattr(V15_DECOMPOSER, "transport") and hasattr(V15_DECOMPOSER.transport, "policy"):
    _v15_policy_c1 = V15_DECOMPOSER.transport.policy
    V15_DECOMPOSER.transport.policy = TransportPolicy(
        request_delay_seconds=float(_v15_policy_c1.request_delay_seconds),
        max_transport_retries=1,
        base_backoff_seconds=float(_v15_policy_c1.base_backoff_seconds),
        max_backoff_seconds=float(_v15_policy_c1.max_backoff_seconds),
        jitter_seconds=float(_v15_policy_c1.jitter_seconds),
        consecutive_429_cooldown_threshold=1,
        cooldown_seconds=HCX_CIRCUIT_COOLDOWN_SECONDS_C1,
        timeout_seconds=float(_v15_policy_c1.timeout_seconds),
    )

_V15_DECOMPOSE_RAW_C1 = V15_DECOMPOSER.decompose


def _v15_decompose_circuit_guard_c1(
    question: str,
    expected_businesses: Sequence[str],
) -> dict[str, Any]:
    remaining = HCX_SHARED_GATE_V3.cooldown_remaining_seconds()
    if remaining > 0:
        raise HCXCircuitOpenErrorC1(
            f"HCX 429 보호 대기 중이므로 구조화 분해를 호출하지 않습니다. 약 {remaining:.1f}초 남았습니다."
        )
    row = _V15_DECOMPOSE_RAW_C1(question, expected_businesses)
    if int(row.get("http_status") or 0) == 429:
        HCX_SHARED_GATE_V3.cooldown_until_monotonic = max(
            HCX_SHARED_GATE_V3.cooldown_until_monotonic,
            time.monotonic() + HCX_CIRCUIT_COOLDOWN_SECONDS_C1,
        )
        raise HCXCircuitOpenErrorC1(
            "HCX-007 구조화 분해에서 429가 반복되어 이번 실행을 중단했습니다. "
            f"{HCX_CIRCUIT_COOLDOWN_SECONDS_C1:.0f}초 후 자동 해제됩니다."
        )
    return row


V15_DECOMPOSER.decompose = _v15_decompose_circuit_guard_c1


def hcx_circuit_status_c1() -> dict[str, Any]:
    remaining = HCX_SHARED_GATE_V3.cooldown_remaining_seconds()
    return {
        "open": remaining > 0,
        "cooldown_remaining_seconds": remaining,
        "max_attempts_per_logical_call": HCX_CIRCUIT_MAX_ATTEMPTS_C1,
        "cooldown_seconds": HCX_CIRCUIT_COOLDOWN_SECONDS_C1,
    }


def _stable_json_hash_c1(value: Any) -> str:
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _common_request_key_c1(question: str, state: Mapping[str, Any]) -> str:
    payload = {
        "question": _clean_text(question),
        "state": copy.deepcopy(dict(state)),
        "dense_weight": DENSE_WEIGHT,
        "bm25_weight": BM25_WEIGHT,
        "candidate_depth": CANDIDATE_DEPTH,
        "reranker_model": RERANKER_MODEL_NAME,
        "parent_context_max_chars": PARENT_CONTEXT_MAX_CHARS,
        "dense_cache_version": DENSE_CACHE_VERSION,
        "fact_index_sha256": FACT_INDEX_DOCUMENT_C1["_sha256"],
    }
    return _stable_json_hash_c1(payload)


def _answer_cache_key_c1(
    variant: str,
    common: Mapping[str, Any],
    augmented_pack: Mapping[str, Any] | None,
) -> str:
    return _stable_json_hash_c1({
        "variant": variant,
        "resolved_question": common.get("resolved_question"),
        "base_pack_sha256": common.get("evidence_pack_sha256"),
        "augmented_pack_sha256": (
            _stable_json_hash_c1(augmented_pack) if augmented_pack is not None else None
        ),
        "prompt_version": "b-c-d-individual-c1",
    })


def new_bcd_controller_state_c1() -> dict[str, Any]:
    return {
        "conversation": new_bd_comparison_state(),
        "current_question": "",
        "common_request_key": "",
        "common": None,
        "common_created_at": None,
        "answer_cache": {},
        "committed": False,
        "committed_variant": None,
        "events": [],
        "running": False,
        "ignored_duplicate_events": 0,
    }


print({
    "hcx_circuit_breaker": True,
    "max_attempts": HCX_CIRCUIT_MAX_ATTEMPTS_C1,
    "cooldown_seconds": HCX_CIRCUIT_COOLDOWN_SECONDS_C1,
    "automatic_release": True,
    "hcx007_transport_max_retries": 1,
})


## 21. B/C/D v3 개별 실행과 공통 검색 Trace

공통 검색에서 발생한 질문 임베딩의 호출간격 대기와 429 재시도 대기를 저장하고, 답변안별 세부 측정값과 함께 출력합니다.


In [ ]:
def _variant_usage_v3(payload: Mapping[str, Any]) -> dict[str, int]:
    usage = payload.get("usage") or {}
    return {
        "prompt_tokens": int(usage.get("prompt_tokens") or 0),
        "completion_tokens": int(usage.get("completion_tokens") or 0),
        "total_tokens": int(usage.get("total_tokens") or 0),
    }


def _answer_cache_key_v3(
    variant: str,
    common: Mapping[str, Any],
    augmented_pack: Mapping[str, Any] | None,
) -> str:
    return _stable_json_hash_c1({
        "variant": variant,
        "resolved_question": common.get("resolved_question"),
        "base_pack_sha256": common.get("evidence_pack_sha256"),
        "augmented_pack_sha256": (
            _stable_json_hash_c1(augmented_pack) if augmented_pack is not None else None
        ),
        "prompt_version": "b-c-d-detailed-latency-markdown-v3",
    })


def _trace_summary_since_v3(start_index: int) -> dict[str, Any]:
    traces = HCX_SHARED_GATE_V3.history[start_index:]
    attempts = [attempt for trace in traces for attempt in trace.get("attempts") or []]
    return {
        "logical_api_calls": len(traces),
        "physical_http_attempts": len(attempts),
        "rate_limit_429_count": sum(
            1 for attempt in attempts if attempt.get("status") == "RATE_LIMIT_429"
        ),
        "pacing_wait_ms": sum(float(trace.get("pacing_wait_ms") or 0) for trace in traces),
        "retry_wait_ms": sum(float(trace.get("retry_wait_ms") or 0) for trace in traces),
        "traces": copy.deepcopy(traces),
    }


def _prepare_or_reuse_common_v3(
    question: str,
    holder: dict[str, Any],
) -> tuple[dict[str, Any], bool, float]:
    cleaned = _clean_text(question)
    if holder.get("common") is not None and cleaned == holder.get("current_question"):
        return holder["common"], True, 0.0

    request_key = _common_request_key_c1(cleaned, holder["conversation"])
    gate_start = len(HCX_SHARED_GATE_V3.history)
    started = time.perf_counter()
    common = prepare_common_retrieval_v1(cleaned, state=holder["conversation"])
    wall_ms = (time.perf_counter() - started) * 1000
    common["common_hcx_trace_v3"] = _trace_summary_since_v3(gate_start)
    holder.update({
        "current_question": cleaned,
        "common_request_key": request_key,
        "common": common,
        "common_created_at": time.time(),
        "answer_cache": {},
        "committed": False,
        "committed_variant": None,
    })
    return common, False, wall_ms


def execute_bcd_variant_v3(
    variant: str,
    question: str,
    holder: dict[str, Any],
    *,
    force_answer_regeneration: bool = False,
) -> dict[str, Any]:
    variant = str(variant).upper()
    if variant not in {"B", "C", "D"}:
        raise ValueError(f"지원하지 않는 답변안: {variant}")
    click_started = time.perf_counter()
    gate_start_index = len(HCX_SHARED_GATE_V3.history)
    common, common_cache_hit, common_this_click_ms = _prepare_or_reuse_common_v3(
        question, holder
    )
    if common.get("route") != "RETRIEVE":
        return {
            "variant": variant,
            "common": common,
            "route": common.get("route"),
            "route_message": common.get("route_message"),
            "common_cache_hit": common_cache_hit,
            "answer_cache_hit": False,
            "latency": {
                "common_this_click_ms": common_this_click_ms,
                "answer_ms": 0.0,
                "click_wall_ms": (time.perf_counter() - click_started) * 1000,
            },
            "api_trace": _trace_summary_since_v3(gate_start_index),
        }

    fact_started = time.perf_counter()
    matched_records, fact_audit = match_fact_index_c1(common)
    augmented_pack = build_fact_augmented_pack_c1(common["evidence_pack"], matched_records)
    fact_ms = (time.perf_counter() - fact_started) * 1000
    cache_key = _answer_cache_key_v3(
        variant,
        common,
        augmented_pack if variant == "C" else None,
    )
    cached = holder["answer_cache"].get(cache_key)
    if cached is not None and not force_answer_regeneration:
        payload = copy.deepcopy(cached)
        answer_cache_hit = True
        answer_ms = 0.0
    else:
        answer_cache_hit = False
        answer_started = time.perf_counter()
        if variant == "B":
            payload = generate_answer_b_v3(
                common["resolved_question"], common["evidence_pack"]
            )
        elif variant == "C":
            payload = generate_answer_c_v3(common["resolved_question"], augmented_pack)
        else:
            payload = generate_answer_d_v3(
                common["resolved_question"], common["evidence_pack"]
            )
        answer_ms = (time.perf_counter() - answer_started) * 1000
        holder["answer_cache"][cache_key] = copy.deepcopy(payload)

    if not holder.get("committed"):
        holder["conversation"].setdefault("turns", []).extend([
            {"role": "user", "content": _clean_text(question)},
            {"role": "assistant", "content": normalize_answer_markdown_v3(payload.get("answer"))},
        ])
        holder["committed"] = True
        holder["committed_variant"] = variant

    trace = _trace_summary_since_v3(gate_start_index)
    usage = _variant_usage_v3(payload)
    stage = payload.get("stage_latency_ms") or {}
    result = {
        "variant": variant,
        "route": "RETRIEVE",
        "common": common,
        "payload": payload,
        "matched_fact_records": matched_records,
        "fact_audit": fact_audit,
        "augmented_pack": augmented_pack if variant == "C" else None,
        "common_cache_hit": common_cache_hit,
        "answer_cache_hit": answer_cache_hit,
        "committed_variant": holder.get("committed_variant"),
        "latency": {
            "stored_common_pipeline_ms": float(
                (common.get("latency_ms") or {}).get("공통 준비 전체") or 0
            ),
            "common_this_click_ms": common_this_click_ms,
            "fact_index_match_ms": fact_ms if variant == "C" else 0.0,
            "answer_ms": answer_ms,
            "click_wall_ms": (time.perf_counter() - click_started) * 1000,
        },
        "api_trace": trace,
        "usage": usage,
        "circuit": hcx_circuit_status_c1(),
    }
    event = {
        "question": _clean_text(question),
        "variant": variant,
        "common_cache_hit": common_cache_hit,
        "answer_cache_hit": answer_cache_hit,
        "fact_index_count": len(matched_records) if variant == "C" else 0,
        "fact_index_ids": ", ".join(
            str(row.get("fact_index_id") or "") for row in matched_records
        ) if variant == "C" else "",
        "stored_common_pipeline_ms": result["latency"]["stored_common_pipeline_ms"],
        "common_this_click_ms": common_this_click_ms,
        "fact_index_match_ms": result["latency"]["fact_index_match_ms"],
        "answer_ms": answer_ms,
        "skeleton_api_ms": float(stage.get("skeleton_api_wall_ms") or 0),
        "final_api_ms": float(stage.get("final_api_wall_ms") or 0),
        "click_wall_ms": result["latency"]["click_wall_ms"],
        **{key: value for key, value in trace.items() if key != "traces"},
        **usage,
        "coverage_status": payload.get("coverage_status"),
        "answer_chars": len(str(payload.get("answer") or "")),
    }
    holder["events"].append(event)
    result["event"] = event
    return result


print("B/C/D v3 상세 레이턴시 실행기 준비 완료")


## 23. Action Link 공통 검색 캐시 결합

공통 검색이 처음 실행될 때 한 번만 규칙 매칭하고 B·C·D가 같은 `action_links` 목록을 재사용합니다. 답변 모델이 만든 URL은 사용자 답변에서 제거하고, 검증된 레지스트리 URL만 일반 링크로 렌더링합니다.


In [ ]:
_PREPARE_COMMON_BEFORE_ACTION_LINK_V4 = _prepare_or_reuse_common_v3
_NORMALIZE_MARKDOWN_BEFORE_ACTION_LINK_V4 = normalize_answer_markdown_v3
_EXECUTE_BCD_BEFORE_ACTION_LINK_V4 = execute_bcd_variant_v3


def normalize_answer_markdown_v3(text: Any) -> str:
    return sanitize_answer_urls_v1(
        _NORMALIZE_MARKDOWN_BEFORE_ACTION_LINK_V4(text)
    )


def _prepare_or_reuse_common_v3(
    question: str,
    holder: dict[str, Any],
) -> tuple[dict[str, Any], bool, float]:
    common, cache_hit, common_this_click_ms = _PREPARE_COMMON_BEFORE_ACTION_LINK_V4(
        question, holder
    )
    if "action_links" not in common:
        started = time.perf_counter()
        selected, audit = resolve_action_links_v1(common)
        action_ms = (time.perf_counter() - started) * 1000
        common["action_links"] = selected
        common["action_link_audit"] = audit
        common["action_link_registry_version"] = ACTION_LINK_REGISTRY_DOCUMENT_V1.get(
            "registry_version"
        )
        common.setdefault("latency_ms", {})["Action Link Registry"] = action_ms
        if "공통 준비 전체" in common.get("latency_ms", {}):
            common["latency_ms"]["공통 준비 전체"] = (
                float(common["latency_ms"].get("공통 준비 전체") or 0) + action_ms
            )
        common_this_click_ms += action_ms
    return common, cache_hit, common_this_click_ms


def execute_bcd_variant_v3(
    variant: str,
    question: str,
    holder: dict[str, Any],
    *,
    force_answer_regeneration: bool = False,
) -> dict[str, Any]:
    result = _EXECUTE_BCD_BEFORE_ACTION_LINK_V4(
        variant,
        question,
        holder,
        force_answer_regeneration=force_answer_regeneration,
    )
    common = result.get("common") or {}
    payload = result.get("payload") or {}
    result["answer"] = normalize_answer_markdown_v3(
        payload.get("answer") or result.get("route_message") or ""
    )
    official_sources = []
    seen_source_urls = set()
    for source in (common.get("evidence_pack") or {}).get("sources") or []:
        url = str(source.get("source_url") or "").strip()
        if not url or url in seen_source_urls:
            continue
        seen_source_urls.add(url)
        official_sources.append({
            "title": str(source.get("title") or "공식 출처"),
            "url": url,
        })
    result["official_sources"] = official_sources
    result["action_links"] = action_links_for_streamlit_v1(
        common.get("action_links") or []
    )
    result["action_link_audit"] = copy.deepcopy(
        common.get("action_link_audit") or []
    )
    result["action_link_registry_version"] = common.get(
        "action_link_registry_version"
    )
    event = result.get("event")
    if isinstance(event, dict):
        event["action_link_count"] = len(result["action_links"])
        event["action_link_ids"] = " | ".join(
            str(row.get("link_id") or "") for row in result["action_links"]
        )
    return result


print({
    "action_link_integration": "enabled",
    "answer_url_policy": "LLM URLs stripped; registry links only",
    "colab_renderer": "Markdown official action links",
    "streamlit_adapter": "action_links_for_streamlit_v1",
})


## 24. Action Link 출력과 상세 레이턴시 UI

답변 본문과 공식 출처는 기존 방식으로 유지합니다. 사용자가 신청·조회·서류·상담 행동을 명확히 요청한 경우에만 Action Link Registry가 검증된 공식 페이지를 일반 Markdown 링크로 추가합니다. 동일한 `action_links` 결과는 추후 `action_links_for_streamlit_v1()`로 변환하여 Streamlit 버튼에 연결할 수 있습니다.


In [ ]:
import html

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, JSON, Markdown, display


def _fact_sources_markdown_v3(result: Mapping[str, Any]) -> str:
    base_urls = {
        str(row.get("source_url") or "")
        for row in result["common"]["evidence_pack"].get("sources") or []
    }
    lines = ["### Fact Index 추가 공식 근거", ""]
    seen: set[str] = set()
    for record in result.get("matched_fact_records") or []:
        title = str(record.get("document_title") or record.get("fact_index_id") or "공식 근거")
        for url in record.get("source_urls") or []:
            url = str(url)
            if url and url not in seen and url not in base_urls:
                seen.add(url)
                lines.append(f"- [{title}]({url})")
    return "\n".join(lines) if len(lines) > 2 else ""


def _latency_row_v3(
    section: str,
    stage: str,
    value: float,
    *,
    included: bool,
    parent: str = "",
    note: str = "",
) -> dict[str, Any]:
    return {
        "구간": section,
        "측정 단계": stage,
        "지연시간(ms)": float(value),
        "이번 클릭": bool(included),
        "포함되는 상위 구간": parent,
        "설명": note,
    }


def detailed_latency_rows_v3(result: Mapping[str, Any]) -> list[dict[str, Any]]:
    common = result.get("common") or {}
    common_latency = common.get("latency_ms") or {}
    common_included = not bool(result.get("common_cache_hit"))
    common_trace = common.get("common_hcx_trace_v3") or {}
    query_traces = [
        trace for trace in common_trace.get("traces") or []
        if trace.get("stage") == "query_embedding"
    ]
    query_pacing = sum(float(trace.get("pacing_wait_ms") or 0) for trace in query_traces)
    query_retry = sum(float(trace.get("retry_wait_ms") or 0) for trace in query_traces)
    query_wall = float(common_latency.get("질문 임베딩") or 0)
    rows = [
        _latency_row_v3("공통", "문맥 처리", common_latency.get("문맥 처리", 0), included=common_included, parent="질의분석"),
        _latency_row_v3("공통", "질의분석 전체 Wall", common_latency.get("질의분석", 0), included=common_included),
        _latency_row_v3("검색", "질문 임베딩 전체 Wall", query_wall, included=common_included),
        _latency_row_v3("검색", "└ 호출간격 대기", query_pacing, included=common_included, parent="질문 임베딩 전체 Wall", note="중복 합산 금지"),
        _latency_row_v3("검색", "└ 429 재시도 대기", query_retry, included=common_included, parent="질문 임베딩 전체 Wall", note="중복 합산 금지"),
        _latency_row_v3("검색", "└ 임베딩 처리 추정", max(0.0, query_wall - query_pacing - query_retry), included=common_included, parent="질문 임베딩 전체 Wall", note="Wall-대기 추정값"),
        _latency_row_v3("검색", "Dense 계산", common_latency.get("Dense 계산", 0), included=common_included),
        _latency_row_v3("검색", "BM25", common_latency.get("BM25", 0), included=common_included),
        _latency_row_v3("검색", "BAAI Reranker", common_latency.get("BAAI Reranker", 0), included=common_included),
        _latency_row_v3("검색", "Parent-Child", common_latency.get("Parent-Child8192", 0), included=common_included),
        _latency_row_v3("검색", "Evidence Pack", common_latency.get("저지연 Evidence Pack", 0), included=common_included),
        _latency_row_v3("후속행동", "Action Link Registry 규칙 매칭", common_latency.get("Action Link Registry", 0), included=common_included),
        _latency_row_v3("공통", "공통 준비 전체 Wall", common_latency.get("공통 준비 전체", 0), included=common_included),
    ]

    variant = str(result.get("variant") or "")
    payload = result.get("payload") or {}
    stage = payload.get("stage_latency_ms") or {}
    if variant in {"B", "C"}:
        if variant == "C":
            rows.append(_latency_row_v3("C", "Fact Index 규칙 매칭", (result.get("latency") or {}).get("fact_index_match_ms", 0), included=True))
        rows.extend([
            _latency_row_v3(variant, "프롬프트 구성", stage.get("prompt_build_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3(variant, "HCX 답변 API 전체 Wall", stage.get("api_wall_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3(variant, "└ 호출간격 대기", stage.get("pacing_wait_ms", 0), included=not result.get("answer_cache_hit"), parent="HCX 답변 API 전체 Wall", note="중복 합산 금지"),
            _latency_row_v3(variant, "└ 429 재시도 대기", stage.get("retry_wait_ms", 0), included=not result.get("answer_cache_hit"), parent="HCX 답변 API 전체 Wall", note="중복 합산 금지"),
            _latency_row_v3(variant, "└ HCX 처리 추정", stage.get("estimated_service_ms", 0), included=not result.get("answer_cache_hit"), parent="HCX 답변 API 전체 Wall", note="Wall-대기 추정값"),
            _latency_row_v3(variant, "JSON 파싱·검증", stage.get("parse_validation_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3(variant, "Markdown·안전 후처리", stage.get("postprocess_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3(variant, f"{variant}안 전체 Wall", stage.get("answer_total_ms", 0), included=not result.get("answer_cache_hit")),
        ])
    elif variant == "D":
        rows.extend([
            _latency_row_v3("D", "Answer Need 추출", stage.get("need_extraction_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "Skeleton 프롬프트 구성", stage.get("skeleton_prompt_build_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "Skeleton API 전체 Wall", stage.get("skeleton_api_wall_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "└ Skeleton 호출간격 대기", stage.get("skeleton_pacing_wait_ms", 0), included=not result.get("answer_cache_hit"), parent="Skeleton API 전체 Wall", note="중복 합산 금지"),
            _latency_row_v3("D", "└ Skeleton 429 대기", stage.get("skeleton_retry_wait_ms", 0), included=not result.get("answer_cache_hit"), parent="Skeleton API 전체 Wall", note="중복 합산 금지"),
            _latency_row_v3("D", "└ Skeleton HCX 처리 추정", stage.get("skeleton_estimated_service_ms", 0), included=not result.get("answer_cache_hit"), parent="Skeleton API 전체 Wall", note="Wall-대기 추정값"),
            _latency_row_v3("D", "Skeleton JSON 검증", stage.get("skeleton_validation_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "Skeleton Evidence 선택", stage.get("evidence_selection_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "최종 프롬프트 구성", stage.get("final_prompt_build_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "최종답변 API 전체 Wall", stage.get("final_api_wall_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "└ 최종답변 호출간격 대기", stage.get("final_pacing_wait_ms", 0), included=not result.get("answer_cache_hit"), parent="최종답변 API 전체 Wall", note="중복 합산 금지"),
            _latency_row_v3("D", "└ 최종답변 429 대기", stage.get("final_retry_wait_ms", 0), included=not result.get("answer_cache_hit"), parent="최종답변 API 전체 Wall", note="중복 합산 금지"),
            _latency_row_v3("D", "└ 최종답변 HCX 처리 추정", stage.get("final_estimated_service_ms", 0), included=not result.get("answer_cache_hit"), parent="최종답변 API 전체 Wall", note="Wall-대기 추정값"),
            _latency_row_v3("D", "Markdown·안전 후처리", stage.get("postprocess_ms", 0), included=not result.get("answer_cache_hit")),
            _latency_row_v3("D", "D안 전체 Wall", stage.get("answer_total_ms", 0), included=not result.get("answer_cache_hit")),
        ])
    rows.append(_latency_row_v3("전체", "이번 버튼 클릭 전체 Wall", (result.get("latency") or {}).get("click_wall_ms", 0), included=True))
    return rows


def render_bcd_result_v3(
    result: Mapping[str, Any],
    holder: Mapping[str, Any],
    *,
    show_technical: bool,
) -> None:
    if result.get("route") != "RETRIEVE":
        display(Markdown(f"## {result.get('route')}\n\n{result.get('route_message') or ''}"))
        return
    variant = str(result["variant"])
    payload = result["payload"]
    display(Markdown(f"## 답변 {variant}안\n\n{normalize_answer_markdown_v3(payload.get('answer'))}"))
    source_md = _sources_from_pack_v1(result["common"]["evidence_pack"])
    if source_md:
        display(Markdown(source_md.replace("공통 공식 출처", "공식 출처")))
    render_action_links_colab_v1(result["common"].get("action_links") or [])
    if variant == "C":
        fact_source_md = _fact_sources_markdown_v3(result)
        if fact_source_md:
            display(Markdown(fact_source_md))
        matches = result.get("matched_fact_records") or []
        display(Markdown(
            "### Fact Index 적용 결과\n\n"
            + (", ".join(str(row.get("fact_index_id")) for row in matches) if matches else "적용 0건 · B안과 동일한 원본 Evidence 범위")
        ))

    display(Markdown("### 상세 단계별 레이턴시"))
    display(Markdown(
        "하위 `호출간격 대기·429 대기·HCX 처리 추정`은 해당 API 전체 Wall에 포함되므로 서로 다시 더하지 않습니다. "
        "`이번 클릭=False`인 공통 검색 행은 이전 버튼에서 측정해 캐시에 보존한 값입니다."
    ))
    display(pd.DataFrame(detailed_latency_rows_v3(result)))

    display(Markdown("### 현재 질문 누적 비교"))
    current_question = str(result["common"].get("question") or "")
    rows = [row for row in holder.get("events") or [] if row.get("question") == current_question]
    display(pd.DataFrame(rows))

    if show_technical:
        display(Markdown("### 질의분석 결과"))
        display(Markdown(analysis_markdown_compare({
            "analyzer_label": "V1.5 관계형 멀티턴 개선",
            "analysis": result["common"]["analysis"],
        })))
        display(Markdown("### 공통 검색 결과"))
        display(Markdown(retrieval_markdown_compare({"search_results": result["common"]["search_results"]})))
        display(Markdown("### Basic Evidence Pack"))
        display(JSON(result["common"]["evidence_pack"], expanded=False))
        if variant == "C":
            display(Markdown("### Fact Index 보강 Evidence Pack"))
            display(JSON(result["augmented_pack"], expanded=False))
            display(Markdown("### Fact Index 매칭 감사표"))
            display(pd.DataFrame(result.get("fact_audit") or []))
        display(Markdown("### Action Link Registry 선택 감사표"))
        display(pd.DataFrame(result["common"].get("action_link_audit") or []))
        if variant == "D":
            display(Markdown("### D안 Answer Skeleton"))
            display(JSON(payload.get("skeleton") or {}, expanded=False))


def launch_bcd_individual_chat_v3() -> dict[str, Any]:
    question = widgets.Textarea(
        placeholder="예: 착오송금 반환 신청은 어떻게 하나요?",
        description="질문",
        layout=widgets.Layout(width="100%", height="90px"),
    )
    show_technical = widgets.Checkbox(value=False, description="기술 정보 표시")
    force_answer = widgets.Checkbox(value=False, description="답변만 다시 생성")
    button_b = widgets.Button(description="B 답변", button_style="info")
    button_c = widgets.Button(description="C 답변", button_style="success")
    button_d = widgets.Button(description="D 답변", button_style="warning")
    clear_button = widgets.Button(description="공통 검색 캐시 지우기")
    reset_button = widgets.Button(description="대화 전체 초기화", button_style="danger")
    output = widgets.Output()
    holder = new_bcd_controller_state_c1()
    buttons = [button_b, button_c, button_d, clear_button, reset_button]

    def set_busy(value: bool) -> None:
        holder["running"] = value
        for button in buttons:
            button.disabled = value

    def run_variant(variant: str) -> None:
        if holder["running"]:
            holder["ignored_duplicate_events"] += 1
            return
        user_text = _clean_text(question.value)
        if not user_text:
            return
        set_busy(True)
        with output:
            display(Markdown(f"---\n\n### 사용자 질문\n\n{user_text}"))
            display(Markdown(f"선택 실행: **{variant}안만 호출**"))
        try:
            result = execute_bcd_variant_v3(
                variant,
                user_text,
                holder,
                force_answer_regeneration=bool(force_answer.value),
            )
            with output:
                render_bcd_result_v3(result, holder, show_technical=bool(show_technical.value))
        except Exception as error:
            with output:
                display(Markdown(
                    f"### {variant}안 실행 오류\n\n"
                    f"`{type(error).__name__}: {html.escape(str(error))}`"
                ))
                display(JSON({"circuit": hcx_circuit_status_c1(), "last_trace": HCX_SHARED_GATE_V3.last_trace}, expanded=False))
        finally:
            set_busy(False)

    def clear_current(_=None) -> None:
        if holder["running"]:
            return
        holder.update({
            "current_question": "",
            "common_request_key": "",
            "common": None,
            "answer_cache": {},
            "committed": False,
            "committed_variant": None,
        })
        with output:
            display(Markdown("공통 검색 캐시와 현재 질문 답변 캐시를 지웠습니다."))

    def reset_all(_=None) -> None:
        if holder["running"]:
            return
        fresh = new_bcd_controller_state_c1()
        holder.clear()
        holder.update(fresh)
        question.value = ""
        output.clear_output()

    button_b.on_click(lambda _: run_variant("B"))
    button_c.on_click(lambda _: run_variant("C"))
    button_d.on_click(lambda _: run_variant("D"))
    clear_button.on_click(clear_current)
    reset_button.on_click(reset_all)
    display(widgets.VBox([
        widgets.HTML("<h3>KDIC V1.5 · B/C/D 개별 호출 · Action Link · 상세 레이턴시</h3>"),
        question,
        widgets.HBox([show_technical, force_answer]),
        widgets.HBox([button_b, button_c, button_d]),
        widgets.HBox([clear_button, reset_button]),
        widgets.HTML(
            "<small>첫 버튼만 공통 검색을 수행합니다. B/C는 1회 호출, D는 Skeleton과 최종답변 2회 호출입니다. "
            "각 답변 아래에 API 대기와 Skeleton 세부시간을 분리하고, 신청·조회 의도가 명확할 때만 검증된 공식 서비스 링크를 표시합니다.</small>"
        ),
        output,
    ]))
    return holder


bcd_chat_state_v3 = launch_bcd_individual_chat_v3()
# v2 결과 저장 셀과의 하위 호환 별칭입니다.
bcd_chat_state_c1 = bcd_chat_state_v3


## 23. 결과 저장


In [ ]:
from pathlib import Path


def export_bcd_events_c1(
    holder: Mapping[str, Any] = bcd_chat_state_c1,
    output_dir: str | Path = "/content/kdic-bcd-individual-results",
) -> dict[str, str]:
    directory = Path(output_dir)
    directory.mkdir(parents=True, exist_ok=True)
    events_path = directory / "2026-08-20-kdic-bcd-individual-latency.csv"
    audit_path = directory / "2026-08-20-kdic-bcd-call-audit.json"
    events = list(holder.get("events") or [])
    pd.DataFrame(events).to_csv(events_path, index=False, encoding="utf-8-sig")
    audit = {
        "fact_index": {
            "schema_version": FACT_INDEX_DOCUMENT_C1["schema_version"],
            "sha256": FACT_INDEX_DOCUMENT_C1["_sha256"],
            "record_count": len(FACT_INDEX_RECORDS_C1),
        },
        "circuit": hcx_circuit_status_c1(),
        "hcx_history": HCX_SHARED_GATE_V3.history,
        "events": events,
    }
    audit_path.write_text(
        json.dumps(audit, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    result = {"events_csv": str(events_path), "call_audit_json": str(audit_path)}
    print(result)
    return result


print("필요할 때 export_bcd_events_c1()을 실행해 비교 결과를 저장하세요.")
